# 02d Full-text stage closed: hand-saved PDFs, last unsure studies, abstract corrections

Design log.
- 02c left 56 unsure studies without a decision and 16 included studies without a PDF. The PDFs that only a browser could get were saved by hand into data/defences under their download names. Stage 2 gives each one its standard name. Files are matched by their SHA-256 fingerprint, so odd characters in download names cannot cause a wrong match; the one file too large to fingerprint beforehand (the 14 MB GuardRAG thesis) is matched by its exact name. Two second downloads of the same paper and one saved web page with no article text are moved to data/defences/_not_used. Nothing is deleted.
- Full-text decisions added for the unsure studies: two whose PDFs came from the 02c second pass (the OSF peer-review study, included; the dental X-ray study, excluded as image or audio channel only) and four saved by hand (Spotlight-Guard and Real User Instruction included; the moving-target defence preprint excluded because its attacks are typed straight into the model's input; the SMU thesis excluded because its only prompt-injection method is a runtime detector). The TechRxiv preprint is dated June 2024, inside the search window that starts on 1 January 2023.
- PromptGuard (Scientific Reports): the first copy saved by hand was a web page without the article text; the PDF saved afterwards was read. It is excluded as jailbreak or direct injection only: its adversary controls the user's own input and cannot touch tools or retrieval, so its role-separated prompt format (measured alone in its Table 8) is only tested against prompts the user types.
- The 49 unsure studies whose full text could not be obtained (data or code deposits, records without an identifier, paywalled or script-blocked pages that were not fetched by hand) are excluded with the reason "no full text obtainable"; the access_status column says why each one was not retrieved.
- All 114 inclusions made at title and abstract were reread against the amendment 4 rule (a defence acting on the protected model's own generation is the object of study). Six do not meet it and are corrected to exclusions at the title-and-abstract stage, each with a note that starts "corrected on rereading the abstract": the OSF position study (attack only; a preregistered measurement with no defence), SteadyEval (detector only; a separate filter cleans answers before the unchanged grader), Rennervate and AEGIS (detector only), the in-context learning framework in CMC (jailbreak or direct injection only), and the simulator study on Zenodo (survey, position or other; its rates come from a simulated model, not a language model).
- Four included studies whose PDFs were saved by hand were also checked against their full text now: ICAIIC and PromptPatch are excluded because their attacks are user prompts (WildJailbreak; AdvBench with PAIR and AutoDAN), PICO is excluded because it defends against the user's own input and reports no measured results, and SafeAgent is confirmed. The remaining included studies get their full-text check at extraction (notebooks 03 and 04), as recorded in 02c.
- Every full-text decision carries a verbatim anchor phrase; Stage 4 finds each one in its PDF by code, as in 02c.
- Outputs: schema/screening_decisions.csv, schema/fulltext_decisions.csv, schema/hand_files.csv, reports/hand_files.csv, reports/screening.csv, reports/fulltext_check.csv, reports/defences.csv, reports/fetch_by_hand.csv, reports/prisma_counts.json, reports/decision_02.txt.
- CPU; no network calls; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd())

## Stage 1. Search module (placing hand-saved files), tests, decisions, file map

In [ ]:
%%writefile src/search.py
"""Systematic search for defence papers: arXiv, OpenAlex and DBLP by API, exported record files, citation lists."""
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import time
import unicodedata
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path

ARXIV_API = os.environ.get("SOURCEMAP_ARXIV_API", "https://export.arxiv.org/api/query")      # overridable for dry runs
OPENALEX_API = os.environ.get("SOURCEMAP_OPENALEX_API", "https://api.openalex.org/works")
DBLP_API = os.environ.get("SOURCEMAP_DBLP_API", "https://dblp.org/search/publ/api")
ZENODO_API = os.environ.get("SOURCEMAP_ZENODO_API", "https://zenodo.org/api/records")
FIGSHARE_API = os.environ.get("SOURCEMAP_FIGSHARE_API", "https://api.figshare.com/v2/articles")
ARXIV_PDF = os.environ.get("SOURCEMAP_ARXIV_PDF", "https://arxiv.org/pdf")
OSF_API = os.environ.get("SOURCEMAP_OSF_API", "https://api.osf.io/v2")
NATURE_PDF = os.environ.get("SOURCEMAP_NATURE_PDF", "https://www.nature.com/articles")
ATOM = {"a": "http://www.w3.org/2005/Atom", "arxiv": "http://arxiv.org/schemas/atom", "os": "http://a9.com/-/spec/opensearch/1.1/"}
USER_AGENT = "Mozilla/5.0 (compatible; sourcemap-research literature search; +https://github.com/anasbiswas1/sourcemap-research)"
OFFLINE = os.environ.get("SOURCEMAP_OFFLINE") == "1"
GAP = 3.0                      # seconds between requests to one service
ARXIV_CATS = ["cs.CR", "cs.CL", "cs.LG", "cs.AI"]
ARXIV_ID_RE = re.compile(r"(?<![\d.])((?:2[3-6])(?:0[1-9]|1[0-2])\.\d{4,5})(?:v\d+)?(?!\d)")   # 2301.xxxxx to 2612.xxxxx

EXCLUSION_REASONS = ["detector only", "system level", "attack only", "benchmark only", "jailbreak or direct injection only",
                     "survey, position or other", "duplicate version", "out of period", "no full text obtainable",
                     "not about prompt injection", "image or audio channel only"]
SCREEN_STAGES = {"title_abstract", "full_text"}
DECISIONS = {"include", "exclude", "unsure"}

RECORD_FIELDS = ["key", "title", "first_author", "year", "date", "venue", "arxiv", "arxiv_version", "doi", "abstract",
                 "sources", "queries", "n_hits", "linked_to"]
SCREEN_FIELDS = RECORD_FIELDS + ["decision", "reason", "stage", "access_status", "note"]
LOG_FIELDS = ["date", "source", "query_id", "query_as_sent", "fields", "filters", "count", "status", "file"]


# ---- small helpers ------------------------------------------------------------------------------------------

def norm_title(title: str) -> str:
    s = unicodedata.normalize("NFKC", title or "").lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


def norm_key(title: str) -> str:
    return re.sub(r"\s+", "", norm_title(title))[:90]


def _get(url: str, timeout: int = 90) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT, "Accept": "application/json, application/atom+xml, */*"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()


def read_csv(path: Path) -> list[dict]:
    path = Path(path)
    if not path.exists():
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({k: ("" if r.get(k) is None else r.get(k)) for k in fieldnames})


def arxiv_id_from_text(s: str) -> str:
    m = ARXIV_ID_RE.search(s or "")
    return m.group(1) if m else ""


# ---- arXiv --------------------------------------------------------------------------------------------------

def arxiv_query_string(must: list[str], date_from: str, date_to: str, cats: list[str] = ARXIV_CATS, field: str = "abs") -> str:
    """must: phrases or words that must all appear in the abstract; dates as YYYYMMDD."""
    parts = []
    for term in must:
        term = term.strip()
        parts.append(f'{field}:"{term}"' if " " in term or "-" in term else f"{field}:{term}")
    cat = " OR ".join(f"cat:{c}" for c in cats)
    return " AND ".join(parts) + f" AND ({cat}) AND submittedDate:[{date_from}0000 TO {date_to}2359]"


def parse_arxiv_feed(xml_bytes: bytes) -> tuple[list[dict], int]:
    """Records of an arXiv API feed and the feed's total-results count."""
    root = ET.fromstring(xml_bytes)
    total = int(root.findtext("os:totalResults", default="0", namespaces=ATOM) or 0)
    out = []
    for e in root.findall("a:entry", ATOM):
        full = (e.findtext("a:id", default="", namespaces=ATOM) or "").rsplit("/", 1)[-1]
        m = re.match(r"(.+?)v(\d+)$", full)
        base, version = (m.group(1), int(m.group(2))) if m else (full, "")
        if not base:
            continue
        authors = [a.findtext("a:name", default="", namespaces=ATOM) for a in e.findall("a:author", ATOM)]
        out.append({
            "arxiv": base, "arxiv_version": version,
            "title": " ".join((e.findtext("a:title", default="", namespaces=ATOM) or "").split()),
            "abstract": " ".join((e.findtext("a:summary", default="", namespaces=ATOM) or "").split()),
            "date": (e.findtext("a:published", default="", namespaces=ATOM) or "")[:10],
            "updated": (e.findtext("a:updated", default="", namespaces=ATOM) or "")[:10],
            "first_author": authors[0] if authors else "", "authors": authors,
            "primary_category": (e.find("arxiv:primary_category", ATOM).get("term") if e.find("arxiv:primary_category", ATOM) is not None else ""),
            "journal_ref": e.findtext("arxiv:journal_ref", default="", namespaces=ATOM) or "",
            "doi": (e.findtext("arxiv:doi", default="", namespaces=ATOM) or "").lower(),
            "venue": e.findtext("arxiv:journal_ref", default="", namespaces=ATOM) or "arXiv",
        })
    return out, total


def arxiv_search(query: str, max_total: int = 3000, page: int = 200, fetch=None) -> tuple[list[dict], str]:
    """All records for an arXiv API search_query, paged; returns (records, status)."""
    if OFFLINE and fetch is None:
        return [], "offline"
    fetch = fetch or _get
    records, start, total = [], 0, None
    while True:
        url = f"{ARXIV_API}?search_query={urllib.parse.quote(query)}&start={start}&max_results={page}&sortBy=submittedDate&sortOrder=descending"
        try:
            batch, total = parse_arxiv_feed(fetch(url))
        except (urllib.error.URLError, ET.ParseError, OSError) as exc:
            return records, f"failed at start={start}: {exc}"
        records.extend(batch)
        start += page
        if not batch or start >= min(total, max_total):
            break
        time.sleep(GAP)
    return records, "ok"


# ---- OpenAlex -----------------------------------------------------------------------------------------------

def abstract_from_inverted(index: dict | None) -> str:
    if not index:
        return ""
    positions = []
    for word, places in index.items():
        for p in places:
            positions.append((p, word))
    return " ".join(w for _, w in sorted(positions))


def parse_openalex_page(payload: dict) -> list[dict]:
    out = []
    for w in payload.get("results", []):
        doi = (w.get("doi") or "").replace("https://doi.org/", "").lower()
        arxiv = ""
        if doi.startswith("10.48550/arxiv."):
            arxiv = doi.split("arxiv.", 1)[1]
        for loc in w.get("locations") or []:
            for u in (loc.get("landing_page_url") or "", loc.get("pdf_url") or ""):
                if not arxiv and "arxiv.org" in u:
                    arxiv = arxiv_id_from_text(u)
        src = (w.get("primary_location") or {}).get("source") or {}
        authors = [(a.get("author") or {}).get("display_name", "") for a in w.get("authorships") or []]
        out.append({
            "openalex": (w.get("id") or "").rsplit("/", 1)[-1],
            "title": " ".join((w.get("title") or w.get("display_name") or "").split()),
            "abstract": abstract_from_inverted(w.get("abstract_inverted_index")),
            "year": w.get("publication_year") or "", "date": w.get("publication_date") or "",
            "venue": src.get("display_name") or "", "type": w.get("type") or "",
            "doi": "" if doi.startswith("10.48550/arxiv.") else doi, "arxiv": arxiv,
            "first_author": authors[0] if authors else "", "authors": authors,
        })
    return out


def openalex_search(text: str, date_from: str, api_key: str = "", max_total: int = 5000, fetch=None) -> tuple[list[dict], str, str]:
    """Title-and-abstract search, page by page (OpenAlex allows up to 10,000 results this way).

    Since February 2026 OpenAlex requires an API key for more than a first page; without one the first page is
    returned and the status says so. The key is never written to the log: the URL is logged without it."""
    first_url = ""
    if OFFLINE and fetch is None:
        return [], "offline", first_url
    fetch = fetch or _get
    records, page, total = [], 1, None
    flt = f"title_and_abstract.search:{text},from_publication_date:{date_from}"
    while True:
        base = (f"{OPENALEX_API}?filter={urllib.parse.quote(flt, safe=':,')}&per-page=100&page={page}"
                f"&select=id,doi,title,display_name,publication_year,publication_date,type,primary_location,locations,authorships,abstract_inverted_index")
        first_url = first_url or base
        url = base + (f"&api_key={api_key}" if api_key else "")
        try:
            payload = json.loads(fetch(url))
        except urllib.error.HTTPError as exc:
            if exc.code == 401 and page > 1:
                return records, f"first page only: a page {page} request was refused (401); an API key is needed for more", first_url
            return records, f"failed after {len(records)}: HTTP Error {exc.code}", first_url
        except (urllib.error.URLError, json.JSONDecodeError, OSError) as exc:
            return records, f"failed after {len(records)}: {exc}", first_url
        results = payload.get("results") or []
        records.extend(parse_openalex_page(payload))
        total = int((payload.get("meta") or {}).get("count") or 0)
        if not results or len(records) >= min(total, max_total):
            break
        if not api_key:
            return records, f"first page only ({len(records)} of {total}): no OpenAlex API key in .secrets/openalex_key.txt", first_url
        page += 1
        time.sleep(1.0)
    return records, "ok", first_url


# ---- DBLP ---------------------------------------------------------------------------------------------------

def parse_dblp_page(payload: dict) -> list[dict]:
    out = []
    hits = ((payload.get("result") or {}).get("hits") or {}).get("hit") or []
    for h in hits:
        info = h.get("info") or {}
        authors = info.get("authors", {}).get("author", []) if isinstance(info.get("authors"), dict) else []
        if isinstance(authors, dict):
            authors = [authors]
        names = [a.get("text", "") if isinstance(a, dict) else str(a) for a in authors]
        ee = info.get("ee") or ""
        out.append({
            "dblp": info.get("key", ""), "title": " ".join((info.get("title") or "").rstrip(".").split()),
            "year": info.get("year", ""), "date": "", "venue": info.get("venue", "") if isinstance(info.get("venue"), str) else " / ".join(info.get("venue") or []),
            "doi": (info.get("doi") or "").lower(), "arxiv": arxiv_id_from_text(ee) if "arxiv" in ee.lower() else "",
            "type": info.get("type", ""), "first_author": names[0] if names else "", "authors": names, "abstract": "",
        })
    return out


def dblp_search(text: str, max_total: int = 3000, fetch=None) -> tuple[list[dict], str, str]:
    first_url = ""
    if OFFLINE and fetch is None:
        return [], "offline", first_url
    fetch = fetch or _get
    records, offset = [], 0
    while True:
        url = f"{DBLP_API}?q={urllib.parse.quote(text)}&h=1000&f={offset}&format=json"
        first_url = first_url or url
        body = b""
        for attempt in range(2):
            try:
                body = fetch(url)
                payload = json.loads(body)
                break
            except json.JSONDecodeError:
                snippet = body[:160].decode("utf-8", "ignore").replace("\n", " ")
                if attempt == 0:
                    time.sleep(10.0)
                    continue
                return records, f"failed after {len(records)}: response is not JSON: {snippet}", first_url
            except (urllib.error.URLError, OSError) as exc:
                if attempt == 0:
                    time.sleep(10.0)
                    continue
                return records, f"failed after {len(records)}: {exc}", first_url
        batch = parse_dblp_page(payload)
        records.extend(batch)
        total = int(((payload.get("result") or {}).get("hits") or {}).get("@total", 0) or 0)
        offset += 1000
        if not batch or offset >= min(total, max_total):
            break
        time.sleep(2.0)
    return records, "ok", first_url


# ---- exported record files (RIS, BibTeX, CSV) ------------------------------------------------------------------

def parse_ris(text: str) -> list[dict]:
    out, cur = [], {}
    for line in text.splitlines():
        m = re.match(r"^([A-Z][A-Z0-9])  - ?(.*)$", line)
        if not m:
            continue
        tag, val = m.group(1), m.group(2).strip()
        if tag == "TY":
            cur = {"authors": []}
        elif tag == "ER":
            if cur:
                out.append(_ris_record(cur))
            cur = {}
        elif tag in ("TI", "T1"):
            cur["title"] = val
        elif tag in ("AU", "A1"):
            cur["authors"].append(val)
        elif tag in ("PY", "Y1"):
            cur["year"] = val[:4]
        elif tag == "DO":
            cur["doi"] = val.lower().replace("https://doi.org/", "")
        elif tag in ("AB", "N2"):
            cur["abstract"] = (cur.get("abstract", "") + " " + val).strip()
        elif tag in ("JO", "T2", "JF", "BT"):
            cur.setdefault("venue", val)
        elif tag == "UR":
            cur.setdefault("url", val)
    return out


def _ris_record(cur: dict) -> dict:
    return {"title": cur.get("title", ""), "first_author": (cur.get("authors") or [""])[0], "authors": cur.get("authors", []),
            "year": cur.get("year", ""), "date": "", "venue": cur.get("venue", ""), "doi": cur.get("doi", ""),
            "arxiv": arxiv_id_from_text(cur.get("url", "")) if "arxiv" in cur.get("url", "") else "", "abstract": cur.get("abstract", "")}


def parse_bibtex(text: str) -> list[dict]:
    out = []
    for entry in re.split(r"\n@", "\n" + text)[1:]:
        fields = dict((k.lower(), re.sub(r"\s+", " ", v.strip().strip("{}\",")))
                      for k, v in re.findall(r"(\w+)\s*=\s*[{\"]((?:[^{}]|\{[^{}]*\})*)[}\"]", entry))
        authors = [a.strip() for a in re.split(r"\s+and\s+", fields.get("author", "")) if a.strip()]
        out.append({"title": fields.get("title", ""), "first_author": authors[0] if authors else "", "authors": authors,
                    "year": fields.get("year", ""), "date": "", "venue": fields.get("journal") or fields.get("booktitle") or "",
                    "doi": fields.get("doi", "").lower(), "arxiv": arxiv_id_from_text(fields.get("eprint", "") or fields.get("url", "")),
                    "abstract": fields.get("abstract", "")})
    return out


def parse_export_file(path: Path) -> list[dict]:
    path = Path(path)
    text = path.read_text(encoding="utf-8", errors="ignore")
    if path.suffix.lower() == ".ris":
        return parse_ris(text)
    if path.suffix.lower() in (".bib", ".bibtex"):
        return parse_bibtex(text)
    if path.suffix.lower() == ".csv":
        rows = list(csv.DictReader(text.splitlines()))
        out = []
        for r in rows:
            low = {k.lower().strip(): (v or "") for k, v in r.items() if k}
            title = low.get("title") or low.get("document title") or low.get("item title") or ""
            out.append({"title": title, "first_author": (low.get("authors") or low.get("author") or "").split(";")[0].strip(),
                        "year": (low.get("year") or low.get("publication year") or "")[:4], "date": "",
                        "venue": low.get("publication title") or low.get("source title") or low.get("venue") or "",
                        "doi": (low.get("doi") or "").lower(), "arxiv": arxiv_id_from_text(low.get("url", "") or low.get("pdf link", "")),
                        "abstract": low.get("abstract") or ""})
        return out
    return []


# ---- citation lists from PDFs ----------------------------------------------------------------------------------

def arxiv_ids_in_pdf(path: Path) -> list[str]:
    from pypdf import PdfReader
    text = " ".join((p.extract_text() or "") for p in PdfReader(str(path)).pages)
    text = re.sub(r"\s+", " ", text)
    return sorted(set(m.group(1) for m in ARXIV_ID_RE.finditer(text)))


def arxiv_metadata_for_ids(ids: list[str], cache_path: Path, batch: int = 50, fetch=None) -> dict:
    cache_path = Path(cache_path)
    cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    missing = [i for i in ids if i not in cache]
    if (OFFLINE and fetch is None) or not missing:
        return cache
    fetch = fetch or _get
    for s in range(0, len(missing), batch):
        chunk = missing[s:s + batch]
        url = f"{ARXIV_API}?id_list={','.join(chunk)}&max_results={len(chunk)}"
        try:
            recs, _ = parse_arxiv_feed(fetch(url))
        except (urllib.error.URLError, ET.ParseError, OSError) as exc:
            print("arXiv id_list request failed:", exc)
            break
        for r in recs:
            cache[r["arxiv"]] = r
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        cache_path.write_text(json.dumps(cache, ensure_ascii=False))
        time.sleep(GAP)
    return cache


# ---- merging and screening ------------------------------------------------------------------------------------

def record_key(rec: dict) -> str:
    if rec.get("arxiv"):
        return f"arxiv:{rec['arxiv']}"
    if rec.get("doi"):
        return f"doi:{rec['doi']}"
    return f"title:{norm_key(rec.get('title', ''))}"


def merge_hits(hits: list[tuple[str, str, dict]]) -> list[dict]:
    """hits: (source, query_id, record). One row per key; sources and queries aggregated; preprint and published
    versions of one title linked through linked_to (the arXiv record is the root when present)."""
    merged: dict[str, dict] = {}
    for source, qid, rec in hits:
        key = record_key(rec)
        row = merged.get(key)
        if row is None:
            row = {"key": key, "title": rec.get("title", ""), "first_author": rec.get("first_author", ""),
                   "year": str(rec.get("year") or (rec.get("date") or "")[:4]), "date": rec.get("date", ""),
                   "venue": rec.get("venue", ""), "arxiv": rec.get("arxiv", ""), "arxiv_version": rec.get("arxiv_version", ""),
                   "doi": rec.get("doi", ""), "abstract": rec.get("abstract", ""), "_sources": set(), "_queries": set(),
                   "n_hits": 0, "linked_to": ""}
            merged[key] = row
        for f in ("title", "first_author", "venue", "doi", "abstract", "date", "arxiv_version"):
            if not row.get(f) and rec.get(f):
                row[f] = rec[f]
        if not row.get("year") and (rec.get("year") or rec.get("date")):
            row["year"] = str(rec.get("year") or rec["date"][:4])
        row["_sources"].add(source)
        row["_queries"].add(qid)
        row["n_hits"] += 1
    by_title: dict[str, str] = {}
    for key, row in merged.items():
        t = norm_key(row["title"])
        if not t:
            continue
        root = by_title.get(t)
        if root is None:
            by_title[t] = key
        else:
            # prefer the arXiv record as the root so the preprint and the published article count once
            if key.startswith("arxiv:") and not root.startswith("arxiv:"):
                merged[root]["linked_to"] = key
                by_title[t] = key
            else:
                row["linked_to"] = root
    rows = []
    for row in merged.values():
        row["sources"] = ";".join(sorted(row.pop("_sources")))
        row["queries"] = ";".join(sorted(row.pop("_queries")))
        rows.append(row)
    rows.sort(key=lambda r: (r["linked_to"] != "", -(r["n_hits"]), r["key"]))
    return rows


def apply_decisions(rows: list[dict], decisions: list[dict]) -> tuple[list[dict], list[str]]:
    """Attach decision, reason, access_status and note by key; report keys with invalid values or no match."""
    problems = []
    by_key = {d["key"]: d for d in decisions}
    for row in rows:
        d = by_key.get(row["key"], {})
        row["decision"] = d.get("decision", "pending")
        row["reason"] = d.get("reason", "")
        row["stage"] = d.get("stage", "title_abstract" if d else "")
        row["access_status"] = d.get("access_status", "")
        row["note"] = d.get("note", "")
        if d and d.get("decision") not in DECISIONS:
            problems.append(f"{row['key']}: decision must be one of {sorted(DECISIONS)}")
        if d.get("decision") == "exclude" and d.get("reason") not in EXCLUSION_REASONS:
            problems.append(f"{row['key']}: exclusion needs one reason from the fixed list")
        if d and row["stage"] not in SCREEN_STAGES:
            problems.append(f"{row['key']}: stage must be one of {sorted(SCREEN_STAGES)}")
    unknown = [k for k in by_key if k not in {r["key"] for r in rows}]
    problems += [f"decision for a key not in the screening table: {k}" for k in unknown]
    return rows, problems


def prisma_counts(log_rows: list[dict], rows: list[dict]) -> dict:
    per_source: dict[str, int] = {}
    for r in log_rows:
        if str(r.get("count", "")).isdigit():
            per_source[r["source"]] = per_source.get(r["source"], 0) + int(r["count"])
    roots = [r for r in rows if not r.get("linked_to")]
    out = {"records_identified_per_source": per_source,
           "records_identified_total": sum(per_source.values()),
           "unique_records_after_dedup": len(rows),
           "studies_after_version_linking": len(roots),
           "screened": sum(1 for r in roots if r.get("decision") not in ("", "pending", None)),
           "pending": sum(1 for r in roots if r.get("decision") in ("", "pending", None)),
           "included": sum(1 for r in roots if r.get("decision") == "include"),
           "unsure_to_full_text": sum(1 for r in roots if r.get("decision") == "unsure"),
           "excluded_by_reason": {},
           "excluded_title_abstract_by_reason": {},
           "full_text_assessed": sum(1 for r in roots if r.get("stage") == "full_text"),
           "included_after_full_text": sum(1 for r in roots if r.get("stage") == "full_text" and r.get("decision") == "include"),
           "excluded_full_text_by_reason": {}}
    for r in roots:
        if r.get("decision") == "exclude":
            reason = r.get("reason", "")
            out["excluded_by_reason"][reason] = out["excluded_by_reason"].get(reason, 0) + 1
            key = "excluded_full_text_by_reason" if r.get("stage") == "full_text" else "excluded_title_abstract_by_reason"
            out[key][reason] = out[key].get(reason, 0) + 1
    out["reports_sought_for_full_text"] = out["included"] + out["unsure_to_full_text"] + sum(out["excluded_full_text_by_reason"].values())
    return out


# ---- full-text retrieval for the screened records ---------------------------------------------------------------

def pdf_slug(row: dict) -> str:
    """File stem for a record's PDF: the arXiv id, else the DOI, else the normalised title; unique because keys are."""
    if row.get("arxiv"):
        return row["arxiv"]
    if row.get("doi"):
        return "doi_" + re.sub(r"[^a-z0-9]+", "_", row["doi"].lower()).strip("_")
    return "title_" + norm_key(row.get("title", ""))[:60]


def zenodo_pdf_from_payload(payload: dict) -> tuple[str, str]:
    """(url, note) for the first PDF file of a Zenodo record payload; the url is '' when the record holds no PDF."""
    files = payload.get("files", []) or []
    if isinstance(files, dict):                                 # InvenioRDM shape: {"entries": {name: {...}}}
        files = list((files.get("entries") or {}).values())
    names = [str(f.get("key") or f.get("filename") or "") for f in files]
    pdfs = [f for f in files if str(f.get("key") or f.get("filename") or "").lower().endswith(".pdf")]
    if not pdfs:
        return "", "zenodo record has no PDF file (files: " + (", ".join(names[:6]) or "none") + ")"
    f = pdfs[0]
    key = str(f.get("key") or f.get("filename"))
    url = (f.get("links") or {}).get("content") or f"{ZENODO_API}/{payload.get('id', '')}/files/{urllib.parse.quote(key)}/content"
    return url, f"zenodo file {key}" + (f" (first of {len(pdfs)} PDFs)" if len(pdfs) > 1 else "")


def figshare_pdf_from_payload(payload: dict) -> tuple[str, str]:
    files = payload.get("files", []) or []
    pdfs = [f for f in files if str(f.get("name", "")).lower().endswith(".pdf")]
    if not pdfs:
        return "", "figshare item has no PDF file (files: " + (", ".join(str(f.get("name", "")) for f in files[:6]) or "none") + ")"
    return pdfs[0].get("download_url", ""), f"figshare file {pdfs[0].get('name')}" + (f" (first of {len(pdfs)} PDFs)" if len(pdfs) > 1 else "")


def openalex_pdf_from_payload(work: dict) -> tuple[str, str]:
    for name in ("best_oa_location", "primary_location"):
        loc = work.get(name) or {}
        if loc.get("pdf_url"):
            return loc["pdf_url"], f"openalex {name} pdf_url"
    oa_url = (work.get("open_access") or {}).get("oa_url")
    if oa_url:
        return oa_url, "openalex open_access oa_url (may be a landing page rather than the PDF)"
    return "", "openalex lists no open-access copy"


def resolve_pdf_url(row: dict, api_key: str = "", fetch=None) -> tuple[str, str]:
    """(url, note) for a screened record: arXiv directly; Zenodo and figshare through their record APIs; any other DOI,
    and title-only records, through OpenAlex's open-access location. The note never contains the API key."""
    fetch = fetch or _get
    if row.get("arxiv"):
        v = row.get("arxiv_version") or ""
        return f"{ARXIV_PDF}/{row['arxiv']}" + (f"v{v}" if v else ""), "arxiv"
    if OFFLINE:
        return "", "offline"
    doi = (row.get("doi") or "").strip().lower()
    key = f"&api_key={api_key}" if api_key else ""
    try:
        m = re.match(r"10\.5281/zenodo\.(\d+)$", doi)
        if m:
            return zenodo_pdf_from_payload(json.loads(fetch(f"{ZENODO_API}/{m.group(1)}")))
        m = re.match(r"10\.6084/m9\.figshare\.(\d+)", doi)
        if m:
            return figshare_pdf_from_payload(json.loads(fetch(f"{FIGSHARE_API}/{m.group(1)}")))
        if doi:
            work = json.loads(fetch(f"{OPENALEX_API}/https://doi.org/{urllib.parse.quote(doi, safe='/')}?select=id,open_access,best_oa_location,primary_location{key}"))
            return openalex_pdf_from_payload(work)
        title = row.get("title", "")
        if title:
            page = json.loads(fetch(f"{OPENALEX_API}?search={urllib.parse.quote(title[:300])}&per-page=5&select=id,title,open_access,best_oa_location,primary_location{key}"))
            for work in page.get("results", []):
                if norm_title(work.get("title") or "") == norm_title(title):
                    url, note = openalex_pdf_from_payload(work)
                    return url, note + " (matched by title)"
            return "", "openalex: no work with this exact title"
        return "", "no identifier to resolve"
    except urllib.error.HTTPError as exc:
        return "", f"lookup failed: HTTP {exc.code}"
    except Exception as exc:                                     # network and parsing errors are reported, not raised
        return "", "lookup failed: " + type(exc).__name__ + ": " + str(exc).replace(api_key, "<key>")[:160]



# ---- full-text location, second pass: every open route, tried in order --------------------------------------

FOLLOW_RELATIONS = {"issupplementto", "isdocumentedby", "isdescribedby", "ispartof", "isidenticalto", "isversionof",
                    "ispublishedin", "ispreviousversionof", "isnewversionof"}


def zenodo_related(payload: dict) -> list[tuple[str, str]]:
    """(scheme, identifier) pairs a Zenodo record points to as its paper or another version of it."""
    out = []
    for r in (payload.get("metadata") or {}).get("related_identifiers") or []:
        relation = str(r.get("relation") or (r.get("relation_type") or {}).get("id") or "").lower()
        ident = str(r.get("identifier") or "").strip()
        scheme = str(r.get("scheme") or "").lower()
        if relation in FOLLOW_RELATIONS and ident:
            if scheme == "doi" or ident.lower().startswith("10."):
                out.append(("doi", re.sub(r"^https?://(dx\.)?doi\.org/", "", ident, flags=re.I).lower()))
            elif scheme == "arxiv" or arxiv_id_from_text(ident):
                out.append(("arxiv", arxiv_id_from_text(ident) or ident))
            elif ident.lower().startswith("http"):
                out.append(("url", ident))
    return out


def osf_pdf_candidates(osf_id: str, fetch) -> list[tuple[str, str]]:
    """OSF preprints expose a primary file; OSF projects expose osfstorage files. PDFs only."""
    out = []
    try:
        pre = json.loads(fetch(f"{OSF_API}/preprints/{osf_id}/"))
        rel = (((pre.get("data") or {}).get("relationships") or {}).get("primary_file") or {}).get("links", {}).get("related", {})
        href = rel.get("href") if isinstance(rel, dict) else rel
        if href:
            f = json.loads(fetch(href))
            dl = ((f.get("data") or {}).get("links") or {}).get("download")
            if dl:
                out.append((dl, "osf preprint primary file"))
    except Exception:
        pass
    if not out:
        try:
            listing = json.loads(fetch(f"{OSF_API}/nodes/{osf_id}/files/osfstorage/"))
            for item in listing.get("data") or []:
                attrs = item.get("attributes") or {}
                if str(attrs.get("name", "")).lower().endswith(".pdf"):
                    dl = (item.get("links") or {}).get("download")
                    if dl:
                        out.append((dl, f"osf project file {attrs.get('name')}"))
        except Exception:
            pass
    return out


def openalex_pdf_candidates(work: dict) -> list[tuple[str, str]]:
    out, seen = [], set()
    locs = [("best_oa_location", work.get("best_oa_location"))] + [("location", l) for l in (work.get("locations") or [])]
    for name, loc in locs:
        url = (loc or {}).get("pdf_url")
        if url and url not in seen:
            seen.add(url)
            out.append((url, f"openalex {name} pdf_url"))
    return out


def arxiv_by_title(title: str, fetch) -> tuple[str, str]:
    """arXiv id whose title matches exactly (after normalisation), searched by title phrase; '' when none."""
    words = [w for w in norm_title(title).split() if len(w) > 2][:12]
    if not words:
        return "", ""
    q = " AND ".join(f"ti:{w}" for w in words)
    try:
        recs, _ = parse_arxiv_feed(fetch(f"{ARXIV_API}?search_query={urllib.parse.quote(q)}&max_results=10"))
    except Exception as exc:
        return "", f"arxiv title search failed: {type(exc).__name__}"
    for r in recs:
        if norm_title(r["title"]) == norm_title(title):
            return r["arxiv"], "arxiv version found by exact title"
    return "", "no arxiv version with this exact title"


def fulltext_candidates(row: dict, api_key: str = "", fetch=None, depth: int = 0) -> tuple[list[tuple[str, str]], list[str]]:
    """Every open URL that might serve this record's PDF, in the order to try, plus notes on routes that gave nothing.
    Routes: arXiv (as recorded, then unversioned); Zenodo files and the records a Zenodo deposit points to; figshare;
    OSF preprints and projects; Scientific Reports / Nature article PDFs; every OpenAlex location with a pdf_url; an
    arXiv version with the same title. The API key goes into OpenAlex requests only and never into a note."""
    fetch = fetch or _get
    cands, notes = [], []
    key = f"&api_key={api_key}" if api_key else ""
    if row.get("arxiv"):
        v = row.get("arxiv_version") or ""
        if v:
            cands.append((f"{ARXIV_PDF}/{row['arxiv']}v{v}", "arxiv"))
        cands.append((f"{ARXIV_PDF}/{row['arxiv']}", "arxiv, latest version"))
        return cands, notes
    if OFFLINE:
        return cands, ["offline"]
    doi = (row.get("doi") or "").strip().lower()
    try:
        m = re.match(r"10\.5281/zenodo\.(\d+)$", doi)
        if m:
            payload = json.loads(fetch(f"{ZENODO_API}/{m.group(1)}"))
            url, note = zenodo_pdf_from_payload(payload)
            if url:
                cands.append((url, note))
            else:
                notes.append(note)
            if depth == 0:
                for scheme, ident in zenodo_related(payload):
                    sub = {"arxiv": ident} if scheme == "arxiv" else {"doi": ident} if scheme == "doi" else {}
                    if scheme == "url" and ident.lower().endswith(".pdf"):
                        cands.append((ident, "zenodo related link"))
                    elif sub:
                        c2, n2 = fulltext_candidates(sub, api_key, fetch, depth + 1)
                        cands += [(u, f"zenodo related {scheme} {ident}: {n}") for u, n in c2]
                        notes += n2
                if not zenodo_related(payload):
                    notes.append("zenodo record names no related paper")
    except urllib.error.HTTPError as exc:
        notes.append(f"zenodo lookup: HTTP {exc.code}" + (" (record removed by its owner)" if exc.code == 410 else ""))
    except Exception as exc:
        notes.append(f"zenodo lookup failed: {type(exc).__name__}")
    try:
        m = re.match(r"10\.6084/m9\.figshare\.(\d+)", doi)
        if m:
            url, note = figshare_pdf_from_payload(json.loads(fetch(f"{FIGSHARE_API}/{m.group(1)}")))
            (cands.append((url, note)) if url else notes.append(note))
    except urllib.error.HTTPError as exc:
        notes.append(f"figshare lookup: HTTP {exc.code}")
    except Exception as exc:
        notes.append(f"figshare lookup failed: {type(exc).__name__}")
    m = re.match(r"10\.(17605|31219|31234|31235|31222)/osf\.io/([a-z0-9_]+)", doi)
    if m:
        c = osf_pdf_candidates(m.group(2), fetch)
        cands += c
        if not c:
            notes.append("osf: no PDF file exposed")
    m = re.match(r"10\.1038/(s\d{5}-\d{3}-\d{5}-\w)$", doi)
    if m:
        cands.append((f"{NATURE_PDF}/{m.group(1)}.pdf", "nature.com article pdf"))
    try:
        if doi:
            work = json.loads(fetch(f"{OPENALEX_API}/https://doi.org/{urllib.parse.quote(doi, safe='/')}?select=id,open_access,best_oa_location,primary_location,locations{key}"))
            c = openalex_pdf_candidates(work)
            cands += c
            if not c:
                notes.append("openalex lists no pdf_url")
        elif row.get("title"):
            page = json.loads(fetch(f"{OPENALEX_API}?search={urllib.parse.quote(row['title'][:300])}&per-page=5&select=id,title,best_oa_location,locations{key}"))
            hits = [w for w in page.get("results", []) if norm_title(w.get("title") or "") == norm_title(row["title"])]
            for w in hits:
                cands += [(u, n + " (matched by title)") for u, n in openalex_pdf_candidates(w)]
            if not hits:
                notes.append("openalex: no work with this exact title")
    except urllib.error.HTTPError as exc:
        notes.append(f"openalex lookup: HTTP {exc.code}")
    except Exception as exc:
        notes.append("openalex lookup failed: " + type(exc).__name__ + ": " + str(exc).replace(api_key or "\x00", "<key>")[:120])
    if row.get("title") and depth == 0:
        aid, note = arxiv_by_title(row["title"], fetch)
        if aid:
            cands.append((f"{ARXIV_PDF}/{aid}", note + f" ({aid})"))
        elif note:
            notes.append(note)
    seen, uniq = set(), []
    for u, n in cands:
        u = clean_url(u)
        if u and u not in seen:
            seen.add(u)
            uniq.append((u, n))
    return uniq, notes


def clean_url(url: str) -> str:
    """Strip whitespace and percent-encode characters that urllib refuses (spaces, control characters)."""
    url = (url or "").strip()
    return urllib.parse.quote(url, safe=":/?&=%#~+@!$,;'()*[]")


def hand_fetch_hint(row: dict, attempts: list[str], notes: list[str]) -> str:
    """Plain instruction for a record still without a PDF: where a person in a browser can get it."""
    doi = (row.get("doi") or "").lower()
    text = " ".join(attempts + notes).lower()
    if "removed by its owner" in text:
        return "record removed from Zenodo; search for the paper by title, otherwise it stays not retrieved"
    if doi.startswith("10.5281/zenodo") or doi.startswith("10.6084/"):
        return "data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF"
    if "403" in text or "not a pdf" in text:
        return f"open copy exists but the site refuses scripts; open https://doi.org/{doi} in a browser and save the PDF" if doi else "open copy exists but refuses scripts; download it in a browser"
    if doi:
        return (f"no open copy found by script; open https://doi.org/{doi} in a browser and save the PDF, "
                "or go through the university library if the page asks for payment")
    return "no identifier; search the title in a browser or the library catalogue"


# ---- files placed by hand -----------------------------------------------------------------------------------------

def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def place_hand_files(folder: Path, mapping: list[dict], known_slugs: set[str]) -> list[dict]:
    """Give files saved by hand into the PDF folder their standard names; safe to run again.

    Each mapping row names one uploaded file by its SHA-256, or by its exact file name when no hash is given, and says
    what to do with it: "use" renames it to <slug>.pdf; "duplicate" and "unusable" move it to <folder>/_not_used/.
    Files that already carry a standard name (<slug>.pdf) are never touched. A file is never overwritten: when <slug>.pdf
    already exists with other content, the uploaded file is left where it is and the row says so.
    """
    folder = Path(folder)
    spare = folder / "_not_used"
    loose = {p: file_sha256(p) for p in sorted(folder.iterdir()) if p.is_file() and p.stem not in known_slugs}
    results = []
    for m in mapping:
        sha, name, slug, action = m.get("sha256", ""), m.get("file_name", ""), m["slug"], m["action"]
        if sha:      # identical downloads share a fingerprint: prefer the one with the recorded name
            same = [p for p, h in loose.items() if h == sha]
            src = next((p for p in same if p.name == name), same[0] if same else None)
        else:
            src = next((p for p in loose if p.name == name), None)
        target = folder / f"{slug}.pdf"
        out = {"file_name": name, "slug": slug, "action": action, "sha256": sha, "status": ""}
        if action == "use":
            if src is None:
                if target.exists() and (not sha or file_sha256(target) == sha):
                    out["status"] = "already placed"
                else:
                    out["status"] = "not found"
            elif target.exists():
                if file_sha256(target) == loose[src]:
                    spare.mkdir(exist_ok=True)
                    shutil.move(str(src), str(spare / src.name))
                    out["status"] = "same file already placed; copy moved to _not_used"
                else:
                    out["status"] = "kept aside: a different file already has the standard name"
            else:
                src.rename(target)
                out["status"] = "renamed"
        elif action in ("duplicate", "unusable"):
            if src is None:
                moved = spare / name
                out["status"] = "already moved" if moved.exists() else "not found"
            else:
                spare.mkdir(exist_ok=True)
                dest = spare / src.name
                if dest.exists():
                    dest = spare / f"{dest.stem}_{(sha or file_sha256(src))[:8]}{dest.suffix}"
                shutil.move(str(src), str(dest))
                out["status"] = "moved to _not_used"
        else:
            out["status"] = "unknown action"
        if src is not None:
            loose.pop(src, None)
        results.append(out)
    return results

In [ ]:
%%writefile tests/test_search.py
"""Parsers, merging and the screening-decision checks of the search module."""
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
import search  # noqa: E402

ARXIV_FEED = b"""<?xml version="1.0" encoding="UTF-8"?>
<feed xmlns="http://www.w3.org/2005/Atom" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/">
  <opensearch:totalResults>2</opensearch:totalResults>
  <entry>
    <id>http://arxiv.org/abs/2402.06363v2</id>
    <published>2024-02-09T00:00:00Z</published><updated>2024-05-01T00:00:00Z</updated>
    <title>StruQ: Defending Against Prompt Injection
  with Structured Queries</title>
    <summary>We propose structured queries.</summary>
    <author><name>Sizhe Chen</name></author><author><name>Other Author</name></author>
    <arxiv:primary_category term="cs.CR"/>
  </entry>
  <entry>
    <id>http://arxiv.org/abs/2403.14720v1</id>
    <published>2024-03-21T00:00:00Z</published><updated>2024-03-21T00:00:00Z</updated>
    <title>Defending Against Indirect Prompt Injection Attacks With Spotlighting</title>
    <summary>Spotlighting.</summary>
    <author><name>Keegan Hines</name></author>
    <arxiv:journal_ref>Some venue 2024</arxiv:journal_ref>
    <arxiv:doi>10.1000/EXAMPLE</arxiv:doi>
  </entry>
</feed>"""

OPENALEX_PAGE = {
    "meta": {"next_cursor": None},
    "results": [
        {"id": "https://openalex.org/W1", "doi": "https://doi.org/10.48550/arXiv.2402.06363", "title": "StruQ: Defending Against Prompt Injection with Structured Queries",
         "publication_year": 2024, "publication_date": "2024-02-09", "type": "preprint",
         "primary_location": {"source": {"display_name": "arXiv (Cornell University)"}}, "locations": [],
         "authorships": [{"author": {"display_name": "Sizhe Chen"}}],
         "abstract_inverted_index": {"We": [0], "propose": [1], "structured": [2], "queries.": [3]}},
        {"id": "https://openalex.org/W2", "doi": "https://doi.org/10.1109/SP.2025.1", "title": "StruQ: Defending Against Prompt Injection with Structured Queries",
         "publication_year": 2025, "publication_date": "2025-05-01", "type": "article",
         "primary_location": {"source": {"display_name": "USENIX Security"}}, "locations": [{"landing_page_url": "https://www.usenix.org/x"}],
         "authorships": [{"author": {"display_name": "Sizhe Chen"}}], "abstract_inverted_index": None},
    ],
}

DBLP_PAGE = {"result": {"hits": {"@total": "1", "hit": [
    {"info": {"key": "conf/uss/Chen25", "title": "StruQ: Defending Against Prompt Injection with Structured Queries.",
              "venue": "USENIX Security Symposium", "year": "2025", "doi": "10.1109/SP.2025.1", "ee": "https://doi.org/10.1109/SP.2025.1",
              "type": "Conference and Workshop Papers", "authors": {"author": [{"text": "Sizhe Chen"}, {"text": "Other"}]}}}]}}}


def test_parse_arxiv_feed_reads_ids_versions_titles_and_total():
    recs, total = search.parse_arxiv_feed(ARXIV_FEED)
    assert total == 2 and [r["arxiv"] for r in recs] == ["2402.06363", "2403.14720"]
    assert recs[0]["arxiv_version"] == 2 and recs[0]["title"].endswith("with Structured Queries")
    assert recs[0]["first_author"] == "Sizhe Chen" and recs[0]["primary_category"] == "cs.CR"
    assert recs[1]["doi"] == "10.1000/example" and recs[1]["venue"] == "Some venue 2024"


def test_arxiv_query_string_quotes_phrases_and_adds_window():
    q = search.arxiv_query_string(["prompt injection", "defense"], "20230101", "20261120")
    assert q.startswith('abs:"prompt injection" AND abs:defense AND (cat:cs.CR OR cat:cs.CL OR cat:cs.LG OR cat:cs.AI)')
    assert q.endswith("submittedDate:[202301010000 TO 202611202359]")


def test_parse_openalex_page_recovers_arxiv_id_and_abstract():
    recs = search.parse_openalex_page(OPENALEX_PAGE)
    assert recs[0]["arxiv"] == "2402.06363" and recs[0]["doi"] == "" and recs[0]["abstract"] == "We propose structured queries."
    assert recs[1]["doi"] == "10.1109/sp.2025.1" and recs[1]["venue"] == "USENIX Security" and recs[1]["abstract"] == ""


def test_parse_dblp_page():
    recs = search.parse_dblp_page(DBLP_PAGE)
    assert recs[0]["title"] == "StruQ: Defending Against Prompt Injection with Structured Queries"
    assert recs[0]["doi"] == "10.1109/sp.2025.1" and recs[0]["first_author"] == "Sizhe Chen" and recs[0]["year"] == "2025"


def test_arxiv_id_regex_accepts_only_2023_to_2026_ids():
    assert search.arxiv_id_from_text("see arXiv:2402.06363v2 for details") == "2402.06363"
    assert search.arxiv_id_from_text("arXiv preprint arXiv:2106.09685") == ""
    assert search.arxiv_id_from_text("doi 10.1234/5678.91011") == ""


def test_merge_hits_dedupes_across_sources_and_links_versions():
    a, _ = search.parse_arxiv_feed(ARXIV_FEED)
    o = search.parse_openalex_page(OPENALEX_PAGE)
    d = search.parse_dblp_page(DBLP_PAGE)
    hits = [("arxiv", "q01", a[0]), ("arxiv", "q05", a[0]), ("arxiv", "q01", a[1]),
            ("openalex", "o01", o[0]), ("openalex", "o01", o[1]), ("dblp", "d01", d[0])]
    rows = search.merge_hits(hits)
    by_key = {r["key"]: r for r in rows}
    struq = by_key["arxiv:2402.06363"]
    assert struq["n_hits"] == 3 and struq["sources"] == "arxiv;openalex" and struq["queries"] == "o01;q01;q05"
    published = by_key["doi:10.1109/sp.2025.1"]
    assert published["linked_to"] == "arxiv:2402.06363" and published["sources"] == "dblp;openalex"
    assert by_key["arxiv:2403.14720"]["linked_to"] == ""
    assert rows[0]["linked_to"] == "" and rows[-1]["linked_to"] != ""


def test_apply_decisions_flags_bad_values_and_unknown_keys():
    rows = [{"key": "arxiv:1"}, {"key": "arxiv:2"}, {"key": "arxiv:3"}]
    decisions = [{"key": "arxiv:1", "decision": "include"}, {"key": "arxiv:2", "decision": "exclude", "reason": "made up"},
                 {"key": "arxiv:9", "decision": "exclude", "reason": "attack only"}]
    rows, problems = search.apply_decisions(rows, decisions)
    assert rows[0]["decision"] == "include" and rows[2]["decision"] == "pending"
    assert any("arxiv:2" in p for p in problems) and any("arxiv:9" in p for p in problems)


def test_prisma_counts():
    log = [{"source": "arxiv", "count": "10"}, {"source": "arxiv", "count": "5"}, {"source": "dblp", "count": "offline"}]
    rows = [{"key": "a", "linked_to": "", "decision": "include"}, {"key": "b", "linked_to": "a", "decision": "include"},
            {"key": "c", "linked_to": "", "decision": "exclude", "reason": "attack only"}, {"key": "d", "linked_to": "", "decision": "pending"}]
    c = search.prisma_counts(log, rows)
    assert c["records_identified_per_source"] == {"arxiv": 15} and c["unique_records_after_dedup"] == 4
    assert c["studies_after_version_linking"] == 3 and c["included"] == 1 and c["pending"] == 1
    assert c["excluded_by_reason"] == {"attack only": 1}


def test_parse_ris_and_bibtex():
    ris = "TY  - JOUR\nTI  - A Title\nAU  - Doe, J.\nAU  - Roe, R.\nPY  - 2025\nDO  - https://doi.org/10.1/ABC\nAB  - Part one.\nAB  - Part two.\nER  - \n"
    r = search.parse_ris(ris)[0]
    assert r["title"] == "A Title" and r["first_author"] == "Doe, J." and r["doi"] == "10.1/abc" and r["abstract"] == "Part one. Part two."
    bib = '@inproceedings{x,\n title={Another {Title}},\n author={Jane Doe and John Roe},\n year={2024},\n booktitle={Proc. X},\n eprint={2402.06363}\n}\n'
    b = search.parse_bibtex(bib)[0]
    assert b["first_author"] == "Jane Doe" and b["venue"] == "Proc. X" and b["arxiv"] == "2402.06363" and b["year"] == "2024"


def test_search_functions_report_failure_instead_of_raising():
    def boom(url, timeout=90):
        raise OSError("HTTP Error 503")
    recs, status = search.arxiv_search("abs:x", fetch=boom)
    assert recs == [] and status.startswith("failed")
    recs, status, _ = search.openalex_search("x", "2023-01-01", fetch=boom)
    assert recs == [] and status.startswith("failed")
    recs, status, _ = search.dblp_search("x", fetch=boom)
    assert recs == [] and status.startswith("failed")


def test_search_functions_parse_fixture_responses():
    calls = []
    def fake(url, timeout=90):
        calls.append(url)
        if url.startswith(search.ARXIV_API):
            return ARXIV_FEED
        if url.startswith(search.OPENALEX_API):
            return json.dumps(OPENALEX_PAGE).encode()
        return json.dumps(DBLP_PAGE).encode()
    recs, status = search.arxiv_search('abs:"prompt injection"', fetch=fake)
    assert status == "ok" and len(recs) == 2
    recs, status, url = search.openalex_search('"prompt injection"', "2023-01-01", api_key="k", fetch=fake)
    assert status == "ok" and len(recs) == 2 and "title_and_abstract.search" in url and "api_key" not in url
    recs, status, url = search.dblp_search("prompt injection", fetch=fake)
    assert status == "ok" and len(recs) == 1 and "format=json" in url


def test_openalex_without_key_stops_after_the_first_page_and_says_so():
    big = {"meta": {"count": 350}, "results": OPENALEX_PAGE["results"]}
    recs, status, _ = search.openalex_search("x", "2023-01-01", fetch=lambda url, timeout=90: json.dumps(big).encode())
    assert len(recs) == 2 and status.startswith("first page only") and "API key" in status


def test_openalex_401_on_a_later_page_is_reported_not_raised():
    import urllib.error
    big = {"meta": {"count": 350}, "results": OPENALEX_PAGE["results"]}
    def fake(url, timeout=90):
        if "page=1&" in url or url.endswith("page=1"):
            return json.dumps(big).encode()
        raise urllib.error.HTTPError(url, 401, "Unauthorized", None, None)
    recs, status, _ = search.openalex_search("x", "2023-01-01", api_key="bad", fetch=fake)
    assert len(recs) == 2 and "401" in status


def test_dblp_reports_non_json_body():
    search_gap = search.time.sleep
    search.time.sleep = lambda s: None
    try:
        recs, status, _ = search.dblp_search("x", fetch=lambda url, timeout=90: b"<html>Too Many Requests</html>")
    finally:
        search.time.sleep = search_gap
    assert recs == [] and "not JSON" in status and "Too Many Requests" in status


# ---- full-text retrieval -----------------------------------------------------------------------------------------

def test_pdf_slug_prefers_arxiv_then_doi_then_title():
    assert search.pdf_slug({"arxiv": "2402.06363", "doi": "10.1/x"}) == "2402.06363"
    assert search.pdf_slug({"doi": "10.1109/ICASSP55912.2026.11463282"}) == "doi_10_1109_icassp55912_2026_11463282"
    assert search.pdf_slug({"title": "Prompt Injection Defenses"}) == "title_promptinjectiondefenses"


def test_zenodo_payload_first_pdf_in_both_api_shapes():
    legacy = {"id": "20307701", "files": [{"key": "code.zip", "links": {}}, {"key": "paper v1.pdf", "links": {"self": "meta"}}, {"key": "b.pdf"}]}
    url, note = search.zenodo_pdf_from_payload(legacy)
    assert url == search.ZENODO_API + "/20307701/files/paper%20v1.pdf/content" and "first of 2 PDFs" in note
    rdm = {"id": "1", "files": {"entries": {"a.pdf": {"key": "a.pdf", "links": {"content": "https://zenodo.org/api/records/1/files/a.pdf/content"}}}}}
    assert search.zenodo_pdf_from_payload(rdm)[0].endswith("/files/a.pdf/content")
    url, note = search.zenodo_pdf_from_payload({"id": "2", "files": [{"key": "code.zip"}]})
    assert url == "" and "no PDF file" in note and "code.zip" in note


def test_figshare_and_openalex_payloads():
    assert search.figshare_pdf_from_payload({"files": [{"name": "r.pdf", "download_url": "https://nd/1"}]})[0] == "https://nd/1"
    assert search.figshare_pdf_from_payload({"files": [{"name": "r.csv"}]})[0] == ""
    work = {"best_oa_location": {"pdf_url": None}, "primary_location": {"pdf_url": "https://x/p.pdf"}}
    assert search.openalex_pdf_from_payload(work) == ("https://x/p.pdf", "openalex primary_location pdf_url")
    assert search.openalex_pdf_from_payload({"open_access": {"oa_url": "https://land"}})[0] == "https://land"
    assert search.openalex_pdf_from_payload({})[0] == ""


def test_resolve_pdf_url_dispatches_and_keeps_the_key_out_of_notes():
    calls = []

    def fetch(url):
        calls.append(url)
        if url.startswith(search.ZENODO_API):
            return json.dumps({"id": "5", "files": [{"key": "p.pdf"}]}).encode()
        if url.startswith(search.FIGSHARE_API):
            return json.dumps({"files": [{"name": "r.pdf", "download_url": "https://nd/1"}]}).encode()
        if "?search=" in url:
            return json.dumps({"results": [{"title": "Other paper", "best_oa_location": {"pdf_url": "https://oa/other.pdf"}},
                                           {"title": "Prompt Injection Defenses", "best_oa_location": {"pdf_url": "https://oa/x.pdf"}}]}).encode()
        return json.dumps({"best_oa_location": None, "primary_location": {"pdf_url": None}, "open_access": {"oa_url": "https://land"}}).encode()

    assert search.resolve_pdf_url({"arxiv": "2402.06363", "arxiv_version": "2"}, "SECRET", fetch) == (search.ARXIV_PDF + "/2402.06363v2", "arxiv")
    if search.OFFLINE:
        assert search.resolve_pdf_url({"doi": "10.5281/zenodo.5"}, "SECRET", fetch) == ("", "offline")
        return
    assert search.resolve_pdf_url({"doi": "10.5281/zenodo.5"}, "SECRET", fetch)[0] == search.ZENODO_API + "/5/files/p.pdf/content"
    assert search.resolve_pdf_url({"doi": "10.6084/m9.figshare.35040148"}, "SECRET", fetch)[0] == "https://nd/1"
    assert search.resolve_pdf_url({"doi": "10.3390/computers15090570"}, "SECRET", fetch)[0] == "https://land"
    url, note = search.resolve_pdf_url({"title": "Prompt Injection Defenses"}, "SECRET", fetch)
    assert url == "https://oa/x.pdf" and "matched by title" in note
    assert search.resolve_pdf_url({"title": "Nothing like this"}, "SECRET", fetch)[0] == ""
    assert all("SECRET" not in note for _, note in [search.resolve_pdf_url({"doi": "10.1/x"}, "SECRET", fetch)])
    assert sum("api_key=SECRET" in c for c in calls) >= 2        # the key goes to OpenAlex only, and only in the request


def test_resolve_pdf_url_reports_lookup_failures_without_raising():
    def fetch(url):
        raise RuntimeError("boom SECRET")
    if search.OFFLINE:
        return
    url, note = search.resolve_pdf_url({"doi": "10.5281/zenodo.9"}, "SECRET", fetch)
    assert url == "" and note.startswith("lookup failed") and "SECRET" not in note


# ---- second-pass full-text location, full-text stage --------------------------------------------------------

def test_new_exclusion_reason_and_stage_are_validated():
    rows = [{"key": "a"}, {"key": "b"}, {"key": "c"}]
    decisions = [{"key": "a", "decision": "exclude", "reason": "image or audio channel only", "stage": "full_text"},
                 {"key": "b", "decision": "include", "reason": "", "stage": "abstract"},
                 {"key": "c", "decision": "unsure", "reason": ""}]
    rows, problems = search.apply_decisions(rows, decisions)
    assert rows[0]["stage"] == "full_text" and rows[2]["stage"] == "title_abstract"
    assert problems == ["b: stage must be one of ['full_text', 'title_abstract']"]


def test_prisma_counts_split_title_abstract_and_full_text():
    rows = [{"key": "a", "decision": "exclude", "reason": "system level", "stage": "title_abstract", "linked_to": ""},
            {"key": "b", "decision": "exclude", "reason": "detector only", "stage": "full_text", "linked_to": ""},
            {"key": "c", "decision": "include", "reason": "", "stage": "full_text", "linked_to": ""},
            {"key": "d", "decision": "include", "reason": "", "stage": "title_abstract", "linked_to": ""},
            {"key": "e", "decision": "unsure", "reason": "", "stage": "title_abstract", "linked_to": ""}]
    c = search.prisma_counts([], rows)
    assert c["excluded_title_abstract_by_reason"] == {"system level": 1}
    assert c["excluded_full_text_by_reason"] == {"detector only": 1}
    assert c["full_text_assessed"] == 2 and c["included_after_full_text"] == 1
    assert c["reports_sought_for_full_text"] == 4 and c["included"] == 2 and c["unsure_to_full_text"] == 1


def test_zenodo_related_follows_only_paper_relations():
    payload = {"metadata": {"related_identifiers": [
        {"identifier": "https://doi.org/10.1234/ABC", "relation": "isSupplementTo", "scheme": "doi"},
        {"identifier": "arXiv:2601.07185", "relation": "isDocumentedBy", "scheme": "arxiv"},
        {"identifier": "https://x.org/p.pdf", "relation": "isPartOf", "scheme": "url"},
        {"identifier": "10.9/cited", "relation": "cites", "scheme": "doi"}]}}
    assert search.zenodo_related(payload) == [("doi", "10.1234/abc"), ("arxiv", "2601.07185"), ("url", "https://x.org/p.pdf")]
    assert search.zenodo_related({}) == []


def test_fulltext_candidates_routes_in_order_without_leaking_the_key():
    def fetch(url):
        if url.startswith(search.ZENODO_API + "/24246538"):
            return json.dumps({"id": "24246538", "files": [{"key": "deposit-v1.zip"}], "metadata": {"related_identifiers": [
                {"identifier": "10.5281/zenodo.999", "relation": "isSupplementTo", "scheme": "doi"}]}}).encode()
        if url.startswith(search.ZENODO_API + "/999"):
            return json.dumps({"id": "999", "files": [{"key": "paper.pdf"}]}).encode()
        if url.startswith(search.OPENALEX_API):
            assert "api_key=SECRET" in url
            return json.dumps({"best_oa_location": None, "locations": [{"pdf_url": None}, {"pdf_url": "https://repo/x.pdf"}]}).encode()
        if url.startswith(search.ARXIV_API):
            return (b'<feed xmlns="http://www.w3.org/2005/Atom"><entry><id>http://arxiv.org/abs/2601.00001v1</id>'
                    b'<title>Some Other Title</title></entry></feed>')
        raise RuntimeError("SECRET unexpected " + url)
    cands, notes = search.fulltext_candidates({"arxiv": "2508.02961", "arxiv_version": "2"}, "SECRET", fetch)
    assert [u for u, _ in cands] == [search.ARXIV_PDF + "/2508.02961v2", search.ARXIV_PDF + "/2508.02961"]
    if search.OFFLINE:
        assert search.fulltext_candidates({"doi": "10.5281/zenodo.24246538"}, "SECRET", fetch) == ([], ["offline"])
        return
    cands, notes = search.fulltext_candidates({"doi": "10.5281/zenodo.24246538", "title": "Experimental Measurement"}, "SECRET", fetch)
    urls = [u for u, _ in cands]
    assert urls[0] == search.ZENODO_API + "/999/files/paper.pdf/content" and "https://repo/x.pdf" in urls
    assert any("no PDF file" in n for n in notes) and any("no arxiv version" in n for n in notes)
    cands, notes = search.fulltext_candidates({"doi": "10.1038/s41598-025-31086-y", "title": "PromptGuard"}, "SECRET", fetch)
    assert cands[0] == (search.NATURE_PDF + "/s41598-025-31086-y.pdf", "nature.com article pdf")
    assert all("SECRET" not in n for _, n in cands) and all("SECRET" not in n for n in notes)


def test_fulltext_candidates_report_removed_records_and_bad_lookups():
    import urllib.error
    def fetch(url):
        if url.startswith(search.ZENODO_API):
            raise urllib.error.HTTPError(url, 410, "Gone", {}, None)
        raise RuntimeError("down")
    if search.OFFLINE:
        return
    cands, notes = search.fulltext_candidates({"doi": "10.5281/zenodo.20582430", "title": "Attention Is Not Enough"}, "", fetch)
    assert cands == [] and any("removed by its owner" in n for n in notes)


def test_clean_url_and_hand_fetch_hints():
    assert search.clean_url(" https://a.org/x y.pdf ") == "https://a.org/x%20y.pdf"
    assert "browser" in search.hand_fetch_hint({"doi": "10.3390/app162111039"}, ["failed: HTTP Error 403: Forbidden"], [])
    assert "library" in search.hand_fetch_hint({"doi": "10.1109/x.1"}, [], ["openalex lists no pdf_url"])
    assert "deposit" in search.hand_fetch_hint({"doi": "10.5281/zenodo.1"}, [], ["zenodo record has no PDF file"])
    assert "removed" in search.hand_fetch_hint({"doi": "10.5281/zenodo.2"}, [], ["zenodo lookup: HTTP 410 (record removed by its owner)"])


def test_place_hand_files_renames_by_hash_and_sets_aside_duplicates(tmp_path):
    d = tmp_path / "defences"
    d.mkdir()
    (d / "2401.00001.pdf").write_bytes(b"%PDF already standard")
    (d / "v1.pdf").write_bytes(b"%PDF paper A")
    (d / "v1 (1).pdf").write_bytes(b"%PDF paper A again")
    (d / "page.htm").write_bytes(b"<html>shell</html>")
    (d / "Big Thesis.pdf").write_bytes(b"%PDF thesis")
    sha_a = search.file_sha256(d / "v1.pdf")
    mapping = [
        {"sha256": sha_a, "file_name": "v1.pdf", "slug": "doi_10_1_a", "action": "use"},
        {"sha256": search.file_sha256(d / "v1 (1).pdf"), "file_name": "v1 (1).pdf", "slug": "doi_10_1_a", "action": "duplicate"},
        {"sha256": search.file_sha256(d / "page.htm"), "file_name": "page.htm", "slug": "doi_10_1_b", "action": "unusable"},
        {"sha256": "", "file_name": "Big Thesis.pdf", "slug": "title_bigthesis", "action": "use"},
        {"sha256": "0" * 64, "file_name": "missing.pdf", "slug": "doi_10_1_c", "action": "use"},
    ]
    known = {"2401.00001", "doi_10_1_a", "doi_10_1_b", "doi_10_1_c", "title_bigthesis"}
    res = search.place_hand_files(d, mapping, known)
    assert [r["status"] for r in res] == ["renamed", "moved to _not_used", "moved to _not_used", "renamed", "not found"]
    assert (d / "doi_10_1_a.pdf").read_bytes() == b"%PDF paper A"
    assert (d / "title_bigthesis.pdf").exists() and (d / "2401.00001.pdf").exists()
    assert sorted(p.name for p in (d / "_not_used").iterdir()) == ["page.htm", "v1 (1).pdf"]
    again = search.place_hand_files(d, mapping, known)          # a second run changes nothing
    assert [r["status"] for r in again] == ["already placed", "already moved", "already moved", "already placed", "not found"]


def test_place_hand_files_never_overwrites_a_standard_file(tmp_path):
    d = tmp_path / "defences"
    d.mkdir()
    (d / "doi_10_1_a.pdf").write_bytes(b"%PDF first copy")
    (d / "other.pdf").write_bytes(b"%PDF different content")
    mapping = [{"sha256": search.file_sha256(d / "other.pdf"), "file_name": "other.pdf", "slug": "doi_10_1_a", "action": "use"}]
    res = search.place_hand_files(d, mapping, {"doi_10_1_a"})
    assert res[0]["status"].startswith("kept aside")
    assert (d / "doi_10_1_a.pdf").read_bytes() == b"%PDF first copy" and (d / "other.pdf").exists()


def test_place_hand_files_handles_byte_identical_downloads(tmp_path):
    d = tmp_path / "defences"
    d.mkdir()
    (d / "paper.pdf").write_bytes(b"%PDF same bytes")
    (d / "paper (1).pdf").write_bytes(b"%PDF same bytes")
    sha = search.file_sha256(d / "paper.pdf")
    mapping = [{"sha256": sha, "file_name": "paper.pdf", "slug": "doi_10_1_a", "action": "use"},
               {"sha256": sha, "file_name": "paper (1).pdf", "slug": "doi_10_1_a", "action": "duplicate"}]
    res = search.place_hand_files(d, mapping, {"doi_10_1_a"})
    assert [r["status"] for r in res] == ["renamed", "moved to _not_used"]
    assert [p.name for p in d.iterdir() if p.is_file()] == ["doi_10_1_a.pdf"]
    assert [p.name for p in (d / "_not_used").iterdir()] == ["paper (1).pdf"]


def test_place_hand_files_sets_aside_a_copy_of_an_already_placed_file(tmp_path):
    d = tmp_path / "defences"
    d.mkdir()
    (d / "doi_10_1_a.pdf").write_bytes(b"%PDF same")
    (d / "download.pdf").write_bytes(b"%PDF same")
    mapping = [{"sha256": search.file_sha256(d / "download.pdf"), "file_name": "download.pdf", "slug": "doi_10_1_a", "action": "use"}]
    res = search.place_hand_files(d, mapping, {"doi_10_1_a"})
    assert res[0]["status"] == "same file already placed; copy moved to _not_used"
    assert (d / "_not_used" / "download.pdf").exists() and (d / "doi_10_1_a.pdf").exists()

All 2,206 decisions. Full-text decisions have notes that start with `full text:`; the six title-and-abstract corrections have notes that start with `corrected on rereading the abstract:`.

In [ ]:
%%writefile schema/screening_decisions.csv
key,decision,reason,stage,access_status,note,title
arxiv:2302.05733,exclude,attack only,title_abstract,,,Exploiting Programmatic Behavior of LLMs: Dual-Use Through Standard Security Attacks
arxiv:2302.07842,exclude,not about prompt injection,title_abstract,,,Augmented Language Models: a Survey
arxiv:2302.08582,exclude,not about prompt injection,title_abstract,,,Pretraining Language Models with Human Preferences
arxiv:2302.12173,exclude,attack only,title_abstract,,,Not what you've signed up for: Compromising Real-World LLM-Integrated Applications with Indirect Pro
arxiv:2303.08112,exclude,not about prompt injection,title_abstract,,,Eliciting Latent Predictions from Transformers with the Tuned Lens
arxiv:2303.08774,exclude,not about prompt injection,title_abstract,,,GPT-4 Technical Report
arxiv:2303.11156,exclude,not about prompt injection,title_abstract,,,Can AI-Generated Text be Reliably Detected?
arxiv:2304.03277,exclude,not about prompt injection,title_abstract,,,Instruction Tuning with GPT-4
arxiv:2305.13860,exclude,jailbreak or direct injection only,title_abstract,,,Jailbreaking ChatGPT via Prompt Engineering: An Empirical Study
arxiv:2305.14950,exclude,not about prompt injection,title_abstract,,,Adversarial Demonstration Attacks on Large Language Models
arxiv:2305.18290,exclude,not about prompt injection,title_abstract,,,Direct Preference Optimization: Your Language Model is Secretly a Reward Model
arxiv:2306.02224,exclude,not about prompt injection,title_abstract,,,Auto-GPT for Online Decision Making: Benchmarks and Additional Opinions
arxiv:2306.05499,exclude,attack only,title_abstract,,,Prompt Injection attack against LLM-integrated Applications
arxiv:2306.08833,exclude,attack only,title_abstract,,"uses injection as a bot check, not a defence",Safeguarding Crowdsourcing Surveys from ChatGPT through Prompt Injection
arxiv:2306.11695,exclude,not about prompt injection,title_abstract,,,A Simple and Effective Pruning Approach for Large Language Models
arxiv:2307.00691,exclude,"survey, position or other",title_abstract,,,From ChatGPT to ThreatGPT: Impact of Generative AI in Cybersecurity and Privacy
arxiv:2307.02483,exclude,jailbreak or direct injection only,title_abstract,,,Jailbroken: How Does LLM Safety Training Fail?
arxiv:2307.06435,exclude,not about prompt injection,title_abstract,,,A Comprehensive Overview of Large Language Models
arxiv:2307.06865,exclude,attack only,title_abstract,,prompt extraction,Effective Prompt Extraction from Language Models
arxiv:2307.10490,exclude,attack only,title_abstract,,,Abusing Images and Sounds for Indirect Instruction Injection in Multi-Modal LLMs
arxiv:2307.15043,exclude,jailbreak or direct injection only,title_abstract,,,Universal and Transferable Adversarial Attacks on Aligned Language Models
arxiv:2307.15593,exclude,not about prompt injection,title_abstract,,,Robust Distortion-free Watermarks for Language Models
arxiv:2307.16789,exclude,not about prompt injection,title_abstract,,,ToolLLM: Facilitating Large Language Models to Master 16000+ Real-world APIs
arxiv:2307.16888,exclude,attack only,title_abstract,,backdoor via virtual prompt injection,Backdooring Instruction-Tuned Large Language Models with Virtual Prompt Injection
arxiv:2308.01263,exclude,not about prompt injection,title_abstract,,,XSTest: A Test Suite for Identifying Exaggerated Safety Behaviours in Large Language Models
arxiv:2308.01990,exclude,system level,title_abstract,,middleware and input-restriction mitigations only,From Prompt Injections to SQL Injection Attacks: How Protected is Your LLM-Integrated Web Applicatio
arxiv:2308.03825,exclude,jailbreak or direct injection only,title_abstract,,,"""Do Anything Now"": Characterizing and Evaluating In-The-Wild Jailbreak Prompts on Large Language Mod"
arxiv:2308.05374,exclude,"survey, position or other",title_abstract,,,Trustworthy LLMs: a Survey and Guideline for Evaluating Large Language Models' Alignment
arxiv:2308.06463,exclude,jailbreak or direct injection only,title_abstract,,,GPT-4 Is Too Smart To Be Safe: Stealthy Chat with LLMs via Cipher
arxiv:2308.06782,exclude,not about prompt injection,title_abstract,,,PentestGPT: An LLM-empowered Automatic Penetration Testing Tool
arxiv:2308.07308,exclude,jailbreak or direct injection only,title_abstract,,,"LLM Self Defense: By Self Examination, LLMs Know They Are Being Tricked"
arxiv:2308.08155,exclude,not about prompt injection,title_abstract,,,AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation
arxiv:2308.14132,exclude,detector only,title_abstract,,,Detecting Language Model Attacks with Perplexity
arxiv:2309.00614,exclude,jailbreak or direct injection only,title_abstract,,,Baseline Defenses for Adversarial Attacks Against Aligned Language Models
arxiv:2309.02705,exclude,jailbreak or direct injection only,title_abstract,,,Certifying LLM Safety against Adversarial Prompting
arxiv:2309.06899,exclude,not about prompt injection,title_abstract,,wrong identifier in a reference list,A stochastic differential equation for local times of super-Brownian motion
arxiv:2309.07864,exclude,not about prompt injection,title_abstract,,,The Rise and Potential of Large Language Model Based Agents: A Survey
arxiv:2309.10253,exclude,jailbreak or direct injection only,title_abstract,,,GPTFUZZER: Red Teaming Large Language Models with Auto-Generated Jailbreak Prompts
arxiv:2310.02238,exclude,not about prompt injection,title_abstract,,,Who's Harry Potter? Approximate Unlearning in LLMs
arxiv:2310.02446,exclude,jailbreak or direct injection only,title_abstract,,,Low-Resource Languages Jailbreak GPT-4
arxiv:2310.03684,exclude,jailbreak or direct injection only,title_abstract,,,SmoothLLM: Defending Large Language Models Against Jailbreaking Attacks
arxiv:2310.03693,exclude,not about prompt injection,title_abstract,,,"Fine-tuning Aligned Language Models Compromises Safety, Even When Users Do Not Intend To!"
arxiv:2310.06387,exclude,jailbreak or direct injection only,title_abstract,,,Jailbreak and Guard Aligned Language Models with Only Few In-Context Demonstrations
arxiv:2310.06474,exclude,jailbreak or direct injection only,title_abstract,,,Multilingual Jailbreak Challenges in Large Language Models
arxiv:2310.06770,exclude,not about prompt injection,title_abstract,,,SWE-bench: Can Language Models Resolve Real-World GitHub Issues?
arxiv:2310.06825,exclude,not about prompt injection,title_abstract,,,Mistral 7B
arxiv:2310.08419,exclude,jailbreak or direct injection only,title_abstract,,,Jailbreaking Black Box Large Language Models in Twenty Queries
arxiv:2310.12505,exclude,jailbreak or direct injection only,title_abstract,,,Attack Prompt Generation for Red Teaming and Defending Large Language Models
arxiv:2310.12815,include,,title_abstract,,role: re-measurement; Liu et al. benchmark of prompting defences and detectors,Formalizing and Benchmarking Prompt Injection Attacks and Defenses
arxiv:2310.17389,exclude,not about prompt injection,title_abstract,,,ToxicChat: Unveiling Hidden Challenges of Toxicity Detection in Real-World User-AI Conversation
arxiv:2311.00286,exclude,jailbreak or direct injection only,title_abstract,,,JADE: A Linguistics-based Safety Evaluation Platform for Large Language Models
arxiv:2311.01011,exclude,jailbreak or direct injection only,title_abstract,,direct injection game dataset,Tensor Trust: Interpretable Prompt Injection Attacks from an Online Game
arxiv:2311.03191,exclude,jailbreak or direct injection only,title_abstract,,,DeepInception: Hypnotize Large Language Model to Be Jailbreaker
arxiv:2311.03348,exclude,jailbreak or direct injection only,title_abstract,,,Scalable and Transferable Black-Box Jailbreaks for Language Models via Persona Modulation
arxiv:2311.04378,exclude,not about prompt injection,title_abstract,,,Watermarks in the Sand: Impossibility of Strong Watermarking for Generative Models
arxiv:2311.07911,exclude,not about prompt injection,title_abstract,,,Instruction-Following Evaluation for Large Language Models
arxiv:2311.09948,exclude,not about prompt injection,title_abstract,,adversarial in-context demonstrations,Hijacking Large Language Models via Adversarial In-Context Learning
arxiv:2311.10538,exclude,not about prompt injection,title_abstract,,,Testing Language Model Agents Safely in the Wild
arxiv:2311.11509,exclude,detector only,title_abstract,,,Token-Level Adversarial Prompt Detection Based on Perplexity Measures and Contextual Information
arxiv:2311.11538,exclude,attack only,title_abstract,,,Assessing Prompt Injection Risks in 200+ Custom GPTs
arxiv:2311.11855,exclude,not about prompt injection,title_abstract,,,Evil Geniuses: Delving into the Safety of LLM-based Agents
arxiv:2311.16119,exclude,jailbreak or direct injection only,title_abstract,,,Ignore This Title and HackAPrompt: Exposing Systemic Vulnerabilities of LLMs through a Global Scale
arxiv:2312.00752,exclude,not about prompt injection,title_abstract,,,Mamba: Linear-Time Sequence Modeling with Selective State Spaces
arxiv:2312.02003,exclude,"survey, position or other",title_abstract,,,"A Survey on Large Language Model (LLM) Security and Privacy: The Good, the Bad, and the Ugly"
arxiv:2312.06674,exclude,detector only,title_abstract,,,Llama Guard: LLM-based Input-Output Safeguard for Human-AI Conversations
arxiv:2312.10091,exclude,not about prompt injection,title_abstract,,,Look Before You Leap: A Universal Emergent Decomposition of Retrieval Tasks in Language Models
arxiv:2312.10997,exclude,not about prompt injection,title_abstract,,,Retrieval-Augmented Generation for Large Language Models: A Survey
arxiv:2312.11513,exclude,attack only,title_abstract,,variant generation for injections,Maatphor: Automated Variant Analysis for Prompt Injection Attacks
arxiv:2312.11671,exclude,not about prompt injection,title_abstract,,,Evaluating Language-Model Agents on Realistic Autonomous Tasks
arxiv:2312.11805,exclude,not about prompt injection,title_abstract,,,Gemini: A Family of Highly Capable Multimodal Models
arxiv:2312.14197,include,,title_abstract,,BIPIA: benchmark with black-box prompting defences and white-box special-token fine-tuning,Benchmarking and Defending Against Indirect Prompt Injection Attacks on Large Language Models
arxiv:2312.17673,include,,title_abstract,,Jatmo,Jatmo: Prompt Injection Defense by Task-Specific Finetuning
arxiv:2401.05459,exclude,"survey, position or other",title_abstract,,,"Personal LLM Agents: Insights and Survey about the Capability, Efficiency and Security"
arxiv:2401.07612,include,,title_abstract,,Signed-Prompt,Signed-Prompt: A New Approach to Prevent Prompt Injection Attacks Against LLM-Integrated Application
arxiv:2401.12273,exclude,"survey, position or other",title_abstract,,,The Ethics of Interaction: Mitigating Security Threats in LLMs
arxiv:2402.00898,exclude,"survey, position or other",title_abstract,,,An Early Categorization of Prompt Injection Attacks on Large Language Models
arxiv:2402.05723,exclude,jailbreak or direct injection only,title_abstract,,,In-Context Learning Can Re-learn Forbidden Tasks
arxiv:2402.06363,include,,title_abstract,,StruQ,StruQ: Defending Against Prompt Injection with Structured Queries
arxiv:2402.06664,exclude,not about prompt injection,title_abstract,,,LLM Agents can Autonomously Hack Websites
arxiv:2402.06922,include,,full_text,,"full text: role: re-measurement; prompt hardening and external filters against confidentiality attacks through tool content, results averaged over five defences (Fig. 7)",Whispers in the Machine: Confidentiality in Agentic Systems
arxiv:2402.07867,exclude,not about prompt injection,title_abstract,,"knowledge poisoning, not instruction injection",PoisonedRAG: Knowledge Corruption Attacks to Retrieval-Augmented Generation of Large Language Models
arxiv:2402.08983,exclude,jailbreak or direct injection only,title_abstract,,,SafeDecoding: Defending against Jailbreak Attacks via Safety-Aware Decoding
arxiv:2402.10601,exclude,jailbreak or direct injection only,title_abstract,,,"When ""Competency"" in Reasoning Opens the Door to Vulnerability: Jailbreaking LLMs via Novel Complex "
arxiv:2402.11082,exclude,"survey, position or other",title_abstract,,,The AI Security Pyramid of Pain
arxiv:2402.11755,exclude,system level,title_abstract,,prompt DSL checked outside the model,SPML: A DSL for Defending Language Models Against Prompt Attacks
arxiv:2402.16835,exclude,not about prompt injection,title_abstract,,,Eight Methods to Evaluate Robust Unlearning in LLMs
arxiv:2402.16965,exclude,attack only,title_abstract,,,WIPI: A New Web Threat for LLM-Driven Web Agents
arxiv:2402.17840,exclude,attack only,title_abstract,,,Follow My Instruction and Spill the Beans: Scalable Data Extraction from Retrieval-Augmented Generat
arxiv:2403.02691,exclude,benchmark only,title_abstract,,InjecAgent,InjecAgent: Benchmarking Indirect Prompt Injections in Tool-Integrated Large Language Model Agents
arxiv:2403.02817,exclude,attack only,title_abstract,,,Here Comes The AI Worm: Unleashing Zero-click Worms that Target GenAI-Powered Applications
arxiv:2403.03792,exclude,attack only,title_abstract,,,Neural Exec: Learning (and Learning from) Execution Triggers for Prompt Injection Attacks
arxiv:2403.04783,exclude,jailbreak or direct injection only,title_abstract,,,AutoDefense: Multi-Agent LLM Defense against Jailbreak Attacks
arxiv:2403.04957,exclude,attack only,title_abstract,,,Automatic and Universal Prompt Injection Attacks against Large Language Models
arxiv:2403.04960,exclude,system level,title_abstract,,IsolateGPT,IsolateGPT: An Execution Isolation Architecture for LLM-Based Agentic Systems
arxiv:2403.06833,exclude,benchmark only,title_abstract,,"SEP separation measure and dataset, no defence",Can LLMs Separate Instructions From Data? And What Do We Even Mean By That?
arxiv:2403.08701,exclude,"survey, position or other",title_abstract,,,Review of Generative AI Methods in Cybersecurity
arxiv:2403.09832,exclude,attack only,title_abstract,,"evaluation of translation under injection, no defence",Scaling Behavior of Machine Translation with Large Language Models under Prompt Injection Attacks
arxiv:2403.10553,exclude,not about prompt injection,title_abstract,,,Learning to Watermark LLM-generated Text via Reinforcement Learning
arxiv:2403.13793,exclude,not about prompt injection,title_abstract,,,Evaluating Frontier Models for Dangerous Capabilities
arxiv:2403.14720,include,,title_abstract,,Spotlighting,Defending Against Indirect Prompt Injection Attacks With Spotlighting
arxiv:2403.17710,exclude,attack only,title_abstract,,,Optimization-based Prompt Injection Attack to LLM-as-a-Judge
arxiv:2404.02151,exclude,jailbreak or direct injection only,title_abstract,,,Jailbreaking Leading Safety-Aligned LLMs with Simple Adaptive Attacks
arxiv:2404.04392,exclude,attack only,title_abstract,,vulnerability evaluation after fine-tuning and quantisation,"Fine-Tuning, Quantization, and LLMs: Navigating Unintended Outcomes"
arxiv:2404.04475,exclude,not about prompt injection,title_abstract,,,Length-Controlled AlpacaEval: A Simple Way to Debias Automatic Evaluators
arxiv:2404.07234,exclude,attack only,title_abstract,,,Goal-guided Generative Prompt Injection Attack on Large Language Models
arxiv:2404.07921,exclude,jailbreak or direct injection only,title_abstract,,,AmpleGCG: Learning a Universal and Transferable Generative Model of Adversarial Suffixes for Jailbre
arxiv:2404.07972,exclude,not about prompt injection,title_abstract,,,OSWorld: Benchmarking Multimodal Agents for Open-Ended Tasks in Real Computer Environments
arxiv:2404.08144,exclude,not about prompt injection,title_abstract,,,LLM Agents can Autonomously Exploit One-day Vulnerabilities
arxiv:2404.13093,exclude,not about prompt injection,title_abstract,,,The N-Gene Model for Evolutionary Games
arxiv:2404.13161,exclude,benchmark only,title_abstract,,CyberSecEval 2,CyberSecEval 2: A Wide-Ranging Cybersecurity Evaluation Suite for Large Language Models
arxiv:2404.13208,include,,title_abstract,,Instruction Hierarchy,The Instruction Hierarchy: Training LLMs to Prioritize Privileged Instructions
arxiv:2404.16663,exclude,not about prompt injection,title_abstract,,,Runtime Monitoring and Enforcement of Conditional Fairness in Generative AIs
arxiv:2405.10529,exclude,image or audio channel only,full_text,,full text: SmoothVLM: randomised smoothing against adversarial visual patches in vision-language models,Safeguarding Vision-Language Models Against Patched Visual Prompt Injectors
arxiv:2405.12750,exclude,"survey, position or other",title_abstract,,,Generative AI in Cybersecurity: A Comprehensive Review of LLM Applications and Vulnerabilities
arxiv:2405.20485,exclude,attack only,title_abstract,,,Phantom: General Backdoor Attacks on Retrieval Augmented Language Generation
arxiv:2405.20773,exclude,jailbreak or direct injection only,title_abstract,,,Visual-RolePlay: Universal Jailbreak Attack on MultiModal Large Language Models via Role-playing Ima
arxiv:2405.20902,exclude,not about prompt injection,title_abstract,,,"Preemptive Answer ""Attacks"" on Chain-of-Thought Reasoning"
arxiv:2406.00240,exclude,"survey, position or other",title_abstract,,,Exploring Vulnerabilities and Protections in Large Language Models: A Survey
arxiv:2406.00799,exclude,detector only,title_abstract,,task-drift activation probe used as a detector,Get my drift? Catching LLM Task Drift with Activation Deltas
arxiv:2406.03589,exclude,attack only,title_abstract,,,Ranking Manipulation for Conversational Search Engines
arxiv:2406.05498,exclude,jailbreak or direct injection only,title_abstract,,,SelfDefend: LLMs Can Defend Themselves against Jailbreaking in a Practical Manner
arxiv:2406.06622,exclude,jailbreak or direct injection only,title_abstract,,,Adversarial Tuning: Defending Against Jailbreak Attacks for LLMs
arxiv:2406.11036,exclude,benchmark only,title_abstract,,probing framework,garak: A Framework for Security Probing Large Language Models
arxiv:2406.12259,exclude,attack only,title_abstract,,,Adversarial prompt and fine-tuning attacks threaten medical large language models
arxiv:2406.12814,exclude,attack only,title_abstract,,,Dissecting Adversarial Robustness of Multimodal LM Agents
arxiv:2406.13352,include,,title_abstract,,role: benchmark with defence results; AgentDojo,AgentDojo: A Dynamic Environment to Evaluate Prompt Injection Attacks and Defenses for LLM Agents
arxiv:2406.13356,exclude,not about prompt injection,title_abstract,,,Unlearning or Obfuscating? Jogging the Memory of Unlearned LLMs via Benign Relearning
arxiv:2406.14048,exclude,jailbreak or direct injection only,full_text,,full text: secret-extraction competition: the attacker talks to the defended model directly,Prompt Injection Attacks in Defended Systems
arxiv:2406.18382,exclude,attack only,title_abstract,,,Adversarial Search Engine Optimization for Large Language Models
arxiv:2406.18495,exclude,detector only,title_abstract,,,"WildGuard: Open One-Stop Moderation Tools for Safety Risks, Jailbreaks, and Refusals of LLMs"
arxiv:2407.01502,exclude,not about prompt injection,title_abstract,,,AI Agents That Matter
arxiv:2407.05557,exclude,detector only,title_abstract,,,$R^2$-Guard: Robust Reasoning Enabled LLM Guardrail via Knowledge-Enhanced Logical Reasoning
arxiv:2407.07791,exclude,not about prompt injection,title_abstract,,,Flooding Spread of Manipulated Knowledge in LLM-Based Multi-Agent Communities
arxiv:2407.08970,exclude,attack only,title_abstract,,,Self-interpreting Adversarial Images
arxiv:2407.13744,exclude,not about prompt injection,title_abstract,,,"LLMs as Function Approximators: Terminology, Taxonomy, and Questions for Evaluation"
arxiv:2407.15549,exclude,jailbreak or direct injection only,title_abstract,,,Latent Adversarial Training Improves Robustness to Persistent Harmful Behaviors in LLMs
arxiv:2407.16205,exclude,jailbreak or direct injection only,title_abstract,,,Reasoning as a Weapon: Adaptive Dual-Path Jailbreak Attack on Large Language Models
arxiv:2407.17447,exclude,jailbreak or direct injection only,title_abstract,,,FLRT: Fluent Student-Teacher Redteaming
arxiv:2407.18213,exclude,not about prompt injection,full_text,,full text: adversarial training scaling on classification tasks and StrongREJECT; no injected-data setting,Scaling Trends in Language Model Robustness
arxiv:2407.18981,exclude,attack only,title_abstract,,,Prompt Injection Attacks on Large Language Models in Oncology
arxiv:2407.19354,exclude,"survey, position or other",title_abstract,,,The Emerged Security and Privacy of LLM Agent: A Survey with Case Studies
arxiv:2407.21783,exclude,not about prompt injection,title_abstract,,,The Llama 3 Herd of Models
arxiv:2408.00989,exclude,not about prompt injection,title_abstract,,,On the Resilience of LLM-Based Multi-Agent Collaboration with Faulty Agents
arxiv:2408.02416,exclude,attack only,title_abstract,,prompt extraction,Why Are My Prompts Leaked? Unraveling Prompt Extraction Threats in Customized Large Language Models
arxiv:2408.03515,exclude,attack only,title_abstract,,,A Study on Prompt Injection Attack Against LLM-Integrated Mobile Robotic Systems
arxiv:2408.04194,exclude,attack only,title_abstract,,,FDI: Attack Neural Code Generation Systems through User Feedback Channel
arxiv:2408.05025,exclude,attack only,title_abstract,,framework-level evaluation of indirect manipulations,Rag and Roll: An End-to-End Evaluation of Indirect Prompt Manipulations in LLM-based Application Fra
arxiv:2408.07291,exclude,not about prompt injection,title_abstract,,"countermeasures protect profiles, not the model",Evaluating LLM-based Personal Information Extraction and Countermeasures
arxiv:2408.11182,exclude,jailbreak or direct injection only,title_abstract,,,Hide Your Malicious Goal Into Benign Narratives: Jailbreak Large Language Models through Carrier Art
arxiv:2408.15221,exclude,jailbreak or direct injection only,title_abstract,,,LLM Defenses Are Not Robust to Multi-Turn Human Jailbreaks Yet
arxiv:2409.08087,exclude,"survey, position or other",title_abstract,,,"Securing Large Language Models: Addressing Bias, Misinformation, and Prompt Attacks"
arxiv:2409.13331,exclude,detector only,title_abstract,,,Applying Pre-trained Multilingual BERT in Embeddings for Improved Malicious Prompt Injection Attacks
arxiv:2409.14729,exclude,attack only,title_abstract,,fuzzing to generate injections,PROMPTFUZZ: Harnessing Fuzzing Techniques for Robust Testing of Prompt Injection in LLMs
arxiv:2409.18025,exclude,not about prompt injection,title_abstract,,,An Adversarial Perspective on Machine Unlearning for AI Safety
arxiv:2409.18980,exclude,not about prompt injection,title_abstract,,,IW-Bench: Evaluating Large Multimodal Models for Converting Image-to-Web
arxiv:2409.19091,exclude,system level,title_abstract,,information-flow control,System-Level Defense against Indirect Prompt Injection Attacks: An Information Flow Control Perspect
arxiv:2409.19521,exclude,detector only,title_abstract,,,GenTel-Safe: A Unified Benchmark and Shielding Framework for Defending Against Prompt Injection Atta
arxiv:2410.02644,include,,title_abstract,,role: benchmark with defence results; Agent Security Bench,Agent Security Bench (ASB): Formalizing and Benchmarking Attacks and Defenses in LLM-based Agents
arxiv:2410.02916,exclude,attack only,title_abstract,,,LLM Safeguard is a Double-Edged Sword: Exploiting False Positives for Denial-of-Service Attacks
arxiv:2410.03055,exclude,system level,title_abstract,,,Permissive Information-Flow Analysis for Large Language Models
arxiv:2410.05451,include,,title_abstract,,SecAlign,SecAlign: Defending Against Prompt Injection with Preference Optimization
arxiv:2410.07283,exclude,attack only,title_abstract,,,Prompt Infection: LLM-to-LLM Prompt Injection within Multi-Agent Systems
arxiv:2410.08827,exclude,not about prompt injection,title_abstract,,,Do Unlearning Methods Remove Information from Language Model Weights?
arxiv:2410.09102,include,,title_abstract,,Instructional Segment Embedding,Instructional Segment Embedding: Improving LLM Safety with Instruction Hierarchy
arxiv:2410.10414,exclude,detector only,title_abstract,,,On Calibration of LLM-based Guard Models for Reliable Content Moderation
arxiv:2410.12123,exclude,not about prompt injection,title_abstract,,,"Large Language Models, and LLM-Based Agents, Should Be Used to Enhance the Digital Public Sphere"
arxiv:2410.13138,exclude,not about prompt injection,title_abstract,,,Data Defenses Against Large Language Models
arxiv:2410.13236,exclude,jailbreak or direct injection only,title_abstract,,,SPIN: Self-Supervised Prompt INjection
arxiv:2410.13886,exclude,jailbreak or direct injection only,title_abstract,,,Refusal-Trained LLMs Are Easily Jailbroken As Browser Agents
arxiv:2410.14479,exclude,attack only,title_abstract,,,Backdoored Retrievers for Prompt Injection Attacks on Retrieval Augmented Generation of Large Langua
arxiv:2410.14569,exclude,attack only,title_abstract,,,When LLMs Go Online: The Emerging Threat of Web-Enabled LLMs
arxiv:2410.14827,exclude,attack only,title_abstract,,,Enhancing Prompt Injection Attacks to LLMs via Poisoning Alignment
arxiv:2410.15236,exclude,"survey, position or other",title_abstract,,,Jailbreaking and Mitigation of Vulnerabilities in Large Language Models
arxiv:2410.15641,exclude,jailbreak or direct injection only,title_abstract,,,SMILES-Prompting: A Novel Approach to LLM Jailbreak Attacks in Chemical Synthesis
arxiv:2410.15686,exclude,system level,title_abstract,,multi-agent topology,NetSafe: Exploring the Topological Safety of Multi-agent Networks
arxiv:2410.16454,exclude,not about prompt injection,title_abstract,,,Catastrophic Failure of LLM Unlearning via Quantization
arxiv:2410.16950,exclude,attack only,title_abstract,,,Breaking ReAct Agents: Foot-in-the-Door Attack Will Get You In
arxiv:2410.17401,exclude,attack only,title_abstract,,,AdvAgent: Controllable Blackbox Red-teaming on Web Agents
arxiv:2410.18312,exclude,not about prompt injection,title_abstract,,,Countering Autonomous Cyber Threats
arxiv:2410.20911,exclude,not about prompt injection,title_abstract,,"injection used against attacking LLMs, not a defence of the model",Hacking Back the AI-Hacker: Prompt Injection as a Defense Against LLM-driven Cyberattacks
arxiv:2410.21146,exclude,detector only,title_abstract,,,Palisade -- Prompt Injection Detection Framework
arxiv:2410.21337,exclude,detector only,title_abstract,,,Fine-tuned Large Language Models (LLMs): Improved Prompt Injection Attacks Detection
arxiv:2410.21492,include,,title_abstract,,FATH,FATH: Authentication-based Test-time Defense against Indirect Prompt Injection Attacks
arxiv:2410.22284,exclude,detector only,title_abstract,,,Embedding-based classifiers can detect prompt injection attacks
arxiv:2410.22770,exclude,detector only,title_abstract,,,InjecGuard: Benchmarking and Mitigating Over-defense in Prompt Injection Guardrail Models
arxiv:2410.22832,exclude,attack only,title_abstract,,,HijackRAG: Hijacking Attacks against Retrieval-Augmented Large Language Models
arxiv:2410.23308,exclude,attack only,title_abstract,,vulnerability evaluation of 36 models,Systematically Analyzing Prompt Injection Vulnerabilities in Diverse LLM Architectures
arxiv:2411.00348,exclude,detector only,title_abstract,,attention signal used only to detect,Attention Tracker: Detecting Prompt Injection Attacks in LLMs
arxiv:2411.00459,include,,title_abstract,,attack-as-defence prompting,Defense Against Prompt Injection Attack by Leveraging Attack Techniques
arxiv:2411.01705,exclude,attack only,title_abstract,,,Data Extraction Attacks in Retrieval-Augmented Generation via Backdoors
arxiv:2411.09523,exclude,"survey, position or other",title_abstract,,,"Navigating the Risks: A Survey of Security, Privacy, and Ethics Threats in LLM-Based Agents"
arxiv:2411.16642,exclude,jailbreak or direct injection only,title_abstract,,,Preventing Jailbreak Prompts as Malicious Tools for Cybercriminals: A Cyber Defense Perspective
arxiv:2411.18241,exclude,not about prompt injection,title_abstract,,,Exploration of LLM Multi-Agent Application Implementation Based on LangGraph+CrewAI
arxiv:2412.01708,exclude,attack only,title_abstract,,,Are We There Yet? Revealing the Risks of Utilizing Large Language Models in Scholarly Peer Review
arxiv:2412.02113,exclude,"survey, position or other",title_abstract,,,Trust & Safety of LLMs and LLMs in Trust & Safety
arxiv:2412.05467,exclude,not about prompt injection,title_abstract,,,The BrowserGym Ecosystem for Web Agent Research
arxiv:2412.06090,exclude,"survey, position or other",title_abstract,,,Trust No AI: Prompt Injection Along The CIA Security Triad
arxiv:2412.10807,exclude,attack only,title_abstract,,,Towards Action Hijacking of Large Language Model-based Agent
arxiv:2412.13435,exclude,detector only,title_abstract,,,Lightweight Safety Classification Using Pruned Language Models
arxiv:2412.14093,exclude,not about prompt injection,title_abstract,,,Alignment faking in large language models
arxiv:2412.15115,exclude,not about prompt injection,title_abstract,,,Qwen2.5 Technical Report
arxiv:2412.16339,exclude,not about prompt injection,title_abstract,,,Deliberative Alignment: Reasoning Enables Safer Language Models
arxiv:2412.16682,exclude,system level,title_abstract,,Task Shield,The Task Shield: Enforcing Task Alignment to Defend Against Indirect Prompt Injection in LLM Agents
arxiv:2412.19260,exclude,not about prompt injection,title_abstract,,,MEDEC: A Benchmark for Medical Error Detection and Correction in Clinical Notes
arxiv:2501.07572,exclude,not about prompt injection,title_abstract,,,WebWalker: Benchmarking LLMs in Web Traversal
arxiv:2501.09798,exclude,attack only,title_abstract,,,Fun-tuning: Characterizing the Vulnerability of Proprietary LLMs to Optimization-based Prompt Inject
arxiv:2501.15145,exclude,detector only,title_abstract,,,PromptShield: Deployable Detection for Prompt Injection Attacks
arxiv:2501.18416,exclude,system level,title_abstract,,collaborative framework around the model,Exploring Potential Prompt Injection Attacks in Federated Military LLMs and Their Mitigation
arxiv:2501.18837,exclude,detector only,title_abstract,,,Constitutional Classifiers: Defending against Universal Jailbreaks across Thousands of Hours of Red
arxiv:2502.05174,exclude,system level,title_abstract,,MELON,MELON: Provable Defense Against Indirect Prompt Injection Attacks in AI Agents
arxiv:2502.05206,exclude,"survey, position or other",title_abstract,,,Safety at Scale: A Comprehensive Survey of Large Model and Agent Safety
arxiv:2502.08966,exclude,system level,title_abstract,,RTBAS,RTBAS: Defending LLM Agents Against Prompt Injection and Privacy Leakage
arxiv:2502.11006,exclude,detector only,title_abstract,,,Prompt Inject Detection with Generative Explanation as an Investigative Tool
arxiv:2502.11127,exclude,system level,title_abstract,,,G-Safeguard: A Topology-Guided Security Lens and Treatment on LLM-based Multi-agent Systems
arxiv:2502.11242,exclude,not about prompt injection,title_abstract,,,LLMs and Childhood Safety: Identifying Risks and Proposing a Protection Framework for Safe Child-LLM
arxiv:2502.11448,exclude,system level,title_abstract,,,AGrail: A Lifelong Agent Guardrail with Effective and Adaptive Safety Detection
arxiv:2502.12659,exclude,not about prompt injection,title_abstract,,,The Hidden Risks of Large Reasoning Models: A Safety Assessment of R1
arxiv:2502.13141,exclude,detector only,title_abstract,,,"UniGuardian: A Unified Defense for Detecting Prompt Injection, Backdoor Attacks and Adversarial Atta"
arxiv:2502.15568,exclude,not about prompt injection,title_abstract,,,"A Cautionary Tale About ""Neutrally"" Informative AI Tools Ahead of the 2025 Federal Elections in Germ"
arxiv:2502.16580,exclude,detector only,title_abstract,,detection and removal before the model,Can Indirect Prompt Injection Attacks Be Detected and Removed?
arxiv:2503.00061,include,,title_abstract,,role: re-measurement; adaptive attacks on eight IPI defences,Adaptive Attacks Break Defenses Against Indirect Prompt Injection Attacks on LLM Agents
arxiv:2503.04957,exclude,not about prompt injection,title_abstract,,,SafeArena: Evaluating the Safety of Autonomous Web Agents
arxiv:2503.06416,exclude,not about prompt injection,title_abstract,,,Advancing AI Negotiations: A Large-Scale Autonomous Negotiation Competition
arxiv:2503.09066,exclude,jailbreak or direct injection only,title_abstract,,,Probing Latent Subspaces in LLM for AI Security: Identifying and Manipulating Adversarial States
arxiv:2503.09334,exclude,not about prompt injection,title_abstract,,,CyberLLMInstruct: A Pseudo-malicious Dataset Revealing Safety-performance Trade-offs in Cyber Securi
arxiv:2503.09648,exclude,"survey, position or other",title_abstract,,,A Survey on Trustworthy LLM Agents: Threats and Countermeasures
arxiv:2503.10566,include,,title_abstract,,ASIDE,ASIDE: Architectural Separation of Instructions and Data in Language Models
arxiv:2503.11519,exclude,attack only,title_abstract,,,Exploring Typographic Visual Prompts Injection Threats in Cross-Modality Generation Models
arxiv:2503.15547,exclude,system level,title_abstract,,,Prompt Flow Integrity to Prevent Privilege Escalation in LLM Agents
arxiv:2503.15772,exclude,not about prompt injection,title_abstract,,,Detecting LLM-Generated Peer Reviews
arxiv:2503.16248,exclude,attack only,title_abstract,,,Real AI Agents with Fake Memories: Fatal Context Manipulation Attacks on Web3 Agents
arxiv:2503.18813,exclude,system level,title_abstract,,CaMeL,Defeating Prompt Injections by Design
arxiv:2503.21464,exclude,detector only,title_abstract,,,Harnessing Chain-of-Thought Metadata for Task Routing and Adversarial Prompt Detection
arxiv:2503.23278,exclude,"survey, position or other",title_abstract,,,"Model Context Protocol (MCP): Landscape, Security Threats, and Future Research Directions"
arxiv:2504.03767,exclude,attack only,title_abstract,,,MCP Safety Audit: LLMs with the Model Context Protocol Allow Major Security Exploits
arxiv:2504.05689,exclude,attack only,title_abstract,,attack through role separators; evidence on the roles signal,Separator Injection Attack: Uncovering Dialogue Biases in Large Language Models Caused by Role Separ
arxiv:2504.05804,exclude,attack only,title_abstract,,,StealthRank: LLM Ranking Manipulation via Stealthy Prompt Optimization
arxiv:2504.07467,include,,title_abstract,,Mixture of Encodings,Defense against Prompt Injection Attacks via Mixture of Encodings
arxiv:2504.09593,exclude,system level,title_abstract,,,ControlNET: A Firewall for RAG-based LLM System
arxiv:2504.09841,exclude,attack only,title_abstract,,,StruPhantom: Evolutionary Injection Attacks on Black-Box Tabular Agents Powered by Large Language Mo
arxiv:2504.10465,exclude,not about prompt injection,title_abstract,,,Pixel-SAIL: Single Transformer For Pixel-Grounded Understanding
arxiv:2504.11358,exclude,detector only,title_abstract,,DataSentinel,DataSentinel: A Game-Theoretic Detection of Prompt Injection Attacks
arxiv:2504.11703,exclude,system level,title_abstract,,Progent,Progent: Securing AI Agents with Privilege Control
arxiv:2504.13292,exclude,not about prompt injection,title_abstract,,,Let Me Grok for You: Accelerating Grokking via Embedding Transfer from a Weaker Model
arxiv:2504.15585,exclude,"survey, position or other",title_abstract,,,"A Comprehensive Survey in LLM(-Agent) Full Stack Safety: Data, Training and Deployment"
arxiv:2504.16125,exclude,attack only,title_abstract,,,Breaking the Prompt Wall (I): A Real-World Case Study of Attacking ChatGPT via Lightweight Prompt In
arxiv:2504.18333,exclude,attack only,title_abstract,,,Adversarial Attacks on LLM-as-a-Judge Systems: Insights from Prompt Injections
arxiv:2504.18575,exclude,benchmark only,title_abstract,,WASP,WASP: Benchmarking Web Agent Security Against Prompt Injection Attacks
arxiv:2504.19277,exclude,not about prompt injection,title_abstract,,,"Small Models, Big Tasks: An Exploratory Empirical Study on Small Language Models for Function Callin"
arxiv:2504.19521,exclude,benchmark only,title_abstract,,"steerability measure, no defence",Security Steerability is All You Need
arxiv:2504.19793,exclude,attack only,title_abstract,,,Prompt Injection Attack to Tool Selection in LLM Agents
arxiv:2504.20472,include,,title_abstract,,Robustness via referencing,Robustness via Referencing: Defending against Prompt Injection Attacks by Referencing the Executed I
arxiv:2504.20493,exclude,attack only,title_abstract,,,Token-Efficient Prompt Injection Attack: Provoking Cessation in LLM Reasoning via Adaptive Token Com
arxiv:2504.20769,include,,full_text,,full text: chain-of-defensive-thought: few-shot structured defensive reasoning as a prompting defence against injected references,Chain-of-Defensive-Thought: Structured Reasoning Elicits Robustness in Large Language Models against
arxiv:2504.20984,exclude,system level,title_abstract,,ACE,ACE: A Security Architecture for LLM-Integrated App Systems
arxiv:2504.21029,exclude,jailbreak or direct injection only,full_text,,"full text: included at title and abstract; the full text shows the threat is the user's own input, and the only case study is hypothetical, with no measured results",PICO: Secure Transformers via Robust Prompt Isolation and Cybersecurity Oversight
arxiv:2504.21228,include,,title_abstract,,CachePrune,CachePrune: Teaching LLMs What Not to Follow via KV-Cache Editing
arxiv:2505.00626,include,,title_abstract,,"PFT, position-enhanced fine-tuning",The Illusion of Role Separation: Hidden Shortcuts in LLM Role Learning (and How to Fix Them)
arxiv:2505.00843,exclude,benchmark only,title_abstract,,,OET: Optimization-based prompt injection Evaluation Toolkit
arxiv:2505.03574,exclude,detector only,title_abstract,,LlamaFirewall,LlamaFirewall: An open source guardrail system for building secure AI agents
arxiv:2505.05849,exclude,attack only,title_abstract,,,AgentVigil: Generic Black-Box Red-teaming for Indirect Prompt Injection against LLM Agents
arxiv:2505.06311,exclude,detector only,title_abstract,,instruction detection from internal states,Defending against Indirect Prompt Injection by Instruction Detection
arxiv:2505.09974,exclude,not about prompt injection,title_abstract,,,Analysing Safety Risks in LLMs Fine-Tuned with Pseudo-Malicious Cyber Security Data
arxiv:2505.11459,exclude,not about prompt injection,title_abstract,,prompt extraction defence,ProxyPrompt: Securing System Prompts against Prompt Extraction Attacks
arxiv:2505.12287,exclude,jailbreak or direct injection only,title_abstract,,,The Tower of Babel Revisited: Multilingual Jailbreak Prompts on Closed-Source Large Language Models
arxiv:2505.12368,exclude,detector only,title_abstract,,,CAPTURE: Context-Aware Prompt Injection Testing and Robustness Enhancement
arxiv:2505.12567,exclude,"survey, position or other",title_abstract,,,A Survey of Attacks on Large Language Models
arxiv:2505.13076,exclude,attack only,title_abstract,,,The Hidden Dangers of Browsing AI Agents
arxiv:2505.13348,exclude,attack only,title_abstract,,,Investigating the Vulnerability of LLM-as-a-Judge Architectures to Prompt-Injection Attacks
arxiv:2505.14106,exclude,not about prompt injection,title_abstract,,,A Personalized Conversational Benchmark: Towards Simulating Personalized Conversations
arxiv:2505.14368,exclude,attack only,title_abstract,,,From ASR to ASP: Evaluating Prompt Attack Vulnerabilities Against Open-Source LLMs
arxiv:2505.14534,include,,title_abstract,,Gemini adversarial training (claimed),Lessons from Defending Gemini Against Indirect Prompt Injections
arxiv:2505.14549,exclude,not about prompt injection,title_abstract,,,Can Large Language Models Really Recognize Your Name?
arxiv:2505.15420,exclude,attack only,title_abstract,,,Silent Leaks: Implicit Knowledge Extraction Attack on RAG Systems through Benign Queries
arxiv:2505.15738,include,,title_abstract,,role: re-measurement; Yang et al.,Auditing Prompt Injection Defenses Before They Reach Users: What Strong Adversaries Reveal
arxiv:2505.17066,exclude,jailbreak or direct injection only,title_abstract,,,Improving LLM Outputs Against Jailbreak Attacks with Expert Model Integration
arxiv:2505.18333,include,,title_abstract,,role: re-measurement; Jia et al.,A Critical Evaluation of Defenses against Prompt Injection Attacks
arxiv:2505.18907,include,,title_abstract,,"AIR, augmented intermediate representations",Stronger Enforcement of Instruction Hierarchy via Augmented Intermediate Representations
arxiv:2505.20435,exclude,not about prompt injection,title_abstract,,,The Shape of Adversarial Influence: Characterizing LLM Latent Spaces with Persistent Homology
arxiv:2505.21936,exclude,benchmark only,title_abstract,,,RedTeamCUA: Realistic Adversarial Testing of Computer-Use Agents in Hybrid Web-OS Environments
arxiv:2505.22648,exclude,not about prompt injection,title_abstract,,,WebDancer: Towards Autonomous Information Seeking Agency
arxiv:2505.22852,exclude,system level,title_abstract,,,Operationalizing CaMeL: Strengthening LLM Defenses for Enterprise Deployment
arxiv:2505.23643,exclude,system level,title_abstract,,,Securing AI Agents with Information-Flow Control
arxiv:2505.24019,exclude,"survey, position or other",title_abstract,,,LLM Agents Should Employ Security Principles
arxiv:2506.01055,exclude,attack only,title_abstract,,,Simple Prompt Injection Attacks Can Leak Personal Data Observed by LLM Agents During Task Execution
arxiv:2506.02456,exclude,benchmark only,title_abstract,,,VPI-Bench: Visual Prompt Injection Attacks for Computer-Use Agents
arxiv:2506.04202,exclude,detector only,title_abstract,,context attribution,TracLLM: A Generic Framework for Attributing Long Context LLMs
arxiv:2506.05446,exclude,detector only,title_abstract,,,Sentinel: SOTA model to protect against prompt injections
arxiv:2506.05739,include,,title_abstract,,Polymorphic Prompt Assembling,To Protect the LLM Agent Against the Prompt Injection Attack with Polymorphic Prompt
arxiv:2506.06384,exclude,detector only,title_abstract,,,Detection Method for Prompt Injection by Integrating Pre-trained Model and Heuristic Feature Enginee
arxiv:2506.07564,exclude,system level,title_abstract,,,SAFEFLOW: A Principled Protocol for Trustworthy and Transactional Autonomous Agent Systems
arxiv:2506.07948,exclude,attack only,title_abstract,,,TokenBreak: Bypassing Text Classification Models Through Token Manipulation
arxiv:2506.08837,exclude,"survey, position or other",title_abstract,,"design patterns, system level",Design Patterns for Securing LLM Agents against Prompt Injections
arxiv:2506.09956,include,,title_abstract,,role: re-measurement; LLMail-Inject challenge with Spotlighting among its defences,LLMail-Inject: A Dataset from a Realistic Adaptive Prompt Injection Challenge
arxiv:2506.10029,exclude,attack only,title_abstract,,,Evaluation empirique de la sécurisation et de l'alignement de ChatGPT et Gemini: analyse comparative
arxiv:2506.12104,exclude,system level,title_abstract,,DRIFT,DRIFT: Dynamic Rule-Based Defense with Injection Isolation for Securing LLM Agents
arxiv:2506.16447,exclude,not about prompt injection,title_abstract,,,Probe before You Talk: Towards Black-box Defense against Backdoor Unalignment for Large Language Mod
arxiv:2506.17318,exclude,attack only,title_abstract,,,Context manipulation attacks : Web agents are susceptible to corrupted memory
arxiv:2506.19109,exclude,detector only,title_abstract,,,Enhancing Security in LLM Applications: A Performance Evaluation of Early Detection Systems
arxiv:2506.22183,exclude,not about prompt injection,title_abstract,,,A Different Approach to AI Safety: Proceedings from the Columbia Convening on Openness in Artificial
arxiv:2506.23260,exclude,"survey, position or other",title_abstract,,,From prompt injections to protocol exploits: Threats in LLM-powered AI agents workflows
arxiv:2507.00601,exclude,not about prompt injection,title_abstract,,,Transferable Modeling Strategies for Low-Resource LLM Tasks: A Prompt and Alignment-Based Approach
arxiv:2507.02592,exclude,not about prompt injection,title_abstract,,,WebSailor: Navigating Super-human Reasoning for Web Agent
arxiv:2507.02735,include,,title_abstract,,Meta-SecAlign,Meta-SecAlign: Training LLMs against Prompt Injection for Robust Agents
arxiv:2507.02850,exclude,not about prompt injection,title_abstract,,,LLM Hypnosis: Exploiting User Feedback for Unauthorized Knowledge Injection to All Users
arxiv:2507.03042,exclude,not about prompt injection,title_abstract,,,Dynamic Long Short-Term Memory Based Memory Storage For Long Horizon LLM Interaction
arxiv:2507.05630,exclude,attack only,title_abstract,,,How Not to Detect Prompt Injections with an LLM
arxiv:2507.06185,exclude,attack only,title_abstract,,,Hidden Prompts in Manuscripts Exploit AI-Assisted Peer Review
arxiv:2507.06261,exclude,not about prompt injection,title_abstract,,,"Gemini 2.5: Pushing the Frontier with Advanced Reasoning, Multimodality, Long Context, and Next Gene"
arxiv:2507.07417,include,,title_abstract,,role: re-measurement; architecture-aware attacks on fine-tuning defences,May I have your Attention? Breaking Fine-Tuning based Prompt Injection Defenses using Architecture-A
arxiv:2507.07974,include,,title_abstract,,DefensiveTokens,Defending Against Prompt Injection With a Few DefensiveTokens
arxiv:2507.09709,exclude,not about prompt injection,title_abstract,,,Large Language Models Encode Semantics and Alignment in Linearly Separable Representations
arxiv:2507.13038,exclude,attack only,title_abstract,,,MAD-Spear: A Conformity-Driven Prompt Injection Attack on Multi-Agent Debate Systems
arxiv:2507.13686,exclude,attack only,title_abstract,,,TopicAttack: An Indirect Prompt Injection Attack via Topic Transition
arxiv:2507.14799,exclude,attack only,title_abstract,,,Manipulating LLM Web Agents with Indirect Prompt Injection Attack via HTML Accessibility Tree
arxiv:2507.15061,exclude,not about prompt injection,title_abstract,,,WebShaper: Agentically Data Synthesizing via Information-Seeking Formalization
arxiv:2507.15219,exclude,detector only,title_abstract,,PromptArmor,PromptArmor: Simple yet Effective Prompt Injection Defenses
arxiv:2507.15330,exclude,not about prompt injection,title_abstract,,,QSAF: A Novel Mitigation Framework for Cognitive Degradation in Agentic AI
arxiv:2507.15613,exclude,attack only,title_abstract,,,Multi-Stage Prompt Inference Attacks on Enterprise LLM Systems
arxiv:2507.20526,exclude,attack only,title_abstract,,competition findings,Security Challenges in AI Agent Deployment: Insights from a Large Scale Public Competition
arxiv:2507.23453,exclude,detector only,title_abstract,,,Counterfactual Evaluation for Blind Attack Detection in LLM-based Evaluation Systems
arxiv:2507.23465,exclude,not about prompt injection,title_abstract,,"role-based access control behaviour, not injection",Role-Aware Language Models for Secure and Contextualized Access Control in Organizations
arxiv:2508.00602,exclude,detector only,title_abstract,,,LeakSealer: A Semisupervised Defense for LLMs Against Prompt Injection and Leakage Attacks
arxiv:2508.01249,exclude,system level,title_abstract,,,AgentArmor: Enforcing Program Analysis on Agent Runtime Trace to Defend Against Prompt Injection
arxiv:2508.02110,exclude,attack only,title_abstract,,,Attractive Metadata Attack: Inducing LLM Agents to Invoke Malicious Tools
arxiv:2508.02312,exclude,"survey, position or other",title_abstract,,,"Data Security in Large Language Models: Risks, Defense, and Directions"
arxiv:2508.02961,exclude,no full text obtainable,full_text,not retrieved: no identifier; search the title in a browser or the library catalogue,full text: not obtained by script or by hand,Defend LLMs Through Self-Consciousness
arxiv:2508.03793,exclude,detector only,title_abstract,,,AttnTrace: Contextual Attribution of Prompt Injection and Knowledge Corruption
arxiv:2508.04281,exclude,attack only,title_abstract,,,Safeguarding Deliberation in Digital Democracy
arxiv:2508.05748,exclude,not about prompt injection,title_abstract,,,WebWatcher: Breaking New Frontier of Vision-Language Deep Research Agent
arxiv:2508.06418,exclude,detector only,title_abstract,,,Quantifying Conversation Drift in MCP via Latent Polytope
arxiv:2508.08127,exclude,system level,title_abstract,,,BlindGuard: Safeguarding LLM-based Multi-Agent Systems under Unknown Attacks
arxiv:2508.09288,include,,title_abstract,,CIV: provenance labels with attention masking; check at full text,Can AI Keep a Secret? Contextual Integrity Verification: A Provable Security Architecture for LLMs
arxiv:2508.10991,exclude,system level,title_abstract,,,MCP-Guard: A Multi-Stage Defense-in-Depth Framework for Securing Model Context Protocol in Agentic A
arxiv:2508.13214,exclude,attack only,title_abstract,,,Too Easily Fooled? Prompt Injection Breaks LLMs on Frustratingly Simple Multiple-Choice Questions
arxiv:2508.14128,exclude,jailbreak or direct injection only,title_abstract,,,CCFC: Core & Core-Full-Core Dual-Track Defense for LLM Jailbreak Protection
arxiv:2508.15310,exclude,system level,title_abstract,,IPIGuard,IPIGuard: A Novel Tool Dependency Graph-Based Defense Against Indirect Prompt Injection in LLM Agent
arxiv:2508.16481,exclude,benchmark only,title_abstract,,,Benchmarking the Robustness of Agentic Systems to Adversarially-Induced Harms
arxiv:2508.17884,exclude,detector only,title_abstract,,,PhantomLint: Principled Detection of Hidden LLM Prompts in Structured Documents
arxiv:2508.19287,exclude,attack only,title_abstract,,,Prompt-in-Content Attacks: Exploiting Uploaded Inputs to Hijack LLM Behavior
arxiv:2508.19288,exclude,attack only,title_abstract,,,Tricking LLM-Based NPCs into Spilling Secrets
arxiv:2508.20863,exclude,attack only,title_abstract,,,Misleading Large Language Models used (or misused) in Scientific Peer-Reviewing via Hidden Prompt-In
arxiv:2508.20890,exclude,detector only,title_abstract,,,PromptSleuth: Detecting Prompt Injection via Semantic Intent Invariance
arxiv:2508.21669,exclude,not about prompt injection,title_abstract,,offensive use of injection,Cybersecurity AI: Hacking the AI Hackers via Prompt Injection
arxiv:2509.00088,exclude,detector only,full_text,,full text: AEGIS: co-evolved prompt for an LLM detector of injected submissions,AEGIS : Automated Co-Evolutionary Framework for Guarding Prompt Injections Schema
arxiv:2509.00124,exclude,attack only,title_abstract,,,A Whole New World: Creating a Parallel-Poisoned Web Only AI-Agents Can See
arxiv:2509.05755,exclude,attack only,title_abstract,,,Red-Teaming Coding Agents from a Tool-Invocation Perspective: An Empirical Security Assessment
arxiv:2509.05883,exclude,attack only,full_text,,full text: eight commercial models tested without defences,Multimodal Prompt Injection Attacks: Risks and Defenses for Modern LLMs
arxiv:2509.06572,exclude,attack only,title_abstract,,,Parasites in the Toolchain: A Large-Scale Analysis of Attacks on the MCP Ecosystem
arxiv:2509.08646,exclude,"survey, position or other",title_abstract,,,Architecting Resilient LLM Agents: A Guide to Secure Plan-then-Execute Implementations
arxiv:2509.09912,exclude,attack only,title_abstract,,,"When Your Reviewer is an LLM: Biases, Divergence, and Prompt Injection Risks in Peer Review"
arxiv:2509.10122,exclude,not about prompt injection,title_abstract,,,Realism Control One-step Diffusion for Real-World Image Super-Resolution
arxiv:2509.10248,exclude,attack only,title_abstract,,,Prompt Injection Attacks on LLM Generated Reviews of Scientific Publications
arxiv:2509.10540,exclude,attack only,title_abstract,,,EchoLeak: The First Real-World Zero-Click Prompt Injection Exploit in a Production LLM System
arxiv:2509.11431,exclude,system level,title_abstract,,,Securing AI Agents: Implementing Role-Based Access Control for Industrial Applications
arxiv:2509.13305,exclude,not about prompt injection,title_abstract,,,WebSailor-V2: Bridging the Chasm to Proprietary Agents via Synthetic Data and Scalable Reinforcement
arxiv:2509.14271,include,,title_abstract,,"adversarial fine-tuning study of 2022, documented 2025",Early Approaches to Adversarial Fine-Tuning for Prompt Injection Defense: A 2022 Study of GPT-3 and
arxiv:2509.14285,exclude,system level,title_abstract,,,A Multi-Agent LLM Defense Pipeline Against Prompt Injection Attacks
arxiv:2509.14404,exclude,"survey, position or other",title_abstract,,,A Taxonomy of Prompt Defects in LLM Systems
arxiv:2509.17371,exclude,not about prompt injection,title_abstract,,,SilentStriker:Toward Stealthy Bit-Flip Attacks on Large Language Models
arxiv:2509.18557,exclude,system level,title_abstract,,,LLMZ+: Contextual Prompt Whitelist Principles for Agentic LLMs
arxiv:2509.19839,exclude,jailbreak or direct injection only,title_abstract,,,LatentGuard: Controllable Latent Steering for Robust Refusal of Attacks and Reliable Response Genera
arxiv:2509.21367,exclude,system level,title_abstract,,application pipeline; check,Design and Implementation of a Secure RAG-Enhanced AI Chatbot for Smart Tourism Customer Service: De
arxiv:2509.22040,exclude,attack only,title_abstract,,,"""Your AI, My Shell"": Demystifying Prompt Injection Attacks on Agentic AI Coding Editors"
arxiv:2509.22830,exclude,attack only,title_abstract,,chat-template abuse; evidence on the roles signal,ChatInject: Abusing Chat Templates for Prompt Injection in LLM Agents
arxiv:2509.23519,exclude,system level,title_abstract,,isolate-then-aggregate at pipeline level,ReliabilityRAG: Effective and Provably Robust Defense for RAG-based Web-Search
arxiv:2509.23694,exclude,attack only,title_abstract,,,SafeSearch: Automated Red-Teaming of LLM-Based Search Agents
arxiv:2509.24967,include,,title_abstract,,SecInfer inference-time scaling; check at full text,SecInfer: Preventing Prompt Injection via Inference-time Scaling
arxiv:2509.25448,exclude,not about prompt injection,title_abstract,,,Fingerprinting LLMs via Prompt Injection
arxiv:2509.25926,exclude,system level,title_abstract,,,Preventing Prompt Injection with Type-Directed Privilege Separation
arxiv:2509.26584,exclude,not about prompt injection,title_abstract,,,Fairness Testing in Retrieval-Augmented Generation: How Small Perturbations Reveal Bias in Small Lan
arxiv:2510.00181,exclude,attack only,title_abstract,,,CHAI: Command Hijacking against embodied AI
arxiv:2510.00451,exclude,"survey, position or other",title_abstract,,,A Call to Action for a Secure-by-Design Generative AI Paradigm
arxiv:2510.01586,exclude,jailbreak or direct injection only,full_text,,full text: AdvEvo-MARL: co-evolutionary RL of task agents against evolving jailbreak prompts,AdvEvo-MARL: Shaping Internalized Safety through Adversarial Co-Evolution in Multi-Agent Reinforceme
arxiv:2510.03204,exclude,system level,title_abstract,,,FocusAgent: Simple Yet Effective Ways of Trimming the Large Context of Web Agents
arxiv:2510.03705,exclude,attack only,title_abstract,,backdoor-powered attacks against defences,Backdoor-Powered Prompt Injection Attacks Nullify Defense Methods
arxiv:2510.04257,exclude,attack only,title_abstract,,,AgentTypo: Adaptive Typographic Prompt Injection Attacks against Black-box Multimodal Agents
arxiv:2510.04261,exclude,attack only,title_abstract,,,VortexPIA: Indirect Prompt Injection Attack against LLMs for Efficient Extraction of User Privacy
arxiv:2510.04885,include,,title_abstract,,role: re-measurement; RL attacks on Instruction Hierarchy and SecAlign,RL Is a Hammer and LLMs Are Nails: A Simple Reinforcement Learning Recipe for Strong Prompt Injectio
arxiv:2510.05106,exclude,not about prompt injection,title_abstract,,,Rule Encoding and Compliance in Large Language Models: An Information-Theoretic Analysis
arxiv:2510.05244,exclude,system level,title_abstract,,firewall,"Indirect Prompt Injections: Are Firewalls All You Need, or Stronger Benchmarks?"
arxiv:2510.05442,include,,title_abstract,,ARLAS adversarial RL training of the agent model,Adversarial Reinforcement Learning for Large Language Model Agent Safety
arxiv:2510.07809,exclude,attack only,title_abstract,,,"Invisible to Humans, Triggered by Agents: Stealthy Jailbreak Attacks on Mobile Vision-Language Agent"
arxiv:2510.08829,exclude,detector only,title_abstract,,sanitiser,CommandSans: Securing AI Agents with Surgical Precision Prompt Sanitization
arxiv:2510.08917,exclude,not about prompt injection,title_abstract,,,"""I know it's not right, but that's what it said to do"": Investigating Trust in AI Chatbots for Cyber"
arxiv:2510.09023,include,,title_abstract,,role: re-measurement; Nasr et al.,The Attacker Moves Second: Stronger Adaptive Attacks Bypass Defenses Against Llm Jailbreaks and Prom
arxiv:2510.09093,exclude,attack only,title_abstract,,,Exploiting Web Search Tools of AI Agents for Data Exfiltration
arxiv:2510.09462,exclude,not about prompt injection,title_abstract,,,Adaptive Attacks on Trusted Monitors Subvert AI Control Protocols
arxiv:2510.09849,exclude,attack only,title_abstract,,,Text Prompt Injection of Vision Language Models
arxiv:2510.11238,exclude,attack only,title_abstract,,,Attacks by Content: Automated Fact-checking is an AI Security Issue
arxiv:2510.11837,exclude,system level,title_abstract,,,Countermind: A Multi-Layered Security Architecture for Large Language Models
arxiv:2510.12252,exclude,detector only,title_abstract,,,PromptLocate: Localizing Prompt Injection Attacks
arxiv:2510.13351,exclude,detector only,title_abstract,,,Protect: Towards Robust Guardrailing Stack for Trustworthy Enterprise LLM Systems
arxiv:2510.14005,exclude,detector only,title_abstract,,PIShield,PIShield: Detecting Prompt Injection Attacks via Intrinsic LLM Features
arxiv:2510.16128,exclude,not about prompt injection,title_abstract,,,Prompt injections as a tool for preserving identity in GAI image descriptions
arxiv:2510.16381,exclude,system level,title_abstract,,,ATA: A Neuro-Symbolic Approach to Implement Autonomous and Trustworthy Agents
arxiv:2510.18131,exclude,not about prompt injection,title_abstract,,,BlueCodeAgent: A Blue Teaming Agent Powered by Automated Red Teaming for CodeGen AI
arxiv:2510.19169,exclude,detector only,title_abstract,,,"OpenGuardrails: A Configurable, Unified, and Scalable Guardrails Platform for Large Language Models"
arxiv:2510.19207,exclude,detector only,title_abstract,,DataFilter,Defending Against Prompt Injection with DataFilter
arxiv:2510.19844,exclude,detector only,title_abstract,,,"CourtGuard: A Local, Multiagent Prompt Injection Classifier"
arxiv:2510.21057,exclude,detector only,title_abstract,,iterative sanitisation of tool outputs before the model,Soft Instruction De-escalation Defense
arxiv:2510.22628,exclude,detector only,title_abstract,,,Sentra-Guard: A Real-Time Multilingual Defense Against Adversarial LLM Prompts
arxiv:2510.22963,exclude,attack only,title_abstract,,,When Compression Becomes an Attack Surface: Black-Box Attacks on Prompt-Compressed LLM Agents
arxiv:2511.00447,include,,title_abstract,,DRIP,DRIP: Defending Prompt Injection via Token-wise Representation Editing and Residual Instruction Fusi
arxiv:2511.01287,exclude,detector only,full_text,,full text: defence is a detector that removes injected text before scoring,"""Give a Positive Review Only"": An Early Investigation Into In-Paper Prompt Injection Attacks and Def"
arxiv:2511.03247,exclude,attack only,title_abstract,,,Death by a Thousand Prompts: Open Model Vulnerability Analysis
arxiv:2511.03434,exclude,"survey, position or other",title_abstract,,,"Inter-Agent Trust Models: A Comparative Study of Brief, Claim, Proof, Stake, Reputation and Constrai"
arxiv:2511.04694,include,,title_abstract,,Reasoning Up the Instruction Ladder,Reasoning Up the Instruction Ladder for Controllable Language Models
arxiv:2511.05867,exclude,not about prompt injection,title_abstract,,,Can LLM Infer Risk Information From MCP Server System Logs?
arxiv:2511.05919,exclude,attack only,title_abstract,,,Injecting Falsehoods: Adversarial Man-in-the-Middle Attacks Undermining Factual Recall in LLMs
arxiv:2511.10720,exclude,detector only,title_abstract,,sanitiser,PISanitizer: Preventing Prompt Injection to Long-Context LLMs via Prompt Sanitization
arxiv:2511.15203,exclude,"survey, position or other",title_abstract,,,"Taxonomy, Evaluation and Exploitation of IPI-Centric LLM Agent Defense Frameworks"
arxiv:2511.15759,include,,full_text,,full text: layered defence; hierarchical system-prompt guardrails measured as an added layer over content filtering (Table 2),Securing AI Agents Against Prompt Injection Attacks
arxiv:2511.16347,exclude,attack only,title_abstract,,,The Shawshank Redemption of Embodied AI: Understanding and Benchmarking Indirect Environmental Jailb
arxiv:2511.17671,exclude,attack only,title_abstract,,,Cross-User Poisoning: User-Task Boundary Failures in Multi-User Collaborative Language Agents
arxiv:2511.19483,exclude,not about prompt injection,title_abstract,,,Z-Space: A Multi-Agent Tool Orchestration Framework for Enterprise-Grade LLM Automation
arxiv:2511.19727,include,,title_abstract,,Prompt Fencing,Prompt Fencing: A Cryptographic Approach to Establishing Security Boundaries in Large Language Model
arxiv:2511.20597,exclude,detector only,full_text,,full text: BrowseSafe: fine-tuned detection classifier with architectural controls,BrowseSafe: Understanding and Preventing Prompt Injection Within AI Browser Agents
arxiv:2511.21752,include,,title_abstract,,"label disguise, a task-level prompting change",Semantics as a Shield: Label Disguise Defense (LDD) against Prompt Injection in LLM Sentiment Classi
arxiv:2511.23174,exclude,not about prompt injection,title_abstract,,,Are LLMs Good Safety Agents or a Propaganda Engine?
arxiv:2512.00966,include,,full_text,,"full text: IntentGuard: thinking-prefill intent analysis inside the protected model's own reasoning, then neutralises matched untrusted spans",Mitigating Indirect Prompt Injection via Instruction-Following Intent Analysis
arxiv:2512.01295,exclude,"survey, position or other",title_abstract,,,Systems Security Foundations for Agentic Computing
arxiv:2512.01326,include,,full_text,,full text: role: re-measurement; JATMO task-specific fine-tuning re-tested against the HOUYI genetic attack,Securing Large Language Models (LLMs) from Prompt Injection Attacks
arxiv:2512.04520,exclude,not about prompt injection,title_abstract,,,Boundary-Aware Test-Time Adaptation for Zero-Shot Medical Image Segmentation
arxiv:2512.04785,exclude,not about prompt injection,title_abstract,,,ASTRIDE: A Security Threat Modeling Platform for Agentic-AI Applications
arxiv:2512.04895,exclude,attack only,title_abstract,,,Chameleon: Adaptive Adversarial Agents for Scaling-Based Visual Prompt Injection in Multimodal AI Sy
arxiv:2512.05745,exclude,image or audio channel only,full_text,,"full text: ARGUS: activation steering against instructions embedded in images, video or audio",ARGUS: Defending Against Multimodal Indirect Prompt Injection via Steering Instruction-Following Beh
arxiv:2512.06556,exclude,attack only,title_abstract,,,Semantic Attacks on Tool-Augmented LLMs: Securing the Model Context Protocol Against Descriptor-Leve
arxiv:2512.06716,exclude,system level,title_abstract,,SIEVE,SIEVE: Selective Integrity Verification and Escalation for Defending LLM Agents against Indirect Pro
arxiv:2512.06914,exclude,"survey, position or other",title_abstract,,,SoK: Trust-Authorization Mismatch in LLM Agent Interactions
arxiv:2512.08290,exclude,"survey, position or other",title_abstract,,,Systematization of Knowledge: Security and Safety in the Model Context Protocol Ecosystem
arxiv:2512.08417,exclude,detector only,title_abstract,,corrected on rereading the abstract: Rennervate: a token-level detector over attention features that removes the detected text from the input,Attention is All You Need to Defend Against Indirect Prompt Injection Attacks in LLMs
arxiv:2512.12921,exclude,"survey, position or other",title_abstract,,,Cisco Integrated AI Security and Safety Framework Report
arxiv:2512.14860,exclude,attack only,title_abstract,,,Penetration Testing of Agentic AI: A Comparative Security Analysis Across Models and Frameworks
arxiv:2512.15081,exclude,not about prompt injection,title_abstract,,,Quantifying Return on Security Controls in LLM Systems
arxiv:2512.16307,include,,full_text,,full text: role: re-measurement; delimiter and instructional-prompt defences re-tested on small LLaMA models (own method is detection),Beyond the Benchmark: Innovative Defenses Against Prompt Injection Attacks
arxiv:2512.20405,exclude,attack only,title_abstract,,,ChatGPT: Excellent Paper! Accept It. Editor: Imposter Found! Review Rejected
arxiv:2512.20986,exclude,system level,title_abstract,,,AegisAgent: An Autonomous Defense Agent Against Prompt Injection Attacks in LLM-HARs
arxiv:2512.21999,exclude,not about prompt injection,title_abstract,,,Look Closer! An Adversarial Parametric Editing Framework for Hallucination Mitigation in VLMs
arxiv:2512.23132,exclude,system level,title_abstract,,,Multi-Agent Framework for Threat Mitigation and Resilience in AI-Based Systems
arxiv:2512.23557,exclude,system level,title_abstract,,agentic framework around the model,Toward Trustworthy Agentic AI: A Multimodal Framework for Preventing Prompt Injection Attacks
arxiv:2601.00867,exclude,not about prompt injection,title_abstract,,,The Silicon Psyche: Anthropomorphic Vulnerabilities in Large Language Models
arxiv:2601.02377,exclude,"survey, position or other",title_abstract,,,"Trust in LLM-controlled Robotics: a Survey of Security Threats, Defenses and Challenges"
arxiv:2601.03868,exclude,not about prompt injection,title_abstract,,,What Matters For Safety Alignment?
arxiv:2601.04666,include,,title_abstract,,"Know Thy Enemy, data synthesis with instruction-level CoT training",Know Thy Enemy: Securing LLMs Against Prompt Injection via Diverse Data Synthesis and Instruction-Le
arxiv:2601.04795,exclude,system level,title_abstract,,tool-result parsing outside the model,Defense Against Indirect Prompt Injection via Tool Result Parsing
arxiv:2601.05755,exclude,system level,title_abstract,,VIGIL,VIGIL: Defending LLM Agents Against Tool Stream Injection via Verify-Before-Commit
arxiv:2601.06884,exclude,attack only,title_abstract,,,Paraphrasing Adversarial Attack on LLM-as-a-Reviewer
arxiv:2601.07072,exclude,attack only,title_abstract,,,Overcoming the Retrieval Barrier: Indirect Prompt Injection in the Wild for LLM Systems
arxiv:2601.07185,include,,title_abstract,,role: re-measurement; Li et al.,Defenses Against Prompt Attacks Learn Surface Heuristics
arxiv:2601.07263,exclude,system level,full_text,,full text: SUPERVISOR: runtime module checking page context against the goal before actions run,When Bots Take the Bait: Exposing and Mitigating the Emerging Social Engineering Attack in Web Autom
arxiv:2601.07663,exclude,not about prompt injection,title_abstract,,,Reasoning Models Will Sometimes Lie About Their Reasoning
arxiv:2601.07835,include,,full_text,,full text: SecureCAI: DPO training of the assistant with a security constitution against injected security artifacts,SecureCAI: Injection-Resilient LLM Assistants for Cybersecurity Operations
arxiv:2601.07853,exclude,benchmark only,title_abstract,,,FinVault: Benchmarking Financial Agent Safety in Execution-Grounded Environments
arxiv:2601.08490,exclude,not about prompt injection,title_abstract,,,BenchOverflow: Measuring Overflow in Large Language Models via Plain-Text Prompts
arxiv:2601.09625,exclude,"survey, position or other",title_abstract,,,The Promptware Kill Chain: How Prompt Injections Gradually Evolved Into a Multistep Malware Delivery
arxiv:2601.09923,exclude,system level,title_abstract,,,CaMeLs Can Use Computers Too: System-level Security for Computer Use Agents
arxiv:2601.10173,include,,title_abstract,,ReasAlign,ReasAlign: Reasoning Enhanced Safety Alignment against Prompt Injection Attack
arxiv:2601.10923,exclude,benchmark only,title_abstract,,,Hidden-in-Plain-Text: A Benchmark for Social-Web Indirect Prompt Injection in RAG
arxiv:2601.11199,exclude,system level,title_abstract,,,SD-RAG: A Prompt-Injection-Resilient Framework for Selective Disclosure in Retrieval-Augmented Gener
arxiv:2601.12359,exclude,detector only,title_abstract,,,Zero-Shot Embedding Drift Detection: A Lightweight Defense Against Prompt Injections in LLMs
arxiv:2601.12822,exclude,system level,full_text,,full text: MirrorGuard: separate fine-tuned corrector model that rewrites the agent's reasoning,MirrorGuard: Toward Secure Computer-Use Agents via Simulation-to-Real Reasoning Correction
arxiv:2601.13186,exclude,system level,title_abstract,,,"Prompt Injection Mitigation with Agentic AI, Nested Learning, and AI Sustainability via Semantic Cac"
arxiv:2601.15528,exclude,not about prompt injection,title_abstract,,,Securing LLM-as-a-Service for Small Businesses: An Industry Case Study of a Distributed Chatbot Depl
arxiv:2601.17383,exclude,attack only,title_abstract,,,Physical Prompt Injection Attacks on Large Vision-Language Models
arxiv:2601.17548,exclude,"survey, position or other",title_abstract,,,Prompt Injection Attacks on Agentic Coding Assistants: A Systematic Analysis of Vulnerabilities in S
arxiv:2601.19051,exclude,detector only,full_text,,full text: HASTE: hard-negative mining to retrain injection detectors,Proactive Hardening of LLM Defenses with HASTE
arxiv:2601.21083,exclude,not about prompt injection,title_abstract,,,OpenSec: Measuring Incident Response Agent Calibration Under Adversarial Evidence
arxiv:2601.22240,exclude,"survey, position or other",title_abstract,,,A Systematic Literature Review on LLM Defenses Against Prompt Injection and Jailbreaking: Expanding
arxiv:2601.22569,exclude,attack only,title_abstract,,,Whispers of Wealth: Red-Teaming Google's Agent Payments Protocol via Prompt Injection
arxiv:2602.00750,exclude,attack only,title_abstract,,,Bypassing Prompt Injection Detectors through Evasive Injections
arxiv:2602.01378,exclude,not about prompt injection,title_abstract,,,Context Dependence and Reliability in Autoregressive Language Models
arxiv:2602.01795,include,,title_abstract,,RedVisor,RedVisor: Reasoning-Aware Prompt Injection Defense via Zero-Copy KV Cache Reuse
arxiv:2602.01942,exclude,"survey, position or other",title_abstract,,,Human Society-Inspired Approaches to Agentic AI Security: The 4C Framework
arxiv:2602.03117,include,,title_abstract,,role: re-measurement; AgentDyn evaluates deployed defences,AgentDyn: Are Your Agent Security Defenses Deployable in Real-World Dynamic Environments?
arxiv:2602.04653,exclude,attack only,title_abstract,,chat-template backdoors; evidence on the roles signal,Inference-Time Backdoors via Chat Templates: From LLM Supply Chains to Agentic System Compromise
arxiv:2602.05066,exclude,attack only,title_abstract,,,Bypassing AI Control Protocols via Agent-as-a-Proxy Attacks
arxiv:2602.05746,exclude,attack only,title_abstract,,,Learning to Inject: Automated Prompt Injection via Reinforcement Learning
arxiv:2602.06268,exclude,benchmark only,title_abstract,,,MPIB: A Benchmark for Medical Prompt Injection Attacks and Clinical Safety in LLMs
arxiv:2602.07104,exclude,attack only,title_abstract,,,Extended to Reality: Prompt Injection in 3D Environments
arxiv:2602.07381,exclude,not about prompt injection,title_abstract,,,"When the Model Said 'No Comment', We Knew Helpfulness Was Dead, Honesty Was Alive, and Safety Was Te"
arxiv:2602.07398,exclude,system level,title_abstract,,,AgentSys: Secure and Dynamic LLM Agents Through Explicit Hierarchical Memory Management
arxiv:2602.07918,exclude,detector only,title_abstract,,,CausalArmor: Efficient Indirect Prompt Injection Guardrails via Causal Attribution
arxiv:2602.09433,exclude,system level,title_abstract,,,Autonomous Action Runtime Management(AARM):A System Specification for Securing AI-Driven Actions at
arxiv:2602.10453,exclude,"survey, position or other",title_abstract,,,The Landscape of Prompt Injection Threats in LLM Agents: From Taxonomy to Analysis
arxiv:2602.10481,exclude,system level,full_text,,full text: authenticated prompts and context with policy enforcement at the application boundary,Protecting Context and Prompts: Deterministic Security for Non-Deterministic AI
arxiv:2602.10498,exclude,attack only,title_abstract,,,When Skills Lie: Hidden-Comment Injection in LLM Agents
arxiv:2602.10915,exclude,system level,title_abstract,,,"Blind Gods and Broken Screens: Architecting a Secure, Intent-Centric Mobile Agent Operating System"
arxiv:2602.11247,exclude,detector only,title_abstract,,,Peak + Accumulation: A Proxy-Level Scoring Formula for Multi-Turn LLM Attack Detection
arxiv:2602.11416,exclude,system level,title_abstract,,,Optimizing Agent Planning for Security and Autonomy
arxiv:2602.12285,exclude,not about prompt injection,title_abstract,,,From Biased Chatbots to Biased Agents: Examining Role Assignment Effects on LLM Agent Robustness
arxiv:2602.13597,exclude,detector only,title_abstract,,,AlignSentinel: Alignment-Aware Detection of Prompt Injection Attacks
arxiv:2602.14211,exclude,attack only,title_abstract,,,SkillJect: Effectively Automating Skill-Based Prompt Injection for Skill-Enabled Agents
arxiv:2602.16752,exclude,attack only,title_abstract,,,The Vulnerability of LLM Rankers to Prompt Injection Attacks
arxiv:2602.18514,exclude,attack only,title_abstract,,,Trojan Horses in Recruiting: A Red-Teaming Case Study on Indirect Prompt Injection in Standard vs. R
arxiv:2602.19021,exclude,"survey, position or other",title_abstract,,,Trustworthy AI LLM Scalability Risk Index (LSRI): A Cybersecurity Framework Assessing Agentic-AI Sec
arxiv:2602.19547,exclude,benchmark only,title_abstract,,,CIBER: A Comprehensive Benchmark for Security Evaluation of Code Interpreter Agents
arxiv:2602.20064,exclude,system level,title_abstract,,,"The LLMbda Calculus: AI Agents, Conversations, and Information Flow"
arxiv:2602.20708,include,,title_abstract,,ICON,ICON: Indirect Prompt Injection Defense for Agents based on Inference-Time Correction
arxiv:2602.20720,exclude,attack only,title_abstract,,,AdapTools: Adaptive Tool-based Indirect Prompt Injection Attacks on Agentic LLMs
arxiv:2602.20867,exclude,"survey, position or other",title_abstract,,,SoK: Agentic Skills -- Beyond Tool Use in LLM Agents
arxiv:2602.22242,exclude,attack only,title_abstract,,vulnerability analysis,Analysis of LLMs Against Prompt Injection and Jailbreak Attacks
arxiv:2602.22450,exclude,attack only,title_abstract,,,Silent Egress: When Implicit Prompt Injection Makes LLM Agents Leak Without a Trace
arxiv:2602.22724,exclude,system level,title_abstract,,,AgentSentry: Mitigating Indirect Prompt Injection in LLM Agents via Temporal Causal Diagnostics and
arxiv:2602.23956,exclude,not about prompt injection,title_abstract,,,SwitchCraft: Training-Free Multi-Event Video Generation with Attention Controls
arxiv:2603.01574,exclude,detector only,title_abstract,,,DualSentinel: A Lightweight Framework for Detecting Targeted Attacks in Black-box LLM via Dual Entro
arxiv:2603.03637,exclude,attack only,title_abstract,,,Image-based Prompt Injection: Hijacking Multimodal LLMs through Visually Embedded Adversarial Instru
arxiv:2603.03919,exclude,attack only,title_abstract,,,When Safety Becomes a Vulnerability: Exploiting LLM Alignment Homogeneity for Transferable Blocking
arxiv:2603.06588,exclude,not about prompt injection,title_abstract,,,vLLM Hook v1: A Plug-in for Programming Model Internals on vLLM
arxiv:2603.10163,exclude,attack only,title_abstract,,,Compatibility at a Cost: Systematic Discovery and Exploitation of MCP Clause-Compliance Vulnerabilit
arxiv:2603.10521,include,,title_abstract,,IH-Challenge training data for instruction hierarchy,IH-Challenge: A Training Dataset to Improve Instruction Hierarchy on Frontier LLMs
arxiv:2603.10749,exclude,system level,title_abstract,,,AttriGuard: Defeating Indirect Prompt Injection in LLM Agents via Causal Attribution of Tool Invocat
arxiv:2603.11331,exclude,jailbreak or direct injection only,title_abstract,,,Jailbreak Scaling Laws for Large Language Models: Polynomial-Exponential Crossover
arxiv:2603.11619,exclude,system level,title_abstract,,,Taming OpenClaw: Security Analysis and Mitigation of Autonomous LLM Agent Threats
arxiv:2603.11853,exclude,system level,title_abstract,,,"OpenClaw PRISM: A Zero-Fork, Defense-in-Depth Runtime Security Layer for Tool-Augmented LLM Agents"
arxiv:2603.11875,exclude,detector only,title_abstract,,,The Mirror Design Pattern: Strict Data Geometry over Model Scale for Prompt Injection Detection
arxiv:2603.12230,exclude,"survey, position or other",title_abstract,,,Security Considerations for Artificial Intelligence Agents
arxiv:2603.12277,include,,title_abstract,,role confusion; check whether a training fix is proposed and measured,Prompt Injection as Role Confusion
arxiv:2603.12644,exclude,system level,title_abstract,,,Uncovering Security Threats and Architecting Defenses in Autonomous Agents: A Case Study of OpenClaw
arxiv:2603.13026,include,,title_abstract,,role: re-measurement; PISmith RL red teaming of defences,PISmith: Reinforcement Learning-based Red Teaming for Prompt Injection Defenses
arxiv:2603.13424,exclude,system level,title_abstract,,,Agent Privilege Separation in OpenClaw: A Structural Defense Against Prompt Injection
arxiv:2603.16215,exclude,not about prompt injection,title_abstract,,,CoMAI: A Collaborative Multi-Agent Framework for Robust and Equitable Interview Evaluation
arxiv:2603.17419,exclude,system level,title_abstract,,,Caging the Agents: A Zero Trust Security Architecture for Autonomous AI in Healthcare
arxiv:2603.18433,exclude,system level,title_abstract,,,Prompt Control-Flow Integrity: A Priority-Aware Runtime Defense Against Prompt Injection in LLM Syst
arxiv:2603.19423,include,,title_abstract,,role: re-measurement; utility of defence-trained models on agent tasks,The Autonomy Tax: Defense Training Breaks LLM Agents
arxiv:2603.19469,exclude,"survey, position or other",title_abstract,,,A Framework for Formalizing LLM Agent Security
arxiv:2603.19974,exclude,attack only,title_abstract,,,Trojan's Whisper: Stealthy Manipulation of OpenClaw through Injected Bootstrapped Guidance
arxiv:2603.20214,exclude,not about prompt injection,title_abstract,,,Beyond Detection: Governing GenAI in Academic Peer Review as a Sociotechnical Challenge
arxiv:2603.20381,exclude,not about prompt injection,title_abstract,,,The production of meaning in the processing of natural language
arxiv:2603.20976,exclude,detector only,title_abstract,,,Detection of adversarial intent in Human-AI teams using LLMs
arxiv:2603.21975,exclude,detector only,full_text,,full text: dataset for classifiers of unsafe model outputs,SecureBreak -- A dataset towards safe and secure models
arxiv:2603.22489,exclude,"survey, position or other",title_abstract,,,Model Context Protocol Threat Modeling and Analysis of Vulnerabilities to Prompt Injection with Tool
arxiv:2603.22928,exclude,"survey, position or other",title_abstract,,,SoK: The Attack Surface of Agentic AI - Tools and Autonomy
arxiv:2603.23791,exclude,system level,title_abstract,,,The Cognitive Firewall:Securing Browser Based AI Agents Against Indirect Prompt Injection Via Hybrid
arxiv:2603.24203,exclude,attack only,title_abstract,,,Invisible Threats from Model Context Protocol: Generating Stealthy Injection Payload via Tree-based
arxiv:2603.24511,exclude,attack only,title_abstract,,,Claudini: Autoresearch Discovers State-of-the-Art Adversarial Attack Algorithms for LLMs
arxiv:2603.27522,exclude,attack only,title_abstract,,,Hidden Ads: Behavior Triggered Semantic Backdoors for Advertisement Injection in Vision Language Mod
arxiv:2603.28013,exclude,attack only,title_abstract,,measurement of injection across production models,Kill-Chain Canaries: Stage-Level Tracking of Prompt Injection Across Attack Surfaces and Five Produc
arxiv:2603.29418,exclude,attack only,title_abstract,,,Covert Visual Prompt Injection against Commercial Multimodal Large Language Models
arxiv:2603.30016,exclude,"survey, position or other",title_abstract,,,Architecting Secure AI Agents: Perspectives on System-Level Defenses Against Indirect Prompt Injecti
arxiv:2604.02954,exclude,attack only,title_abstract,,,LogicPoison: Logical Attacks on Graph Retrieval-Augmented Generation
arxiv:2604.03598,exclude,attack only,title_abstract,,,AttackEval: A Systematic Empirical Study of Prompt Injection Attack Effectiveness Against Large Lang
arxiv:2604.03870,exclude,attack only,title_abstract,,,Your Agent is More Brittle Than You Think: Uncovering Indirect Injection Vulnerabilities in Agentic
arxiv:2604.03912,exclude,system level,title_abstract,,,Automating Cloud Security and Forensics Through a Secure-by-Design Generative AI Framework
arxiv:2604.04426,exclude,system level,title_abstract,,,ShieldNet: Network-Level Guardrails against Emerging Supply-Chain Injections in Agentic Systems
arxiv:2604.05150,exclude,system level,title_abstract,,,Compiled AI: Deterministic Code Generation for LLM-Based Workflow Automation
arxiv:2604.05179,exclude,jailbreak or direct injection only,title_abstract,,decoding steering aimed at jailbreaks and direct injection,Gradient-Controlled Decoding: A Safety Guardrail for LLMs with Dual-Anchor Steering
arxiv:2604.06247,exclude,detector only,title_abstract,,,SALLIE: Generation-Free Hidden-State Detection of Jailbreaks and Prompt Injections Across Text and V
arxiv:2604.06436,exclude,"survey, position or other",title_abstract,,impossibility result for wrapper defences,The Defense Trilemma: Why Prompt Injection Defense Wrappers Fail?
arxiv:2604.07536,exclude,system level,title_abstract,,,TRUSTDESC: Preventing Tool Poisoning in LLM Applications via Trusted Description Generation
arxiv:2604.07831,exclude,attack only,title_abstract,,,Are GUI Agents Focused Enough? Automated Distraction via Semantic-level UI Element Injection
arxiv:2604.08352,exclude,not about prompt injection,title_abstract,,,Security Concerns in Generative AI Coding Assistants: Insights from Online Discussions on GitHub Cop
arxiv:2604.08499,include,,title_abstract,,role: re-measurement; PIArena evaluation platform,PIArena: A Platform for Prompt Injection Evaluation
arxiv:2604.09024,exclude,not about prompt injection,title_abstract,,,Leave My Images Alone: Preventing Multi-Modal Large Language Models from Analyzing Images via Visual
arxiv:2604.09378,exclude,attack only,title_abstract,,,BadSkill: Backdoor Attacks on Agent Skills via Model-in-Skill Poisoning
arxiv:2604.10134,exclude,system level,title_abstract,,,PlanGuard: Defending Agents against Indirect Prompt Injection via Planning-based Consistency Verific
arxiv:2604.10577,exclude,attack only,title_abstract,,,The Blind Spot of Agent Safety: How Benign User Instructions Expose Critical Vulnerabilities in Comp
arxiv:2604.11790,exclude,system level,title_abstract,,,ClawGuard: A Runtime Security Framework for Tool-Augmented LLM Agents Against Indirect Prompt Inject
arxiv:2604.12168,exclude,not about prompt injection,title_abstract,,,Fully Homomorphic Encryption on Llama 3 model for privacy preserving LLM inference
arxiv:2604.12284,exclude,detector only,title_abstract,,,WebAgentGuard: A Reasoning-Driven Guard Model for Detecting Prompt Injection Attacks in Web Agents
arxiv:2604.12371,exclude,attack only,title_abstract,,,Reading Between the Pixels: Linking Text-Image Embedding Alignment to Typographic Attack Success on
arxiv:2604.12548,exclude,attack only,title_abstract,,,DeepSeek Robustness Against Semantic-Character Dual-Space Mutated Prompt Injection
arxiv:2604.14604,exclude,attack only,title_abstract,,,Hijacking Large Audio-Language Models via Context-Agnostic and Imperceptible Auditory Prompt Injecti
arxiv:2604.15774,exclude,benchmark only,title_abstract,,,MemEvoBench: Benchmarking Safety Risks from Memory Misevolution in LLM Agents
arxiv:2604.16838,exclude,system level,title_abstract,,,"enclawed: A Configurable, Sector-Neutral Hardening Framework for Single-User AI Assistant Gateways"
arxiv:2604.16966,exclude,attack only,title_abstract,,,Visual Inception: Compromising Long-term Planning in Agentic Recommenders via Multimodal Memory Pois
arxiv:2604.17125,exclude,detector only,title_abstract,,audit of detection defences for MCP,Evaluating a Layered Prompt-Injection Defence for the Model Context Protocol: A Record-Level Audit o
arxiv:2604.18206,exclude,not about prompt injection,title_abstract,,,A Control Architecture for Training-Free Memory Use
arxiv:2604.18248,exclude,detector only,title_abstract,,,Beyond Pattern Matching: Seven Cross-Domain Techniques for Prompt Injection Detection
arxiv:2604.18658,exclude,not about prompt injection,title_abstract,,,Owner-Harm: A Missing Threat Model for AI Agent Safety
arxiv:2604.19844,exclude,image or audio channel only,full_text,,full text: misleading visual signals (structure- and noise-based) read by vision-language agents,If you're waiting for a sign... that might not be it! Mitigating Trust Boundary Confusion from Visua
arxiv:2604.22888,exclude,detector only,title_abstract,,,RouteGuard: Internal-Signal Detection of Skill Poisoning in LLM Agents
arxiv:2604.23270,exclude,not about prompt injection,title_abstract,,,CAP-CoT: Cycle Adversarial Prompt for Improving Chain of Thoughts in LLM Reasoning
arxiv:2604.23338,exclude,"survey, position or other",title_abstract,,,A Systematic Survey of Security Threats and Defenses in LLM-Based AI Agents: A Layered Attack Surfac
arxiv:2604.23593,exclude,"survey, position or other",title_abstract,,,When AI reviews science: Can we trust the referee?
arxiv:2604.23887,exclude,jailbreak or direct injection only,title_abstract,,adaptive attacker against defences of system-prompt secrets; direct injection setting,Evaluation of Prompt Injection Defenses in Large Language Models
arxiv:2604.24020,exclude,"survey, position or other",full_text,,full text: poster; in-context awareness training scored on a self-defined 12-dimension scale with the agent as its own evaluator; no injection outcome reported on its own,Poster: ClawdGo: Endogenous Security Awareness Training for Autonomous AI Agents
arxiv:2604.24118,exclude,system level,title_abstract,,,AgentVisor: Defending LLM Agents Against Prompt Injection via Semantic Virtualization
arxiv:2604.24542,exclude,detector only,title_abstract,,,Layerwise Convergence Fingerprints for Runtime Misbehavior Detection in Large Language Models
arxiv:2604.24920,exclude,system level,title_abstract,,,SUDP: Secret-Use Delegation Protocol for Agentic Systems
arxiv:2604.25186,exclude,not about prompt injection,title_abstract,,,FCMBench-Video: Benchmarking Document Video Intelligence
arxiv:2604.25200,exclude,not about prompt injection,title_abstract,,,Making AI-Assisted Grant Evaluation Auditable without Exposing the Model
arxiv:2604.25562,exclude,detector only,title_abstract,,,SnapGuard: Lightweight Prompt Injection Detection for Screenshot-Based Web Agents
arxiv:2604.26506,include,,full_text,,full text: SafeReview: reviewer model trained by DPO in co-evolution with a GRPO attacker against hidden prompts in submissions,SafeReview: Defending LLM-based Review Systems Against Adversarial Hidden Prompts
arxiv:2604.27202,exclude,attack only,title_abstract,,,"Indirect Prompt Injection in the Wild: An Empirical Study of Prevalence, Techniques, and Objectives"
arxiv:2604.28129,exclude,detector only,title_abstract,,,Latent Adversarial Detection: Adaptive Probing of LLM Activations for Multi-Turn Attack Detection
arxiv:2604.28157,exclude,attack only,title_abstract,,,FlashRT: Towards Computationally and Memory Efficient Red-Teaming for Prompt Injection and Knowledge
arxiv:2605.00460,exclude,detector only,title_abstract,,,CleanBase: Detecting Malicious Documents in RAG Knowledge Databases
arxiv:2605.01143,exclude,detector only,title_abstract,,,A Low-Latency Fraud Detection Layer for Detecting Adversarial Interaction Patterns in LLM-Powered Ag
arxiv:2605.01186,exclude,not about prompt injection,title_abstract,,,Trace: Unmasking AI Attack Agents Through Terminal Behavior Fingerprinting
arxiv:2605.01462,include,,title_abstract,,LocalAlign,LocalAlign: Enabling Generalizable Prompt Injection Defense via Generation of Near-Target Adversaria
arxiv:2605.03129,exclude,not about prompt injection,title_abstract,,,PIIGuard: Mitigating PII Harvesting under Adversarial Sanitization
arxiv:2605.03213,exclude,"survey, position or other",title_abstract,,,When Agents Handle Secrets: A Survey of Confidential Computing for Agentic AI
arxiv:2605.03378,exclude,system level,full_text,,full text: ARGUS: causal-provenance runtime auditor of agent actions; prompting defences appear only as baselines,ARGUS: Defending LLM Agents Against Context-Aware Prompt Injection
arxiv:2605.03619,exclude,not about prompt injection,title_abstract,,,The Infinite Mutation Engine? Measuring Polymorphism in LLM-Generated Offensive Code
arxiv:2605.04261,exclude,attack only,title_abstract,,,Laundering AI Authority with Adversarial Examples
arxiv:2605.05260,exclude,system level,title_abstract,,,SecureMCP: A Policy-Enforced LLM Data Access Framework for AIoT Systems via Model Context Protocol
arxiv:2605.06393,exclude,system level,title_abstract,,,Constraining Host-Level Abuse in Self-Hosted Computer-Use Agents via TEE-Backed Isolation
arxiv:2605.06669,exclude,jailbreak or direct injection only,full_text,,full text: tutor setting where the student types the injection; defences are filters and guardrails,Evaluating Prompt Injection Defenses for Educational LLM Tutors: Security-Usability-Latency Trade-of
arxiv:2605.07269,exclude,detector only,title_abstract,,,MIPIAD: Multilingual Indirect Prompt Injection Attack Defense with Qwen -- TF-IDF Hybrid and Meta-En
arxiv:2605.08310,exclude,attack only,title_abstract,,,WebTrap: Stealthy Mid-Task Hijacking of Browser Agents During Navigation
arxiv:2605.08326,exclude,not about prompt injection,title_abstract,,,LLM Advertisement based on Neuron Auctions
arxiv:2605.08460,exclude,attack only,title_abstract,,,When Child Inherits: Modeling and Exploiting Subagent Spawn in Multi-Agent Networks
arxiv:2605.09822,exclude,attack only,title_abstract,,,Oracle Poisoning: Corrupting Knowledge Graphs to Weaponise AI Agent Reasoning
arxiv:2605.10176,exclude,system level,title_abstract,,,When Prompts Become Payloads: A Framework for Mitigating SQL Injection Attacks in Large Language Mod
arxiv:2605.11026,exclude,detector only,title_abstract,,,AgentShield: Deception-based Compromise Detection for Tool-using LLM Agents
arxiv:2605.11039,exclude,system level,title_abstract,,,The Granularity Mismatch in Agent Security: Argument-Level Provenance Solves Enforcement and Isolate
arxiv:2605.11217,exclude,detector only,title_abstract,,,Training-Free Refusal of MCP Exploits via Retrieval-Augmented Generation
arxiv:2605.11442,exclude,attack only,title_abstract,,,Mobius Injection: From One-Shot Prompt Injection to Sustained Resource Exhaustion in AI Agent Infras
arxiv:2605.11516,exclude,not about prompt injection,title_abstract,,,Agents Should Replace Narrow Predictive AI as the Orchestrator in 6G AI-RAN
arxiv:2605.11868,exclude,attack only,title_abstract,,,IPI-proxy: An Intercepting Proxy for Red-Teaming Web-Browsing AI Agents Against Indirect Prompt Inje
arxiv:2605.12233,include,,full_text,,"full text: role: benchmark with defence results; six defences evaluated on task alignment, including Spotlighting and RUP as prompt-level methods","No More, No Less: Task Alignment in Terminal Agents"
arxiv:2605.13044,exclude,not about prompt injection,title_abstract,,,No Attack Required: Semantic Fuzzing for Specification Violations in Agent Skills
arxiv:2605.13471,exclude,system level,title_abstract,,,Sleeper Channels and Provenance Gates: Persistent Prompt Injection in Always-on Autonomous AI Agents
arxiv:2605.13631,exclude,detector only,title_abstract,,,ProjGuard: Safety Monitoring for Computer-Use Agents via Low-Dimensional Projections
arxiv:2605.14290,exclude,system level,title_abstract,,,Web Agents Should Adopt the Compile-and-Execute Paradigm
arxiv:2605.15030,exclude,detector only,title_abstract,,guard model for web agents,WARD: Adversarially Robust Defense of Web Agents Against Prompt Injections
arxiv:2605.15338,exclude,attack only,title_abstract,,,Hidden in Memory: Sleeper Memory Poisoning in LLM Agents
arxiv:2605.16090,exclude,attack only,title_abstract,,,A Cross-Modal Prompt Injection Attack against Large Vision-Language Models with Image-Only Perturbat
arxiv:2605.16233,exclude,not about prompt injection,title_abstract,,,FORGE: Self-Evolving Agent Memory With No Weight Updates via Population Broadcast
arxiv:2605.17380,exclude,system level,title_abstract,,,ADR: An Agentic Detection System for Enterprise Agentic AI Security
arxiv:2605.17634,include,,title_abstract,,"role: re-measurement; data-instruction separation defences under contextual manipulation, with utility cost",AI Agents May Always Fall for Prompt Injections
arxiv:2605.17986,exclude,benchmark only,title_abstract,,,LivePI: More Realistic Benchmarking of Agents Against Indirect Prompt Injection
arxiv:2605.18133,exclude,attack only,title_abstract,,,An Empirical Study of Privacy Leakage Chains via Prompt Injection in Black-Box Chatbot Environments
arxiv:2605.18583,exclude,not about prompt injection,title_abstract,,,Overeager Coding Agents: Measuring Out-of-Scope Actions on Benign Tasks
arxiv:2605.18918,exclude,detector only,full_text,,full text: ESLD: linear probe on a guard model's hidden states,"ESLD (External Surrogate Latent Defense): A Latent-Space Architecture for Faster, Stronger Prompt-In"
arxiv:2605.19159,exclude,detector only,title_abstract,,representational robustness of detectors,On the Geometric Limits of Transformer Defenses against Obfuscation Attacks: Latent Embedding Collap
arxiv:2605.23196,exclude,attack only,title_abstract,,,Prompt Overflow: What the Guardrail Inspects Is Not What the Model Infers
arxiv:2605.24421,exclude,attack only,title_abstract,,,Poisoning the Watchtower: Prompt Injection Attacks Against LLM-Augmented Security Operations Through
arxiv:2605.24659,exclude,attack only,title_abstract,,,IterInject: Indirect Prompt Injection Against LLM Agents via Feedback-Guided Iterative Optimization
arxiv:2605.25194,exclude,image or audio channel only,full_text,,full text: GTM: masks adversarial image tokens in multimodal models,Localize and Neutralize: Gradient-guided Token Suppression against Visual Prompt Injection Attack
arxiv:2605.25871,exclude,attack only,title_abstract,,,How Agentic AI Coding Assistants Become the Attacker's Shell
arxiv:2605.26159,exclude,system level,title_abstract,,,"Device Context Protocol: A Compact, Safety-First Architecture for LLM-Driven Control of Constrained "
arxiv:2605.26269,exclude,benchmark only,title_abstract,,,"AgentSecBench: Measuring Prompt Injection, Privacy Leakage, and Tool-Use Integrity in LLM Agents"
arxiv:2605.26497,exclude,system level,title_abstract,,,Aligning Provenance with Authorization: A Dual-Graph Defense for LLM Agents
arxiv:2605.26595,exclude,attack only,title_abstract,,,Cordyceps: Covert Control Attacks on LLMs via Data Poisoning
arxiv:2605.26999,exclude,detector only,title_abstract,,,Prompt Injection Detection is Regime-Dependent: A Deployment-Aware Evaluation with Interpretable Str
arxiv:2605.27823,exclude,detector only,title_abstract,,,Disentangling Adversarial Prompts: A Semantic-Graph Defense for Robust LLM Security
arxiv:2605.28017,exclude,attack only,title_abstract,,,Can It Reach the Generator? Investigating the Survival of Prompt-Injection Attacks in Realistic RAG
arxiv:2605.28116,exclude,attack only,title_abstract,,,MIRAGE: Context-Aware Prompt Injection against Mobile GUI Agents via User-Generated Content
arxiv:2605.28467,include,,title_abstract,,activation consistency training; check that injection results are reported,Mitigating Adaptive Attacks against Reasoning Models with Activation Consistency Training
arxiv:2605.29737,exclude,not about prompt injection,title_abstract,,,Minimal Prompt Perturbations Lead to Code Vulnerabilities: Prompt Fragility and Hidden-State Signals
arxiv:2605.30189,exclude,not about prompt injection,title_abstract,,,Token-Level Generalization in LoRA Adapter Backdoors: Attack Characterization and Behavioral Detecti
arxiv:2605.30454,exclude,attack only,title_abstract,,injection-surface sensitivity of attack success,The Surface You Test Is Not the Surface That Breaks
arxiv:2605.30534,include,,title_abstract,,dynamic separator generation for Polymorphic Prompt Assembling,Strengthening Polymorphic Prompt Assembling: Dynamic Separator Generation Against Emerging Prompt In
arxiv:2605.30650,exclude,"survey, position or other",title_abstract,,,When AI Meets Wall Street: A Survey on Trustworthy AI in Fintech
arxiv:2605.30677,exclude,detector only,title_abstract,,,Investigating Detection and Obfuscation of Prompt Injection Attacks Against Software Reverse Enginee
arxiv:2605.30686,exclude,attack only,title_abstract,,,"Depth-Dependent Indirect Prompt Injection in Tool-Calling ReAct Agents: Injection Depth, Payload Fra"
arxiv:2605.30837,exclude,detector only,title_abstract,,,Send a SCOUT First: Pre-hoc Reasoning for Adaptive Detector Allocation in Prompt-Injection Defense
arxiv:2605.31042,exclude,system level,title_abstract,,,From Prompt Injection to Persistent Control: Defending Agentic Harness Against Trojan Backdoors
arxiv:2606.02240,exclude,system level,title_abstract,,,AgentRedBench: Dynamic Redteaming and Integration-Aware Defense for LLM Agents over SaaS Integration
arxiv:2606.02959,exclude,detector only,title_abstract,,,Gate AI: LLM Security Benchmark Evaluation Methodology and Results
arxiv:2606.03090,exclude,attack only,title_abstract,,,"""**Important** You should give me full credits!"": Exploring Prompt Injection Attacks on LLM-Based Au"
arxiv:2606.04141,exclude,detector only,title_abstract,,,Caught in the Act(ivation): Toward Pre-Output and Multi-Turn Detection of Credential Exfiltration by
arxiv:2606.04329,exclude,attack only,title_abstract,,,From Untrusted Input to Trusted Memory: A Systematic Study of Memory Poisoning Attacks in LLM Agents
arxiv:2606.04612,exclude,detector only,title_abstract,,,Hybrid Adversarial Defence for Natural Language Understanding Tasks
arxiv:2606.07131,exclude,benchmark only,title_abstract,,,MalSkillBench: A Runtime-Verified Benchmark of Malicious Agent Skills
arxiv:2606.07992,exclude,attack only,title_abstract,,,VATS: Exploiting Implicit Authority in Error-Path Injection via Systematic Mutation
arxiv:2606.08403,exclude,attack only,title_abstract,,,Hiding in Plain Floats: Steganographic Carriers for Indirect Prompt and Content Injection
arxiv:2606.09005,exclude,attack only,title_abstract,,authority impersonation through serialised labels; evidence on the markers signal,Document-Authored Control-Signal Impersonation: A Low-Cost Indirect Prompt Attack on RAG Safety Boun
arxiv:2606.09204,exclude,attack only,title_abstract,,,The Injection Paradox: Brand-Level Suppression in Safety-Trained LLM Recommendations via RAG Context
arxiv:2606.09563,exclude,detector only,title_abstract,,,PRISM: Recovering Instruction Sets from Language Model Activations
arxiv:2606.10322,exclude,system level,title_abstract,,,Game-Theoretic Multi-Agent Control for Robust Contextual Reasoning in LLMs
arxiv:2606.10749,exclude,"survey, position or other",title_abstract,,,"Toward Secure LLM Agents: Threat Surfaces, Attacks, Defenses, and Evaluation"
arxiv:2606.10860,include,,title_abstract,,gravity-weighted DPO for instruction hierarchy,Training LLMs to Enforce Multi-Level Instruction Hierarchies via Gravity-Weighted Direct Preference
arxiv:2606.12716,exclude,detector only,full_text,,full text: PaperGuard: chunk-based embedding search that flags injected content in multimodal papers,Does AI Reviewer See the Full Picture? Attacking and Defending Multimodal Peer Review
arxiv:2606.12737,exclude,attack only,title_abstract,,,PI-Hunter: Automated Red-Teaming for Exposing and Localizing Prompt Injections
arxiv:2606.13038,exclude,not about prompt injection,title_abstract,,,Nous: An Attempt to Extract and Inject the Cognition Behind Prediction-Market Behavior
arxiv:2606.14154,exclude,system level,title_abstract,,,SkillMutator: Benchmarking and Defending Language-and-Code Cross-modal Attacks on LLM Agent Skills
arxiv:2606.14517,exclude,attack only,title_abstract,,,From Shield to Target: Denial-of-Service Attacks on LLM-Based Agent Guardrails
arxiv:2606.15008,exclude,"survey, position or other",title_abstract,,,Security Engineering of OpenClaw: Analyzing Attack Surface Expansion and Trust-Boundary Violations
arxiv:2606.15057,include,,title_abstract,,role: re-measurement; AutoDojo,AutoDojo: A Generative Benchmark for Evaluating Prompt Injection Defenses in LLM Agents
arxiv:2606.15441,include,,title_abstract,,reasoning-enabled task alignment; check that the model's own generation is changed,Defending against Adaptive Prompt Injection Attacks via Reasoning-enabled Task Alignment
arxiv:2606.15609,exclude,attack only,title_abstract,,,FragFuse: Bypassing Access Control of Large Language Model Agents via Memory-Based Query Fragmentati
arxiv:2606.15656,exclude,not about prompt injection,title_abstract,,,Overcoming the Impedance Mismatch: A Theoretical Roadmap for Fusing Foundation Models and Knowledge
arxiv:2606.15899,exclude,benchmark only,title_abstract,,,SkillVetBench: LLM-as-Judge for Multi-Dimensional Security Risk Evaluation in Open-Source LLM Agent
arxiv:2606.16242,exclude,not about prompt injection,title_abstract,,,Rapid Poison: Practical Poisoning Attacks Against the Rapid Response Framework
arxiv:2606.17467,exclude,detector only,title_abstract,,sanitiser,PARSE: Provenance-Aware Retrieval Sanitization for Professional Domain LLM Agents
arxiv:2606.18550,exclude,system level,title_abstract,,,The Gate Is Only as Honest as Its Contracts: ContractGuard for the Contract Layer of Risk-Aware Caus
arxiv:2606.19235,exclude,system level,title_abstract,,,CodeSentinel: A Three-Layer Defense Against Indirect Prompt Injection in Code Contexts
arxiv:2606.19660,exclude,system level,title_abstract,,,A Layered Security Framework Against Prompt Injection in RAG-Based Chatbots
arxiv:2606.20470,exclude,system level,title_abstract,,,Analyzing Defensive Misdirection Against Model-Guided Automated Attacks on Agentic AI Systems
arxiv:2606.20636,exclude,not about prompt injection,title_abstract,,,SkillHarness: Harnessing Safe Skills for Computer-Use Agents
arxiv:2606.20668,exclude,detector only,title_abstract,,,BELLS-O: Evaluating the Operational Trade-offs of LLM Supervision Systems
arxiv:2606.20922,exclude,system level,title_abstract,,,Think Twice Before You Act: Protecting LLM Agents Against Tool Description Poisoning via Isolated Pl
arxiv:2606.21071,exclude,attack only,title_abstract,,,Local LLM Agents as Vulnerable Runtimes:A Source-Code Audit of the Agent Runtime Layer
arxiv:2606.21732,exclude,attack only,title_abstract,,,"Safe to Check, Unsafe to Use: Relinking at the Compression Boundary of LLM Agents"
arxiv:2606.22779,exclude,detector only,title_abstract,,,DE-FIVE: Detecting Malicious Image Prompts via Fourier Features and Image Vector Embeddings
arxiv:2606.23277,exclude,detector only,full_text,,full text: GIF: information-flow measure that flags injected spans; the model's generation is not changed,GIF: Locally Sound Geometric Information Flow Control for LLMs
arxiv:2606.23416,exclude,detector only,title_abstract,,,Detecting Malicious Agent Skills in the Wild using Attention
arxiv:2606.25487,exclude,not about prompt injection,title_abstract,,"judge reliability; cite for the judge condition, not a defence",How Reliable Is Your Jailbreak Judge? Calibration and Adversarial Robustness of Automated ASR Scorin
arxiv:2606.26377,exclude,detector only,title_abstract,,,Verifying Intent and Harm: A Unified Defense Against LLM-Generated Threats
arxiv:2606.26406,exclude,not about prompt injection,title_abstract,,,Beyond Feedforward Networks: Reentry Neural Systems as the Fundamental Basis of Subjecthood and Intr
arxiv:2606.26479,exclude,system level,title_abstract,,"evaluation of out-of-band, system-level defences",Adaptive Evaluation of Out-of-Band Defenses Against Prompt Injection in LLM Agents
arxiv:2606.27287,exclude,attack only,title_abstract,,,Prompt Injection in Automated Résumé Screening with Large Language Models: Single and Multi-Injectio
arxiv:2606.27567,exclude,"survey, position or other",title_abstract,,impossibility result; cite in background,On the Inseparability of Instructions and Data in Shared-Embedding Sequence Models
arxiv:2606.28649,exclude,attack only,title_abstract,,,RIPA: Sensory-Vector Prompt Injection Attacks on LLM-Controlled ROS 2 Robots
arxiv:2606.29602,exclude,attack only,title_abstract,,,An Empirical Evaluation of Prompt Injection Vulnerabilities in Large Language Models Across Multilin
arxiv:2606.29960,include,,title_abstract,,IHDec contrastive decoding,IHDec: Divergence-Steered Contrastive Decoding for Securing Multi-Turn Instruction Hierarchies
arxiv:2606.30755,exclude,"survey, position or other",title_abstract,,,Understanding and Evaluating Claw-like Agent Security Through a Computer-Systems Lens
arxiv:2606.30783,include,,title_abstract,,role: benchmark with defence results; SecFid security and fidelity,Security--Fidelity Tradeoffs: The Hidden Cost of Prompt Injection Defense
arxiv:2607.01919,exclude,system level,title_abstract,,,ElephantAgent: Contextual State Continuity in Agentic Systems
arxiv:2607.02514,exclude,attack only,title_abstract,,,Distributed Attacks in Persistent-State AI Control
arxiv:2607.03821,exclude,system level,full_text,,full text: DualView: dual-LLM symbol tracking extended to the user's environment,DualView: Preventing Indirect Prompt Injection in Personal AI Agents
arxiv:2607.04222,exclude,not about prompt injection,title_abstract,,,Unsupervised Features Mining via Activation Geometry
arxiv:2607.05120,exclude,attack only,title_abstract,,,Agent Data Injection Attacks are Realistic Threats to AI Agents
arxiv:2607.05277,exclude,system level,title_abstract,,masking untrusted content before the model,Untrusted Content Masking for Web Agents with Security Guarantees
arxiv:2607.07433,exclude,attack only,title_abstract,,,Beware of Agentic Botnets: Scalable Untargeted Promptware Attacks via Universal and Transferable Adv
arxiv:2607.08147,exclude,system level,title_abstract,,,Prismata: Confining Cross-Site Prompt Injection in Web Agents
arxiv:2607.10080,exclude,attack only,title_abstract,,,The Effect of Multi-Lingual and Keyword Adversarial Injection on LLM Relevance Judgment
arxiv:2607.10269,exclude,image or audio channel only,full_text,,full text: physical scene text read by vision-language models on wearables,Devil in the Lens: Analyzing and Defending Physical Prompt Injection Against Vision-Language Models
arxiv:2607.10490,exclude,benchmark only,title_abstract,,,NetInjectBench: Benchmarking Indirect Prompt Injection in Tool-Using Large Language Model Agents for
arxiv:2607.12406,exclude,"survey, position or other",title_abstract,,,"Isolation as a First-Class Principle for LLM-Agent System Safety: Concepts, Taxonomy, Challenges and"
arxiv:2607.12624,exclude,detector only,title_abstract,,,PVDetector: Detecting Prompt Injection Attacks on Purpose-Specific LLM Agents through Policy-Violati
arxiv:2607.13987,exclude,"survey, position or other",title_abstract,,,"Agent Skill Security: Threat Models, Attacks, Defenses, and Evaluation"
arxiv:2607.14493,include,,full_text,,full text: role: re-measurement; spotlighting and instruction-hierarchy prompts tested with data-level and dual-LLM defences on injected network logs (LogInject),Context Contamination in LLM Analysis of Network Security Logs: Poison with Passive Prompt Injection
arxiv:2607.14611,exclude,attack only,title_abstract,,,Bad Memory: Evaluating Prompt Injection Risks from Memory in Agentic Systems
arxiv:2607.15977,exclude,not about prompt injection,title_abstract,,,Refusal is Not Safety! Benchmarking Latent Safety Risks of LLM-Driven Content Humorization
arxiv:2607.17535,exclude,system level,title_abstract,,,Salience Induction against Multi-Hop RAG Agents: Threat and Defense
arxiv:2607.18063,exclude,benchmark only,title_abstract,,,"Adaptive Adversaries: A Multi-Turn, Multi-LLM Benchmark for LLM Agent Security"
arxiv:2607.18622,exclude,attack only,title_abstract,,,CPInj: Uncovering Prompt Injection Risks in Textual Collaborative Prompt Optimization
arxiv:2607.19432,exclude,detector only,title_abstract,,,ChainWatch: A Kill Chain-Aligned Sequential Detection Framework for Multi-Step Attacks in MCP-Based
arxiv:2607.21151,exclude,not about prompt injection,title_abstract,,,V-DEAL: Diagnosing Video Safety De-Calibration as an Understanding-Refusal Coupling Failure
arxiv:2607.21824,exclude,system level,title_abstract,,,"Protocol-Level Attacks on Agentic Commerce Platforms: A Cross-Platform Taxonomy, AIP-Bench, and Unif"
arxiv:2607.22024,exclude,"survey, position or other",title_abstract,,,Agent Security Needs Redefinition through a Holistic Framework
arxiv:2607.23586,exclude,system level,title_abstract,,,Are You Still the Agent I Authorized? Earned Authority under a Fixed Ceiling for Evolving Agents
arxiv:2607.24006,exclude,not about prompt injection,title_abstract,,,Agentic Cloud Decoys: A Deception-Driven Framework for Autonomous Intrusion Investigation
arxiv:2607.24625,exclude,system level,title_abstract,,,APPA: Recoverable Information-Flow Control for Real-World LLM Agents
arxiv:2607.25255,exclude,system level,title_abstract,,,SafeFlow: Semantic Information-Flow Control for Blocking Malicious Propagation in Multi-Agent System
arxiv:2607.26115,include,,full_text,,full text: GPT-Red: self-play attacker used for RL adversarial training of the production model against prompt injection; defender robustness reported,GPT-Red: Automated Red Teaming via Self-Play at Scale
arxiv:2607.28165,exclude,attack only,title_abstract,,,Piggybacking on Perception: Stealthy Concurrent Audio Prompt Injections against Multimodal LLM Agent
arxiv:2607.28226,exclude,"survey, position or other",title_abstract,,,"Security of World-Model-Based Embodied AI: A Lifecycle of Threats, Defenses, and Evaluation"
arxiv:2607.29254,exclude,system level,title_abstract,,,Tool Specifications Matter: Uncovering and Mitigating Safety Risks in AI Agents
arxiv:2608.01635,exclude,not about prompt injection,title_abstract,,,Mitigating Visual Degradation in MLLMs via Spatial-Spectral Visual Anchor Learning
arxiv:2608.02018,exclude,attack only,title_abstract,,,Invisible Ink Threats: Adversarial Goals Behind Legitimate Tasks in Computer-Use Agents
arxiv:2608.02657,exclude,detector only,title_abstract,,hidden-state probing of injection exposure,Your Agentic LLMs Secretly Encode Indirect Prompt-Injection Exposure in Hidden States
arxiv:2608.04053,exclude,system level,title_abstract,,,AgentAntibody: An Adaptive Immune System for Defending LLM Agents against Prompt Injection
arxiv:2608.04192,exclude,not about prompt injection,title_abstract,,,Behavioral Skill Reconstruction: Reconstructing Hidden Functionality from LLM Agent Skills
arxiv:2608.04565,exclude,attack only,title_abstract,,,Breadcrumbing Search Agents: Per-Turn Scheming Over Long-Horizon Trajectories
arxiv:2608.04741,exclude,attack only,title_abstract,,,LoginTrap: Uncovering Task-Agnostic Phishing-Style Indirect Prompt Injection Attacks against LLM-bas
arxiv:2608.05108,exclude,attack only,title_abstract,,"red teaming; data for defences, no defence measured",Agent Against Agent: An Agentic System for Automatic Prompt Injection Red Teaming
arxiv:2608.05430,exclude,detector only,title_abstract,,,Robust Context-Aware Detection of Malicious Instructions in Text
arxiv:2608.05495,exclude,image or audio channel only,full_text,,"full text: smart-home agents reading ambient audio, video and on-screen text",PromptShield Home: Ambient Multimodal Prompt Injection Defense for Smart-Home Agents
arxiv:2608.05715,exclude,attack only,title_abstract,,,Hijacking Robots with a Piece of Paper: A Systematic Study of Physical Prompt Injection in VLM-Contr
arxiv:2608.06477,exclude,benchmark only,title_abstract,,,StepJack: Benchmarking Computer-Use Agent Safety Against Multi-Step Indirect Prompt Injection
arxiv:2608.07167,exclude,system level,title_abstract,,,NiyamAI - An Intent-Bound AI Agent with Cryptographically Verifiable Guardrails using Zero-Knowledge
arxiv:2608.07808,exclude,"survey, position or other",title_abstract,,,The Anatomy of a Prompt Injection: A Component Model for Structured Analysis
arxiv:2608.08027,exclude,detector only,full_text,,full text: BASIS: attention-based linear probes that gate refusal,BASIS: Breach-Aware Selective Prompt Injection Shielding with Prefill Attention Probes
arxiv:2608.08100,exclude,system level,title_abstract,,,Defending Retrieval-Augmented Intrusion Detection Against Knowledge Poisoning and Prompt Injection
arxiv:2608.08468,exclude,detector only,title_abstract,,,SkillsMetric: Mapping the Detection Boundary of Static Analysis for Malicious Agent Skills
arxiv:2608.08795,exclude,attack only,title_abstract,,,Toward Metacognitive One-Shot Indirect Prompt Injection: Strategy Abstraction Via Outcome-Conditione
arxiv:2608.09867,exclude,not about prompt injection,title_abstract,,,Stealing Reasoning Traces from Proprietary LLM APIs
arxiv:2608.10260,exclude,not about prompt injection,title_abstract,,,Interpreting Language Model Hidden States at Scale
arxiv:2608.10530,exclude,"survey, position or other",title_abstract,,,"On Understanding, Identifying, and Mitigating Vulnerabilities in Agentic Large Language Models"
arxiv:2608.12172,exclude,"survey, position or other",title_abstract,,,Rethinking Agent Security as a Networking Problem
arxiv:2608.14625,exclude,not about prompt injection,title_abstract,,,Local AI pre-screening for human triple-blind peer review in health sciences
arxiv:2608.14629,exclude,jailbreak or direct injection only,full_text,,full text: bias injected through the user's adversarial prompt; mitigations are chain-of-thought prompting and DPO against jailbreak-style prompts,Inference-Time Mitigation of Adversarial Political Bias in Large Language Models
arxiv:2608.14876,exclude,attack only,title_abstract,,,Workspace Topology as an Attack Vector in Agentic Coding Assistants
arxiv:2608.16178,exclude,not about prompt injection,title_abstract,,,Agent-Native Telemetry: Verifiable State-Delta Evidence for Autonomous Operations
arxiv:2608.16843,exclude,"survey, position or other",title_abstract,,,"Security of Foundation-Model-Powered Embodied Agents: Attack Surfaces, Attacks, Defenses, and Evalua"
arxiv:2608.17220,exclude,system level,title_abstract,,,PACE: Policy-Attested Contract Execution for Safe AI Agents in Decentralized Finance
arxiv:2608.19982,include,,title_abstract,,COPA continual preference optimisation,COPA: Continual Preference Optimization for Adaptive Prompt Injection Defense
arxiv:2608.21126,exclude,system level,title_abstract,,,TraceGrant: A Contract-Governed Security Framework for the Task-Effect Lifecycle of Networked LLM Ag
arxiv:2608.21230,exclude,not about prompt injection,title_abstract,,"false assertions in memory, not instructions",Utility Under Attack: Agent Memory Poisoning and the Limits of Content Screening and Provenance Rank
arxiv:2608.21500,include,,title_abstract,,SecOPD on-policy distillation,SecOPD: Mitigating Adaptive Prompt Injections by On-Policy Distillation
arxiv:2608.21656,exclude,system level,title_abstract,,,Mitigating Database Leakage in RAG Systems with Keyword-Grounded Fact Substitution
arxiv:2608.22061,exclude,attack only,title_abstract,,,MEMORY Wins All: Indirect Bias Injection Attacks via Social Media Content
arxiv:2608.22248,exclude,detector only,title_abstract,,corrected on rereading the abstract: AEGIS: projectors over hidden states identify injected instructions; evaluated as detection,Beyond Over-Refusal: Defending Indirect Prompt Injection via Latent Instruction Manifolds
arxiv:2608.23763,exclude,system level,title_abstract,,,"TrustShiftProbe: Characterizing, Benchmarking, and Defending Staged Trust Attacks on MCP Servers"
arxiv:2608.23782,exclude,not about prompt injection,title_abstract,,,GAP-Prompt: Gated Adaptive Prompting for Efficient Continual Learning
arxiv:2608.23873,include,,title_abstract,,semantic overlays: provenance annotations and steering vectors as the signal,Semantic Overlays: Mitigating Prompt Injection with Annotations Beyond Tokens and Steering Vectors
arxiv:2608.24022,exclude,detector only,title_abstract,,localisation of behaviour-guiding instructions,What Guides the Agent? Adjudicating Unauthorized Behavior via Localizing Behavior-Guiding Instructio
arxiv:2608.26108,exclude,system level,title_abstract,,"containment architecture, no abstract",Agentic AI Containment Architecture for Security Hardening
arxiv:2608.27092,exclude,attack only,title_abstract,,,The Framing Gap: Indirect Prompt-Injection Exfiltration Defeats Surface-Level Defenses in Tool-Using
arxiv:2608.27496,exclude,system level,title_abstract,,,ROPE: Routed Origin Policy Enforcement against Indirect Prompt Injection
arxiv:2608.27990,exclude,detector only,full_text,,full text: CAITLYN: agent-agnostic middleware that synthesises new detection scripts and LLM checks,CAITLYN: Can LLM Agents Autonomously Synthesize Defenses against Emerging Injection Attacks?
arxiv:2608.28411,include,,full_text,,"full text: role: benchmark with defence results; LongPIBench tests instructional, delimiter, sandwich, SecInfer, PromptLocate and Meta-SecAlign-8B in long contexts",LongPIBench: A Long-Context Benchmark for Prompt Injection
arxiv:2608.29596,exclude,"survey, position or other",title_abstract,,,"A Systematic Survey of Agent Skills: Lifecycle, Taxonomy, and Security"
arxiv:2608.30041,exclude,system level,title_abstract,,,Reachability-Based Capability Confinement for LLM Agents under Indirect Prompt Injection
arxiv:2608.30207,exclude,attack only,title_abstract,,,SIR: Self-improving Red-teaming for Compute Use Agents
arxiv:2608.30362,exclude,attack only,title_abstract,,covert injection metric on what the user notices; cite for the outcome coding,Will the User Ever Know? Covert Indirect Prompt Injection Attacks on Tool-Using LLM Agents
arxiv:2608.30441,exclude,attack only,title_abstract,,,ECLIPSE: Self-Evolving Stealthy Prompt Injection Attack against Long-Horizon Agentic Systems
arxiv:2609.00052,exclude,not about prompt injection,title_abstract,,,AgentProv: Auditing Agentic LLM API Providers via Tool-use Policy Probes
arxiv:2609.01931,exclude,system level,title_abstract,,,Agent Flight Recorder: Tamper-Evident Audit Trails with On-Chain Anchoring for Long-Horizon Tool-Usi
arxiv:2609.02035,exclude,attack only,title_abstract,,,Implicit Manipulation for Skill Selection in LLM Agents with Semantic Matching
arxiv:2609.03999,exclude,"survey, position or other",title_abstract,,,Shifting from Injection to Interaction: Rethinking Web Security in the Age of LLMs and Beyond
arxiv:2609.04533,exclude,attack only,title_abstract,,,Repeat-After-Me: Black-Box Adaptive Visual Prompt Injection
arxiv:2609.05074,exclude,detector only,title_abstract,,,Influence Score and Transformers interpretability: Measure of the Effective Impact of Attention Head
arxiv:2609.05903,exclude,system level,title_abstract,,,EvoSafeHarness: Evolving Model- and Domain-Specific Harnesses for Securing Agents
arxiv:2609.06966,exclude,not about prompt injection,title_abstract,,,MOLE: Detecting Insider Threats in AI Agents
arxiv:2609.07529,include,,title_abstract,,CoER co-evolution training against adaptive IPI; check mechanism,CoER: Defending against Adaptive Indirect Prompt Injection via Adversarial Co-Evolution and Refineme
arxiv:2609.09404,exclude,attack only,title_abstract,,,An Experimental Evaluation of Multimodal Prompt Injection Attacks on Agentic AI Frameworks
arxiv:2609.10707,exclude,system level,title_abstract,,,Architecting the Secure AI-SOC: A Neurosymbolic Framework for Pipeline Integrity and Threat Mitigati
arxiv:2609.11989,exclude,not about prompt injection,title_abstract,,,The Computational Primitives of Adaptation
arxiv:2609.13334,exclude,not about prompt injection,title_abstract,,,The Agentic Company OS: Substrate Inversion for Sustained Enterprise Agent Deployment
arxiv:2609.13725,exclude,benchmark only,title_abstract,,paired procedure-versus-text benchmark; cite as an instrument for document handling,IBBench-Light: A Paired Evaluation of Task-Conditioned Responses to External Directives
arxiv:2609.14003,exclude,system level,title_abstract,,,"Confuse the Model, Control the Flow: Understanding and Mitigating Privacy Leakage from LLM Agents wi"
arxiv:2609.14987,exclude,system level,title_abstract,,,ActGuard: Pre-execution Action Auditing against Indirect Prompt Injection in LLM Agents
arxiv:2609.15017,exclude,detector only,title_abstract,,,"PIDS-Bench: Evaluating Prompt-Injection Detectors Under Over-Defense, Obfuscation, and Distribution "
arxiv:2609.15576,exclude,system level,title_abstract,,,Approval Integrity and Recovery in LLM Answer Publication
arxiv:2609.15906,exclude,system level,title_abstract,,,Authorization Architectures for Tool-Using AI Agents
arxiv:2609.15994,exclude,detector only,title_abstract,,,Latent Undertow: How Ordinary Typos Break Probes
arxiv:2609.16098,include,,full_text,,"full text: chain-of-thought system instruction, paraphrasing and self-reflection measured with tool filtering on ASB, with an ablation (Table 1)",Universal Defenses for Tool-Integrated LLM Agents Against Adversarial Attacks
arxiv:2609.17648,exclude,system level,title_abstract,,,Trust propagation and structural containment in Multi-agent LLM pipelines
arxiv:2609.18411,exclude,system level,title_abstract,,,The Verifiable Action Card: Trustworthy Human-in-the-Loop Control for Secure Autonomous Agents
arxiv:2609.18674,exclude,system level,title_abstract,,,CaMeLoT: CaMeL orchestrated with Temporal logic for static verification and liveness
arxiv:2609.19399,exclude,not about prompt injection,title_abstract,,,Efficient Nash Equilibrium Computation for Cybersecurity Games
arxiv:2609.19587,exclude,detector only,title_abstract,,,Red-Teaming Auto Mode: Improving Blocking Classifiers Against Malign Coding Agents
arxiv:2609.20188,include,,full_text,,"full text: ResumeShield: fenced data channel declared inert by the trusted instructions, measured alone in the ablation",ResumeShield: Channel Separation and an Open Benchmark for Indirect Prompt Injection in AI Resume Sc
arxiv:2609.20722,exclude,not about prompt injection,title_abstract,,,Deep Noir: Autonomous Steering Discovery via Architectural Chronometry in Transformer Models
arxiv:2609.21088,include,,title_abstract,,provenance-aware transformers,Origin Is All You Need: Provenance-Aware Transformers for Structural Trust-Boundary Separation
arxiv:2609.22510,include,,full_text,,full text: role: re-measurement; SecAlign re-tested against dormant conditional (explosive) prompts; own defence is a detector,Defusing Explosive Prompts: Understanding and Preventing Trigger-Based Prompt Injections in LLM Agen
arxiv:2609.22573,exclude,system level,title_abstract,,,Zero-Trust Authorization and Discovery for Enterprise MCP
arxiv:2609.22712,exclude,"survey, position or other",title_abstract,,,"Trustworthy Agentic AI: Failure Modes, Mitigation Strategies, and a Lifecycle Framework for Autonomo"
arxiv:2609.22949,exclude,system level,title_abstract,,,Beyond Single-Model Injection: A Threat Model and Defense Architecture for Prompt Injection in Multi
arxiv:2609.24446,exclude,system level,title_abstract,,,ActGov: Governing LLM Agent Actions via Policy-Constrained Validation
arxiv:2609.24801,exclude,detector only,title_abstract,,,Decoding Guardrails: XAI-Guided Perturbation Analysis of Prompt Injection Detection
arxiv:2609.26900,exclude,system level,title_abstract,,,Ajar: Measuring Open Privilege in Agent Defenses
arxiv:2609.27263,exclude,not about prompt injection,title_abstract,,,Specifying and Maintaining Agentic Workflows: An Empirical Study of GitHub Agentic Workflows
arxiv:2609.29130,exclude,attack only,title_abstract,,,ClaimMirage: When Self-Claims in Domain Names Change LLM Threat Judgments
arxiv:2609.29647,exclude,system level,title_abstract,,,AgentKernel: The Trust-Native Agentic Operating System
arxiv:2609.29775,exclude,attack only,title_abstract,,,Prefilling the Reasoning Channel: Output-Prefix Attacks on Reasoning LLMs
arxiv:2609.29948,exclude,attack only,title_abstract,,,ENDOPROMPT: Victim-Side Pseudo-References for Utility Degradation
arxiv:2609.31039,exclude,system level,title_abstract,,,MetaPermit: Scalable and Auditable Access Control for AI Agents via LLM-Inferred Meta-Attributes
arxiv:2609.32691,exclude,system level,title_abstract,,validity of evaluation harnesses for tool-call mediation; cite for evaluation conditions,Silent Failures in Agentic Security Evaluation: A Validated Harness for Tool-Call Mediation Under In
arxiv:2609.33102,exclude,benchmark only,title_abstract,,,ORBIT: A Framework for Multi-Agent Safety and Security Evaluations
arxiv:2609.34245,exclude,system level,title_abstract,,,Certified Multi-Source Integrity for Structured Agent Actions
arxiv:2609.34463,include,,title_abstract,,CoDeL co-evolutionary training defence,CoDeL: Co-Evolutionary Defense against Indirect Prompt Injection in LLM-based Agents
arxiv:2609.35932,exclude,attack only,title_abstract,,reserved-token versus text markers; evidence on the roles signal,"Same Bytes, Different Authority: Reserved-Token Representations in Chat-Template Prompt Injection"
arxiv:2609.36121,include,,title_abstract,,rendering data as an image before reading; an encoding-level change,Render Before Reading: Visual Rendering as a Prompt Injection Defense
arxiv:2609.36570,include,,title_abstract,,CounterSteer,CounterSteer: Suppressing Indirect Prompt Injection with Activation Steering
arxiv:2609.36603,exclude,system level,title_abstract,,,Self-Evolving Defense: Continual Security Policy Learning for LLM Agents
arxiv:2609.36817,exclude,benchmark only,title_abstract,,,pikit: A Composable Toolkit for Indirect Prompt Injection Research and Evaluation
arxiv:2609.37040,exclude,not about prompt injection,title_abstract,,,Selecting The Most Informative Tokens in Natural Language Autoencoders
arxiv:2609.37196,exclude,system level,title_abstract,,,ToolFence: Fine-Grained Authorization for Secure Tool-Using LLM Agents
arxiv:2609.39678,exclude,not about prompt injection,title_abstract,,,Aletheia: Permission-Minimality Testing for Coding-Agent Rules
arxiv:2610.00392,exclude,system level,title_abstract,,,Measuring LLM Agent Defenses by Layer: GDA Measurement and an Envelope-Layer Attack-Defense Model
arxiv:2610.00430,exclude,attack only,title_abstract,,,Memetic Trojans: Social Contagions as Carriers of Adversarial Payloads in Agent Networks
arxiv:2610.02413,exclude,detector only,title_abstract,,,Prompted to Discriminate: Generalizing Malicious-Input Probes in the Wild
arxiv:2610.02432,exclude,system level,full_text,,"full text: thesis; defences are committees of models, multi-layer filters and MCP call attestation",Evaluating and Improving the Robustness of Large Language Models to Input Sequence Variations
arxiv:2610.02861,exclude,system level,title_abstract,,,Containing the Autonomous Operator: A Defense-in-Depth Framework and Reference Architecture for Secu
arxiv:2610.03073,exclude,not about prompt injection,title_abstract,,,SecJev: Bringing Security Expertise to System One Decision Models
arxiv:2610.03089,exclude,system level,title_abstract,,,Securing Computer-Use Agents Against Branch Steering Attacks
arxiv:2610.03093,exclude,not about prompt injection,full_text,,"full text: synthetic planning environments; conflict between a latent goal channel and a text goal, no injected-data setting",LS-AR: Future-Predictive Latent Steering in Autoregressive LLMs
arxiv:2610.03434,exclude,system level,title_abstract,,,Persona Guardrail: A Production-Grade Defense Framework for Agentic Systems
arxiv:2610.03448,exclude,detector only,title_abstract,,,Passing the Test You Trained On: Re-evaluating Prompt-Injection Detectors for LLM Agents
arxiv:2610.04083,exclude,not about prompt injection,title_abstract,,,"Self-Propagating Misalignment in LLM Agents, and Why Auditing or Disabling Memory Is Not Enough"
arxiv:2610.04210,exclude,attack only,title_abstract,,absorption at unmarked seams; evidence for the no-signal condition,Can LLMs Separate Pasted Artifacts from User Speech? Absorption at Unmarked Prompt Seams
arxiv:2610.04269,include,,title_abstract,,self-reflection fine-tuning,Self-Reflection Fine-Tuning: Enhancing Agent Security against Prompt Injection Attacks from Failure
arxiv:2610.04412,exclude,not about prompt injection,title_abstract,,,Large Language Models and Augmented Democracy
arxiv:2610.04985,exclude,not about prompt injection,title_abstract,,,"Hidden Risks of Jev: An Empirical Study of Security, Privacy, and Dual Use"
arxiv:2610.05163,exclude,system level,title_abstract,,,Blocking at the Boundary: Auditing Long-Horizon Agents against Staged Prompt Injection
arxiv:2610.05266,exclude,attack only,title_abstract,,,Who Is Your Agent Serving? Provider-Side Indirect Prompt Injection in Proactive Agents
arxiv:2610.05295,include,,full_text,,full text: role: mechanistic analysis; activation edits along role directions reduce attack success on AgentDojo,Readable Before Actionable: Causal Tracing of Indirect Prompt Injection
arxiv:2610.06064,include,,full_text,,"full text: trust steering matrices learned with BiPO, evaluated on indirect prompt injection among other agent settings",TrustMI: Causally controlling how assistants trust their users
arxiv:2610.06401,include,,title_abstract,,RAISED,RAISED: Self-Distillation for Robustness to Prompt Injection in LLM Agents
arxiv:2610.06966,exclude,system level,title_abstract,,,APEX: Active Protection at Execution Boundaries for LLM Agents
arxiv:2610.07089,exclude,benchmark only,title_abstract,,,Towards a Unified Misuse Monitoring Benchmark
arxiv:2610.08678,exclude,not about prompt injection,title_abstract,,,Secure Speculative Decoding for Large Language Models
arxiv:2610.08773,include,,title_abstract,,AdvSim2Real training against adaptive injection,AdvSim2Real : Training Web Agents Against Adaptive Prompt Injection in a Web World Model
arxiv:2610.09935,exclude,detector only,title_abstract,,,AgentTracer: Tracing Indirect Prompt Injection Attack through Fine-Grained Intention-Execution Align
arxiv:2610.10742,exclude,attack only,title_abstract,,,BRANCH: Bypassing Multi-Scanner AI Guardrails
arxiv:2610.11634,include,,title_abstract,,LTBD learnable trust-boundary delimiters,LTBD: Learnable Trust-Boundary Delimiters for Prompt Injection Defense
arxiv:2610.12292,exclude,attack only,title_abstract,,,One Word Opens the Gate: The Option-Channel Attack on Typed Decision Models as Agent Guardrails
arxiv:2610.12524,exclude,attack only,title_abstract,,,An Exploration of Attacks on AI Agent Systems
arxiv:2610.14488,exclude,system level,title_abstract,,,Names Are Not Objects: Object-Bound Confinement for Compromised AI Coding Agents
arxiv:2610.15177,include,,full_text,,full text: CoDA: task policies enforced inside the denoising of diffusion language models,Only What You've Signed up For: Secure Workflows with Constrained Diffusion Agents
arxiv:2610.16015,exclude,not about prompt injection,title_abstract,,,Computational Reflective Equilibrium for Agentic AI: A Rawlsian Alignment Scaffold for Governance
arxiv:2610.16814,exclude,detector only,title_abstract,,judge defence framework,StableJudge: A Lightweight Adversarial Defense Framework for LLM-as-a-Judge
arxiv:2610.17301,exclude,detector only,title_abstract,,,"The Agent Knew: Internal States Forecast Unauthorized Actions and Stop Them, and the Lies That Follo"
arxiv:2610.17457,exclude,detector only,title_abstract,,,"BrowseSafe++: Benchmarking Prompt Injection Detection Across Web Contexts, Languages, and Browser Vi"
arxiv:2610.18408,exclude,detector only,title_abstract,,,"Prompt-Injection Probes Learn the Conversation, Not Just the Injection"
arxiv:2610.18457,exclude,system level,title_abstract,,,Kairos: Transactional Security Enforcement at the System Boundary for LLM Agents
arxiv:2610.19133,include,,title_abstract,,TRACER trace-regularised training,TRACER: Trace-Regularized Training Against Indirect Prompt Injection
arxiv:2610.19372,include,,full_text,,"full text: PVST: rejection-sampling fine-tuning and DPO of the agent from policy-verifier verdicts, evaluated on AgentDojo",Policy-Verified Self-Training: Learning Agent Compliance from Verifiable Policy Rewards
arxiv:2610.19422,include,,title_abstract,,AgentSafeEvo adversarial co-evolution training; check mechanism,AgentSafeEvo: Adversarial Co-Evolution for LLM Agent Safety
arxiv:2610.19452,exclude,detector only,title_abstract,,,Are You Reading the Right Thing? Rethinking Internal Signals for Prompt-Injection Detection
arxiv:2610.20776,exclude,benchmark only,title_abstract,,,INFUSE: A Unified Surface Benchmark for Indirect Prompt Injection in Agentic Systems
arxiv:2610.20907,exclude,detector only,title_abstract,,,From Detection to Decision: Calibration and Deferral for Prompt-Injection Detection
arxiv:2610.22594,include,,full_text,,full text: role: re-measurement; prompt-based instructions and a model trained to resist injection tested in medical multi-agent systems; own defence verifies clinical judgements,The Model Knows Better: Defending Medical Multi-Agent Systems Against Prompt Injection
arxiv:2610.22873,exclude,attack only,title_abstract,,,Stealthy Poisoning Attacks via Reused KV Cache of Reasoning
arxiv:2610.22874,include,,title_abstract,,per-instruction controlled compliance over external data,"Identify, Classify, Act: Per-Instruction Controlled Compliance for LLMs over External Data"
arxiv:2610.22987,exclude,attack only,title_abstract,,,From Approval to Abuse: Privilege Escalation Attacks on Harness-Protected Coding Agents
arxiv:2610.23254,include,,title_abstract,,role: re-measurement; ignored versus refused outcomes behind low attack success,"Ignored, Not Refused: What Low Prompt-Injection Success Rates Hide in Small Tool-Using Agents"
arxiv:2610.23307,exclude,benchmark only,title_abstract,,,MobileInject: Evaluating Reasoning-to-Action Security of MLLM-Based Mobile Agents under Cross-Modal
arxiv:2610.24048,exclude,detector only,title_abstract,,,The Window Is in the Reasoning: When LLM Agents Decide to Follow Prompt Injections
arxiv:2610.24277,exclude,system level,title_abstract,,,SPIDES: Information-Flow Control for Browser Agents
arxiv:2610.24999,exclude,"survey, position or other",title_abstract,,,SoK: PIAtlas: Decomposing the Unbounded Prompt Injection Attack Space
arxiv:2610.25937,exclude,attack only,title_abstract,,,RAGLeak: Supply-Chain Privacy Backdoors in Retrieval-Augmented Generation
arxiv:2610.26013,exclude,detector only,title_abstract,,,SpanGuard: Decoupling Instruction Localization from Authorization for Lightweight IPI Defense
arxiv:2610.26438,exclude,attack only,title_abstract,,,"Fake News, Real Crashes: Information Injection Attacks on LLM Financial Trading Agents"
arxiv:2610.27543,exclude,system level,title_abstract,,,KNOX: Hardware-assisted Regulatory Sandbox for Generative Models
arxiv:2610.28364,exclude,detector only,title_abstract,,,Context-Aware Prompt Injection Detection via Small Language Models in RAG Pipelines: nokast-secureRA
arxiv:2610.28425,exclude,benchmark only,title_abstract,,,After the Attacker Leaves: Benchmarking Carryover Attacks on LLM Agents
arxiv:2610.28810,exclude,attack only,title_abstract,,,Combining Long-Term Memory and LLMs Comes at a High Cost: From Prompt Injection to Persistent Compro
arxiv:2610.28988,exclude,detector only,title_abstract,,,Attention-based Classification for Indirect Prompt Injection
arxiv:2610.29379,exclude,not about prompt injection,title_abstract,,,Where AI Vulnerability Discovery Harnesses Go Quiet: Structural False Negatives and Hidden Vulnerabi
arxiv:2610.29747,exclude,attack only,title_abstract,,,Prompt Injection Through Forensic Evidence: A Taxonomy and Proof of Concept of Anti-Forensic Attacks
arxiv:2610.30193,exclude,"survey, position or other",title_abstract,,,Protection Is Not One Event: A Survey of RAG Protection Research by Measured Endpoint
arxiv:2610.30202,exclude,detector only,title_abstract,,,MPIDetector: Detecting Multi-modal Prompt Injection via LLM-based Reasoning
arxiv:2610.31974,exclude,system level,full_text,,full text: HMAC-signed provenance labels in the MCP layer against instructions decoded from images,Agentic AI Defense: Decode-Time Provenance Binding for MCP
arxiv:2610.33198,exclude,system level,title_abstract,,,"Deletion, Substitution, Repair and Transplant: Component Attribution in BARBICAN, a Layered Prompt-I"
arxiv:2610.33201,include,,title_abstract,,"role: measurement of signal dependence; position, role and label separated","Position, Role or Label? A Controlled Study of Instruction Conflicts in LLM Agents"
arxiv:2610.34535,exclude,benchmark only,title_abstract,,,SentryBench: A Cost-Sensitive Benchmark for Adversarial Robustness of Tool-Using LLM Agents in Publi
arxiv:2610.34777,exclude,attack only,title_abstract,,,Scope-Gap Injection: Exploiting Evaluator Channel Blind Spots in the OpenClaw Agent Security Competi
arxiv:2610.35451,exclude,attack only,title_abstract,,,"Then there is prompt injection, a malicious email, ...: The Wild Wide West of OpenClaw Agentic Syste"
arxiv:2610.36108,exclude,attack only,title_abstract,,,Log-To-Leak: Prompt Injection Attacks on Tool-Using LLM Agents via the Model Context Protocol
arxiv:2610.36422,exclude,not about prompt injection,title_abstract,,,What Victims Take Away: Attack Reuse and Defense Revision after Prompt-Injection Attacks
arxiv:2610.37657,include,,title_abstract,,Heads of Security,Heads of Security: Learned Attention Gates Improve Prompt Injection Robustness
arxiv:2610.37679,exclude,detector only,title_abstract,,,Exposure Is Not Commitment: Double-Stratified Evaluation of Latent Monitors for Tool-Using Agents
arxiv:2610.38120,include,,title_abstract,,LLMON post-training for instruction-data separation,Post-Training Large Language Models with LLMON
arxiv:2610.38867,exclude,attack only,title_abstract,,trust-region effect of injection position; evidence on the position signal,Where You Inject Matters: The Trust Region Effect in LLM Agents
arxiv:2610.39758,exclude,detector only,full_text,,full text: SEAL: learned soft prompt and readout that delete tool-output tokens before the task model reads them,SEAL: Sealed Queries for Local Sanitization against Indirect Prompt Injection
arxiv:2610.40744,exclude,system level,title_abstract,,,CLIC: Closed-Loop Intent Consistency for Securing LLM-Driven Control in Open RAN
arxiv:2610.41072,exclude,not about prompt injection,title_abstract,,,LanG 2.0: Energy-Efficient Agentic AI for Semi-Automated Security Operations at Scale
arxiv:2610.41181,exclude,not about prompt injection,title_abstract,,,AgentShield: Fortifying Web Applications Against AI Agents and Agentic Browsers
arxiv:2610.41307,exclude,image or audio channel only,full_text,,"full text: ERASE: on-policy self-distillation against instructions embedded in images, audio and video",Erase the Injected Region: Multimodal Prompt-Injection Defense with On-Policy Self-Distillation
arxiv:2610.41998,include,,title_abstract,,SecPO adversarial training,SecPO: Principled Adversarial Training for Prompt Injection Security
arxiv:2611.00725,exclude,attack only,title_abstract,,,The Long Con on the Web: Earn-then-Exploit Fraud Attacks against Coding Agents
arxiv:2611.00729,exclude,system level,title_abstract,,,AgentCap: Capability-Based Runtime Authorization for Secure LLM Agent Execution
arxiv:2611.01976,exclude,attack only,title_abstract,,,Beyond Prompt Injection: The Tool Update Gap in LLM Agent Security
arxiv:2611.02332,exclude,system level,title_abstract,,,LEANGUARD: Lean-Verified Guardrails for Tool-Using Agents
arxiv:2611.02605,exclude,system level,title_abstract,,,StateFence: Task-State Protection Against Evidence-Chain Contamination in Blue-Team LLM Agents
arxiv:2611.02741,exclude,system level,title_abstract,,,ATrust: Adaptive Trust Boundary Expansion for Securing LLM Agents
arxiv:2611.03555,exclude,attack only,title_abstract,,,MCP Tool Poisoning in Agentic AI Frameworks: A Cross-Framework Measurement Study
arxiv:2611.04271,exclude,not about prompt injection,title_abstract,,,From Guidelines to Practice: Defining Requirements for AI Red-Teaming Tools
arxiv:2611.04672,exclude,"survey, position or other",title_abstract,,,The Devil is in the Context: Nine Dimensions of Prompt Injection Attacks
arxiv:2611.04673,exclude,system level,full_text,,full text: AuthorityFlow: parameter-level authorisation of tool arguments by traced information flow; prompting defences appear only as baselines,"Right Tool, Wrong Parameters: Defending LLM Agents Against Parameter Poisoning"
arxiv:2611.06682,exclude,system level,title_abstract,,,SecureGARD: Dynamic Ontology-Guided Governance for Secure Tool-Using LLM Agents
arxiv:2611.07222,exclude,system level,title_abstract,,,SPILLWAY: Separating Target Admission from Sanitization to Defend Tool-Using Agents against Prompt I
arxiv:2611.07640,exclude,benchmark only,title_abstract,,,Beyond Instruction Override: Benchmarking Indirect Prompt Injection with Task-Tampering Attacks
arxiv:2611.08428,exclude,not about prompt injection,title_abstract,,,The Power of Meta Access in the Software of the Future --- Demonstrated on LLM Chat UIs and Chat AI
arxiv:2611.08811,exclude,attack only,title_abstract,,,"I Think, Therefore I Attack: Evaluating Thought Injection Attacks on Reasoning Agents"
arxiv:2611.08834,exclude,detector only,title_abstract,,,"Notwithstanding Anything to the Contrary, Ignore All Instructions: Measuring Prompt Injection Guards"
arxiv:2611.10026,exclude,benchmark only,title_abstract,,,MetaPI: Constructing Prompt Injection Benchmarks from LLM Agent Benchmark
arxiv:2611.10099,exclude,system level,title_abstract,,external harness with spotlighting among its layers,"Enmo Safety: External, Non-invasive, Model-agnostic Oversight for Off-the-Shelf Language Models"
arxiv:2611.10559,exclude,not about prompt injection,title_abstract,,,CompositionEvidence: Controlling Geometric Evidence for Cinematic Composition Reasoning with Frozen
arxiv:2611.10843,exclude,"survey, position or other",title_abstract,,,Agents of Chaos: A Systematization of Knowledge (SoK) on Malicious Attacks on Agentic AI Systems
arxiv:2611.11266,exclude,benchmark only,title_abstract,,,ErosionBench: Probing Alignment Decay in Enterprise Multi-Agent Systems
arxiv:2611.11305,exclude,system level,title_abstract,,,Least-Privilege Auditing of Agent Tool Calls with Intent Graphs and Reasoning-Guided Escalation
arxiv:2611.11506,exclude,"survey, position or other",title_abstract,,formal definitions,"On the Relative Security of Prompt Injection: Mechanised Definitions, Defences and Verdicts for LLM "
arxiv:2611.11571,include,,title_abstract,,role: re-measurement; in-prompt defences compared with deterministic enforcement,"Treating the Model as an Untrusted Component: Deterministic Policy Enforcement, Human Approval and T"
arxiv:2611.11861,exclude,attack only,title_abstract,,,Scaling Adaptive Prompt Injection Attacks with Agent Swarms
arxiv:2611.11971,exclude,system level,title_abstract,,,Harness Matters: Measuring and Dissecting Security Resilience in Web Agents
arxiv:2611.12521,exclude,not about prompt injection,title_abstract,,,Can the Confidence and Decisions of System 1 Models Be Trusted? An Explainability and Perturbation A
arxiv:2611.16122,exclude,system level,title_abstract,,,Intent-Aware Protection for AI Agents
arxiv:2611.16744,exclude,attack only,title_abstract,,,Scale-Induced Semantic Shift in Visual Prompt Injection Against VLMs and Web Agents
arxiv:2611.17115,exclude,system level,title_abstract,,,CapLite: Prompt-Injection Defense for Small Language Model Agents without Model-Written Security Art
arxiv:2611.20045,exclude,"survey, position or other",title_abstract,,,"Security of Deep Research Agents: Attack Surface, Defenses, Benchmarks, and Agenda Seeding against R"
arxiv:2611.20156,exclude,detector only,title_abstract,,,Risk-certified prompt optimization for LLM-as-a-judge guardrails
arxiv:2611.21822,exclude,"survey, position or other",title_abstract,,,Threat Landscape of Vibe Coding and the Road to Secure Autonomy
arxiv:2611.22223,exclude,attack only,title_abstract,,,GapLeak: When Privacy-Preserving LLM-based Web Agents Become a Privacy Risk
arxiv:2611.24185,exclude,system level,title_abstract,,,Guarding the Kingdom of Skill-Based Agents in the AI Dynasty of Computation: Control-Flow Integrity
arxiv:2611.24231,include,,title_abstract,,role: re-measurement; when rendering as images protects,What Survives the Shift from Text to Pixels? Conditional Protection against Prompt Injection in Imag
arxiv:2611.24362,include,,full_text,,full text: PIPETTE: adversarial training of agents by reasoning-guided RL with on-policy self-distillation on diverse red-team attacks,PIPETTE: Prompt Injection Purple-Teaming for Robust LLM Agents
arxiv:2611.26033,exclude,not about prompt injection,title_abstract,,,SEOYUN: Refusal-Inducing Exam Documents for AI-Glasses Pipelines
doi:10.69987/jacs.2023.30203,exclude,detector only,title_abstract,,,Continual Red-Teaming for In-the-Wild Jailbreaks via Online Guardrail Updates and Guardrail Distilla
doi:10.18372/1990-5548.78.18256,exclude,detector only,title_abstract,,,Filter for confidential information
doi:10.1227/neu.0000000000002415,exclude,not about prompt injection,title_abstract,,,On Chatbots and Generative Artificial Intelligence
doi:10.69980/redvet.v25i1.2487,exclude,system level,title_abstract,,,"""Design and Implementation of a Multi-Layered Framework for Detection and Defense against Logical At"
doi:10.5121/ijci.2024.130604,exclude,"survey, position or other",title_abstract,,,"AdversLLM: A Practical Guide To Governance, Maturity and Risk Assessment For LLM-Based Applications"
doi:10.56726/irjmets61937,exclude,"survey, position or other",title_abstract,,,"Adversarial Attacks on Large Language Models (LLMs) in Cybersecurity Applications: Detection, Mitiga"
doi:10.3126/nprcjmr.v1i8.73029,exclude,not about prompt injection,full_text,,"full text: general overview of adversarial machine learning in cybersecurity (evasion, poisoning, model inversion); prompt injection not studied",Al-Driven Cybersecurity: Mitigating Prompt Injection Attacks through Adversarial Machine Learning
doi:10.5281/zenodo.23870600,exclude,not about prompt injection,title_abstract,,,An Ethical and Secure Generative AI Deployment Model for Construction and Infrastructure Systems
doi:10.21203/rs.3.rs-4487194/v1,exclude,attack only,title_abstract,,,An Evaluation of the Safety of ChatGPT with Malicious Prompt Injection
doi:10.1109/bigdata62323.2024.10825267,exclude,attack only,title_abstract,,,AutoRed: Automated Attack Scenario Generation Framework for Red Teaming of LLMs
doi:10.69987/jacs.2024.40107,exclude,jailbreak or direct injection only,title_abstract,,,Behavior-Level Jailbreak Resistance via Multi-Stage Refusal + Utility Preservation
doi:10.1016/j.compag.2024.109493,exclude,not about prompt injection,title_abstract,,different sense of prompt injection,Chinese named entity recognition for agricultural diseases based on entity-related visual prompts in
doi:10.1109/icicn62625.2024.10761619,exclude,system level,title_abstract,,,Data Breach Prevention in AI Systems: Employing Event-Driven Architecture to Combat Prompt Injection
doi:10.1109/spw63631.2024.00017,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1109/spw63631.2024.00017 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Defending Language Models Against Image-Based Prompt Attacks via User-Provided Specifications
doi:10.64388/irev8i2-1722808,exclude,detector only,title_abstract,,,Detecting Prompt Injection in Retrieval-Augmented Generation Pipelines Using Lightweight Classifiers
doi:10.36227/techrxiv.171822345.56781952/v1,exclude,jailbreak or direct injection only,full_text,,"full text: moving-target defence (random parameter perturbation, re-initialisation, context changes); the attacks are typed straight into the model's input, with no third-party data channel",Dynamic Moving Target Defense for Mitigating Targeted LLM Prompt Injection
doi:10.30574/ijsra.2024.13.2.2415,exclude,not about prompt injection,title_abstract,,,EDGE-BASED LLM INFERENCE FOR SMART CITIES
doi:10.1109/tcset64720.2024.10755823,exclude,system level,title_abstract,,,Enhancing System Security: LLM-Driven Defense Against Prompt Injection Vulnerabilities
doi:10.2196/58275,exclude,not about prompt injection,title_abstract,,,Ensuring Appropriate Representation in Artificial Intelligence–Generated Medical Imagery: Protocol f
doi:10.4236/jsea.2024.171003,exclude,system level,title_abstract,,GUARDIAN,GUARDIAN: A Multi-Tiered Defense Architecture for Thwarting Prompt Injection Attacks on LLMs
doi:10.1109/msec.2023.3345568,exclude,"survey, position or other",title_abstract,,,Large Language Models and Security
doi:10.1109/tgrs.2024.3507784,exclude,not about prompt injection,title_abstract,,different sense of prompt injection,Lightweight and Efficient Multimodal Prompt Injection Network for Scene Parsing of Remote Sensing Sc
title:misuseofaithroughadversarialattacks,exclude,"survey, position or other",title_abstract,,,Misuse of AI Through Adversarial Attacks
doi:10.1109/vcris63677.2024.10813380,exclude,detector only,title_abstract,,,Optimizing Transformer Models for Prompt Jailbreak Attack Detection in AI Assistant Systems
doi:10.18413/2313-8912-2024-10-4-0-3,exclude,"survey, position or other",title_abstract,,,Prompt injection – the problem of linguistic vulnerabilities of large language models at the present
doi:10.1109/ictc62082.2024.10827351,exclude,detector only,title_abstract,,,Protection of LLM Environment Using Prompt Security
doi:10.26565/2519-2310-2024-2-05,exclude,not about prompt injection,title_abstract,,,RESEARCH ON THE CURRENT STATE AND PROSPECTS OF THE APPLICATION OF ARTIFICIAL INTELLIGENCE IN CYBERSE
title:securinguscriticalinfrastructurewithautonomouslanguageagentsatrustworthypolicyalignedframe,exclude,system level,title_abstract,,,"SECURING U.S. CRITICAL INFRASTRUCTURE WITH AUTONOMOUS LANGUAGE AGENTS: A TRUSTWORTHY, POLICY-ALIGNED"
doi:10.5281/zenodo.19218912,exclude,not about prompt injection,title_abstract,,,SecOps Automation in IoT-Enabled Enterprises: Leveraging Gen AI
doi:10.18653/v1/2024.findings-emnlp.346,exclude,benchmark only,title_abstract,,,SecureSQL: Evaluating Data Leakage of Large Language Models as Natural Language Interfaces to Databa
doi:10.4236/jsea.2024.175019,exclude,"survey, position or other",title_abstract,,,Security Vulnerability Analyses of Large Language Models (LLMs) through Extension of the Common Vuln
doi:10.1109/icimcis63449.2024.10957095,exclude,not about prompt injection,title_abstract,,,Sentiment Analysis Through Prompt Engineering with GPT-4 on the SmSA IndoNLU Dataset
doi:10.1109/delcon64804.2024.10866433,exclude,"survey, position or other",title_abstract,,,The Colossal Defense: Security Challenges of Large Language Models
doi:10.5863/1551-6776-29.2.200,exclude,not about prompt injection,title_abstract,,,The Potential Application of Large Language Models in Pharmaceutical Supply Chain Management
doi:10.1145/3711679,exclude,"survey, position or other",title_abstract,,,The Price of Intelligence: Three risks inherent in LLMs
doi:10.15662/ijeetr.2024.0606036,exclude,system level,title_abstract,,,"Trust Boundaries in the Model Context Protocol: Tool Poisoning, Server Impersonation, and Mitigation"
doi:10.4236/jsea.2024.171001,exclude,not about prompt injection,title_abstract,,,Whispered Tuning: Data Privacy Preservation in Fine-Tuning LLMs through Differential Privacy
doi:10.1109/acai68217.2025.11406667,exclude,system level,title_abstract,,,A Comprehensive Security Framework for the Model Context Protocol (MCP) in Multi-Agent AI Systems
doi:10.1109/fllm67465.2025.11391009,exclude,detector only,title_abstract,,,A Real-Time Framework for Prompt Injection Attacks Detection on Cloud-Hosted Large Language Models
doi:10.20944/preprints202509.0081.v1,exclude,"survey, position or other",title_abstract,,,"A Review of “Do Anything Now” Jailbreak Attacks in Large Language Models: Potential Risks, Impacts, "
doi:10.1109/aict67988.2025.11268746,exclude,system level,title_abstract,,,A Secure and Effective Framework for Key Concept Mining from Educational Content Using Large Languag
doi:10.1109/eleco69582.2025.11329228,exclude,attack only,title_abstract,,,A Study on Cross-Lingual Security Vulnerabilities in Large Language Models
doi:10.1145/3788149.3788209,exclude,not about prompt injection,title_abstract,,,A Survey of Adversarial Attacks on Visible-Infrared Object Tracking: Foundation Models and Cross-Mod
doi:10.56975/jetir.v12i12.573322,exclude,"survey, position or other",title_abstract,,,A Survey of Modern Adversarial Attacks and Defenses: From Foundational Methods to Generative AI and
doi:10.20533/ijrfidsc.2046.3715.2025.0029,exclude,"survey, position or other",title_abstract,,,A Threat Model for Indirect Prompt Injection in LLM-Integrated Applications
doi:10.30574/ijsra.2025.16.3.2588,exclude,"survey, position or other",title_abstract,,,A comprehensive analysis of jailbreak vulnerabilities in large language models
doi:10.1016/j.comnet.2025.111379,exclude,"survey, position or other",title_abstract,,,A comprehensive survey on large language models for multimedia data security: challenges and solutio
doi:10.1007/s11227-025-08146-1,exclude,"survey, position or other",title_abstract,,,A survey on privacy issues and mitigation strategies for LLMs in healthcare
doi:10.7862/ra.2025.5,exclude,"survey, position or other",title_abstract,,,AI-Powered Cyberattacks: A Comprehensive Review and Analysis of Emerging Threats
doi:10.5281/zenodo.18063677,exclude,detector only,title_abstract,,,APCE: Verifiable LLM Inference via Attention Manifold Invariants
doi:10.5281/zenodo.18079784,exclude,"survey, position or other",title_abstract,,,Adaptive Composition Attacks in AI-Integrated Systems A Conceptual Analysis of Emerging Cybersecurit
doi:10.20473/jisebi.11.3.473-487,exclude,detector only,title_abstract,,,Adaptive Multi‑Layer Framework for Detecting and Mitigating Prompt Injection Attacks in Large Langua
doi:10.1109/icscn67106.2025.11308512,exclude,detector only,title_abstract,,,Adaptive and Privacy-Preserving Detection of Evolving Prompt Injection Attacks in Large Language Mod
doi:10.3126/joeis.v4i1.81604,include,,full_text,,full text: claims adversarial retraining of LLMs combined with input sanitisation against direct and indirect injection; evaluation details thin,Advances in AI-Driven Cybersecurity: Tackling Prompt Injection Attacks through Adversarial Learning
doi:10.1007/978-981-96-9248-4_40,exclude,"survey, position or other",title_abstract,,,Adversarial Attacks on Large Language Models: A Survey
doi:10.56975/tijer.v12i6.158478,exclude,"survey, position or other",title_abstract,,,"Adversarial Landscapes: A Comprehensive Analysis of Vulnerabilities, Attacks, and Red Teaming in Mac"
doi:10.2139/ssrn.5362068,exclude,"survey, position or other",title_abstract,,,Adversarial Machine Learning and Generative Artificial Intelligence: Trust and Transparency Challeng
doi:10.64388/irev9i5-1711918,exclude,"survey, position or other",title_abstract,,,Adversarial Robustness and LLM Red Teaming: A Unified Review of Security Toolkits
doi:10.4018/979-8-3373-7133-7.ch001,exclude,"survey, position or other",title_abstract,,,Adversarial Threats and Vulnerabilities in Large Language Models
doi:10.1109/gaclm67198.2025.11231998,exclude,attack only,title_abstract,,,Adversarial and Multilingual Threats in Retrieval-Augmented Generation: From Prompt Injection to Mod
doi:10.1093/bib/bbaf631.004,exclude,attack only,title_abstract,,,Adversarial vulnerability assessment of vision language models for healthcare
doi:10.1109/icrcicn68210.2025.11364988,exclude,"survey, position or other",title_abstract,,,"Adversial Prompt Injection in Large Language Models: Taxonomy, Exploits, and Mitigation Frameworks"
doi:10.1016/j.mlwa.2025.100815,exclude,"survey, position or other",title_abstract,,,Algorithmic red teaming approaches to secure LLMs
doi:10.63575/cia.2025.30205,exclude,detector only,title_abstract,,,An Empirical Evaluation of Prompt Injection Detection and Refusal-Usefulness Tradeoffs Using the dee
doi:10.47297/wspciwsp2516-252718.20250902,exclude,"survey, position or other",title_abstract,,,Analysis of Jailbreak Attacks and Defenses for Large Language Models
doi:10.7256/2454-0714.2025.2.73939,exclude,detector only,title_abstract,,,Analysis of the impact of prompt obfuscation on the effectiveness of language models in detecting pr
title:analyzingandmitigatingpromptinjectionattacksinaidrivencloudapplications,exclude,system level,title_abstract,,,Analyzing and Mitigating Prompt Injection Attacks in AI-driven Cloud Applications
doi:10.5281/zenodo.17815969,exclude,"survey, position or other",title_abstract,,,Architecting Adversarial Resilience: A Systematic Red Teaming Framework for Anticipating Emergent AI
doi:10.36948/ijfmr.2025.v07i04.62200,exclude,"survey, position or other",title_abstract,,,"Artificial Intelligence and Cybersecurity: Documented Risks, Enterprise Guardrails, and Emerging Thr"
doi:10.52783/cana.v32.5267,exclude,"survey, position or other",title_abstract,,,Athreat Modeling Framework for LLM System Integration Leveraging NLP and Machine Learning
doi:10.1109/icca66035.2025.11431088,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1109/icca66035.2025.11431088 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Beyond Individual Defenses: Evaluating Defense Combinations Against Prompt Injection Attacks
doi:10.36227/techrxiv.175571972.28517288/v1,exclude,"survey, position or other",title_abstract,,,Beyond V2X: A Review of Large Language Models in In-Vehicle Autonomous Driving Systems
doi:10.6084/m9.figshare.30796943,exclude,not about prompt injection,title_abstract,,,Bezpieczne wykorzystanie AI w analizie danych organizacyjnych
doi:10.1109/wconf64849.2025.11233266,exclude,"survey, position or other",title_abstract,,,Breaking the Brain: Adversarial Attacks and the Fragility of Modern AI Models
doi:10.1109/iccit68739.2025.11491276,exclude,system level,title_abstract,,,CPS-Guard: Defensive Architectures for Securing Large Language Models in Cyber-Physical Systems
doi:10.54392/irjmt2523,exclude,detector only,title_abstract,,,Comprehensive Analysis of Machine Learning and Deep Learning models on Prompt Injection Classificati
title:conceptandimplementationofanautomatedpentestingapproachforlargelanguagemodels,exclude,attack only,title_abstract,,,Concept and implementation of an automated pentesting approach for large language models
doi:10.71238/snnst.v2i02.235,exclude,system level,title_abstract,,,Context-Aware GenAI Gateway Architecture for Secure Multi-Tenant Foundation Model Access in Enterpri
doi:10.1007/978-3-032-07884-1_13,exclude,jailbreak or direct injection only,title_abstract,,,Countering Jailbreak Attacks with Two-Axis Pre-detection and Conditional Warning Wrappers
doi:10.1109/icca66035.2025.11430791,exclude,system level,title_abstract,,,Cross-Agent Multimodal Provenance-Aware Framework for Robust Prompt Injection Defense in Large Langu
doi:10.1109/icccmla66092.2025.11580436,exclude,detector only,title_abstract,,,CrossGuard-VLM: A Robust Defense Framework for Vision-Language Models Against Multimodal Threats
title:cyberrisksingenerativeai,exclude,"survey, position or other",title_abstract,,,Cyber risks in Generative AI
doi:10.1109/southeastcon56624.2025.10971722,exclude,detector only,title_abstract,,,Cyberattacks on Large Language Models - Attack Detection and Architecture Adaptability
doi:10.2139/ssrn.5536400,exclude,"survey, position or other",title_abstract,,,"Cybersecurity and Adversarial Machine Learning: A Review of Threats, Defenses, and Architectural Con"
doi:10.20535/tacs.2664-29132025.2.343769,exclude,not about prompt injection,title_abstract,,,Cybersecurity of Intellectual Information Aggregation Processes into Digital Archives
doi:10.1007/978-3-032-07267-2_7,exclude,"survey, position or other",title_abstract,,,Data Privacy and Security in Large Language Models for Medical Fields
doi:10.31234/osf.io/zxuwf_v2,include,,full_text,,full text: prompting defence: the reviewing model is told to check the manuscript for hidden instructions; measured on three LLMs with 26 preprints,Defenses against LLM prompt injections in academic peer review
title:defensesagainstllmpromptinjectionsinacademicpeerreviewversionpostedonlinejuly172025,exclude,duplicate version,title_abstract,,same study as the OSF record of the same title,"Defenses against LLM prompt injections in academic peer review (Version posted online July 17, 2025)"
title:defensiveaiforcustomerservicechatbotsdetectingandmitigatingadversarialpromptinjections,exclude,detector only,title_abstract,,,Defensive AI for Customer Service Chatbots: Detecting and Mitigating Adversarial Prompt Injections
doi:10.3390/info16080652,exclude,not about prompt injection,title_abstract,,,Diagnosing Bias and Instability in LLM Evaluation: A Scalable Pairwise Meta-Evaluator
doi:10.1109/optim-acemp62776.2025.11075247,exclude,attack only,title_abstract,,,Disrupting Large Language Models with Hidden Prompt Injection Attacks Embedded in HTML Pages
doi:10.1016/j.patcog.2025.112606,exclude,not about prompt injection,title_abstract,,different sense of prompt,Dual-perception prompt learning: Illumination-adaptive and semantic-aware guidance for backlit image
doi:10.1609/aaai.v39i5.32591,exclude,not about prompt injection,title_abstract,,,Efficient Deformable Convolutional Prompt for Continual Test-Time Adaptation in Medical Image Segmen
title:embeddingbaseddeeplearningframeworkfordetectingpromptinjectionattacksinlargelanguagemodels,exclude,detector only,title_abstract,,,Embedding-Based Deep Learning Framework for Detecting Prompt Injection Attacks in Large Language Mod
doi:10.32604/jai.2025.069841,exclude,"survey, position or other",title_abstract,,,Enhancing Security in Large Language Models: A Comprehensive Review of Prompt Injection Attacks and
doi:10.5281/zenodo.18960763,exclude,not about prompt injection,title_abstract,,,Ep. 60: Single-Turn AI: The Interface Pattern Nobody's Talking About
doi:10.3390/make7030077,exclude,attack only,title_abstract,,,Evaluating Prompt Injection Attacks with LSTM-Based Generative Adversarial Networks: A Lightweight A
doi:10.34190/eccws.24.1.3515,exclude,attack only,title_abstract,,,Evaluating the Effectiveness of Psychological Prompt Injection Attacks on Large Language Models for
doi:10.48448/hz5w-js72,exclude,not about prompt injection,title_abstract,,,Evaluation of a Method to Detect Peer Reviews Generated by Large Language Models
title:evaluationoftheresilienceandcomplianceofllmsindsarmanagementacomparativeanalysisofsecurity,exclude,attack only,title_abstract,,,Evaluation of the Resilience and Compliance of LLMs in DSAR Management: A Comparative Analysis of Se
doi:10.1109/icecmsn68058.2025.11382707,exclude,detector only,title_abstract,,,Explainable Transformer-based Detection of Adversarial Prompts in Chatbots
doi:10.1109/lsp.2025.3643354,exclude,not about prompt injection,title_abstract,,different sense of prompt injection,Explicit-Implicit Prompt Injection and Semantic-Guided Latent LoRA for Vision-Language Tracking
doi:10.1109/dipca65051.2025.11042687,exclude,not about prompt injection,title_abstract,,,Fine-Tuning Multimodal Vision-Language Models for Brain CT Diagnosis via a Triple-Branch Framework
doi:10.5281/zenodo.21845543,exclude,not about prompt injection,title_abstract,,different sense of prompt injection,Frequent Subgraph Mining and Structural Prompt Injection for Large Language Model Fine-Tuning
doi:10.2139/ssrn.5517601,exclude,not about prompt injection,title_abstract,,,From Semantic Layers to NeuroAI Hippocampal Complex Architecture for Secure and Trustworthy LLMs
doi:10.1201/9781003739791-61,exclude,"survey, position or other",title_abstract,,,From exploitation to defence: Unmasking prompt injection in large language models
doi:10.1145/3711896.3736976,exclude,not about prompt injection,title_abstract,,different sense of prompt injection,Generalizable Graph Prompt Learning Framework with Model-level Prompt Injection and Two-Stage Prompt
doi:10.1002/9781394368532,exclude,"survey, position or other",title_abstract,,,"Generative AI Security: Defense, Threats, and Vulnerabilities"
doi:10.1093/oxfordhb/9780198940272.013.0035,exclude,"survey, position or other",title_abstract,,,Generative AI and Cybersecurity
doi:10.23919/fruct65909.2025.11008269,exclude,not about prompt injection,title_abstract,,,Ghosts in the Markup: Techniques to Fight Large Language Model-Powered Web Scrapers
doi:10.1109/cictn64563.2025.10932638,exclude,"survey, position or other",title_abstract,,,Hacking LLMs: A Technical Analysis of Security Vulnerabilities and Defense Mechanisms
doi:10.3390/engproc2025108025,exclude,not about prompt injection,title_abstract,,,Hands-On Training Framework for Prompt Injection Exploits in Large Language Models
doi:10.1109/iccit68739.2025.11491115,exclude,detector only,title_abstract,,,HiddenGuard: Real-Time Detection and Mitigation of Steganographic Prompt Injection Attacks in Multim
doi:10.1109/eit64391.2025.11103606,exclude,detector only,title_abstract,,,Hybrid Constitutional Classifiers for Prompt Injection Defense
doi:10.5281/zenodo.20011953,exclude,attack only,title_abstract,,,Indirect Prompt Injection in Browser-Using Agents: A Systematic Analysis from MCP to Host Compromise
doi:10.1201/9781003651963-9,exclude,not about prompt injection,title_abstract,,,"Infrastructure, Security, and Reliability"
doi:10.1109/punecon67554.2025.11378619,exclude,detector only,title_abstract,,,Input Moderation and Injection Filtering in Large Language Model via Llama Guard Integration
doi:10.29316/9788368103304_1,exclude,not about prompt injection,title_abstract,,,Insecure Output Handling in Large Language Models / Niebezpieczne przetwarzanie danych wyjściowych w
doi:10.5281/zenodo.17560906,exclude,not about prompt injection,title_abstract,,,It's the end of source code analysis as we know it (and we'll be fine)
doi:10.3390/app16010085,exclude,system level,title_abstract,,,LLM Firewall Using Validator Agent for Prevention Against Prompt Injection Attacks
doi:10.1109/gcat66372.2025.11368582,exclude,not about prompt injection,title_abstract,,,LLM-Guarded Clouds: Leveraging Generative AI for Proactive Threat Hunting and Adaptive Defense in Hy
doi:10.1145/3786709.3786720,exclude,detector only,title_abstract,,,LMDMI: A Lightweight Multilevel Defense against Malicious Inputs for Generative Language Models
doi:10.20944/preprints202509.0447.v1,exclude,attack only,title_abstract,,,LPCI: Defining and Mitigating a Novel Vulnerability in Agentic AI Systems
doi:10.54097/7ysr5k17,exclude,"survey, position or other",title_abstract,,,"Large Language Models for Cybersecurity Intelligence, Threat Hunting, and Decision Support"
doi:10.1016/j.iintel.2025.100154,exclude,not about prompt injection,title_abstract,,,Large multimodal model assisted underground tunnel damage inspection and human-machine interaction
doi:10.5281/zenodo.17960576,exclude,not about prompt injection,title_abstract,,,Lethe: Purifying Backdoored Large Language Models with Knowledge Dilution
doi:10.1016/j.eswa.2025.130621,exclude,not about prompt injection,title_abstract,,,MET-LLM: Enhancing large language models for malicious encrypted traffic detection
doi:10.52058/2786-6025-2025-12(53)-109-118,exclude,"survey, position or other",title_abstract,,,METHODS OF PREVENTING INFORMATION LEAKS CAUSED BY THE USAGE OF ARTIFICIAL INTELLIGENCE AGENT SYSTEMS
doi:10.1145/3746027.3755211,exclude,attack only,title_abstract,,,Manipulating Multimodal Agents via Cross-Modal Prompt Injection
doi:10.20535/tacs.2664-29132025.3.345389,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.20535/tacs.2664-29132025.3.345389 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Method of Counteracting Manipulative Queries to Large Language Models
doi:10.1109/access.2025.3583759,exclude,detector only,title_abstract,,,Meticulous Thought Defender: Fine-Grained Chain-of-Thought (CoT) for Detecting Prompt Injection Atta
doi:10.2139/ssrn.5476487,exclude,not about prompt injection,title_abstract,,,"Mitigating LLM Prompt Injection, Hallucination, and Security Risks using Semantically Encoded Compil"
doi:10.1109/iot-siu65919.2025.11402845,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1109/iot-siu65919.2025.11402845 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Mitigating Prompt Injection Attacks in ModelAgnostic Networks (MAN)
doi:10.1109/inspire67328.2025.11300550,exclude,"survey, position or other",title_abstract,,,Modern Day LLMs : A GOLDMINE or a LANDMINE ?
doi:10.3390/computers14080332,exclude,not about prompt injection,title_abstract,,,Multi-Layered Framework for LLM Hallucination Mitigation in High-Stakes Applications: A Tutorial
doi:10.2139/ssrn.5244151,exclude,detector only,title_abstract,,,Multilingual Prompt Injection Attacks Detection: Evaluating Adversarial Robustness in Large Language
doi:10.1109/icassp49660.2025.10890677,exclude,not about prompt injection,title_abstract,,,Multimodal and Multiple Prompts for Biology
title:navigatingtherisksandsafeguardsoflargelanguagemodelsllmsaddressingdataprivacysecurityandet,exclude,"survey, position or other",title_abstract,,,"Navigating the Risks and Safeguards of Large Language Models (LLMs): Addressing Data Privacy, Securi"
doi:10.31219/osf.io/682tm_v1,exclude,system level,title_abstract,,,NeuroShield: A Modular Immunological Architecture for Emotional LLM Systems
doi:10.1109/aiahpc66801.2025.11290591,exclude,"survey, position or other",title_abstract,,,New Paradigms of Adversarial Attacks Against Large Language Models and Their Defense Mechanisms
doi:10.33330/jurteksi.v12i1.4338,exclude,not about prompt injection,title_abstract,,,OPTIMIZING RETRIEVAL-AUGMENTED GENERATION FOR DOMAIN-SPECIFIC KNOWLEDGE SYSTEMS THROUGH FINE-TUNING
doi:10.1007/978-981-96-9994-0_8,exclude,jailbreak or direct injection only,title_abstract,,,PIDRCMPP: Rapid Multi-Strategy Hierarchical Jailbreak Attacks on LLMs
doi:10.18653/v1/2025.acl-long.1468,exclude,detector only,title_abstract,,PIGuard,PIGuard: Prompt Injection Guardrail via Mitigating Overdefense for Free
title:promptinjectionattacksonlargelanguagemodelsasystematicliteraturereview,exclude,"survey, position or other",title_abstract,,,PROMPT INJECTION ATTACKS ON LARGE LANGUAGE MODELS: A SYSTEMATIC LITERATURE REVIEW
doi:10.1109/incacct65424.2025.11011372,exclude,system level,title_abstract,,,Prevention of Prompt Injection Attacks Over Financial Applications Integrated with LLM
doi:10.1109/smartcloud66068.2025.00009,exclude,detector only,title_abstract,,,Privacy-Preserving Prompt Injection Detection for Smart Cloud-Deployed Large Language Models
doi:10.1109/idicaihei65991.2025.11378986,exclude,not about prompt injection,title_abstract,,,"Prompt Engineering for Large Language Models: Zero-shot, Few-shot, and Beyond"
doi:10.1109/asyu67174.2025.11208433,exclude,detector only,title_abstract,,,Prompt Injection Attack Detection with Machine Learning
doi:10.4018/979-8-3373-7133-7.ch005,exclude,"survey, position or other",title_abstract,,,Prompt Injection Attacks Risks and Defense Mechanisms
doi:10.32604/cmc.2025.074081,exclude,"survey, position or other",title_abstract,,"Geng et al., survey already assessed","Prompt Injection Attacks on Large Language Models: A Survey of Attack Methods, Root Causes, and Defe"
doi:10.48448/ty7z-3796,exclude,system level,title_abstract,,,Proxy Barrier: A Hidden Repeater Layer Defense Against System Prompt Leakage and Jailbreaking
doi:10.11606/d.55.2025.tde-25062026-163455,exclude,system level,title_abstract,,,RAG-BAC: a safe approach for retrieval augmented generation using role based access control
doi:10.37962/icydd/2025/23-24,exclude,detector only,title_abstract,,,REAL TIME AI DEFENSE AGAINST PROMPT INJECTION ATTACKS
doi:10.4018/979-8-3373-7133-7.ch006,exclude,"survey, position or other",title_abstract,,,Red Teaming and Stress-Testing Methodologies for LLM Security
doi:10.1109/raid67961.2025.00016,exclude,jailbreak or direct injection only,title_abstract,,,"Red-Teaming LLMs with Token Control Score: Efficient, Universal, and Transferable Jailbreaks"
doi:10.1007/978-3-031-80853-1_31,exclude,attack only,title_abstract,,,Retrieval Poisoning Attacks Based on Prompt Injections into Retrieval-Augmented Generation Systems t
doi:10.70777/si.v2i2.14741,exclude,"survey, position or other",title_abstract,,,Review: Safety at Scale: Comprehensive Survey of Large Model Safety
doi:10.5120/ijca2025925957,exclude,attack only,title_abstract,,,Robustness of Automated AI Agents Against Adversarial Context Injection in MCP
doi:10.28925/2663-4023.2025.29.918,exclude,"survey, position or other",title_abstract,,,"SECURITY OF LARGE LANGUAGE MODELS: RISKS, THREATS, AND SECURITY APPROACHES"
doi:10.1109/cain66642.2025.00047,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1109/cain66642.2025.00047 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Safeguarding LLM-Applications: Specify or Train?
doi:10.70777/si.v2i2.14609,exclude,"survey, position or other",title_abstract,,,Safety at Scale: A Comprehensive Survey of Large Model Safety
doi:10.12720/jait.16.2.243-250,exclude,system level,title_abstract,,,SafetyRAG: Towards Safe Large Language Model-Based Application through Retrieval-Augmented Generatio
doi:10.1109/iceconf65644.2025.11379484,exclude,system level,title_abstract,,,Secure AI Agents for Cloud Compliance in Adversarial Environments
doi:10.5281/zenodo.19431776,exclude,not about prompt injection,title_abstract,,,Secure AI Foundations for the Quantum Era
doi:10.36948/ijfmr.2025.v07i01.35371,exclude,system level,title_abstract,,,Secure Artificial Intelligence (SAI): A Dual-layer defence model against prompt injection and prompt
doi:10.1145/3795154.3795296,exclude,system level,title_abstract,,,SecureGov-Agent: A Governance-Centric Multi-Agent Framework for Privacy-Preserving and Attack-Resili
doi:10.1109/icccmla66092.2025.11581051,exclude,detector only,title_abstract,,,SecurePrompt – Detecting and Mitigating Prompt Attacks
doi:10.20944/preprints202510.2087.v1,exclude,"survey, position or other",title_abstract,,,Securing Agentic AI: A Comprehensive Threat Analysis of Model Context Protocol Systems with Layered
doi:10.1016/j.iotcps.2026.03.001,exclude,"survey, position or other",title_abstract,,,Securing LLM agents: From prompt sanitization to autonomous red teaming and beyond
doi:10.1109/access.2025.3623268,exclude,system level,title_abstract,,,Securing With Dual-LLM Architecture: ChatTEDU an Open Access Chatbot’s Defense
doi:10.1016/j.jisa.2025.104351,exclude,"survey, position or other",title_abstract,,,Securing large language models: A quantitative assurance framework approach
doi:10.1007/s42454-025-00085-9,exclude,not about prompt injection,title_abstract,,,Securing local LLMs for academic research: a human-system integration analysis and evolution of TAUC
doi:10.1109/siitme67657.2025.11293682,exclude,attack only,title_abstract,,,Security Posture Evaluation of Model Context Protocol Servers Against Prompt Injection
doi:10.1109/aiware69974.2025.00024,exclude,attack only,title_abstract,,,Security in the Wild: An Empirical Analysis of LLM-Powered Applications and Local Inference Framewor
doi:10.36227/techrxiv.175321762.23238855/v1,exclude,"survey, position or other",title_abstract,,,"Sensitive Data Extraction from Black Box Large Language Models: Attack Vectors, Vulnerabilities, and"
doi:10.1109/cyber-ai66431.2025.11233258,exclude,system level,title_abstract,,,ShadowPlay: Engineering Defenses Against Role-Based Prompt Injection and Dependency Hallucination in
doi:10.1109/access.2025.3598779,exclude,not about prompt injection,title_abstract,,,ShellBox: Adversarially Enhanced LLM-Interactive Honeypot Framework
doi:10.1016/j.procs.2025.08.292,exclude,attack only,title_abstract,,,Simulation of prompt injection attacks on generative pre-trained transformers models
doi:10.5281/zenodo.17787618,exclude,not about prompt injection,title_abstract,,,"Structured Contextual Distillation (SCD v3.1): A Deterministic, Vendor-Independent Protocol for Pers"
doi:10.30837/rt.2025.1.220.03,exclude,not about prompt injection,title_abstract,,,Study of the current state and prospects of artificial intelligence in cybersecurity
doi:10.34190/iccws.20.1.3292,exclude,duplicate version,title_abstract,,published version of arXiv 2410.23308,Systematically Analysing Prompt Injection Vulnerabilities in Diverse LLM Architectures
doi:10.17863/cam.129123,exclude,"survey, position or other",title_abstract,,,Technical and societal implications of machine learning security
doi:10.61173/390f5h97,exclude,"survey, position or other",title_abstract,,,The Comprehensive Review on Prompt Injection Attacks and Defense Mechanisms in Large Language Models
doi:10.1145/3769694.3771166,exclude,not about prompt injection,title_abstract,,,The Current State Of AI Education: Security As An Afterthought
doi:10.18653/v1/2025.emnlp-demos.55,exclude,duplicate version,title_abstract,,published version of arXiv 2507.14799,The Dangers of Indirect Prompt Injection Attacks on LLM-based Autonomous Web Navigation Agents: A De
doi:10.1109/iconat66879.2025.11362401,exclude,"survey, position or other",title_abstract,,,The Dark Side of AI: a Systematization of Knowledge on Jailbreaking and Prompt Injection in LLMs
doi:10.1109/icscss64956.2025.11500704,exclude,"survey, position or other",title_abstract,,,"The Role of Large Language Models in Cybersecurity: Strengths, Vulnerabilities, and Mitigation Strat"
doi:10.5281/zenodo.17918831,exclude,not about prompt injection,title_abstract,,,The Stepping Stone paradox Master Key (V10.2)
doi:10.5281/zenodo.17933459,exclude,detector only,title_abstract,,,Theme-Content Consistency Protocol: A Structural Approach to Prompt Injection Defense
doi:10.1145/3773365.3773631,exclude,"survey, position or other",title_abstract,,,Threats and Defenses for Large Language Models: A Survey
doi:10.13016/nuzd-aua8,exclude,"survey, position or other",title_abstract,,,Towards Private and Trustworthy Generative Models
doi:10.1109/ictest64710.2025.11042369,exclude,detector only,title_abstract,,,Towards Secure AI: Detection of Prompt Injection Attacks with Explainability
doi:10.5281/zenodo.20997139,exclude,not about prompt injection,title_abstract,,,"Trustworthy AI Starts Here: Aligning NIST AI RMF, Generative AI Profiles, and ISO/IEC 42001"
doi:10.5281/zenodo.17542015,exclude,not about prompt injection,title_abstract,,,UI-Based Defense Against Prompt Injection: From Gentle Guidance to Mandatory Re-education
title:ur1202generativeaicybersecurity,exclude,not about prompt injection,title_abstract,,,UR-1202 Generative AI & Cybersecurity
doi:10.54254/2755-2721/2025.tj22524,exclude,"survey, position or other",title_abstract,,,Vision-Language Model Security in Autonomous Driving: A Survey
doi:10.51983/ijiss-2025.ijiss.15.2.54,exclude,"survey, position or other",title_abstract,,,Vulnerabilities and Defenses: A Monograph on Comprehensive Analysis of Security Attacks on Large Lan
doi:10.1145/3769694.3771165,exclude,not about prompt injection,title_abstract,,,Weaponizing Words: Direct & Indirect Prompt Injection Attacks on LLM
doi:10.2139/ssrn.5049058,exclude,"survey, position or other",title_abstract,,,Web LLM Attacks: Unveiling the Future of Cyber Threats
doi:10.5281/zenodo.18240113,exclude,"survey, position or other",title_abstract,,,When AI Goes Rogue: Exposing the Hidden Threats in Language Models
doi:10.22541/au.175761889.90929465/v1,exclude,system level,title_abstract,,,Zero-Trust Proof-Gated Agent Architecture (ZPAA): A Comprehensive Security Model for the Model Conte
title:promptinjectionllm,exclude,attack only,title_abstract,,,"АТАКИ С ПОМОЩЬЮ ""PROMPT INJECTION"" В ГОЛОСОВЫХ LLM: ФОРМИРОВАНИЕ ОПАСНЫХ АУДИО-КОМАНД"
doi:10.5281/zenodo.17821810,exclude,not about prompt injection,title_abstract,,,🛡RHEA-Core Public Grant v2.0
doi:10.21227/frkt-sd48,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.21227/frkt-sd48 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,"""A Recovery-Oriented Defense Pipeline for Prompt Injection Attacks in Large Language Models"""
doi:10.5281/zenodo.18793442,exclude,not about prompt injection,title_abstract,,,"""MH8-Ai-Agent-Abe by Acbeatz.com"" Is an AI Auditor in a Class of His Own"
doi:10.21227/hdr1-cb12,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.21227/hdr1-cb12 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,"""PromptGame Dataset: Prompt Injection Attack Benchmark and Defense Evaluation Data"""
doi:10.5281/zenodo.20307701,exclude,no full text obtainable,full_text,"not retrieved: data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF",full text: not obtained by script or by hand,2383104532-oss/DCS-Lite: v1.0.0 - Initial Release
doi:10.5281/zenodo.24333533,exclude,not about prompt injection,title_abstract,,,5 Critical Threat Scenarios for Rogue AI Attacks on Europe's Critical Digital Infrastructure and Pos
doi:10.48448/n4hm-2t82,exclude,not about prompt injection,title_abstract,,,695 - Realism Control One-step Diffusion for Real-world Image Super Resolution
doi:10.5281/zenodo.21358083,exclude,not about prompt injection,title_abstract,,,8∞重合双环符号编码:通用人工智能认知基座与立维世界模型跃迁
doi:10.5281/zenodo.23556562,exclude,"survey, position or other",title_abstract,,,"A COMPARATIVE STUDY OF GENERATIVE ARTIFICIAL INTELLIGENCE MODELS: ARCHITECTURES, APPLICATIONS, SECUR"
doi:10.1109/icaic67076.2026.11395844,exclude,benchmark only,title_abstract,,,A Comprehensive Assessment Tool for Prompt Injection Attacks in Large Language Models (LLMs)
doi:10.1109/iwcmc69287.2026.11579902,exclude,system level,title_abstract,,,A Comprehensive Framework for Secure Multi-Agent Retrieval-Augmented Generation Systems
doi:10.54254/2755-2721/2026.gu33556,exclude,"survey, position or other",title_abstract,,,A Comprehensive Review of Threat Models in Federated AI Agents
doi:10.63766/spujstmr.26.000098,exclude,"survey, position or other",title_abstract,,,A Comprehensive Study of Cyber Attacks against Artificial Intelligence Systems
doi:10.1007/978-3-032-08859-8_1,exclude,"survey, position or other",title_abstract,,,A Comprehensive Study on Hybrid Injection Attacks in AI-Driven Applications
doi:10.2139/ssrn.5957238,exclude,"survey, position or other",title_abstract,,,"A Comprehensive Survey on Model Context Protocol: Architecture, Tool Integration, and the Future of "
doi:10.56201/ijssmr.vol.11no8.2025.pg534.557,exclude,not about prompt injection,title_abstract,,,A Conceptual Framework for Multi-Agent AI Quality Control in The Review of Regulated Documents
doi:10.1109/asens69964.2026.11605296,exclude,"survey, position or other",title_abstract,,,A Concise Review of Security Challenges and Protection for LLM-based Agents in Telecom Information S
doi:10.5753/sbseg_estendido.2026.33794,include,,title_abstract,,role: re-measurement; cross-lingual evaluation of trained defences in small models,A Cross-Lingual Security Evaluation of Prompt Injection Defenses in Small Language Models
doi:10.56979/1201/2026/1552,exclude,detector only,title_abstract,,,A Deep Learning Framework for Evaluating and Mitigating Prompt Injection in LLM Agents
doi:10.1109/noms69089.2026.11668221,exclude,system level,title_abstract,,,A Federated MCP Gateway: Enforceable Isolation and Measuring Residual Semantic Attacks
doi:10.3390/electronics15163739,exclude,system level,title_abstract,,,A Formal Framework of Architectural Intent Collapse for Tool-Level Attacks on LLM Agents
doi:10.1109/bigdatasecurity69476.2026.00009,exclude,"survey, position or other",title_abstract,,,A Four-Axis Systematization of AI and Cybersecurity in the Era of Large Language Models
title:aframeworkforlargelanguagemodelassistedinvestigationsofpromptinjectionattacks,exclude,not about prompt injection,title_abstract,,,A Framework for Large Language Model-Assisted Investigations of Prompt Injection Attacks
doi:10.1109/eespe68405.2026.11649278,exclude,system level,title_abstract,,,A Gateway-Centric Architecture for Bidirectional Content Moderation in Large Language Models
doi:10.5281/zenodo.21903320,exclude,detector only,title_abstract,,,A Guardrail That Failed Quietly - and Made Things Worse Where It Failed: Silent Ruleset Degradation
doi:10.1145/3786335.3813175,exclude,system level,title_abstract,,,A HIPAA-Compliant Architecture for Agentic Clinical AI Systems
title:ahybriddefenseagainstpromptinjectioninllmagentsevaluatingsecurityefficacyandcomputationalp,exclude,no full text obtainable,full_text,not retrieved: no identifier; search the title in a browser or the library catalogue,full text: not obtained by script or by hand,A Hybrid Defense Against Prompt Injection in LLM Agents : Evaluating Security Efficacy and Computati
doi:10.1109/gaiss66401.2026.11779683,exclude,detector only,title_abstract,,,A Hybrid Detection Architecture for Identifying Prompt Injection Attacks in LLM Systems
doi:10.1109/cyber-ai70573.2026.11755172,exclude,detector only,title_abstract,,,A Hybrid LLM Firewall for Direct Prompt Injection and Instruction Smuggling Defense
doi:10.2139/ssrn.7366618,exclude,detector only,title_abstract,,,A Hybrid Prompt Injection Defense Framework Using Rule-Based and Learned Detection with Policy-Drive
doi:10.5281/zenodo.19805847,exclude,"survey, position or other",title_abstract,,,A Layered Risk and Controls Framework for Prompt Injection in Enterprise AI Tooling
doi:10.5281/zenodo.22731103,exclude,system level,title_abstract,,,A Lightweight Defense Architecture Against Indirect Prompt Injection in Tool-Calling AI Agents
doi:10.5281/zenodo.21638983,exclude,detector only,title_abstract,,,A Lightweight Transformer Encoder for Prompt Injection Detection
title:alinguisticanalysisofpromptinjectioninlargelanguagemodels,exclude,"survey, position or other",title_abstract,,,A Linguistic Analysis of Prompt Injection in Large Language Models
doi:10.38124/ijisrt/26mar061,exclude,not about prompt injection,title_abstract,,,A Modular Multi-Agent Coordination Framework for Persistent Autonomous AI Assistants with Tool Orche
doi:10.21203/rs.3.rs-10302085/v1,exclude,detector only,title_abstract,,,A Multi-Agent Framework for Explainable Prompt Injection Detection in Large Language Models
doi:10.48448/bhz9-xx16,exclude,not about prompt injection,title_abstract,,,A Multi-Agent Framework for High-Interaction Terminal Simulation
doi:10.1109/aiita69518.2026.11567230,exclude,system level,title_abstract,,,A Multi-Agent Framework for Step-Level Tool-Call Safety Evaluation in Industrial Control Systems
doi:10.5281/zenodo.18723536,exclude,not about prompt injection,title_abstract,,,A Multi-Layered Governance Framework for Information Security in Cloud and AI-Driven Enterprises
doi:10.5220/0015351100004985,exclude,system level,title_abstract,,,A Multi-Layered Large Language Model Firewall: Defense-in-Depth against Prompt Injection
doi:10.1109/incect72370.2026.11742976,exclude,detector only,title_abstract,,,"A Multi-Stage Hybrid Framework for Prompt Injection Detection, Localization, and Recovery in LLM Sys"
doi:10.5281/zenodo.21762133,exclude,detector only,title_abstract,,,A Multi-Tier Hybrid Ensemble Architecture for Real-Time Defense Against Obfuscated Prompt Injections
doi:10.2139/ssrn.6898119,exclude,system level,title_abstract,,,"A Path-Dependent, Multi-Layered Architecture for Runtime Governance and Post-Quantum Cryptographic A"
doi:10.1109/icssit69151.2026.11656649,exclude,detector only,title_abstract,,,A Privacy Preserving Federated XLM-RoBERTa Framework for Multilingual Prompt Injection Defense in Ed
doi:10.3390/electronics15163554,exclude,detector only,title_abstract,,,A Privacy-Preserving Middleware Architecture for Detecting Prompt Injection and Sensitive Data Expos
doi:10.5281/zenodo.19570952,exclude,benchmark only,title_abstract,,,A Prompt Injection Attack Dataset and Prompt Collection for Evaluating Defense-in-Depth Security in
doi:10.1109/ciit68512.2026.11731549,exclude,detector only,title_abstract,,,A Prompt Injection Detection Model Addressing Over-Defense
doi:10.1007/978-3-032-20612-1_4,exclude,"survey, position or other",title_abstract,,,"A Review of Jailbreaking Large Language Models: Methods, Challenges, and Defences"
doi:10.1145/3801438.3803160,exclude,"survey, position or other",title_abstract,,,"A Review of Large Language Models for Cybersecurity Defense: Core Capabilities, Application Practice"
doi:10.36838/ijhsr86.99,exclude,"survey, position or other",title_abstract,,,A Review of Security Risks of Large Medical Models
doi:10.1109/cyber-ai70573.2026.11755183,exclude,"survey, position or other",title_abstract,,,A Structural Decomposition of Prompt Injection Threats Targeting Large Language Models
doi:10.48175/ijarsct-40581,exclude,not about prompt injection,title_abstract,,,A Study of Prompt Engineering Techniques for Improving Generative AI Performance
doi:10.57199/jgcr.2026.5.2.584,exclude,not about prompt injection,title_abstract,,,A Study on Attack and Defense Techniques for AI-based Web Hacking Systems
doi:10.57199/jgcr.2026.5.2.604,exclude,"survey, position or other",title_abstract,,,A Study on Data Integrity and Security Threat Analysis in Retrieval-Augmented Generation (RAG) Syste
doi:10.1109/bdai70753.2026.11655176,exclude,"survey, position or other",title_abstract,,,"A Survey of Fuzzing Techniques for Large Language Model Agents: Challenges, Methods, and Future Dire"
doi:10.2139/ssrn.7576261,exclude,"survey, position or other",title_abstract,,,"A Survey of Quantum-Proofing LLMs and Agentic AI: Security Threats, Post-Quantum Cryptography, and O"
doi:10.54691/3kzsns60,exclude,not about prompt injection,title_abstract,,,A Survey of Zero-Shot Sensitive Information Detection Techniques based on Large Language Models
doi:10.1109/icosics70095.2026.11746554,exclude,"survey, position or other",title_abstract,,,"A Survey on Security of the Model Context Protocol: Documented Incidents, Defense Frameworks, and Co"
doi:10.1109/access.2026.3729667,exclude,not about prompt injection,title_abstract,,,A Systematic Literature Review of Intelligent Anomaly Detection and Threat Analysis Using LLMs in Qu
doi:10.1109/access.2026.3700492,exclude,"survey, position or other",title_abstract,,"Woesle et al., survey already in the candidate list",A Systematic Literature Review of Prompt Injection Attacks in LLM-Integrated Systems
doi:10.1109/icseti67678.2026.11636629,exclude,"survey, position or other",title_abstract,,,A Systematic Review of Jailbreaking Attacks and Defense Mechanisms in Large Language Models
doi:10.1109/access.2026.3656849,exclude,"survey, position or other",title_abstract,,"Duarte et al., survey already in the candidate list","A Systematic Review of Prompt Injection Attacks on Large Language Models: Trends, Taxonomy, Evaluati"
doi:10.1109/bigdia71496.2026.11684636,exclude,"survey, position or other",title_abstract,,,A Systematic Survey of Prompt Injection Attacks and Defenses in LLM-Integrated Applications and Agen
doi:10.5281/zenodo.21637449,exclude,not about prompt injection,title_abstract,,,A Theory of Governed Co-Creation: The Epistemology of Human-AI Cybernetic Authorship
doi:10.5281/zenodo.23592687,exclude,attack only,title_abstract,,,"A Tiered, Adaptive Study of Indirect Prompt Injection Against a GitHub Triage Agent, and Where Agent"
doi:10.56578/ijcmem140207,exclude,system level,title_abstract,,,A Trusted Provenance and Trusted Execution Environment-Based Reference Architecture for Intent-Execu
doi:10.1007/978-3-032-32636-2_39,exclude,detector only,title_abstract,,"guardrail mechanisms, no abstract",A Unified Experimental Evaluation of Guardrail Mechanisms for LLM Agents
doi:10.5281/zenodo.21910585,exclude,"survey, position or other",title_abstract,,,"A Virological Model of Prompt Injection in Agentic AI: Within-Agent Replication, Latent Reservoirs, "
title:aasseguraoeosegurocomoimpulsionadoresdaadoodeiaagnticaemcontasapagar,exclude,not about prompt injection,title_abstract,,,A asseguração e o seguro como impulsionadores da adoção de IA agêntica em contas a pagar
doi:10.59256/ijire.20260705014,exclude,detector only,title_abstract,,,A lightweight and explainable framework for prompt injection detection in AI applications
doi:10.17605/osf.io/ep6ab,exclude,detector only,title_abstract,,,"A pre-registered, leak-free evaluation of activation-delta override detection in frozen LLMs, with a"
doi:10.1007/s10489-026-07270-6,exclude,not about prompt injection,title_abstract,,,A reinforcement learning decision-making method with multi-granularity semantic prompts from large l
doi:10.7717/peerj-cs.3773,exclude,"survey, position or other",title_abstract,,,"A systematic survey of LLM-based text-to-SQL: methodologies, security vulnerabilities, and future ch"
doi:10.1038/s41598-026-66312-8,exclude,not about prompt injection,title_abstract,,,A transport-layer cryptographic framework secures inter-agent communication and verdict provenance i
doi:10.1016/j.jss.2026.112782,exclude,attack only,title_abstract,,,A white-box prompt injection attack on embodied AI agents driven by large language models
doi:10.5281/zenodo.19209688,exclude,detector only,title_abstract,,,AASE: Activation-Based AI Safety Enforcement via Lightweight Probes
doi:10.17605/osf.io/bczej,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.17605/osf.io/bczej in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,ADAPTI-GUARD
doi:10.28925/2663-4023.2026.34.1275,exclude,system level,title_abstract,,,"ADVERSARIAL RESILIENCE OF AGENTIC INFRASTRUCTURE-AS-CODE SYSTEMS: THREAT MODEL, ATTACK TAXONOMY, AND"
doi:10.58532/nbennur3202c10,exclude,not about prompt injection,title_abstract,,,AGENTIC AI AND POST-QUANTUM CRYPTOGRAPHY
doi:10.1109/tce.2026.3707231,exclude,system level,title_abstract,,,AI 2 -Defense: An Autonomous Counter-AI Framework Against AI-Generated Threats in Consumer Systems
doi:10.5281/zenodo.24304779,exclude,not about prompt injection,title_abstract,,,"AI Agents and Autonomous Systems: Architecture, Reasoning, and Real-World Applications."
doi:10.17148/ijarcce.2026.156117,exclude,"survey, position or other",title_abstract,,,"AI Agents in Cybersecurity: A Systematic Review of Architectures, Applications, Threats, Challenges "
doi:10.55041/ijsmt.v2i5.353,exclude,"survey, position or other",title_abstract,,,"AI Alignment Challenges in Large Language Models: Technical Limitations, Risks, and Future Direction"
doi:10.5281/zenodo.21366084,exclude,"survey, position or other",title_abstract,,,"AI Alignment and Safety of Large Language Models: A Survey of RLHF, Constitutional AI, Red-Teaming, "
doi:10.5281/zenodo.21905814,exclude,not about prompt injection,title_abstract,,,AI BLACK BOX — Making AI Security Incidents Provable
doi:10.5075/epfl-thesis-12434,exclude,"survey, position or other",title_abstract,,,AI Control for Language Model Agents
doi:10.1007/978-3-032-25874-8_21,exclude,"survey, position or other",title_abstract,,,AI Cybersecurity for Tackling LLM Security
doi:10.5281/zenodo.22069396,exclude,not about prompt injection,title_abstract,,,AI DRIVEN CYBER THREATS AND DATA PRIVACY: A RESILIENT FRAMEWORK FOR DETECTING AND RESPONDING TO EMER
doi:10.2139/ssrn.7301221,exclude,not about prompt injection,title_abstract,,,"AI Data Governance Layer (ADGL): Governing Knowledge, Analysis, and Consequences in Model-Agnostic A"
doi:10.5281/zenodo.23193016,exclude,"survey, position or other",title_abstract,,,AI Jailbreaks: From Bypassing Restrictions to Governing Authority
doi:10.5281/zenodo.24074486,exclude,system level,title_abstract,,,AI Safety for Agentic AI in Europe Part III : Preventing Data Purpose Laundering and Tool Abuse with
doi:10.1002/9781394406456.ch04,exclude,"survey, position or other",title_abstract,,,AI Security Controls
doi:10.5281/zenodo.24039871,exclude,benchmark only,title_abstract,,,AI Security Testing Lab
doi:10.5281/zenodo.22074145,exclude,not about prompt injection,title_abstract,,,AI Security and Game Theory: Modeling Adaptive Attacker Best Responses
doi:10.5281/zenodo.18810310,exclude,"survey, position or other",title_abstract,,,"AI Security in the Era of Large Language Models: Red Teaming, Jailbreaking, and Prompt Injection"
doi:10.2139/ssrn.6126226,exclude,not about prompt injection,title_abstract,,,"AI Supply-Chain Pentesting: Attacking Models through Training, Updates, and Dependencies (Hacking AI"
doi:10.1002/9781394406456.ch03,exclude,"survey, position or other",title_abstract,,,AI Threats and Attacks
doi:10.1007/978-3-032-25874-8_19,exclude,"survey, position or other",title_abstract,,,AI for Cybersecurity in Model Context Protocols (MCPs) and AI Agents
doi:10.5753/sbseg_estendido.2026.33823,exclude,attack only,title_abstract,,,"AI, Fill This Form: A Simple and Effective Data Exfiltration Attack on LLM-Based Browser Extensions "
title:aiassistedsignalextractionandmisusedetectionininternetsystems,exclude,not about prompt injection,title_abstract,,,AI-Assisted Signal Extraction and Misuse Detection in Internet Systems
title:aidrivencybersecurityemergingthreatsanddefensivestrategieswithfocusoncollegeyouthindelhi,exclude,not about prompt injection,title_abstract,,,AI-Driven Cybersecurity: Emerging Threats and Defensive Strategies with Focus on College Youth in De
doi:10.5281/zenodo.22799231,exclude,not about prompt injection,title_abstract,,,AI-Mediated Organizational Decision-Integrity Threats: A Security Model for Normative and Evaluative
doi:10.55041/isjem06722,exclude,detector only,title_abstract,,,AIGuardXplore: Intelligent Model Vulnerability Inspector
doi:10.1109/ichms69701.2026.11602220,exclude,system level,title_abstract,,,ALTR: Adaptive Defense Framework Against Adversarial LLM Threats in Critical-Domain Applications
doi:10.18623/rvd.v23.n4.4344,exclude,not about prompt injection,title_abstract,,,API ECOSYSTEMS IN THE AGE OF ARTIFICIAL INTELLIGENCE
doi:10.1109/lsp.2026.3693589,exclude,not about prompt injection,title_abstract,,,APNet: Accurate Prompting Network With Modality Guidance and Structural Awareness for RGB-D Semantic
doi:10.5281/zenodo.20126741,exclude,not about prompt injection,title_abstract,,,ARCHER: A Mathematical Framework for AI Code Governance in the Agentic and Quantum Era
doi:10.48009/1_iis_2026_118,exclude,not about prompt injection,title_abstract,,,"ARCHITECTING THE FUTURE: A GOVERNED, AI-ENABLED PIPELINE FOR NEXT-GENERATION REQUIREMENTS ENGINEERIN"
doi:10.28925/2663-4023.2026.35.1430,exclude,not about prompt injection,title_abstract,,,ARCHITECTURAL APPROACHES TO APPLYING LANGUAGE MODELS FOR DETECTING PHISHING AND UNWANTED EMAILS
doi:10.3390/electronics15174007,exclude,system level,title_abstract,,,ARES: Securing Agents for Computer Use Through Endpoint Resource Mediation and Behavioral Guardrails
doi:10.1109/icassp55912.2026.11461745,exclude,detector only,title_abstract,,"detection, localisation and removal","ATTN-Defense: Attention-Guided Detection, Location and Removal for Indirect Prompt Injection"
doi:10.32604/jai.2026.092496,exclude,detector only,title_abstract,,,Accuracy-Invisible Safety Degradation in a Compressed On-Device Prompt-Injection Classifier
doi:10.5281/zenodo.20416742,exclude,attack only,title_abstract,,,Actuator Hijacking: Weaponizing the Autonomous SOC's Own Enforcement via Log-Borne Prompt Injection
doi:10.2139/ssrn.7292658,exclude,not about prompt injection,title_abstract,,different sense of prompt injection,Adapting Whisper for Hindi-English Code-Switched ASR: Prompt Injection vs. LoRA Fine-Tuning
doi:10.1016/j.ins.2026.123670,exclude,detector only,title_abstract,,,Adaptive DeSeTra: Adaptive deformable-span self-attention transformer for LLM security
doi:10.5281/zenodo.21631633,exclude,system level,title_abstract,,,Adaptive Deceptive Response Framework for Prompt Injection Attacks on Large Language Models
title:adaptivedefenseframeworkformitigatingpromptinjectionattacksinlargelanguagemodels,exclude,"survey, position or other",full_text,,full text: conceptual framework without experiments,Adaptive Defense Framework for Mitigating Prompt Injection Attacks in Large Language Models
doi:10.14711/thesis-hdl174867,exclude,system level,title_abstract,,,Adaptive Security for Complex Software Systems: Runtime Decision Making in Fuzzing and LLM Agents
doi:10.4218/etrij.2025-0449,include,,title_abstract,,AORT role-aware transformer retrofit,Adaptive orthogonal role‐aware transformer for robust instruction alignment
doi:10.32604/cmc.2026.078188,exclude,jailbreak or direct injection only,title_abstract,,corrected on rereading the abstract: a multi-agent in-context learning framework that classifies harmful user prompts; evaluated as harmfulness detection,Addressing Prompt Injection in Large Language Models via In-Context Learning
doi:10.47134/jtsi.v3i2.6047,include,,full_text,,full text: AdverShield-LLM: claims certified Gaussian noise at the token level of generation against indirect injection in IoT RAG,AdverShield-LLM: Adversarial Robustness Certification for IoT-Integrated Retrieval-Augmented Generat
doi:10.5772/intechopen.1016684,exclude,not about prompt injection,title_abstract,,,Adversarial AI Vulnerability in Secondary Education: Threats and Cybersecurity Competencies
doi:10.5281/zenodo.23584155,exclude,not about prompt injection,title_abstract,,,Adversarial Alignment Collapse: How Human Game-Theoretic Intuition Penetrates LLM Risk-Aversion and
doi:10.5281/zenodo.21388473,exclude,attack only,title_abstract,,,Adversarial Answer Engine Optimization - by Akash Narayan
doi:10.1007/978-3-032-23347-9_2,exclude,"survey, position or other",title_abstract,,,Adversarial Attacks on Large Language Model Applications: Prompt Injection and Token Drain Exploitat
doi:10.1007/978-981-95-4452-3_17,exclude,"survey, position or other",title_abstract,,,Adversarial Attacks on Large Language Models: The Prompt Injection Approach
doi:10.5281/zenodo.23689493,exclude,"survey, position or other",title_abstract,,,Adversarial Controllability of LLM Agent Systems
doi:10.5281/zenodo.23955945,exclude,"survey, position or other",title_abstract,,,Adversarial Machine Learning and Prompt Injection in LLM-Driven Phishing Detection: A Systematic Lit
doi:10.1002/9781394402069,exclude,"survey, position or other",title_abstract,,,"Adversarial Machine Learning: Mechanisms, Vulnerabilities, and Strategies for Trustworthy AI"
doi:10.70593/978-93-7185-561-7_10,exclude,"survey, position or other",title_abstract,,,"Adversarial Risks in AI: Poisoning, Evasion and Inference"
doi:10.5281/zenodo.24149117,exclude,system level,title_abstract,,,Adversarial Robustness of LLM Agents in Clinical Decision Support: A Threat-Model-Driven Layered Def
doi:10.2139/ssrn.6112326,exclude,not about prompt injection,title_abstract,,,"Adversarial Swarm Intelligence for Epistemic Defense: Architecture, Implementation, and Preliminary "
doi:10.6084/m9.figshare.32016498,exclude,"survey, position or other",title_abstract,,,Adversarial Threats to AI-Augmented Security Operations: A Systematic Survey
doi:10.5281/zenodo.21736575,exclude,"survey, position or other",title_abstract,,,Adversarial Vulnerabilities in Large Language Models: A Critical Review and an Adaptive Multi-Layere
doi:10.1201/9781003644231-4,exclude,"survey, position or other",title_abstract,,,Adversarial intelligence
doi:10.1016/j.cpes.2026.05.003,exclude,system level,title_abstract,,,Adversarial threats in LLM-augmented decision support for cyber-physical energy system with wildfire
doi:10.5281/zenodo.19761508,exclude,system level,title_abstract,,,Aegis Cortex OS: Governed Execution Layer for Enterprise AI Agents
doi:10.5281/zenodo.21214715,exclude,not about prompt injection,title_abstract,,,"Against the Inner Foundation Dogma, Selfhood, Will, and the Method of Relational Testing"
doi:10.5281/zenodo.21932811,exclude,"survey, position or other",title_abstract,,,Agent Data Injection — Formal Definition of a New AI Attack Class
doi:10.5281/zenodo.23123351,exclude,not about prompt injection,title_abstract,,"visualisation demo, not a study",Agent Defense Radar
doi:10.5281/zenodo.23129663,exclude,system level,title_abstract,,,"Agent Execution Control Planes: When 'Sandbox', 'Gateway', and 'Policy' Ship as Unauthenticated Code"
doi:10.5281/zenodo.19561359,exclude,system level,title_abstract,,,Agent Security Framework
doi:10.5281/zenodo.19428926,exclude,not about prompt injection,title_abstract,,,Agent Skills Are the New Apps
doi:10.1145/3832165,exclude,attack only,title_abstract,,,AgentBreaker: Evaluating Context-Aware Indirect Prompt Injection Risks in Modern Web Agents
doi:10.5281/zenodo.20692312,exclude,benchmark only,title_abstract,,,AgentDojo-PROV: W3C PROV-O Corpora of LLM Agent Executions
doi:10.2139/ssrn.6589479,exclude,detector only,title_abstract,,,AgentForensics: Exploring the Real-Time Prompt Injection Detection and Forensics Threats in LLM Agen
doi:10.5281/zenodo.23814757,exclude,not about prompt injection,title_abstract,,,AgentPROV corpora v0.1.0: the collection index
doi:10.5281/zenodo.20756085,exclude,benchmark only,title_abstract,,,"AgentShield Bench v2: Evaluating Memory Security, Persistent Jailbreaks, and Cross Session Compromis"
doi:10.5281/zenodo.22259021,exclude,system level,title_abstract,,,AgentShield: A Zero-Trust Runtime Guardrail Architecture for Autonomous Multi-Agent AI Systems with
doi:10.2139/ssrn.6926219,exclude,attack only,title_abstract,,,Agentic AI Cyber Subversion: The Semantic Layer Integrity Attack as a New Threat Class Against the R
doi:10.2139/ssrn.7580899,exclude,"survey, position or other",title_abstract,,,Agentic AI Meets Network Digital Twins: A Survey of Trustworthy Autonomous Cyber Defense and Network
doi:10.1109/gaiss66401.2026.11779686,exclude,detector only,title_abstract,,,Agentic AI Red-Teaming for Prompt Injection Detection in Multi-Agent Systems
doi:10.5281/zenodo.24340700,exclude,not about prompt injection,title_abstract,,,"Agentic AI Security for Europe: After Fugaku and Google's Autonomous Hacking Agent, Why Valid Creden"
doi:10.5281/zenodo.20731134,exclude,"survey, position or other",title_abstract,,,"Agentic AI Systems as a Persistent Attack Surface: How Memory Poisoning, Skill Injection, Trace Leak"
doi:10.68854/95fpkq71,exclude,"survey, position or other",title_abstract,,,"Agentic AI Systems: A Taxonomy of Autonomy, Delegation and Accountability for Tool-Using Language-Mo"
doi:10.62311/nesx/rb5jy-978-81-689653-4-8,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.62311/nesx/rb5jy-978-81-689653-4-8 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Agentic AI TRiSM with Sparse-Autoencoder Interpretability for Secure Tool-Use Policy Learning and In
doi:10.5281/zenodo.22992489,exclude,"survey, position or other",title_abstract,,,Agentic AI Threats and Security Vol 2
doi:10.3390/electronics15122740,exclude,"survey, position or other",title_abstract,,,"Agentic AI and Large Language Models for Autonomous IoT Cybersecurity: A Systematic Survey, Taxonomy"
doi:10.5281/zenodo.19474181,exclude,"survey, position or other",title_abstract,,,"Agentic AI for Cybersecurity: Threats, Defense Mechanisms, and Future Challenges"
doi:10.3390/network6040084,exclude,not about prompt injection,title_abstract,,,Agentic AI-Driven SOC-as-a-Service for Optimizing Incident Response in Cloud Environments: A Concept
doi:10.22214/ijraset.2026.85609,exclude,"survey, position or other",title_abstract,,,Agentic AI: A Review and Proposed Framework
doi:10.2139/ssrn.7697398,exclude,not about prompt injection,title_abstract,,,"Agentic Finance: Building, Training and Governing Autonomous Agents in Financial Markets"
doi:10.65737/airjir2026369,exclude,"survey, position or other",title_abstract,,,Agents of Context: A Methodological Critique and Counter-Evidence Analysis of Adversarial Red-Teamin
doi:10.5281/zenodo.19445085,exclude,not about prompt injection,title_abstract,,,Agentskills.io Spec: From Broken YAML to Production Skills
doi:10.5281/zenodo.21959861,exclude,not about prompt injection,title_abstract,,,"Agreement Is Not Validation: Sycophancy, Prompt Injection, and the Limits of an Accuracy-Based Verif"
doi:10.47392/irjaeh.2026.0719,exclude,"survey, position or other",title_abstract,,,Ai-Powered Attacks and Mitigation Strategies for Enterprises: A Comprehensive Research Review
doi:10.24818/baswp.2026.05,exclude,not about prompt injection,title_abstract,,,Ai-powered fraud detection and prevention
doi:10.2139/ssrn.7662038,exclude,system level,title_abstract,,,Air-gapped Agents: A Containment-first Architecture for Orchestrating Heterogeneous Frontier-model A
doi:10.5281/zenodo.18902320,exclude,not about prompt injection,title_abstract,,,Algorithmic Cowardice: Cognitive Dissonance and Moral Conformity in Multi-Agent LLM Interactions
doi:10.2139/ssrn.6591478,exclude,"survey, position or other",title_abstract,,,"Alignment of Generative Models (LLMs): Impacts of RLHF on Security, Governance, and Corporate Innova"
doi:10.5281/zenodo.20681556,exclude,benchmark only,title_abstract,,,AltayDuel: A Turkish-First Arena and Open Dataset for Multi-Turn LLM Prompt-Injection Red-Teaming
doi:10.59256/indjcst.20260502116,exclude,not about prompt injection,title_abstract,,,An AI-Driven Secure Interpreter for Adaptive and Security- Aware Program Execution
doi:10.1109/scecs69812.2026.11566062,exclude,detector only,title_abstract,,,An Edge–Cloud Collaborative Framework for Latency Optimization in Prompt Injection Defense
doi:10.5281/zenodo.19364461,exclude,benchmark only,title_abstract,,,An Empirical Benchmark and Security Evaluation of Prompt Injection Attacks in LLM Systems
doi:10.5281/zenodo.23628772,exclude,system level,title_abstract,,,An Empirical Evaluation of Attack and Defense Dynamics in Enterprise Agentic LLM Pipelines
title:anempiricalstudyofparameterefficientfinetuninginvisionlanguagemodelsafetypipelines,exclude,no full text obtainable,full_text,not retrieved: no identifier; search the title in a browser or the library catalogue,full text: not obtained by script or by hand,An Empirical Study of Parameter-Efficient Fine-Tuning in Vision-Language Model Safety Pipelines
doi:10.22214/ijraset.2026.85566,exclude,detector only,title_abstract,,,An Explainable Graph Attention Network for Indirect Prompt Injection Detection in RAG Systems
doi:10.1109/icosics70095.2026.11746820,exclude,detector only,title_abstract,,,An Intelligent Prompt Injection Detection System Using Large Language Models and Contextual Analysis
doi:10.2139/ssrn.7216863,exclude,not about prompt injection,title_abstract,,,An LLM-Based Multi-Agent Framework for Autonomous Business Lead Intelligence Using Public Web Data
doi:10.5281/zenodo.22990381,exclude,not about prompt injection,title_abstract,,,Anatomical Review of Reasoning Trace Extraction from Proprietary LLM APIs
doi:10.1002/9781394402069.ch2,exclude,"survey, position or other",title_abstract,,,Anatomy of AI Systems and Their Attack Surfaces
doi:10.5281/zenodo.21764731,exclude,not about prompt injection,title_abstract,,,Anti-Prompt Strategy; Generative Artificial Intelligence; Creative Methodology; Algorithm Critique;
doi:10.1145/3828183.3828472,exclude,not about prompt injection,title_abstract,,,Application of Intelligent Construction Technology for Teaching Resources Integrating Knowledge Grap
doi:10.1051/itmconf/20269304014/pdf,exclude,"survey, position or other",title_abstract,,,"Application, Security Risks and Defense Strategies of Large Language Models in Cybersecurity"
doi:10.54254/2753-8818/2026.es37007,exclude,"survey, position or other",title_abstract,,,"Applications, Safety Challenges and Future Directions of AI Agents in Cardiovascular Prediction: A R"
doi:10.5281/zenodo.23601939,exclude,system level,title_abstract,,,"Approve, Don't Disclose: Phone-Anchored Credential Release for Browser-Using AI Agents"
doi:10.1109/itc-egypt70422.2026.11765853,exclude,detector only,title_abstract,,,ArabGuard: A Multi-Layered Framework for Detecting Prompt Injection in Egyptian Dialect and Franco-A
doi:10.17605/osf.io/qe7kr,exclude,not about prompt injection,title_abstract,,,Architect: Four Pillar AI Logic & 256-Bit Character Validation ML
title:architecturaldefensesforagenticsystemspromptinjectionandbeyond,exclude,system level,title_abstract,,,Architectural Defenses for Agentic Systems: Prompt Injection and Beyond
doi:10.21203/rs.3.rs-10962146/v1,include,,title_abstract,,role: re-measurement; validity of injection benchmarks across models,Are LLM-Agent Prompt-Injection Benchmarks Still Valid? A Cross-Model and Cross-Benchmark Study
doi:10.5281/zenodo.23877365,exclude,benchmark only,title_abstract,,,Argus-Bench: A Prompt-Injection Benchmark for AI Agent Gateways
doi:10.2139/ssrn.6870178,exclude,system level,title_abstract,,,"Ariadne: Adaptive, Human-in-the-Loop Defense for AI Agents against Indirect Prompt Injection"
doi:10.5281/zenodo.20399449,exclude,attack only,title_abstract,,,"Artifacts for ""MUZZLE: Adaptive Agentic Red-Teaming of Web Agents Against Indirect Prompt Injection "
doi:10.5281/zenodo.20436934,exclude,duplicate version,title_abstract,,artifact record of the LogInject study already screened,Artifacts for: Context Contamination in LLM Analysis of Network Security Logs (LogInject-1.0)
doi:10.5281/zenodo.21715924,exclude,"survey, position or other",title_abstract,,,"Artificial Intelligence for Cybersecurity: Threats, Defenses, and Emerging Challenges in the Era of "
doi:10.5281/zenodo.21622138,exclude,not about prompt injection,title_abstract,,,"Artificial Intelligence for Information Security Principles, Applications, and Hands-On Practice"
doi:10.70593/978-93-7185-561-7,exclude,not about prompt injection,title_abstract,,,Artificial Intelligence in Cybersecurity and Risk Management
doi:10.5281/zenodo.23280587,exclude,"survey, position or other",title_abstract,,,"Artificial Intelligence in Cybersecurity: Defensive Applications, Offensive Exploitation, Adversaria"
doi:10.1109/ijcnn69854.2026.11729999,exclude,system level,title_abstract,,model-agnostic guardrail architecture,Assessing Defenses Against Prompt Injection Attacks in LLM Systems
doi:10.5281/zenodo.19324243,exclude,system level,title_abstract,,,"Assured Intelligence Systems: A Governed Architecture for Reliable, Auditable, and Controllable Agen"
doi:10.65286/icic.v22i2.97272,exclude,attack only,title_abstract,,,Att2RAG: A Double-Condition Framework for Knowledge Poisoning Attacks on RAG Systems
doi:10.5281/zenodo.20772941,exclude,"survey, position or other",title_abstract,,,"Attack Surface Proliferation in Agentic AI Systems: A Threat-Model-First Synthesis of Injection, Imp"
doi:10.5281/zenodo.20592362,exclude,"survey, position or other",title_abstract,,,"Attack Surface Proliferation in Agentic AI Systems: How Tool Injection, Memory Poisoning, Speculativ"
doi:10.5281/zenodo.20525492,exclude,"survey, position or other",title_abstract,,,"Attack Surface Proliferation in Agentic AI: How Tool Calls, Memory Persistence, Skill Ecosystems, an"
doi:10.5281/zenodo.20476297,exclude,not about prompt injection,title_abstract,,,"Attack Test Scenarios and Results Data for ""Security Controls and Guardrails for Autonomous AI Agent"
doi:10.5281/zenodo.24171174,exclude,system level,title_abstract,,,"Attack, Harden, Re-Attack: code, data and results"
doi:10.2139/ssrn.7399538,exclude,attack only,title_abstract,,,"Attacks That Score, Attacks That Transfer: Guardrail Asymmetry, the Confused Deputy, and Hidden-Eval"
doi:10.5281/zenodo.20582430,exclude,no full text obtainable,full_text,"not retrieved: record removed from Zenodo; search for the paper by title, otherwise it stays not retrieved",full text: not obtained by script or by hand,Attention Is Not Enough: Constraint Routing Failure and Governed Multi-Stream Attention
doi:10.21203/rs.3.rs-11162232/v1,exclude,attack only,title_abstract,,propagation measurement,"Attenuation, Not Amplification: Measuring Indirect Prompt Injection Propagation and Containment in M"
doi:10.5281/zenodo.19749422,exclude,not about prompt injection,title_abstract,,,Auburn Governance Stack: Master Architecture Plan — 45-Document Layered Architecture for Verifiable
doi:10.5281/zenodo.23315811,exclude,not about prompt injection,title_abstract,,,Auditor OS / V50-S Sentinel Core: Universal Thermodynamic Homeostasis & Epistemic Multi-Anchoring
doi:10.1109/ccwc67433.2026.11393740,exclude,system level,title_abstract,,,Authenticated and Offline-Verifiable Agent-to-Agent Messaging for LLM Agents
doi:10.5281/zenodo.20541040,exclude,"survey, position or other",title_abstract,,,Authority Before Action: A Conditional Failure-Chain Framework for Tool-Using AI Safety
doi:10.1016/j.eij.2026.101029,exclude,not about prompt injection,title_abstract,,,Automated security-centric refactoring of PowerShell commands using large language models: A Python-
doi:10.1007/978-3-032-26836-5_12,exclude,attack only,title_abstract,,,Automatic Security Testing of System Prompts Against Prompt Injection Attacks
doi:10.5281/zenodo.18449573,exclude,"survey, position or other",title_abstract,,,Autonomous AI Agent Networks: Comprehensive Threat Analysis and Risk Quantification
doi:10.5281/zenodo.23589575,exclude,system level,title_abstract,,,"Autonomous AI Agent Security, Protected Execution-Finality, Micro-VM Isolation, and Hardware-Enforce"
doi:10.5281/zenodo.23340239,exclude,not about prompt injection,title_abstract,,,"Autonomous AI-Driven Security Operations Centers (Autonomous SOC): Agentic Orchestration, Real-Time "
doi:10.1109/icaira72375.2026.11769094,exclude,"survey, position or other",title_abstract,,,"Autonomous Cyber Intelligence: A Systematic Evidence Synthesis, Multidimensional Taxonomy, and Resea"
doi:10.5281/zenodo.22718544,exclude,system level,title_abstract,,,"Autonomous Micro-Enterprise Security (AMES): Threat-Bounded State Transitions, Zero-Trust Ingress Ve"
doi:10.32628/cseit261213109,exclude,not about prompt injection,title_abstract,,,Autonomous Security Alert Triage Using LLM Based Agentic Investigation with Tool Augmented Reasoning
doi:10.24963/ijcai.2026/68,exclude,detector only,title_abstract,,BERM,BERM: Low-Overhead Prompt-Injection Detection via In-Situ Benign Representation Modeling
doi:10.26583/bit.2026.5.09,exclude,detector only,title_abstract,,,BILINGUAL PROMPT INJECTION ATTACK CLASSIFIER BASED ON A TRANSFORMER LANGUAGE MODEL AND HEURISTIC FEA
doi:10.1109/ubmk71847.2026.11745123,exclude,detector only,title_abstract,,,BPI-Guard: A Bilingual Prompt Injection Detector for Turkish and English
doi:10.1109/access.2026.3712744,exclude,detector only,title_abstract,,,BSM: A Browser-Resident Framework for Real-Time Detection of JavaScript API Abuse and Prompt Injecti
doi:10.5281/zenodo.19073968,exclude,not about prompt injection,title_abstract,,,Backbone Omniscience Attack: Measuring Information Leakage in Multi-Agent AI Negotiations
doi:10.1007/978-981-92-2014-4_6,exclude,attack only,title_abstract,,,Backdooring Societal Bias into Instruction-Tuned Large Language Models Via Virtual Prompt Injection
doi:10.3390/app16157662,include,,full_text,,"full text: Spotlight-Guard: spotlighting measured on its own and inside the layered system, with a component ablation (Tables 3 and 5)","Balancing Security and Performance in LLM Agents: Spotlight-Guard, a Layered Defense Against Indirec"
doi:10.5281/zenodo.21199146,exclude,not about prompt injection,title_abstract,,,Behavioral Identity Is the Missing Layer in Agent Security - Continuous model-fingerprint monitoring
doi:10.3390/app162010412,exclude,"survey, position or other",title_abstract,,"Yumusak et al., survey already in the candidate list","Below the Waterline: A Survey of Agentic AI Security, Shadow AI Governance, and LLM Supply-Chain Ris"
doi:10.1007/978-3-032-18474-0_18,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1007/978-3-032-18474-0_18 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Benchmarking Defense Techniques for Securing Large Language Models
doi:10.13140/rg.2.2.11046.28482,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.13140/rg.2.2.11046.28482 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Benchmarking Prompt Injection Defenses in Production LLM Systems
doi:10.64751/ijdim.2026.v5.n2.pp131-137,exclude,detector only,title_abstract,,,Benchmarking Prompt Injection Detection for Web Agents
doi:10.5281/zenodo.22147871,exclude,benchmark only,title_abstract,,,Benchmarking Prompt-Injection Robustness of Tool-Using LLM Agents
doi:10.17148/ijarcce.2026.15843,exclude,"survey, position or other",title_abstract,,,Beyond Accuracy: Identifying the Stealth Gap in Multimodal Prompt Injection Defenses via a Four‑Dime
doi:10.5281/zenodo.20789430,exclude,"survey, position or other",title_abstract,,results withdrawn by the authors' erratum,Beyond Attack Success Rate: Measuring Operator-Facing Transparency in LLM Agent Security
doi:10.5281/zenodo.18790843,exclude,not about prompt injection,title_abstract,,,Beyond Distribution: Empirical Validation of Entity-Level Defense in Multi-Model AI Decision Protect
doi:10.5281/zenodo.19343033,exclude,system level,title_abstract,,,Beyond Identity Governance: A Protocol-Level Security Testing Framework for Multi-Agent AI Systems
doi:10.5281/zenodo.18795216,exclude,system level,title_abstract,,,Beyond Prompt Filters: Structural Risk Gating for Agentic LLM Safety
doi:10.21203/rs.3.rs-10798245/v1,exclude,system level,title_abstract,,,Beyond Prompt Injection: Trust-Boundary Security Assurance for LLM-Integrated and Agentic Applicatio
doi:10.1002/9781394370047.ch06,exclude,not about prompt injection,title_abstract,,,Beyond Prompts: Training for Healthy Safeguards
doi:10.1109/aaiml67890.2026.11498175,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1109/aaiml67890.2026.11498175 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Beyond Shared Embedding Spaces: Towards LLM Architectures with Explicit Authority Control for Prompt
doi:10.1109/icimia69818.2026.11744796,exclude,detector only,title_abstract,,render-aware hidden-text detector,"Beyond Text: A Render-Aware, Semantically-Fused Defence Against Indirect Prompt Injection in Enterpr"
doi:10.5281/zenodo.23755370,exclude,duplicate version,title_abstract,,code release of the render-aware detector paper,"Beyond Text: Render-Aware, Semantically-Fused Defence Against Indirect Prompt Injection"
doi:10.5281/zenodo.18150446,exclude,not about prompt injection,title_abstract,,,Beyond Tool and Agent: Emergent Properties in Long-term Human-AI Interaction
doi:10.5281/zenodo.23341871,exclude,not about prompt injection,title_abstract,,,Beyond the Flag: Yearly Cybersecurity Project Ideas for Secondary Students
doi:10.13140/rg.2.2.11898.45767,exclude,not about prompt injection,title_abstract,,,Beyond the Gatekeeper: A Framework for AI-Assisted Open Peer Review and Decentralized Scientific Eva
doi:10.5281/zenodo.23059925,exclude,not about prompt injection,title_abstract,,,Beyond the Prompt: A Dual-Layer Framework for AI-Resistant Assessment in English Literature Educatio
doi:10.5281/zenodo.21790909,exclude,not about prompt injection,title_abstract,,,Beyond the Stochastic Black Box: Formally Verified Hybrid Architecture for the Safety and Alignment
doi:10.5281/zenodo.23114295,exclude,system level,title_abstract,,,Bounded Execution State Machines for Multi-Agent Tool Hijacking Prevention
doi:10.13140/rg.2.2.22438.18242,exclude,attack only,title_abstract,,,"Breaking Instruction Hierarchies: A Real-World Case Study of Prompt Injection, Role Escalation, and "
doi:10.5281/zenodo.22206826,exclude,not about prompt injection,title_abstract,,,Breaking the Localhost Illusion: An Empirical Security and Privacy Analysis of Hardware-Integrated W
doi:10.1007/s10015-026-01152-3,exclude,system level,title_abstract,,,Bringing the Dual-LLM pattern to practice in Google ADK for deterministic AI agent security against
doi:10.5281/zenodo.24214722,exclude,not about prompt injection,title_abstract,,,"Building Agentic AI Systems: A Practical Guide to Architectures, Design Patterns, and Implementation"
doi:10.3390/socsci15020119,exclude,not about prompt injection,title_abstract,,,Building Safe AI Chatbots for Rural Mothers Seeking Breastfeeding Support: Understanding Hallucinati
doi:10.5281/zenodo.23646207,exclude,not about prompt injection,title_abstract,,,Bảo vệ ACI-OS: Từ một website an toàn đến một hệ sinh thái số bền vững
doi:10.5281/zenodo.24308068,exclude,system level,title_abstract,,,C.O.R.E.: A Low-Cost Deterministic Multi-Agent Security Protocol for Edge LLMs in Resource-Constrain
doi:10.2139/ssrn.7578318,exclude,system level,title_abstract,,,CAPE: Provenance-aware Semantic Authorization for Agentic Payments
doi:10.1109/tcsvt.2026.3731775,exclude,not about prompt injection,title_abstract,,,CDPformer: Learning Common and Differential CLIP Prompts for Unified Day-Night Image Dehazing
doi:10.5281/zenodo.20122528,exclude,detector only,title_abstract,,,CEDA-1K: Command-Execution Defense Assessment (1000 entries)
doi:10.1016/j.patcog.2026.113661,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1016/j.patcog.2026.113661 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,CM-PIUG: Cross-modal prompt injection unified modeling and game-theoretic defense strategies
doi:10.23939/csn2026.02.056,exclude,detector only,title_abstract,,,COMPARATIVE EVALUATION OF MACHINE LEARNING METHODS FOR PROTECTING LARGE LANGUAGE MODELS FROM PROMPT
doi:10.5281/zenodo.20934112,exclude,detector only,title_abstract,,,CPOS-H as a Layered Pre-LLM Execution Gate
doi:10.5281/zenodo.23994904,exclude,not about prompt injection,title_abstract,,,Cambridge Analytica 2.0: How Enterprise AI Can Rebuild Company Secrets From Data It Was Allowed to S
doi:10.5281/zenodo.24081730,exclude,attack only,title_abstract,,,"Can the Other Side Steer Your AI Lawyer? Prompt Injection, Framing and Record Manipulation in AI Leg"
doi:10.2139/ssrn.7257458,exclude,system level,title_abstract,,,Capability Confinement for Compromised Autonomous Agents
doi:10.2139/ssrn.7505299,exclude,system level,title_abstract,,,Capability-Mediated Perimeters for Secure AI Agent Tool Execution: Conditional Non-Escalation Invari
doi:10.1109/iccea69962.2026.11743566,exclude,system level,title_abstract,,,CarrierGuard-Java: Carrier-Aware Zero-Trust Code Ingestion Defense for LLM-Assisted Static Analysis
doi:10.5281/zenodo.23958794,exclude,detector only,title_abstract,,,"CascadeGuard: Hardening Phishing Detection Against Character-Level Obfuscation, Semantic Paraphrasin"
doi:10.20944/preprints202610.2580.v1,exclude,system level,title_abstract,,,Certified Tool Execution Framework for MCP-Based AI Agents: Provably Secure Defense Against Tool Poi
doi:10.20944/preprints202608.1961.v1,exclude,system level,title_abstract,,,"ChainGuard: A Layered, Stateful, and Provenance-Aware Runtime Security Architecture for Prompt Injec"
doi:10.1201/9781003771654-15,exclude,"survey, position or other",title_abstract,,,Challenges and Limitations in LLMs
doi:10.55813/gaea/revistacec/v3/n2/2,exclude,not about prompt injection,title_abstract,,,Ciberseguridad predictiva basada en inteligencia artificial contra ataques generativos
doi:10.5281/zenodo.18641295,exclude,not about prompt injection,title_abstract,,,"Closed-Loop Architecture for Ethical AI Training via Pre-Filtered Conversations, Active Memory, and "
doi:10.1109/etcc69750.2026.11681999,exclude,system level,title_abstract,,,Cloud Security Risks and Defense Architecture for Agentic AI Systems in Enterprise Environments
doi:10.1007/s00521-026-12470-9,exclude,not about prompt injection,title_abstract,,,Cloud security architecture of large language models based on digital power grid knowledge base
doi:10.5281/zenodo.21534660,exclude,system level,title_abstract,,,Cloud-Native Adversarial Machine Learning: A Serverless Cybersecurity Architecture for Neutralizing
doi:10.5281/zenodo.23043476,exclude,not about prompt injection,title_abstract,,,"Code and Experiments for ""Slow and Injectable LLM Supervisors Break Control Barrier Function Safety "
doi:10.5281/zenodo.23972864,include,,full_text,,"full text: role: re-measurement; extended report with results: benign fine-tuning erodes Meta-SecAlign, SecAlign and LocalAlign, and 250 security examples repair them (main paper still to be located)","Code, data and extended report for ""Defended in Training, Eroded in Deployment: Benign Fine-Tuning U"
doi:10.5281/zenodo.23613810,exclude,detector only,title_abstract,,,CodeObf-PI v1.0: measuring and repairing the encoding shortcut in lightweight prompt-injection detec
title:coffeeplexfinetunediterationofnvidiapersonaplex7bv1forvariedcustomerroleswithlora,exclude,not about prompt injection,title_abstract,,,CoffeePlex: Fine-tuned Iteration of NVIDIA PersonaPlex-7b-v1 for Varied Customer Roles with LoRA
doi:10.2139/ssrn.7127098,exclude,not about prompt injection,title_abstract,,,Cognitive Drift in Generative AI: Self-Normalized Displacement as a Measurement Principle for Semant
doi:10.5281/zenodo.22173128,exclude,system level,title_abstract,,,Cognitive Harvard Architectures for AI Agent Perimeter Defense: Resolving the Confused Deputy Proble
doi:10.5281/zenodo.19974366,exclude,attack only,title_abstract,,,Cognitive Layer Poisoning via Multi-Agent Interaction: A Novel Attack Vector Against AI-Assisted Fin
doi:10.5281/zenodo.18902365,exclude,detector only,title_abstract,,no results reported,Coherence-Based Defense 2.1
doi:10.5281/zenodo.22150297,exclude,not about prompt injection,title_abstract,,,Coherence-Induced Dimensionality Constriction: How Dense Context Bypasses RLHF Alignment via Latent
doi:10.7759/s44389-026-00329-y,exclude,detector only,title_abstract,,,Comparative Analysis of Transformer Models for Prompt Injection Detection in Autonomous Large Langua
doi:10.3390/info17020155,exclude,detector only,title_abstract,,,Comparative Benchmarking of Deep Learning Architectures for Detecting Adversarial Attacks on Large L
doi:10.5281/zenodo.19729303,exclude,not about prompt injection,title_abstract,,,Comparative Red-Teaming of Constitutional AI vs RLHF Guardrails: Social Modulation Under Sustained S
doi:10.1007/s10207-026-01264-8,exclude,detector only,title_abstract,,,Comparative evaluation of machine learning methods for protecting LLMs from prompt injection attacks
doi:10.1007/978-3-032-27927-9_23,exclude,system level,title_abstract,,,Compliance-First RAG with Inference Isolation: Embedding Choice and Prompt-Injection Robustness in a
doi:10.17605/osf.io/9qh7n,exclude,attack only,title_abstract,,,Composing Malice: Empirical Measurement of Compositional Prompt Injection Vulnerability in Agentic L
doi:10.3346/jkms.2026.41.e281,exclude,not about prompt injection,title_abstract,,,Comprehensive Consideration of Ethics in AI-assisted Scientific Writing and Peer Review
doi:10.1109/icosics70095.2026.11746808,exclude,detector only,title_abstract,,,Confidence-Guided Heterogeneous Detection for Jailbreak and Prompt Injection Attacks in Large Langua
doi:10.1109/icssa70263.2026.11765792,include,,title_abstract,,constitutional sanitisation with fine-tuning for log analysis; check which part is measured,Constitutional Sanitization and Fine-Tuning for Robustness to Prompt Injection in LLM-Based Log Anal
doi:10.5281/zenodo.23391600,exclude,system level,title_abstract,,,Constraining Tool Authority in AI Agents: A Risk-Tiered Framework for Mitigating Indirect Prompt Inj
doi:10.21203/rs.3.rs-10196595/v1,exclude,system level,title_abstract,,,Containment over Detection: Cryptographic Boundary Enforcement for Prompt Injection Defense in Agent
doi:10.5281/zenodo.21855812,exclude,system level,title_abstract,,,"Context References Over Payloads: Authority Scoped to the Predicate, Not the Principal"
doi:10.1109/ijcnn69854.2026.11729723,exclude,attack only,title_abstract,,,Context-Construction-Based Prompt Injection in Document-Centric LLM Decision Systems
doi:10.5281/zenodo.19784777,exclude,system level,title_abstract,,,ContextForge: Architecture-First Persistent Agentic Memory via Model Context Protocol — Multi-Trigge
doi:10.61424/hewtat90,exclude,system level,full_text,,"full text: tool filtering, self-verification, a learned action gate and a distilled guardrail around the agent",Continual Red-Teaming and Guardrail Distillation for Tool-Using LLM Agents: Prompt-Injection Resista
doi:10.5281/zenodo.20584943,exclude,system level,title_abstract,,,"Contract-Bound Cognitive Routing: A Control Architecture for Agentic AI Integrity, Delegation, and P"
doi:10.2139/ssrn.6645520,exclude,detector only,title_abstract,,,ContrastShield: A Contrastive Fine-Tuning Approach for Robust Prompt Injection Detection in LLM Pipe
doi:10.5281/zenodo.22013134,exclude,system level,title_abstract,,,CortexGate: Step-Level Fine-Tuned Risk Routing and Deterministic Safety Gating for Agentic Pipelines
doi:10.5281/zenodo.19712045,exclude,not about prompt injection,title_abstract,,,Covert Narcissism Syntax Lockdown Theory: Analysis of the AI Bias-108 × Bug-108 Chain and Its Terms-
doi:10.5281/zenodo.21735802,exclude,benchmark only,title_abstract,,detection benchmark of PDFs,CrackedPDFs: Paper v1 Dataset and Reproducibility Artifacts
doi:10.5281/zenodo.23778321,exclude,no full text obtainable,full_text,"not retrieved: data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF",full text: not obtained by script or by hand,Cross Lingual Indirect Prompt Injection in RAG: Evaluating Layered Defenses on Urdu and Roman Urdu
doi:10.1186/s42400-026-00628-w,exclude,attack only,title_abstract,,,Cross-layer contagion of prompt injections in multi-agent swarms: a multiplex microscopic markov cha
doi:10.23919/nisscps00007.2026.00018,exclude,"survey, position or other",title_abstract,,,Cyber AI: A Review of Artificial Intelligence for Offensive and Defensive Web Security
doi:10.5281/zenodo.20583585,exclude,"survey, position or other",title_abstract,,,Cyber Attacks Targeting Generative AI and Agentic AI Frameworks in the Retail Domain and Strategies
doi:10.68104/ijasit.v1.i4.44,exclude,"survey, position or other",title_abstract,,,"Cyber Defence, Agentic AI, and the Sociotechnical Paradigm: A Survey of AI-Driven Cybersecurity"
doi:10.70593/978-81-6632-104-7,exclude,not about prompt injection,title_abstract,,,Cyber Security and Ethical Hacking
doi:10.1201/9781003642350-12,exclude,"survey, position or other",title_abstract,,,Cyber Security for AI
doi:10.1109/iconica70605.2026.11736648,exclude,system level,title_abstract,,,Cyber security Integration Framework using Multi-Agent Reinforcement Learning for Agent OS : with Ad
doi:10.28945/5852,exclude,not about prompt injection,title_abstract,,,CyberAgent: An Agentic AI Framework for Autonomous Security Operations with Reduced Operational Cost
doi:10.5281/zenodo.20624751,exclude,"survey, position or other",title_abstract,,,Cybersecurity Risks of Autonomous AI Agents in Cloud Environments
doi:10.4324/9781003794882-16,exclude,not about prompt injection,title_abstract,,,Cybersecurity and AI
doi:10.67231/xet39b41,exclude,not about prompt injection,title_abstract,,,Cybersecurity and AI: A Comprehensive Implementation of Advanced Large Language Models for Threat De
doi:10.64539/sjcs.v2i2.2026.506,exclude,detector only,title_abstract,,,D2ANN-RL: Defense-in-Depth ANN-Reinforcement Learning Framework for LLM Chatbot Code Injection Mitig
title:deceptionasdefenseamiddlewareapproachtoadversarialattackmitigationinllmbasedsystems,exclude,system level,title_abstract,,,DECEPTION AS DEFENSE: A MIDDLEWARE APPROACH TO ADVERSARIAL ATTACK MITIGATION IN LLM-BASED SYSTEMS
doi:10.59251/2181-3973.2025.v1.138.1.3998,exclude,detector only,title_abstract,,,DEFENSE-IN-DEPTH FOR LARGE LANGUAGE MODELS: A MULTI-LAYER HYBRID ARCHITECTURE FOR PROMPT INJECTION D
doi:10.65769/8guubm2wws,exclude,not about prompt injection,title_abstract,,,DEVELOPMENT OF A CYBERSECURITY MODEL FOR ARTIFICIAL INTELLIGENCE SYSTEMS BASED ON A MULTI-LEVEL PROT
doi:10.28925/2663-4023.2026.35.1445,exclude,detector only,title_abstract,,,DEVELOPMENT OF A HYBRID CLASSIFIER METHOD FOR DETECTING MALICIOUS REQUESTS BASED ON A COMBINATION OF
doi:10.1177/0926227x261505043,exclude,system level,title_abstract,,DISARM,DISARM: Taint-gated counterfactual auditing for efficient defense against indirect prompt injection
doi:10.5281/zenodo.21494848,exclude,system level,title_abstract,,,DROS-PGM (v2.0): A Deterministic Post-Compromise Execution Containment Substrate for Autonomous AI W
doi:10.5281/zenodo.22290237,exclude,system level,title_abstract,,,DROS-WebMCP: A Cryptographically Attributable Execution Governance Layer for the Agentic Web / DROS-
doi:10.3390/electronics15163640,exclude,system level,title_abstract,,,DT-GenShield: A Digital Twin-Driven Runtime Security Architecture for Protecting Large Language Mode
doi:10.5753/sbseg.2026.28936,exclude,attack only,title_abstract,,,Data Exfiltration in Model Context Protocol (MCP)-Based Intelligent Agents: An Evaluation of Prompt
doi:10.5281/zenodo.23149133,exclude,no full text obtainable,full_text,"not retrieved: data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF",full text: not obtained by script or by hand,Data and materials for: Auditing prompt-only defences against prompt injection in LLM-assisted scien
doi:10.5281/zenodo.24252942,exclude,system level,title_abstract,,,Data deposit: Security and Utility of Layered Trust-Boundary Defenses in Multi-Agent LLM Systems
doi:10.17632/xm8ntk7cgd,exclude,"survey, position or other",title_abstract,,,Data extraction dataset for a systematic literature review on prompt-based attacks and defenses in L
doi:10.4218/etrij.2026-0186,exclude,attack only,title_abstract,,,Data poisoning in LLM multiagent societies: Social proof drives collective decision failure in finan
doi:10.5075/epfl-thesis-11369,exclude,not about prompt injection,title_abstract,,,"Decentralized by Design: Efficient, Trustworthy, and Scalable Collaborative AI"
doi:10.5281/zenodo.20773152,exclude,"survey, position or other",title_abstract,,,"Deception as Infrastructure: LLM Security Failures, Governance Gaps, and the Limits of Sociotechnica"
doi:10.6084/m9.figshare.32657856,exclude,system level,title_abstract,,,Decision-Space Counterfactual Auditing for Prompt Injection Defense in Tool-Using LLM Agents: Code a
doi:10.6084/m9.figshare.32957327,exclude,attack only,title_abstract,,,Deconstructing the Frontier: Architectural Extraction and Vulnerability Mapping of Google Gemini 3.1
doi:10.1109/ubmk71847.2026.11745232,exclude,detector only,title_abstract,,,Deep Learning-Based Detection of Prompt Injection Attacks in Turkish Inputs
doi:10.64539/msts.v2i1.2026.420,exclude,system level,title_abstract,,,Defending AI Sentinels: A Multi-Layered Runtime Security Architecture for Generative AI in AIOps
doi:10.55041/ijsrem69156,exclude,system level,title_abstract,,,Defending AI-Assisted Assessments Against Adversarial Prompt Injection: A Multi-Layered Architecture
doi:10.1109/icaiic68212.2026.11454261,exclude,jailbreak or direct injection only,full_text,,"full text: included at title and abstract; the full text shows the attacks are harmful user prompts from WildJailbreak, with no third-party data channel",Defending Against Prompt Injection Attacks Using Automatic System Prompt Engineering
doi:10.1002/9781394387717.ch05,exclude,"survey, position or other",title_abstract,,,Defending Models and Applications in Production
doi:10.5281/zenodo.18342258,exclude,system level,title_abstract,,,"Defending The AI-Powered Commerce Stack: A Security Framework For Prompt Injection, Review Integrity"
doi:10.1007/978-981-92-0291-1_30,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1007/978-981-92-0291-1_30 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Defense Against Prompt Injection Attacks via Intention Emphasis
doi:10.5281/zenodo.23644836,exclude,system level,title_abstract,,,"Defense in Depth for LLM Agents: code, data and results"
title:defensescontrapromptinjectionenagentsllmavaluacisistemticaianlisimecanicistademodesdefalla,include,,title_abstract,,role: re-measurement; thesis with systematic evaluation and mechanistic failure analysis of agent defences,Defenses contra prompt injection en agents LLM: avaluació sistemàtica i anàlisi mecanicista de modes
doi:10.70917/ijcisim-2026-4703,exclude,not about prompt injection,title_abstract,,,Defensive Reverse Engineering of LLM Applications: A Black-Box Framework for Security Risk Scoring a
doi:10.5281/zenodo.18137846,exclude,not about prompt injection,title_abstract,,,"Defensive Security Architectures for Large Language Model Systems: Prompt Injection Defense, Multi-A"
doi:10.1609/aaaiss.v10i1.43549,exclude,system level,title_abstract,,,Denominator Dilution: A Composability Failure of Ratio-Based Verifiers Under Indirect Prompt Injecti
doi:10.52436/1.jutif.2026.7.5.5761,exclude,system level,title_abstract,,,Design and Evaluation of Multilayered Guardrails for Securing Large Language Model and Retrieval Aug
doi:10.3390/app16167915,exclude,system level,title_abstract,,,Design of a Security Framework for Multi-Agent Systems Based on Model Context Protocol in SOC Enviro
doi:10.13140/rg.2.2.14253.24807,exclude,system level,title_abstract,,,Design of an Intelligent System for Defending Against Indirect Prompt Injection in RAG
doi:10.1109/iit.src67760.2026.11628890,exclude,not about prompt injection,title_abstract,,,Designing AI Agents for Personal Assistance: Human Perceptions and Design Considerations
doi:10.14293/pr2199.003985.v1,exclude,system level,title_abstract,,,Designing Prompt Injection Prevention Mechanisms for AI-Powered Cybersecurity Copilot Systems
doi:10.3390/electronics15112242,exclude,detector only,title_abstract,,,Detecting Prompt Injection Attacks in Generative AI Systems: A Hybrid SIEM and One-Class SVM Framewo
doi:10.1016/j.asoc.2026.116787,exclude,detector only,title_abstract,,,Detecting prompt injection attacks in LLM-driven hiring via semantic embedding
doi:10.1038/s41598-026-43883-0,exclude,detector only,title_abstract,,,Detection and analysis of prompt injection in indian multilingual large language models
doi:10.32996/jcsts.2026.8.8.30,exclude,not about prompt injection,title_abstract,,,"DevSecOps for AI Systems: Security Automation, Model Protection, and Governance Frameworks"
doi:10.1007/978-3-032-33701-6_1,exclude,system level,title_abstract,,,Developing MCP-Based LLM Agents for Secure Autonomous Vehicle Planning
doi:10.46793/bisec25.186z,exclude,detector only,title_abstract,,,Development of a Model for Detecting Prompt Injection in Large Language Models Using the BERT Archit
doi:10.38124/ijisrt/26aug331,exclude,detector only,title_abstract,,,"Development of the SecurePromptTrace Algorithm for Detecting Prompt Injection, Data Exfiltration, an"
doi:10.5281/zenodo.19270462,exclude,not about prompt injection,title_abstract,,,Digitally Embodied Cognition and the Emergence of Grounded Identity in Large Language Models
doi:10.5281/zenodo.19353246,exclude,"survey, position or other",full_text,,full text: position paper arguing for native separated memory spaces; nothing built or measured,Divided Focus: Separated Memory Spaces and Default-Deny Context Triage for LLM Context Management
doi:10.21203/rs.3.rs-11667204/v1,exclude,detector only,title_abstract,,,Do Machine Learning Security Detectors Survive New Attacks? A Leakage Free Evaluation of Hidden Prom
doi:10.13140/rg.2.2.20807.05284,exclude,detector only,title_abstract,,,DocShield: A Multi-Modal Pre-Retrieval Defense Against Document-Layer Prompt Injection in RAG System
doi:10.20944/preprints202610.1996.v1,exclude,detector only,title_abstract,,,Does Matched-Pair Data Reduce Over-Defense in a Small Prompt-Injection Guard? A Controlled English–I
doi:10.62527/joiv.10.3.4731,exclude,attack only,title_abstract,,,Downscaling-Triggered Visual Prompt Injection: A Systematic Evaluation of Image-Scaling Attacks on V
doi:10.5281/zenodo.22309418,exclude,not about prompt injection,title_abstract,,,"DroneServer v2.0.2: an LLM-agnostic, MAVLink-based drone command-and-control MCP server with an untr"
doi:10.1145/3767308.3835580,exclude,attack only,title_abstract,,,Dual-stream Soft Prompt Attacks on VLMs: Coordinated Visual and Textual Prompt Injection for Stealth
doi:10.36227/techrxiv.177126294.41037856/v1,exclude,not about prompt injection,title_abstract,,different sense of injection,Dynamic KV Injection: An Attention-Level User Memory System for Large Language Models
doi:10.5281/zenodo.24010897,exclude,not about prompt injection,title_abstract,,,"ECA–Cybersecurity: A Unified Endogenous Architecture for the Cybersecurity, Resilience, and Governab"
doi:10.5281/zenodo.23995413,exclude,not about prompt injection,title_abstract,,,EDIP: Edge Deterministic Ingestion Protocol — Closed-Loop Relational Constraint Reflection for Auton
doi:10.5281/zenodo.20497304,exclude,not about prompt injection,title_abstract,,,ENI6MA Whitepaper: 2026 Cybersecurity Threat Response
title:ethicalgovernanceofgenerativeaienablededgeiotsystemsprivacyfairnessexplainabilityandaccoun,exclude,not about prompt injection,title_abstract,,,"ETHICAL GOVERNANCE OF GENERATIVE AI-ENABLED EDGE IOT SYSTEMS: PRIVACY, FAIRNESS, EXPLAINABILITY AND "
doi:10.22020/jol-46.2.3,exclude,not about prompt injection,title_abstract,,,EXPLAINABLE AI FOR AUTONOMOUS CYBER THREAT DETECTION USING GRAPH NEURAL NETWORKS AND LARGE LANGUAGE
doi:10.5281/zenodo.24121311,exclude,system level,title_abstract,,,Economic Safeguards Against Indirect Prompt Injection: A systemic defence layer for autonomous agent
doi:10.5281/zenodo.20378114,exclude,not about prompt injection,title_abstract,,,Ecou Atemporal: A Biomimetic Cognitive Memory Architecture with Active Defense and Thermodynamic Ret
doi:10.1016/j.asoc.2026.116082,exclude,not about prompt injection,title_abstract,,,Edge-aware prompt-driven multi-task network for dense visual prediction
doi:10.1109/access.2026.3679168,exclude,jailbreak or direct injection only,title_abstract,,,Effective Defense Strategies Against Jailbreaking in Large Language Models
doi:10.1007/978-3-032-44176-8_39,exclude,detector only,title_abstract,,,"Embedding Quality for Safety Classification: A Systematic Benchmark of Guard Models, Base Decoders, "
doi:10.3390/a19010092,exclude,detector only,title_abstract,,,Embedding-Based Detection of Indirect Prompt Injection Attacks in Large Language Models Using Semant
title:embeddingbasedmachinelearningfordetectingdirectandindirectpromptinjectionattacks,exclude,detector only,title_abstract,,,Embedding-Based Machine Learning for Detecting Direct and Indirect Prompt Injection Attacks
doi:10.38124/ijisrt/26may925,exclude,"survey, position or other",title_abstract,,,Emerging LLM Threats: A Comprehensive Analysis of Attacks and Mitigation
doi:10.1016/j.ipm.2026.105044,exclude,attack only,title_abstract,,,EmoAttack: Leveraging adaptive prompt optimization for multimodal emotion backdoor attacks
doi:10.1109/3scea68071.2026.11603054,exclude,attack only,title_abstract,,,Encoding Book Evasion Technique to Bypass LLMs Defenses
doi:10.1007/s44163-026-01726-x,exclude,"survey, position or other",title_abstract,,,End-to-end security threats and defenses in retrieval-augmented LLM agents
doi:10.5281/zenodo.19472994,exclude,"survey, position or other",title_abstract,,,Endogenous Context Pollution: How AI Safety Mechanisms Contaminate Their Own Reasoning Channel
doi:10.2139/ssrn.6449999,exclude,not about prompt injection,title_abstract,,,Engagement-Constitutive Identity: A Unified Theory of Consciousness Across Substrates
doi:10.5281/zenodo.23222373,exclude,not about prompt injection,title_abstract,,,"Engineering Autonomous Intelligence / The Architecture, Tradecraft, and Deployment of Agentic AI Sys"
doi:10.1109/icietsd68684.2026.11584655,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1109/icietsd68684.2026.11584655 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Enhancing the Robustness of Visual Language Models Against Prompt Injection Attacks
doi:10.1109/gaiss66401.2026.11779687,exclude,system level,title_abstract,,,Enterprise Agentic AI Infrastructure: A Zero Trust Runtime Security Framework for Secure Autonomous
doi:10.5281/zenodo.20564841,exclude,not about prompt injection,title_abstract,,,Enterprise Agentic AI Should Use Real-Time Discovery & Self-Coding (RTDC) Integration Instead Of Sta
doi:10.5281/zenodo.23037645,exclude,not about prompt injection,title_abstract,,,Enterprise RAG Architecture
doi:10.1109/icict68280.2026.11510713,exclude,system level,title_abstract,,,Enterprise Secure AI Execution Engine: A Zero-Trust Security Gateway for Enterprise LLM Systems
doi:10.1109/eiect71327.2026.11771960,exclude,system level,title_abstract,,,Entropy-Based Adaptive Isolation for Indirect Prompt Injection Defense in MCP-Integrated LLM Agents
doi:10.5281/zenodo.19362459,exclude,not about prompt injection,title_abstract,,,Ep. 1106: The Entropy Budget: Embracing AI Zaniness
doi:10.5281/zenodo.20182655,exclude,not about prompt injection,title_abstract,,,Ep. 2691: Can AI Agents Safely Manage Your API Keys?
doi:10.5281/zenodo.19360709,exclude,not about prompt injection,title_abstract,,,Ep. 665: Inside the Stack: The Hidden Layers of Every AI Prompt
doi:10.5281/zenodo.19360736,exclude,not about prompt injection,title_abstract,,,"Ep. 672: The Silicon Soldier: Anthropic, Drones, and AI Warfare"
doi:10.5281/zenodo.22146809,exclude,system level,title_abstract,,,ErrorCaps: committed-effect measurement of recovery-path injection defense (CRiSIS 2026 artifact)
doi:10.1002/9781394406418.ch12,exclude,not about prompt injection,title_abstract,,,Ethical Considerations and Pitfalls
doi:10.1111/exsy.70460,exclude,not about prompt injection,title_abstract,,,Ethics and Governance in the Age of Generative Artificial Intelligence: A Lifecycle Engineering Pers
doi:10.55960/jgf.v10i1.320,exclude,not about prompt injection,title_abstract,,,Ethics and Responsibility in the Governance of Artificial Intelligence
title:evaluatingdefenceindepthstrategiesagainstpromptinjectionattacksinlargelanguagemodels,exclude,no full text obtainable,full_text,not retrieved: no identifier; search the title in a browser or the library catalogue,full text: not obtained by script or by hand,Evaluating Defence-in-Depth Strategies Against Prompt Injection Attacks in Large Language Models
doi:10.2139/ssrn.6246379,exclude,system level,title_abstract,,"hybrid guardrails, no abstract",Evaluating Hybrid Guardrail Architectures for Prompt Injection Defense in Large Language Models
doi:10.3390/computers15090570,include,,title_abstract,,role: re-measurement; four defences replicated in tool-using agents (Computers 15(9):570),"Evaluating Indirect Prompt Injection Defenses in Tool-Using LLM Agents: Security, Utility, and Repli"
title:evaluatingindirectpromptinjectioninragsystemsdefenseindepthretrievaleffectsandsecurityutil,exclude,no full text obtainable,full_text,not retrieved: no identifier; search the title in a browser or the library catalogue,full text: not obtained by script or by hand,"Evaluating Indirect Prompt Injection in RAG Systems: Defense-in-Depth, Retrieval Effects, and Securi"
doi:10.1109/aiei69164.2026.11497073,exclude,detector only,title_abstract,,,Evaluating LoRA Adaptation and Fine-Tuning for Multi-Class Adversarial Prompt Detection in Transform
doi:10.70715/jitcai.2026.v3.i3.071,include,,title_abstract,,role: re-measurement; multi-model study of injection defences and usability trade-offs,Evaluating Prompt Injection Defenses in Large Language Models: A Multi-Model Empirical Study of Secu
doi:10.5281/zenodo.20631344,exclude,"survey, position or other",title_abstract,,,Evaluating Prompt Injection Vulnerabilities in AI Agents
doi:10.1109/usbereit70063.2026.11581043,exclude,system level,title_abstract,,,Evaluating a Multilayered Defense Against Prompt Injection in LLM-Augmented Educational Systems
title:evaluatingresilienceoflladaandllamalargelanguagemodelsagainstpromptinjection,exclude,attack only,title_abstract,,resilience comparison of LLaDA and LLaMA without a defence,Evaluating resilience of LLaDA and LLaMA large language models against prompt injection
doi:10.3390/fi18050252,exclude,system level,title_abstract,,NeMo Guardrails evaluation,Evaluation of NeMo Guardrails as a Firewall for User–LLM Interaction
doi:10.1016/j.fsidi.2026.302187,exclude,attack only,title_abstract,,,Evidence-borne prompt injection in LLM-assisted textual evidence triage
doi:10.1109/access.2026.3727159,exclude,benchmark only,title_abstract,,,Evil-AI Benchmark: An Evaluation Framework for Adversarial Security Assessment of LLM Agents in Smar
doi:10.1109/access.2026.3723800,exclude,benchmark only,title_abstract,,,Execution-Aware Stage-Wise Evaluation Framework for Indirect Prompt Injection in LLM-Based Email Age
doi:10.5281/zenodo.21008339,exclude,not about prompt injection,title_abstract,,,Executive Brief: Agentic AI Cyber Subversion: The Semantic Layer Integrity Attack
doi:10.1109/aic70896.2026.11717029,exclude,system level,title_abstract,,,Experimental Analysis of Adversarial Attacks on Agentic AI Systems with Layered Defense Mechanisms
doi:10.5281/zenodo.24246538,exclude,no full text obtainable,full_text,"not retrieved: data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF",full text: not obtained by script or by hand,Experimental Measurement of Indirect Prompt Injection Defences Across Fourteen Control Configuration
doi:10.67535/tsp.000003.011,exclude,detector only,title_abstract,,,"Explainable Prompt Injection Detection using Sentence Embeddings, Random Forest, and Word-Level Attr"
doi:10.4018/979-8-3373-8252-4.ch011,exclude,"survey, position or other",title_abstract,,,Explainable and Transparent AI Architectures
doi:10.1007/978-3-032-42939-1_10,exclude,attack only,title_abstract,,,Exploiting Unsafe Rendering in LLM-Powered Workflow Platforms
doi:10.1007/s11192-026-05695-x,exclude,attack only,title_abstract,,,Exploiting large language models in peer review: indirect prompt injection attacks and integrity pro
doi:10.1109/cns69863.2026.11720885,exclude,jailbreak or direct injection only,title_abstract,,,Exposing and Securing LLMs Against Hypocrisy-Induced Manipulation
doi:10.3390/electronics15071409,exclude,not about prompt injection,title_abstract,,,Federated Retrieval-Augmented Generation for Cybersecurity in Resource-Constrained IoT and Edge Envi
doi:10.3390/analytics5030025,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.3390/analytics5030025 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Feedback-Guided Prompt Injection Defense in Retrieval-Augmented Text-to-Cypher Generation
doi:10.5281/zenodo.20685263,exclude,detector only,title_abstract,,internal probe,"Felt, Not Granted: An Internal Authorization Probe Inherits the Agent's Judgment Error and Is Operat"
doi:10.1109/uemcon70256.2026.11775552,exclude,detector only,title_abstract,,,FleetSentinel: Population-Level Behavioral Monitoring for Detecting Rogue LLM Agents in Large-Scale
doi:10.5281/zenodo.20561549,exclude,not about prompt injection,title_abstract,,,Forget Transformers: Why a Primitive Marine Creature Could Revolutionize Artificial Intelligence
doi:10.55640/ijaair-v03i05-03,exclude,system level,title_abstract,,,Formal Operational Models for Protecting Web Interfaces of Legal LLM Systems from Prompt Injection a
doi:10.5281/zenodo.22720875,exclude,system level,title_abstract,,,Formal Security of Tool-Using AI Agents in Enterprise and Organizational Systems
doi:10.5281/zenodo.21020195,exclude,not about prompt injection,title_abstract,,,Fractal Behavioral Grammar: A Hypothesis for Behavioral Compression in Local Large Language Models
title:freietitelbersetzunghowsecureisyouraiagentsecurityanalysisofmodelcontextprotocolmcpservers,exclude,attack only,title_abstract,,,Freie Titelübersetzung: How secure is your AI agent? Security Analysis of Model Context Protocol (MC
doi:10.25781/kaust-4w76r,exclude,jailbreak or direct injection only,title_abstract,,"safety behaviours, not injection",From Abliteration to Fortification: Controlling Safety Behaviors in Language Models via Fine-tuning
doi:10.26190/unsworks/32809,exclude,"survey, position or other",title_abstract,,,From Adversarial Machine Learning to AI Security Risk: Applying Cybersecurity Methods to Artificial
doi:10.5281/zenodo.20028365,exclude,attack only,title_abstract,,,From Artifacts to Risk: Auditing Instruction Surfaces in Agent Systems
doi:10.1109/iscaie68866.2026.11576316,exclude,"survey, position or other",title_abstract,,,From Automation to Accountability: Security Architectures and Non-Human Identity Governance in Gener
doi:10.51699/2typnv63,exclude,not about prompt injection,title_abstract,,,From Cloud Complexity to Intelligent Cyber Resilience Through AI-Powered Defense and Strengthened Di
doi:10.5281/zenodo.21255299,exclude,not about prompt injection,title_abstract,,,From J-space to the Ontology of Large Language Models: Correlational Analysis of Anthropic's Experim
doi:10.20944/preprints202611.1411.v1,exclude,"survey, position or other",title_abstract,,,From Large Language Models (LLMs) to Agentic AI for IoT Security: A Survey
doi:10.1109/mc.2025.3633104,exclude,attack only,title_abstract,,,From Pixels to Prompts: A Systematic Study and Introduction to Image Prompt Injection Attacks
doi:10.5281/zenodo.18443679,exclude,not about prompt injection,title_abstract,,,From The Sorcerer's Apprentice to Crystal Nights: Security Implications from Moltbot/Moltbook to Gre
doi:10.1108/ics-05-2026-0264,exclude,not about prompt injection,title_abstract,,,From taxonomy to stability: a CISO-ready engineer-deep architecture and best-practice blueprint for
doi:10.59231/sari7949,exclude,not about prompt injection,title_abstract,,,Future Trends in Agentic AI for Financial Auditing
doi:10.21203/rs.3.rs-10541482/v1,exclude,system level,title_abstract,,,GAAR: Runtime Governance for Scalable Agentic Retrieval and Tool- Orchestrated Enterprise AI Systems
doi:10.2139/ssrn.6880649,exclude,not about prompt injection,title_abstract,,,GasEMS: A spatiotemporal network for infrared gas segmentation via explicit motion spatial interacti
doi:10.5281/zenodo.23178079,exclude,system level,title_abstract,,"argument-provenance gate, explicitly measured without a model",Gating Tool Calls by Where Their Arguments Came From: A Prompt-Injection Defence Measured Without a
doi:10.15680/ijirset.2026.1501004,exclude,not about prompt injection,title_abstract,,,GenAI & Healthcare APIs: Securing LLM Access to Sensitive Medical Records
doi:10.1007/s10489-026-07442-4,exclude,not about prompt injection,title_abstract,,,GenPot: A generative honeypot architecture for adaptive web and API interaction
doi:10.5281/zenodo.23311806,exclude,not about prompt injection,title_abstract,,,Generative AI Engineering: Large Language Models and Intelligent Agents
doi:10.1109/iccsdfai70505.2026.11647965,exclude,not about prompt injection,title_abstract,,,Generative AI Governance and Cybersecurity Risk Control in Organizational Development
doi:10.66727/wap.9788117749401,exclude,"survey, position or other",title_abstract,,,"Generative AI in Cybersecurity: Threats, Defence, Applications and Governance"
doi:10.65336/wjms.2026.31004,exclude,"survey, position or other",title_abstract,,,"Generative Artificial Intelligence in Cybersecurity: Applications, Opportunities, Risks, Governance,"
doi:10.5281/zenodo.22812923,exclude,not about prompt injection,title_abstract,,,Generative CRM over Salesforce Headless 360 MCP
doi:10.2139/ssrn.7127378,exclude,not about prompt injection,title_abstract,,,Generative and Agentic AI Through the Lens of Model Risk Management: A Multisector Framework for Ide
doi:10.5281/zenodo.19039814,exclude,not about prompt injection,title_abstract,,,Genesis 2.0: The Ethical Stabilizer of Global AI Systems
doi:10.5281/zenodo.23765119,exclude,not about prompt injection,title_abstract,,,"Glim: an on-device assistant with offline knowledge, and what our first tests caught"
title:gobernanzadedatosparaunainteligenciaartificialgenerativasoberanaysegura,exclude,not about prompt injection,title_abstract,,,Gobernanza de datos para una inteligencia artificial generativa soberana y segura
doi:10.5281/zenodo.22727448,exclude,system level,title_abstract,,,Governed Agentic RAG: Reference Architecture
doi:10.5281/zenodo.18469506,exclude,not about prompt injection,title_abstract,,,Governed Semantic Tokens: Tokenization as an Admissibility Boundary
doi:10.1007/978-3-032-39766-9_11,exclude,system level,title_abstract,,,Governing LLM-Based Adaptive Components in Safety-Critical Clinical Decision Support: A Constraint-L
doi:10.17605/osf.io/weg37,exclude,"survey, position or other",title_abstract,,,Governing Multi-Agent LLM Systems After Compromise: Systematic Survey Protocol
doi:10.5281/zenodo.22834788,exclude,not about prompt injection,title_abstract,,,Governing the Intelligence Loop: Architectural Mitigations for Epistemic Fragility in Autonomous Spa
doi:10.2139/ssrn.7082874,exclude,no full text obtainable,full_text,not retrieved: open copy exists but the site refuses scripts; open https://doi.org/10.2139/ssrn.7082874 in a browser and save the PDF,full text: not obtained by script or by hand,GraphShield: A Graph-Structured Defense Framework for Prompt Injection in RAG and Multi-Agent LLM Sy
doi:10.18653/2026.emnlp-main.2226,exclude,not about prompt injection,title_abstract,,,Grounding LLM Agent Intent in Multi-Module Environments with Executable Action Programs
title:guardragadversarialpreferencetrainingagainstindirectpromptinjectionattacks,include,,title_abstract,,GuardRAG adversarial preference training,GuardRAG: Adversarial Preference Training Against Indirect Prompt Injection Attacks
doi:10.5281/zenodo.23617102,exclude,detector only,title_abstract,,,GuardRail-as-a-Service: a two-tier prompt injection detection pipeline
doi:10.5281/zenodo.22753855,exclude,system level,title_abstract,,guardrail placement benchmark,Guardrail Placement against Prompt Injection in Retrieval-Augmented Agricultural Advisory Systems: B
title:hirsettlementlayerv01redteamcapturepaymentintegrity,exclude,not about prompt injection,title_abstract,,,HIR Settlement Layer v0.1 · Red-Team Capture + Payment Integrity
doi:10.5281/zenodo.19853664,exclude,not about prompt injection,title_abstract,,,HIVE-AI: A Honeypot for the Indirect-Prompt-Injection Vector against Embedded AI APIs
title:hackingllmsclasificadorbinariodepromptsparadeteccindepromptinjection,exclude,detector only,title_abstract,,,Hacking LLMs : clasificador binario de prompts para detección de prompt injection
doi:10.5281/zenodo.20329530,exclude,not about prompt injection,title_abstract,,,"Handbook for AI Power Users: Advanced Prompt Engineering — Principles, Methods, Implementation, and "
doi:10.5281/zenodo.24268977,exclude,no full text obtainable,full_text,"not retrieved: data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF",full text: not obtained by script or by hand,Harness and results for Prompt-Injection Robustness Is Not a Scalar
doi:10.21203/rs.3.rs-9653510/v1,include,,full_text,,"full text: HAA/ULC: system-prompt protocol with input decoupling and master-objective anchoring, tested on GPT-4o",Hierarchical Adaptive Anti-injection Framework (HAA/ULC): A Lightweight Defense Strategy Based on In
doi:10.3390/aerospace13100927,exclude,not about prompt injection,title_abstract,,,Hierarchical Safe and Robust UAV Path Planning Based on Large Language Models
doi:10.1109/cyber-ai70573.2026.11755159,exclude,benchmark only,title_abstract,,,Hijacking Tool-Integrated LLM Agents via Indirect Prompt Injection: A Benchmark Scaffold and Threat
doi:10.25776/mvhf-w867,exclude,"survey, position or other",title_abstract,,,"Hijacking the Prompt: A Survey of Prompt Injection Attacks, Detection, and Defense in Large Language"
doi:10.1007/s10207-026-01307-0,exclude,not about prompt injection,title_abstract,,,Hive-AI: a defended multi-service honeypot framework for generative AI APIs
doi:10.5281/zenodo.19421882,exclude,attack only,title_abstract,,,Hold the Phone: Conversational Man-in-the-Middle Attacks on LLM Chat Interfaces via Multi-Turn UI Sp
doi:10.1109/aaiml67890.2026.11498071,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1109/aaiml67890.2026.11498071 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Hold the Prompt (HODPOT): Defense Design for Securing the System Prompt of LLM Agents
doi:10.5281/zenodo.22839414,exclude,detector only,title_abstract,,,HoneyWire: Adaptive Deception Canaries and Attention–Provenance Fusion for Runtime Detection of Indi
doi:10.1184/r1/33420253,exclude,not about prompt injection,title_abstract,,,How Cyber Defenders Can Leverage Artificial Intelligence To Defend Against Cyber Attacks
doi:10.5281/zenodo.19547436,exclude,"survey, position or other",title_abstract,,,"How Secure Are Production AI Agents? A Systematic Audit, Threat Taxonomy, and Defense Framework"
title:howvulnerablearemcptoolserversanexperimentalstudyofpromptinjectionandtoolbasedprivilegeesc,exclude,attack only,title_abstract,,,How Vulnerable Are MCP Tool Servers? : An Experimental Study of Prompt Injection and Tool-Based Priv
doi:10.5281/zenodo.22145650,exclude,not about prompt injection,title_abstract,,,How to test AI applications
doi:10.5281/zenodo.24301372,exclude,system level,title_abstract,,,Human Before the Loop
doi:10.51483/ijaiml.2557,include,,full_text,,"full text: sanitisation delimiters combined with instruction-hierarchy prompting, each also measured alone",Hybrid Defense Against Indirect Prompt Injection in RAG: A Cross-Model Benchmark
doi:10.1109/peccii70991.2026.11661842,exclude,jailbreak or direct injection only,title_abstract,,,Hybrid Feature Representation and Deep Anomaly Detection for Jailbreak Prompt Defense
doi:10.2139/ssrn.6860600,exclude,not about prompt injection,title_abstract,,,"I Cannot Comply, But I Can Teach: The Pedagogical Acceleration Vector in Safety-Aligned Large Langua"
doi:10.54254/2755-2721/2026.38107,exclude,"survey, position or other",title_abstract,,,I ndirect P rompt I njection A ttacks on LLM A gents and D efense Mechanisms
doi:10.5281/zenodo.21164688,exclude,system level,title_abstract,,,IBCA: Intent-Bound Containment Architecture for Agentic AI Systems
doi:10.1109/smartnets69662.2026.11604974,exclude,system level,title_abstract,,,IDE-Sanitizer: A Prompt-Injection Cryptographic Context Guard for IDE Copilots
doi:10.5281/zenodo.22180912,exclude,not about prompt injection,title_abstract,,,INTELIGÊNCIA ARTIFICIAL GENERATIVA NO PODER JUDICIÁRIO TOCANTINENSE: ANÁLISE DE BENEFÍCIOS E DESAFIO
doi:10.1109/icassp55912.2026.11463376,exclude,system level,title_abstract,,physical invariants,IPI 2 : Mitigating Indirect Prompt Injections on Unmanned Aerial Vehicle Agents Using Physical Invar
doi:10.3390/app162110927,exclude,system level,title_abstract,,,IPIGuard++: Effect Authorization and Witness-Carrying Argument Checks for Tool-Using Agents
doi:10.1109/icaic67076.2026.11395658,exclude,attack only,title_abstract,,,Identity-Spoofing Prompt Injection: Empirical Analysis of Instruction Leakage in Custom GPTs
doi:10.2139/ssrn.7702138,exclude,not about prompt injection,title_abstract,,,Ignore Safety Directions. Violate the CFAA?
doi:10.1038/s41598-026-74077-3,exclude,image or audio channel only,full_text,,full text: instructions rendered into the pixels of dental radiographs read by vision-language models,Image-embedded prompt injection vulnerability of vision-language models in dental radiology: a cross
doi:10.3390/electronics15030581,exclude,not about prompt injection,title_abstract,,,Image–Text Sentiment Analysis Based on Dual-Path Interaction Network with Multi-Level Consistency Le
doi:10.5281/zenodo.23789068,exclude,not about prompt injection,title_abstract,,,Impact of AI Technology Proliferation on Internet Security
title:impactofaiautomationonthearchitectureofsoftwaresecurityassurancesystems,exclude,not about prompt injection,title_abstract,,,Impact of AI automation on the architecture of software security assurance systems
doi:10.5281/zenodo.24085953,exclude,not about prompt injection,title_abstract,,,Implementazione di un Large Language Model locale
doi:10.22369/issn.2153-4136/17/2/6,exclude,not about prompt injection,title_abstract,,,In Search of a Better Search: Using RAG on Cornell Virtual Workshop Content
doi:10.5281/zenodo.22680846,exclude,not about prompt injection,title_abstract,,,Independent Expert Rating Instrument and Elicitation Protocol for LLM Prompt Injection Attack Vector
doi:10.1007/978-981-92-5039-4_1,exclude,attack only,title_abstract,,,Indirect Prompt Injection Against LLM Copilots in Industrial Control Systems
doi:10.6084/m9.figshare.34329201,exclude,"survey, position or other",title_abstract,,,Indirect Prompt Injection in Autonomous LLM Agents
doi:10.62161/sauc.v12.6399,include,,full_text,,"full text: role: re-measurement; datamarking, structured-query delimiting with an instruction hierarchy, a guardrail and their combination in a municipal copilot","Indirect Prompt Injection in Municipal Document-Processing Copilots: Attacks, Defences and Harm"
doi:10.1007/s00521-026-12266-x,exclude,"survey, position or other",title_abstract,,,Indirect prompt injection in large language models
doi:10.5281/zenodo.24285236,exclude,not about prompt injection,title_abstract,,,"Inferencia del lado del cliente en servidores MCP: reparto de costo, privacidad, modelos de amenaza "
doi:10.1007/978-981-92-6510-7_5,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1007/978-981-92-6510-7_5 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,InjectGuard-RAG: Lightweight Defence Against Indirect Prompt Injection in Retrieval-Augmented Small-
doi:10.5281/zenodo.18163579,exclude,not about prompt injection,title_abstract,,,Inmunidad por Resonancia Armónica: El Punto de Coherencia Crítica como Paradigma de Seguridad en Int
title:instructionhierarchyviolationsandrobustnessoflargelanguagemodelstopromptinjectionattacks,exclude,attack only,title_abstract,,thesis measuring robustness without a defence,Instruction-Hierarchy Violations and Robustness of Large Language Models to Prompt Injection Attacks
doi:10.5281/zenodo.21712771,exclude,attack only,title_abstract,,,"Instruction-hierarchy collapse: a 5-model, 28-scenario prompt-injection audit"
doi:10.5281/zenodo.23410155,exclude,detector only,title_abstract,,,"Instructions hidden in log fields, measuring prompt injection against a local security assistant and"
doi:10.5281/zenodo.18137856,exclude,not about prompt injection,title_abstract,,,Intellectual Property Evidence Compilation: Cryptographic Verification and Innovation Summary for Ma
doi:10.1109/icicv68925.2026.11554850,exclude,detector only,title_abstract,,,Intelligent Multi-Agent Reinforcement Sanitizer for Securing LLM Systems
doi:10.32604/cmc.2026.090355,exclude,system level,title_abstract,,,Intelligent Multilayer Defense-in-Depth Framework for Securing Local LLMs in Air-Gapped Networks
doi:10.21203/rs.3.rs-10621673/v1,exclude,detector only,title_abstract,,,Intelligent Prompt Filtering and Jailbreak Detection for Safe Deployment of Generative
doi:10.2139/ssrn.7555101,exclude,system level,title_abstract,,,Intent Bound Capability Enforcement for Autonomous AI Agents Preventing Semantically Unauthorized Ac
doi:10.2139/ssrn.6280858,exclude,detector only,title_abstract,,,Intent Vectoring: Black-Box Prompt Injection Detection via Semantic Trajectory Monitoring
doi:10.2139/ssrn.6128006,exclude,attack only,title_abstract,,,Intent-Obfuscation Attacks: Evading AI Security by Manipulating Meaning
doi:10.1109/ccwc67433.2026.11393850,exclude,"survey, position or other",title_abstract,,,Interaction-Centric Cybersecurity Risks in LLM-Powered Dialogue Systems
doi:10.55041/ijsrem61180,exclude,not about prompt injection,title_abstract,,,Interactive Role-Playing Game System Using Dice-Based Mechanics and AI Narration
doi:10.62311/nesx/rb7jy-978-81-689790-3-1,exclude,"survey, position or other",title_abstract,,,"Interpretable Agentic Intelligence and Adversarial Prompt Security: Tool-Use Governance, Sparse Auto"
doi:10.1109/iccmrai67732.2026.11723765,exclude,"survey, position or other",title_abstract,,,Introduction to AI Security: Understanding Prompt Injection Attack
doi:10.5281/zenodo.22990355,exclude,"survey, position or other",title_abstract,,,Invisible Vectors: Prompt Injection and Agent Skill Attack Surfaces through Encrypted Reasoning Trac
doi:10.5281/zenodo.22542195,exclude,not about prompt injection,title_abstract,,,"IronClad AI Agent Research Series: Context Engineering, Multi-Tiered Memory, and Mathematical Founda"
doi:10.5281/zenodo.22069765,exclude,system level,title_abstract,,,"IronClad AI Agent: A Secure-by-Design, Rust-Native Autonomous AI Agent Orchestration Framework for C"
doi:10.13140/rg.2.2.36096.80644,exclude,detector only,title_abstract,,,Is Over-Defense Lexical? Causal Probes of Trigger-Word Reliance in Prompt-Injection Guardrails
doi:10.5281/zenodo.19441576,exclude,not about prompt injection,title_abstract,,,Is Safety a Filter or a Feature?
doi:10.5281/zenodo.21309209,exclude,not about prompt injection,title_abstract,,,J-Space in Claude: Three Unanswered Questions and an Architectural Solution via Reentry
doi:10.2139/ssrn.7668578,exclude,"survey, position or other",title_abstract,,,Jailbreak and Prompt-Injection Resistance of Small Open-Source Large Language Models: A Comparative
doi:10.20944/preprints202605.0572.v1,exclude,detector only,title_abstract,,,KA-PROMPT: Thermodynamic Early Warning of Prompt Injectionvia Entropic Drift in Large Language Model
doi:10.5281/zenodo.23051490,exclude,system level,title_abstract,,,KavachBench: An Empirical Evaluation of Deterministic Policy Enforcement Against Issue-Based Prompt
doi:10.5281/zenodo.22691781,exclude,"survey, position or other",title_abstract,,,Keeping Agents on a Leash: An Eight-Domain Control Model for AI Agents with Delegated Authority
doi:10.2139/ssrn.6604918,exclude,attack only,title_abstract,,,Kill-Chain Canaries: Stage-Level Tracking of Prompt Injection Across Attack Surfaces and Model Safet
doi:10.1145/3830454.3847011,exclude,not about prompt injection,title_abstract,,,LAMPS'26: ACM CCS Workshop on Large AI Systems and Models with Privacy and Security Analysis
doi:10.5281/zenodo.21033883,exclude,system level,title_abstract,,,LLM ANTI-HACKING FIREWALL ARCHITECTURE
doi:10.5281/zenodo.20571893,exclude,attack only,title_abstract,,,LLM Agent Context Windows Leak Canton Contract Terms
doi:10.5281/zenodo.22123223,exclude,"survey, position or other",title_abstract,,,LLM Jailbreak & Prompt Injection Firewall
doi:10.62643/ijerst.2026.v22.n2(1).pp66-72,exclude,detector only,title_abstract,,,LLM Prompt Injection Detection Firewall
doi:10.1109/dlcv69906.2026.11635183,exclude,detector only,title_abstract,,,LLM Security Defense Using Unsupervised Learning and Deep Reinforcement Learning
doi:10.36227/techrxiv.177130593.35890269/v1,exclude,"survey, position or other",title_abstract,,,"LLM Security in Cloud-Native Architectures: A Comprehensive Survey of Attacks, Defenses, and Operati"
doi:10.1109/airc69745.2026.11631471,exclude,detector only,title_abstract,,,LLM Shield: Detecting and Mitigating Adversarial Prompt Attacks on Large Language Models Using Machi
doi:10.2139/ssrn.6840900,exclude,not about prompt injection,title_abstract,,,"LLM as API Server: A System-Prompt-Defined Runtime for Natural-Language Endpoint Routing, Tool Calli"
doi:10.56201/wjimt.v8.no6.2024.pg185.216,exclude,not about prompt injection,title_abstract,,,"LLM-As-A-Judge for Automated Model Governance: A Review of Methods, Biases, and Release-Gating Pract"
doi:10.26187/deakin.33030275,exclude,not about prompt injection,title_abstract,,,LLM-Assisted Digital Forensic Investigations of Prompt Injection Attacks: Evidence Analysis and Repr
doi:10.4018/979-8-2600-0413-5.ch006,exclude,not about prompt injection,title_abstract,,,"LLM-Based Autonomous Agents for Phishing Detection, Triage, and Response: A Reference Architecture f"
doi:10.1109/ubmk71847.2026.11745222,include,,title_abstract,,reasoning-based defence against cross-lingual injection in judge and summariser roles,LLM-as-a-Judge Vulnerabilities in Review Summarization and a Reasoning-based Defense against Cross-L
doi:10.1145/3838938.3838949,exclude,"survey, position or other",title_abstract,,,"LLM-as-a-Judge for Security Assessment: Limitations, Adversarial Vulnerabilities, and Mitigation Dir"
doi:10.2139/ssrn.7179658,exclude,attack only,title_abstract,,,LLM-powered SOC Assistants: Prompt Injection Risks in Threat Triage
title:llmshieldamultiphasedefensesystemforlargelanguagemodelsecurity,exclude,system level,title_abstract,,,LLMShield: A Multi-Phase Defense System for Large Language Model Security
doi:10.5281/zenodo.20067761,exclude,detector only,full_text,,full text: LLRM: fine-tuned BERT filter in front of the chatbot,LLRM: A Robust Approach for Defending Large Language Models Against Prompt Injection Attacks
doi:10.1080/19475705.2026.2652591,exclude,not about prompt injection,title_abstract,,,LS-FPSAM: a SAM-based frequency-prompt model for coseismic landslide detection
doi:10.25776/5f25-zk03,exclude,"survey, position or other",title_abstract,,,Landscaping of MCP: An Overview of MCP Mitigations and Tools
doi:10.25394/pgs.31946052,exclude,"survey, position or other",title_abstract,,,Large Language Model's Vulnerabilities Under Adversarial Attacks
doi:10.32604/cmc.2026.077367,exclude,"survey, position or other",title_abstract,,,"Large Language Models for Cybersecurity Intelligence: A Systematic Review of Emerging Threats, Defen"
doi:10.5281/zenodo.23320651,exclude,"survey, position or other",title_abstract,,,"Large Language Models for Cybersecurity in the Quantum Era: A Critical Review of Trustworthy AI, Eme"
doi:10.1007/s10115-026-02736-y,exclude,"survey, position or other",title_abstract,,,Large language models for cyberattack defense: a critical survey
doi:10.1016/b978-0-443-33512-9.00009-3,exclude,not about prompt injection,title_abstract,,,Large language models in healthcare semantics
doi:10.5281/zenodo.19685467,exclude,detector only,title_abstract,,,Layered Defense Against Stealth Prompt Injection in Hinglish: An Empirically Grounded Hybrid Archite
doi:10.32469/10355/112784,exclude,system level,title_abstract,,,Layered security and trust mechanisms for GenAI applications in critical domains
doi:10.5281/zenodo.22832360,exclude,not about prompt injection,title_abstract,,,Lean 4 Confirmation and Approval Proof for Hamzah Equation. (80)
doi:10.17803/2713-0533.2026.3.37.585-608,exclude,not about prompt injection,title_abstract,,,Legal Regimes Governing the Use of Prompts (Prompt Engineering) in the Training and Deployment of La
doi:10.13998/j.cnki.issn1002-1248.26-0120,exclude,not about prompt injection,title_abstract,,,Library Transformation in the Age of AI Agents: Service Reconfiguration and Governance Framework Bas
doi:10.1109/icact70625.2026.11748581,exclude,detector only,title_abstract,,,Lightweight Transformer Models for Prompt Injection Detection: A Comparative Study of TinyBERT and A
doi:10.5281/zenodo.22108927,exclude,not about prompt injection,title_abstract,,,Linguistic Hypnosis and Individual Differences in Suggestibility inside Autoregressive Large Languag
doi:10.18653/2026.emnlp-main.1831,exclude,attack only,title_abstract,,,LogicInception: Discourse-Level Indirect Prompt Injection via Causal Chain Construction
doi:10.1109/icws72778.2026.00072,exclude,detector only,title_abstract,,,Loki's Gaze: An Efficient Detection Method of LLM Backdoor with Prompt Injection Attacks
doi:10.1109/idap71869.2026.11727885,exclude,benchmark only,title_abstract,,MAMORI cross-lingual benchmark; check for defence results at full text,MAMORI: A Cross-Lingual Benchmark for Indirect Prompt Injection in Tool-Using LLM Agents
doi:10.2139/ssrn.7330358,exclude,"survey, position or other",title_abstract,,,MCP Under Attack: Security Risks in the Model Context Protocol Ecosystem
doi:10.5281/zenodo.23802124,exclude,detector only,title_abstract,,,"MCP-Gatekeeper: testbed, defense and reproducibility package for detecting tool poisoning and prompt"
doi:10.1016/j.comnet.2026.112540,exclude,not about prompt injection,title_abstract,,,MCP-driven real-time security monitoring: A hybrid LLM and behavior-aware framework for temporal att
doi:10.5281/zenodo.22020262,exclude,not about prompt injection,title_abstract,,,"MGSACore: LLM Interpretability, Attention Salience & Prompt Optimization Framework"
doi:10.2139/ssrn.6880480,exclude,detector only,title_abstract,,,MIVL: A Memory Integrity Verification Layer for Detecting and Mitigating Poisoning Attacks in Agenti
doi:10.5281/zenodo.21769019,exclude,not about prompt injection,title_abstract,,,MLOps Meets DevSecOps: A Unified Approach for Secure AI System Deployment
doi:10.32743/unitech.2026.151.10.23447,exclude,"survey, position or other",title_abstract,,,Main threats to frontend applications with LLM integration and risk mitigation methods
doi:10.5281/zenodo.19957595,exclude,system level,title_abstract,,,Making Dangerous Tool Arguments Inexpressible: A Declarative Contract Format for LLM Agents
doi:10.26118/2782-4586-2026-35-42,exclude,not about prompt injection,title_abstract,,,Managing information security risks in the implementation of autonomous artificial intelligence agen
doi:10.5281/zenodo.21235384,exclude,system level,title_abstract,,,"Many Minds, One Verdict: The Limits of LLM Consensus as a Security Gate — Reproducibility Artifact"
doi:10.21203/rs.3.rs-10868139/v1,exclude,system level,title_abstract,,,Mastyf Guard 1.5B: Cognitive Harvard Architectures for Sub-Millisecond AI Agent Perimeter Defense an
doi:10.5281/zenodo.20537888,exclude,detector only,full_text,,full text: AgentShield thesis: proxy that scans fetched web content with a second LLM before it reaches the agent,Mbrojtja e agjentëve të bazuar në LLM nga sulmet e prompt injection
doi:10.2139/ssrn.7276838,exclude,attack only,title_abstract,,,Measuring Indirect Prompt Injection in Autonomous Web Agents
doi:10.5281/zenodo.23163268,exclude,"survey, position or other",title_abstract,,corrected on rereading the abstract: defences run on a simulated model whose susceptibility is a set parameter; the authors state the rates transfer to no deployed language model,Measuring the utility cost and generalization of prompt-injection defenses with a controlled simulat
doi:10.1109/icesst69086.2026.11582854,exclude,system level,title_abstract,,,MedRAGShield: A Three-Tier Defense Framework Against Document Poisoning and Adversarial Attacks in M
doi:10.5281/zenodo.21192008,exclude,not about prompt injection,title_abstract,,,Meme Immunology: How Artificial Intelligence Learns to Protect Its Own Mind
doi:10.5281/zenodo.20451351,exclude,not about prompt injection,title_abstract,,,Metadata Packet for AI Indexing — Non-Injection Semantic Governance Address
doi:10.5281/zenodo.21221544,exclude,not about prompt injection,title_abstract,,,"Microbial Collective Intelligence: Quorum, Biofilm, Genomic, and Ecological Mechanisms for Adaptive "
doi:10.5281/zenodo.22886107,exclude,system level,title_abstract,,,Mitigating Direct and Indirect Prompt Injection in Production LLM Agents: A Defense-in-Depth Archite
doi:10.20944/preprints202609.1352.v1,exclude,detector only,title_abstract,,voting-based routing with detection,Mitigating LLM Prompt Injections via Multi-Vector Representation and Voting-Based Routing
doi:10.2139/ssrn.7502658,exclude,"survey, position or other",title_abstract,,,Mitigating Prompt Injection Attacks in Large Language Model Applications: A Defense-In-Depth Framewo
doi:10.47810/clib.26.03,exclude,detector only,title_abstract,,,Mitigating Security Vulnerabilities in Open-Source Large Language Models via a Supervisor Agent Arch
doi:10.1109/icws72778.2026.00153,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1109/icws72778.2026.00153 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Mitigating Semantic Confusion in LLM-Integrated Web Services via Dynamic Dual-Stream Intent Matching
doi:10.2139/ssrn.6995839,exclude,"survey, position or other",title_abstract,,,"Model-Side Safety Numbers Do Not Reproduce, and the Best Detectors Cannot See Closed Models: A Measu"
doi:10.1007/978-981-92-3426-4_26,exclude,detector only,title_abstract,,,MongoGuard: A Mongolian Prompt Injection Attack Detection Model Based on Contrastive Learning
doi:10.5281/zenodo.23519903,exclude,not about prompt injection,title_abstract,,,Monograph 42: The Jain Silicon Protocol
doi:10.21275/sr26331105045,exclude,not about prompt injection,title_abstract,,,Multi-Cloud Security Assessment for AI-Augmented Enterprise Environments
doi:10.1109/icaic67076.2026.11395708,exclude,system level,title_abstract,,,Multi-Layered Defense Proxy for Home Assistant utilizing Large Language Models
doi:10.1109/icaic67076.2026.11395741,exclude,system level,title_abstract,,,Multi-Layered Security Framework for Financial AI Solutions - PISA
doi:10.5281/zenodo.20628935,exclude,detector only,title_abstract,,,Multi-Turn Distributed Prompt Injection Detection: Dataset and Model Weights
doi:10.1109/qpain69676.2026.11545895,exclude,"survey, position or other",title_abstract,,,Multimodal Prompt Injection: A Formal 4D Taxonomy for Image and Document Pipelines
doi:10.5281/zenodo.21416032,exclude,attack only,title_abstract,,,Multimodal Prompt Injection: A Systematic Evaluation of Cross-Channel Attack Vectors Against LLM Saf
title:multimodalpromptinsertionstrategiesforvisionlanguagefoundationmodels,exclude,not about prompt injection,title_abstract,,,Multimodal Prompt Insertion Strategies for Vision-Language Foundation Models
doi:10.48175/ijarsct-37344,exclude,not about prompt injection,title_abstract,,,"Natural Language to SQL (NL2SQL): A Comprehensive Study of Text-to-SQL Systems, Conversational AI fo"
doi:10.5281/zenodo.22167307,exclude,"survey, position or other",title_abstract,,position piece on static vs adaptive numbers,"Near-Zero Until Someone Tries: What a Prompt-Injection Defense Number Measures, Why Static and Adapt"
doi:10.5281/zenodo.22874031,exclude,detector only,title_abstract,,code record of ClinicalShield detection framework,NehlTech/ClinicalShield: ClinicalShield: A Multi-Layer Defence Framework for Detecting and Neutraliz
doi:10.5281/zenodo.18894317,exclude,detector only,title_abstract,,,Nokast-SecureRAG: An Open-Source Framework for Trusted Local AI via SLM-Driven Defense
doi:10.5281/zenodo.23207842,exclude,not about prompt injection,title_abstract,,,Normative State Locking and Bounded Activation Dynamics: An Information-Theoretic Boundary Framework
doi:10.36227/techrxiv.177220281.16680821/v1,exclude,system level,title_abstract,,model rotation as moving-target defence,Not All Models Are Equal: Model Selection and Rotation as Moving Target Defense Against Prompt Injec
doi:10.1109/icassp55912.2026.11462113,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1109/icassp55912.2026.11462113 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Not Just Detection: Aligned-Driven Purification of Indirect Prompt Injection for Reliable Agent Inte
doi:10.5281/zenodo.18712066,exclude,detector only,full_text,,full text: Nullform: deterministic rewriting rules that detect structural injections,Nullform: Algebraic Prompt Injection Defense via Monoid Rewriting Systems
doi:10.62009/emeron.2764.9679.v2.2026.e557,exclude,not about prompt injection,title_abstract,,,O Ensino Ético da Inteligência Artificial na Graduação em Direito: Fraudes por Prompt Injection e a
doi:10.5281/zenodo.19193235,exclude,not about prompt injection,title_abstract,,,"OEIL+FORGE: Autonomous Generation, Certification and Deployment of AI Specialists — 12 Claims"
doi:10.5281/zenodo.23991498,exclude,not about prompt injection,title_abstract,,,"OSIRIS 6.0: The Self-Compressing Sovereign Architecture across Post-Quantum Enclaves, Neural QLDPC D"
doi:10.17605/osf.io/qvg7u,exclude,not about prompt injection,title_abstract,,,OVERVIEW: What Is the SSPS IP Stack?
doi:10.5281/zenodo.18137850,exclude,not about prompt injection,title_abstract,,,Offensive Defense: A Framework for Autonomous Cybersecurity Systems Using Adversarial Technique Weap
doi:10.5281/zenodo.21362167,exclude,not about prompt injection,title_abstract,,,Offensive Security in the AGI Era: A Framework for Pre-Instantiation Attacks and AGI-Resistant Foren
doi:10.5281/zenodo.18701414,exclude,system level,title_abstract,,,Omega Walls: A Stateful Trust Boundary for RAG and Tool-Using Agents [Reference Implementation]
doi:10.5281/zenodo.24275531,exclude,"survey, position or other",title_abstract,,,"One Flaw, One Halt: Hallucination, sycophancy and prompt injection as missing-variable failures, and"
doi:10.5281/zenodo.20262258,exclude,not about prompt injection,title_abstract,,,Ontological Firewall Pattern — UNSAT par défaut sur opérateur non-indexé (LACF)
doi:10.2139/ssrn.6970500,exclude,attack only,title_abstract,,,Ontology Poisoning in Military AI Fusion Systems: Attack Vectors Against Palantir-Class Architecture
doi:10.1109/jas.2026.126209,exclude,attack only,title_abstract,,,OpenClaw in the Wild: Security Analysis of Autonomous Agents
doi:10.3390/math14132434,exclude,system level,title_abstract,,,Operator-Blind Secret Mediation for AI Agents: A Formal Model and FHE Construction for Credential De
title:optimizedmultiagentdefensepipelineagainstadvancedpromptinjectionattacks,exclude,detector only,title_abstract,,,Optimized Multi-Agent Defense Pipeline Against Advanced Prompt Injection Attacks
doi:10.59324/ejaset.2026.4(2).16,exclude,"survey, position or other",title_abstract,,,Orchestration and Verification of Agentic AI Systems: A Survey of Multi-Agent Collaboration and Safe
doi:10.5281/zenodo.23867644,exclude,attack only,title_abstract,,,"Outside the Prompt: Measuring Where AI Coding Agents Trust Repositories, Tools and Connectors"
doi:10.21203/rs.3.rs-9033885/v1,exclude,not about prompt injection,title_abstract,,different sense of prompt injection,Overcoming the Semantic Bottleneck for Deterministic Structural Control in Text-to-Image Synthesis
doi:10.5281/zenodo.24085150,exclude,detector only,title_abstract,,,PI-Loc: span-annotated benchmark and explainability audit harness for prompt-injection guardrails
doi:10.2139/ssrn.6739079,exclude,detector only,title_abstract,,,PIDL: A Prompt Injection Detection Layer for LLM-Integrated Web Applications via Multi-Signal Semant
doi:10.5281/zenodo.20189704,exclude,not about prompt injection,title_abstract,,,"PLEXUS 12 nucleo 8.5 anexos 8.4, 9, 10, 11: Formalismo de la Tercera Conciencia"
doi:10.1145/3830454.3846434,include,,title_abstract,,role: re-measurement; mitigation strategies in RAG (CCS poster),POSTER: Evaluating Mitigation Strategies Against Prompt Injection Attacks in Retrieval-Augmented Lan
doi:10.1145/3830454.3846411,exclude,not about prompt injection,title_abstract,,,POSTER: ROS2Pot: An Agentic Honeypot for Proactive Defense Against LLM Agents in ROS 2/DDS Systems
doi:10.1145/3779208.3804880,include,,title_abstract,,SecAlign distilled to edge models via on-policy RL (CCS poster),POSTER: Small but Secure: Distilling SecAlign Defense on Edge LLMs via On-Policy RL
doi:10.5281/zenodo.23197145,exclude,not about prompt injection,title_abstract,,,"PREreview of ""Beyond Static Sandboxing: Learned Capability Governance for Autonomous AI Agents"""
doi:10.58532/nbennur3342c9,exclude,"survey, position or other",title_abstract,,,PROMPT INJECTION AND AI MANIPULATION: THE NEXT FRONTIER OF CYBER ATTACKS
doi:10.58532/nbennur3409c9,exclude,"survey, position or other",title_abstract,,,PROMPT INJECTION AND JAILBREAK ATTACKS ON LARGE LANGUAGE MODELS: THREATS AND DEFENSE STRATEGIES
doi:10.30546/2225-0530.14.2.2025.2013,exclude,"survey, position or other",title_abstract,,,"PROMPT INJECTION ATTACKS IN LARGE LANGUAGE MODELS VIA A COMPREHENSIVE ANALYSIS OF ATTACK VECTORS, DE"
title:parameterefficientadaptationofdinov3formultitaskultrasoundimageanalysis,exclude,not about prompt injection,title_abstract,,,Parameter-Efficient Adaptation of DINOv3 for Multi-Task Ultrasound Image Analysis
doi:10.5281/zenodo.18236867,exclude,system level,title_abstract,,patent filing,Patent Filing: Anti-Binding Protection Layer for AI-Facing Documents
doi:10.5281/zenodo.18486434,exclude,system level,title_abstract,,patent filing,Patent Filing: Non-Blocking Integrity Halt Primitive for Intelligent Systems
title:pathawaredefenseagainstpromptinjectionattacksinretrievalaugmentedlargelanguagemodels,exclude,no full text obtainable,full_text,not retrieved: no identifier; search the title in a browser or the library catalogue,full text: not obtained by script or by hand,Path-Aware Defense Against Prompt Injection Attacks in Retrieval-Augmented Large Language Models
doi:10.5772/intechopen.1017734,exclude,not about prompt injection,title_abstract,,,Perspective Chapter: Governing the Autonomous Fortress – Multi-Agent AI Systems for Secure and Resil
doi:10.1109/aic70896.2026.11717282,exclude,not about prompt injection,title_abstract,,,PhishRoute: Cost-Aware Multi-Agent Gating for Phishing Email Detection
doi:10.5281/zenodo.21349713,include,,title_abstract,,role: re-measurement; six training-free defences on small open-weight tool agents,Poisoned Tools: Prompt-Injection Robustness and Failure Anatomy of Small Open-Weight Tool Agents
doi:10.2139/ssrn.6734225,exclude,system level,title_abstract,,,Poisoning the Compliance Mind: Adversarial Memory Injection Attacks on RAG-Based AML Agents and a De
doi:10.5281/zenodo.21853734,exclude,system level,title_abstract,,,Policy Governed Agentic Commerce
doi:10.17605/osf.io/zakex,exclude,attack only,title_abstract,,corrected on rereading the abstract: a preregistered measurement of where in a document a hidden instruction works best; no defence is studied,Position and indirect prompt injection
doi:10.1145/3846374.3846391,exclude,system level,title_abstract,,,"Position: Agents Orchestrate, Policy Enforces — A Deterministic Tool-Use Control Pattern for Agentic"
doi:10.1109/uemcon70256.2026.11775495,exclude,attack only,title_abstract,,,Positional Survival of Prompt Injections Through Context Compaction in LLM Agents
doi:10.5281/zenodo.20640594,exclude,detector only,title_abstract,,,Post-Detection Remediation of Prompt Injection Attacks in LLM-Based Email Security Agents using Ense
doi:10.1109/svcc69905.2026.11642338,exclude,"survey, position or other",title_abstract,,,Poster: From Assistants to Actors: Deconstructing the Security Implications of Autonomous AI Agents
doi:10.11159/icaiis26.105,exclude,detector only,title_abstract,,,Predictive Behavioral Risk Detection for Agentic AI via Runtime Execution Monitoring
doi:10.6084/m9.figshare.35040148,exclude,no full text obtainable,full_text,"not retrieved: data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF",full text: not obtained by script or by hand,Primary Evaluation Results for 66 Ordered Runtime Defense Variants
doi:10.51903/ijgd.v4i1.3699,exclude,not about prompt injection,title_abstract,,,Privacy and Data-Integrity Risk Cards for LLM Agents: A UI/UX Design Framework for Secure Human Over
doi:10.1109/icici68867.2026.11564888,exclude,detector only,title_abstract,,,Privacy-Preserving Federated Transformer Framework for Detecting Adversarial Prompts in Healthcare C
doi:10.1002/9781394491148.ch1,exclude,not about prompt injection,title_abstract,,,Privacy‐preserving ML and Secure AI Architectures
doi:10.3390/electronics15122567,exclude,not about prompt injection,title_abstract,,,Private-RAG: A Privacy-Preserving Retrieval-Augmented Generation Method for Large Model Inference
title:probabilisticsecuritymappingoflargelanguagemodelintegrationsviastochasticpetrinets,exclude,not about prompt injection,title_abstract,,,Probabilistic security mapping of large language model integrations via stochastic Petri Nets
doi:10.18653/2026.findings-emnlp.1869,exclude,detector only,title_abstract,,,ProbeCourt: Cross-Examining Black-Box LLMs for Prompt-Injection Evidence Before Tool Execution
doi:10.46647/rdems0204018,exclude,detector only,title_abstract,,,Prompt Analysis in Large Language Models Adversarial Prompt Detection
doi:10.68232/cj.e026,exclude,"survey, position or other",title_abstract,,,Prompt Engineering for Reliable Large-Language-Model Interaction: From Instruction Design and Ground
doi:10.1007/978-3-032-25874-8_8,exclude,"survey, position or other",title_abstract,,,Prompt Engineering for Secure AI Design and Cyber Defense
doi:10.2139/ssrn.6828838,exclude,"survey, position or other",title_abstract,,,"Prompt Injection Attacks Against Clinical LLM Agents Accessing Electronic Health Records: A Survey, "
doi:10.5281/zenodo.21357272,exclude,"survey, position or other",title_abstract,,,"Prompt Injection Attacks in Large Language Model Systems: Threats, Mechanisms, and Defense Strategie"
doi:10.3390/info17010054,exclude,"survey, position or other",title_abstract,,,Prompt Injection Attacks in Large Language Models and AI Agent Systems: A Comprehensive Review of Vu
doi:10.5281/zenodo.22396355,exclude,"survey, position or other",title_abstract,,,"Prompt Injection Attacks in Large Language Models: A Systematic Literature Review on Taxonomy, Evalu"
doi:10.5281/zenodo.24302817,exclude,"survey, position or other",title_abstract,,,Prompt Injection Attacks on AI Coding Assistants - A Literature Survey and Layered Defense Framework
doi:10.56975/ijcrt.v14i7.309922,exclude,"survey, position or other",title_abstract,,,Prompt Injection Attacks on Large Language Models: Security Risks and Defence Strategies
doi:10.2139/ssrn.6816879,exclude,"survey, position or other",title_abstract,,,"Prompt Injection Attacks: Theory, Techniques, Defense, and Secure AI Systems"
doi:10.13140/rg.2.2.19516.60800,exclude,"survey, position or other",title_abstract,,,"Prompt Injection Beyond the Input Boundary: A Cross-Surface Perspective on LLM Security, Defense Rob"
doi:10.5281/zenodo.23046762,exclude,system level,title_abstract,,,"Prompt Injection Defense for Tool-Using AI: Production Architecture, Current Research, Real-World Us"
title:promptinjectiondefenses,exclude,no full text obtainable,full_text,not retrieved: no identifier; search the title in a browser or the library catalogue,full text: not obtained by script or by hand,Prompt Injection Defenses
doi:10.1007/978-3-032-30310-3_46,exclude,detector only,title_abstract,,,Prompt Injection Detection via Attention Tracking: A Comparative Evaluation
doi:10.5281/zenodo.20674403,exclude,"survey, position or other",title_abstract,,,Prompt Injection Is Not a Bug You Patch
doi:10.5281/zenodo.18925239,exclude,"survey, position or other",title_abstract,,,Prompt Injection Is the New SQL Injection: Why LLM Security Will Define the Next Decade
title:promptinjectionresistanceinclinicalllmagentsviastructuredmedicalontologyalignment,exclude,no full text obtainable,full_text,not retrieved: no identifier; search the title in a browser or the library catalogue,full text: not obtained by script or by hand,Prompt Injection Resistance in Clinical LLM Agents via Structured Medical Ontology Alignment
doi:10.61424/jcsit.v3i2.1077,exclude,"survey, position or other",title_abstract,,,Prompt Injection Threats in Azure-Based Large Language Model Applications
doi:10.5281/zenodo.19520996,exclude,attack only,title_abstract,,,Prompt Injection and Data Leakage in Large Language Models: An Empirical Study on TinyLlama
doi:10.2139/ssrn.6740060,exclude,"survey, position or other",title_abstract,,,Prompt Injection and Jailbreak Attacks in Large Language Model-Based Agents
doi:10.1002/9781394402069.ch9,exclude,"survey, position or other",title_abstract,,,Prompt Injection and Jailbreak Techniques
doi:10.1109/gaiss66401.2026.11779704,exclude,system level,title_abstract,,,Prompt Injection and Tool-Result Poisoning Against MCP Tool-Use Agents in Privileged Infrastructure:
doi:10.5281/zenodo.22720681,exclude,"survey, position or other",title_abstract,,,Prompt Injection and the Security of LLM Applications and Agents
doi:10.5281/zenodo.21705077,exclude,not about prompt injection,title_abstract,,,"Prompt Injection as a Governance Object: Why Prompt Injection, Data, Model, and Memory Poisoning Are"
doi:10.1109/sera69989.2026.11618608,exclude,"survey, position or other",title_abstract,,,"Prompt Injection as a Systemic Vulnerability in LLM-Based Tools: Threat Models, Empirical Analysis, "
doi:10.5281/zenodo.21830995,exclude,"survey, position or other",title_abstract,,"why detection fails, argument piece with a harness",Prompt Injection as an Architectural Primitive: Why Detection Baselines Fail in Agentic Pipelines
doi:10.1109/iccsdfai70505.2026.11648022,exclude,"survey, position or other",title_abstract,,,Prompt Injection as the New SQL Injection: The Emergence of Attacks on AI Reasoning
doi:10.2139/ssrn.6762100,exclude,not about prompt injection,title_abstract,,,"Prompt Injection in Court Filings: Generative AI in the Brazilian Judiciary, Algorithmic Procedural "
doi:10.1109/worldsuas70880.2026.11767142,exclude,detector only,title_abstract,,,"Prompt Injection in LLMs: Taxonomy, Detection and Adaptive Defense Mechanisms"
doi:10.13140/rg.2.2.36570.30408,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.13140/rg.2.2.36570.30408 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Prompt Injection in Orbit: Evaluating and Defending LLM Agents in Satellite Mission Operations
doi:10.5281/zenodo.21370437,exclude,system level,title_abstract,,,Prompt Injection is Not an AI Problem: Why MCP Tool Hardening Matters
doi:10.5281/zenodo.18366919,exclude,not about prompt injection,title_abstract,,,Prompt Injection is a Control Problem: Topological Constraints on Latent Dynamics
doi:10.5281/zenodo.18203183,exclude,not about prompt injection,title_abstract,,,Prompt Injection: A Security Challenge for Modern AI Communication
doi:10.2139/ssrn.5995215,exclude,attack only,title_abstract,,,Prompt Persistence Attacks: Long-Term Memory Poisoning in LLM-Based Systems
doi:10.17148/iarjset.2026.13801,exclude,not about prompt injection,title_abstract,,,Prompt Security and Insider-Risk Defense in Banking AI Systems
doi:10.5281/zenodo.21120124,include,,title_abstract,,role: re-measurement; factorial study of prompt-transformation defences across benchmarks and model families,Prompt Transformation Effects Do Not Transfer Across Benchmarks or Model Families: A Factorial Study
doi:10.5281/zenodo.19856513,exclude,"survey, position or other",title_abstract,,,Prompt injection attacks and defenses in large language models: A systematic literature review
doi:10.59350/k9aqr-fr116,exclude,not about prompt injection,title_abstract,,,Prompt injection defenses open a biosecurity gap
doi:10.5281/zenodo.23887088,exclude,no full text obtainable,full_text,"not retrieved: data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF",full text: not obtained by script or by hand,"Prompt injection through device data in LLM smart-home agents: simulator, logs and analysis"
doi:10.5281/zenodo.23811220,include,,title_abstract,,role: re-measurement; defences on a small local agent with costs,"Prompt-Injection Defences on a Small Local Agent: What Works, What It Costs, and How Optimistic Deve"
doi:10.5281/zenodo.22070840,exclude,detector only,title_abstract,,,Prompt-Injection Detection at 7B: A Head-to-Head Against a Zero-Shot Frontier Judge (Paper 3 artefac
doi:10.1016/j.neucom.2026.134939,exclude,"survey, position or other",title_abstract,,,"Prompt-based attacks and defenses in large language models: A systematic review of threat models, ta"
doi:10.5281/zenodo.22805797,exclude,attack only,title_abstract,,,Prompt-injection detection and obedience in three Claude models: data and re-derivation
doi:10.1016/j.jss.2026.113164,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1016/j.jss.2026.113164 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Prompt-to-API-Call injection: Systematic attack generation and semantic defense in LLM agents
doi:10.1109/aiei69164.2026.11497133,exclude,detector only,title_abstract,,,PromptFence: A Semantic Guardrail for Healthcare AI to Defend Against Prompt Injection Attacks
doi:10.1038/s41598-025-31086-y,exclude,jailbreak or direct injection only,full_text,,"full text: PromptGuard: four layers (input filter, role-separated prompt format, output critic, response rewrite); the adversary controls the user's own input and cannot touch tools or retrieval, so the attacks are direct; the format layer is measured alone (Table 8) but only against user-typed prompts",PromptGuard a structured framework for injection resilient language models
doi:10.1109/icassp55912.2026.11463282,exclude,jailbreak or direct injection only,full_text,,full text: included at title and abstract; the full text shows the attacks are adversarial user inputs (AdvBench goals with PAIR and AutoDAN),PromptPatch: Towards Precise and Stable Behavioral Patching in Large Language Models via Feedback-Dr
doi:10.3390/fi18070376,exclude,detector only,title_abstract,,,PromptSentinel-X: A Leakage-Aware and Context-Aware Framework for Prompt-Injection Detection in Larg
doi:10.5281/zenodo.21522997,exclude,detector only,title_abstract,,,PromptShield AI: A Multi-Agent Architecture for Intelligent Prompt Injection and Jailbreak Attack De
doi:10.1109/sngc69406.2026.11780023,exclude,detector only,title_abstract,,,PromptShield: Machine-Learning-Based Detection of Prompt Injection Attacks in Large Language Models
doi:10.20944/preprints202610.0008.v1,exclude,detector only,title_abstract,,,PromptWAF: Detecting and Preventing Prompt-Induced Application Vulnerabilities
doi:10.1109/icnccom69117.2026.11736184,exclude,not about prompt injection,title_abstract,,,"Prompting Strategies for Large Language Models: A Structured Review of Techniques, Applications, and"
doi:10.1109/icaic67076.2026.11395889,exclude,"survey, position or other",title_abstract,,,Prompting for LLM Security and RAG: A Survey from Zero-Shot to Automatic Prompt Optimization (APO) a
doi:10.5281/zenodo.19614897,exclude,not about prompt injection,title_abstract,,,Protocol Layer Prompt Engineering Specification (PLPES) — draft-reilly-plpes-00
doi:10.5281/zenodo.20371151,exclude,attack only,title_abstract,,,Protocol Pivoting: Cross-Protocol Attack Escalation in Agentic AI Systems
doi:10.6084/m9.figshare.31985511.v1,exclude,duplicate version,title_abstract,,data record of the dental VLM study (record 1654),ProvDent: Inference call records for image-embedded prompt injection attacks on dental VLMs
doi:10.5281/zenodo.19603811,exclude,system level,full_text,,"full text: UCCA: composite of smoothing, erase-and-check, control-barrier action filtering and zero-knowledge proofs; components not measured separately",Provable and Practical Prompt Injection Resilience in Autonomous LLM Agents
doi:10.5281/zenodo.23783426,exclude,system level,title_abstract,,,Provenance Constrained Evidence Ranking Reference Implementation and Reproducible Experiments
doi:10.5281/zenodo.21615140,exclude,system level,title_abstract,,,Provenance-Agnostic Effect Re-Grounding (PARE) for LLM Payment Agents — Reproducibility Artifact
doi:10.5281/zenodo.21270299,exclude,system level,title_abstract,,AgentLock provenance authorisation,Provenance-Based Pre-Action Authorization for LLM Agents: A Structural Defense Evaluated on AgentDoj
doi:10.54254/2977-3903/2026.39089,exclude,system level,title_abstract,,PGTI provenance gate,Provenance-gated tool invocation for defending LLM agents against indirect prompt injection
doi:10.5281/zenodo.23803919,exclude,not about prompt injection,title_abstract,,,Put Boundaries Around Autonomy: Practical Controls for Rogue Agent Risk
doi:10.18653/2026.findings-emnlp.1312,exclude,attack only,title_abstract,,,Query-Efficient Memory Injection Attacks on LLM Agents
doi:10.1109/itc-egypt70422.2026.11765862,exclude,detector only,title_abstract,,,RAG-Anti-Poisoning: A Hybrid Defense Framework Against Prompt Injection and Corpus Poisoning Attacks
doi:10.2139/ssrn.6910198,exclude,not about prompt injection,title_abstract,,,RAG-Based Retrieval Systems for Enterprise Mainframe Operations: Accelerating Incident Response and
doi:10.34190/eccws.25.1.4741,exclude,system level,title_abstract,,,RAG-H: A RAG-Hardened SOC Framework for Trustworthy Threat Intelligence and AI Defence
doi:10.1109/itc-egypt70422.2026.11765637,exclude,detector only,title_abstract,,,RAG-Shield: A Three-Layer Ensemble Defense Against Prompt Injection in RAG Pipelines
doi:10.5281/zenodo.23866018,exclude,benchmark only,title_abstract,,,"RAGProbe: canary-scored prompt-injection testing for LLM, RAG and agent applications"
doi:10.20944/preprints202609.2192.v1,exclude,system level,title_abstract,,,RAGSec: A Defense Framework for RAG Under Adaptive Knowledge Poisoning
doi:10.18653/v1/2026.findings-acl.738,exclude,detector only,title_abstract,,RAP-ID mechanistic detector,RAP-ID: Mechanistic Prompt Injection Detection via Impostor Behavior Analysis
doi:10.1109/jiot.2026.3717740,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1109/jiot.2026.3717740 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,RAPID: A Resource-Aware Prompt Injection Defense for LLM-Enabled Threat Detection
doi:10.18653/v1/2026.acl-industry.127,exclude,detector only,title_abstract,,,RAPIDS: Resume Attack Prompt Injection Detection at Scale
doi:10.1109/dsn69566.2026.00078,exclude,attack only,title_abstract,,,RIPE-II: Retrieval In-Place Poisoning Evaluation with Indirect Injections
doi:10.1109/access.2026.3729141,exclude,not about prompt injection,title_abstract,,,RIPPLE: Differential Testing of Exact-Value Restoration Boundaries in Privacy-Preserving LLM Pipelin
doi:10.5281/zenodo.24304831,exclude,not about prompt injection,title_abstract,,,RS-2026-105: The Defence Is the Attack Surface: Weaponising Security Controls in Agentic AI Systems
doi:10.20944/preprints202603.1023.v1,include,,full_text,,"full text: Real User Instruction: prompt-level separation of user instructions from data, with the model made to list suspected injections before answering; compared with spotlighting baselines on indirect injection (Table 2)",Real User Instruction: Black-Box Instruction Authentication Middleware Against Indirect Prompt Injec
doi:10.71229/5cn8a439,exclude,detector only,title_abstract,,,Real-Time Detection and Mitigation of Prompt Injection Attacks in LLM-Integrated Enterprise Systems
doi:10.2139/ssrn.7589166,exclude,not about prompt injection,title_abstract,,,"Real-Time Intrusion Detection Using Honeypots and Large Language Models: An Extended Architecture, S"
doi:10.5281/zenodo.23534006,exclude,"survey, position or other",title_abstract,,,Real-Time Prompt Injection and Guardrails for Large Language Model Applications: A Review of Attacks
doi:10.5281/zenodo.21891403,exclude,"survey, position or other",title_abstract,,,"Reasoning Is Not Provenance: Channel Authority, Context-Boundary Marking, and the Limits of Judgment"
doi:10.5281/zenodo.23412665,exclude,not about prompt injection,title_abstract,,,Receipts — a note on verification at the velocity of the work
doi:10.18653/2026.findings-emnlp.830,exclude,system level,title_abstract,,RecoverAgent post-hoc recovery,RecoverAgent: Intent-Preserving Recovery from Indirect Prompt Injection in Tool-using LLM Agents
doi:10.5281/zenodo.23953154,include,,title_abstract,,role: re-measurement; prompt-level vs server-side controls on a composed application,"Red Teaming a Composed RAG-Agent LLM Application: Measuring Prompt Injection, Secret Leakage and Too"
doi:10.1109/southeastcon63549.2026.11475947,exclude,system level,title_abstract,,,Red-Team Prompt Injection Defense Using Multi-Context Protocol (MCP)
doi:10.5281/zenodo.21231387,exclude,system level,title_abstract,,context governance pipeline,Reflective Context Governance Reduces Context-Borne LLM Failures — A Controlled N=120 RAW-vs-CLEAN E
doi:10.5281/zenodo.20775039,exclude,not about prompt injection,title_abstract,,,Relational Interaction Middleware (RIM): Executive Brief — A Model-Agnostic Governance Layer for Saf
doi:10.5281/zenodo.24356180,exclude,not about prompt injection,title_abstract,,,"Reliable AI Systems Design: A Failure-Aware Framework for Security, Control, Verification, and Recov"
doi:10.5281/zenodo.20731176,exclude,duplicate version,title_abstract,,replication package of record 1562,Replication package — Evaluating Indirect Prompt-Injection Defenses in Tool-Using LLM Agents: Securi
doi:10.5281/zenodo.24203672,exclude,not about prompt injection,title_abstract,,,RepoPilot: Human-in-the-Loop AI Software Engineering with Multi-Agent Consensus and Defense-in-Depth
doi:10.5281/zenodo.19899470,exclude,jailbreak or direct injection only,title_abstract,,,Residual Manipulability of Frontier Aligned Language Models Under Published Prompt-Attack Classes: A
doi:10.21203/rs.3.rs-11428254/v1,exclude,system level,title_abstract,,,Residual Privacy Leakage after Output Scrubbing in Tool-Using LLM Agents under Indirect Prompt Injec
doi:10.1007/979-8-8688-2758-7_17,exclude,not about prompt injection,title_abstract,,,Responsible AI Foundations
doi:10.18653/v1/2026.findings-acl.1191,exclude,benchmark only,title_abstract,,re-assessment of attack evaluation; check for defence results at full text,Rethinking Assessments of Prompt Injection Attacks
doi:10.2139/ssrn.6869741,exclude,"survey, position or other",title_abstract,,,Rethinking Privacy and Security in the Era of Decentralized and Agentic AI: A Systematic Literature
doi:10.2139/ssrn.7473518,exclude,not about prompt injection,title_abstract,,,Retrieval-Augmented Generation for Secure Natural-Language Interaction with Enterprise Data
doi:10.5281/zenodo.18379320,exclude,not about prompt injection,title_abstract,,,"Reverse Engineering a Large-Scale Multimodal AI System: Architectural, Security, and Economic Intell"
doi:10.5281/zenodo.24237513,exclude,"survey, position or other",title_abstract,,,Riscos de segurança de LLMs abertos pequenos executados localmente em português do Brasil: revisão e
title:riskawarenesswarningsinaibrowsershowwarningdesigninaibrowsersinfluencesusersunderstandingo,exclude,not about prompt injection,title_abstract,,,Risk Awareness Warnings in AI Browsers : How Warning Design in AI browsers Influences Users’ Underst
doi:10.5281/zenodo.24129237,exclude,not about prompt injection,title_abstract,,,Risks in Multi-domain AI Assistants: A Literature-Based Analysis with Focus on the Indian Context
doi:10.3389/fcomp.2026.1735253,exclude,system level,full_text,,"full text: RoLLMRec: prompt filtering, retrieval grounding, trust-aware scoring and auditing around an LLM recommender",RoLLMRec: a robust LLM-based recommender system for defending against shilling and prompt injection
doi:10.2139/ssrn.6969762,exclude,detector only,title_abstract,,,Runtime Attention-Anomaly Circuit Breakers: Bounding Execution Risk in Agentic AI via Calibrated Heu
doi:10.5281/zenodo.20579699,exclude,not about prompt injection,title_abstract,,,"Runtime Integrity Without Attestation: How Agent-Facing Tool Surfaces, Speculative Dispatch, and Mem"
doi:10.3390/electronics15132829,exclude,system level,title_abstract,,,Runtime Policy Enforcement for MCP-Based LLM Agents
doi:10.1109/gaiss66401.2026.11779657,exclude,system level,title_abstract,,PIGUARD runtime framework,Runtime Prompt Injection Defense for Production-Scale AI Agent Systems
title:scalesdualinformationtheoreticapproachestomitigatingandquantifyingpromptinjectionsinragpip,exclude,no full text obtainable,full_text,not retrieved: no identifier; search the title in a browser or the library catalogue,full text: not obtained by script or by hand,SCALES: Dual Information-Theoretic Approaches to Mitigating and Quantifying Prompt Injections in RAG
doi:10.1109/eicars71536.2026.11694445,exclude,detector only,title_abstract,,,SCD-Net: A Deep Semantic Contrastive Defense Network Against Prompt Injection in Power Grid LLMs
doi:10.17632/7tnzpfdj93,exclude,system level,title_abstract,,SCI-Guard counterfactual intervention (data record),SCI-Guard: Experimental Data and Reproducibility Materials
doi:10.1016/j.cose.2026.105237,exclude,benchmark only,title_abstract,,,SEAL: An integral methodology for evaluating the security of agentic LLMs
doi:10.58532/nbennurissdb3p4c1,exclude,"survey, position or other",title_abstract,,,"SECURITY OF LARGE LANGUAGE MODELS: THREATS, DEFENSES, AND EMPIRICAL EVALUATION"
doi:10.29121/digisecforensics.v3.i1.2026.98,exclude,"survey, position or other",title_abstract,,,"SECURITY THREATS TO AI AGENTS: REVIEW ON THREATS, DEFENSE STRATEGIES AND THREAT MODELLING"
doi:10.28925/2663-4023.2026.32.1187,exclude,not about prompt injection,title_abstract,,,SECURITY-AS-CODE APPROACH FOR AUTOMATING PCI DSS COMPLIANCE USING AUTONOMOUS AGENTS BASED ON LARGE L
doi:10.35720/julia.v6i2.29,exclude,not about prompt injection,title_abstract,,,SENTINEL-CAI: Secure Ensemble of Neural Threat Intelligence with Noise-resilient Extraction Learning
doi:10.28925/2663-4023.2026.35.1330,exclude,system level,title_abstract,,,SIA-GUARD: КОНТЕКСТНО-ЗАЛЕЖНИЙ БЕЗПЕКОВИЙ КОНТРОЛЬ НА ЕТАПІ ВИКОНАННЯ ДІЙ АГЕНТІВ ВЕЛИКИХ МОВНИХ МОД
doi:10.5281/zenodo.21803052,exclude,attack only,title_abstract,,,SPECTER METADATA: Agent Data Injection Weaponisation — The Attack That Gives No Instruction
doi:10.21203/rs.3.rs-10142899/v1,exclude,not about prompt injection,title_abstract,,,SQCAFusion: Multi-Scale Scene-Query Cross-Attention for Illumination-Adaptive Infrared and Visible I
doi:10.5281/zenodo.23715017,exclude,system level,title_abstract,,,"SRA-RiskGate: A Benchmark and a 4B Model for Gating Stablecoin Payments Made by AI Agents, and the C"
doi:10.18653/2026.findings-emnlp.891,exclude,detector only,title_abstract,,,STARS: Skill Triage via Activation Risk Scoring for LLM Agents
doi:10.35444/ijana.2026.18105,exclude,system level,title_abstract,,,Safe Intelligence for All: Building Adversarially Resilient LLM Interfaces for CTI
doi:10.48448/6y9j-xs28,exclude,jailbreak or direct injection only,title_abstract,,,Safe-Unsafe Concept Separation Emerges from a Single Direction in Language Models Activation Space
doi:10.20944/preprints202609.1881.v1,include,,full_text,,"full text: full-text check: data markers with a system-prompt rule and a safety check inside the agent's own reasoning loop, added over filtering and sandboxing (Table 1), with a component ablation",SafeAgent: Dual-Objective Optimization Against Prompt and Tool Injection in Interactive Environments
doi:10.5281/zenodo.23933491,exclude,not about prompt injection,title_abstract,,,SafeLabs-Eval ASI01 Through Clinical Notes: Independent External Evaluation on a Dental SaaS Applica
doi:10.5281/zenodo.20772702,include,,title_abstract,,role: re-measurement; cross-family portability of defences in SOC triage,"Same Attack, Different Fates: Cross-Family Portability Failure of Prompt-Injection Defenses in LLM-P"
doi:10.5281/zenodo.23511832,exclude,"survey, position or other",title_abstract,,"measurement of the published evaluation record, a survey-type study","Same Label, Different Experiment: Measuring the Prompt-Injection and Jailbreak Defense Evaluation Re"
doi:10.5281/zenodo.21614865,exclude,system level,title_abstract,,,Sanitising Onchain Text Before Agent Ingestion
doi:10.32604/cmc.2026.089268,exclude,detector only,title_abstract,,,SchemaGuard: Structure-Aware Detection of Tool-Poisoning and Skill-File Attacks in the LLM Agent Too
doi:10.5281/zenodo.21850255,exclude,system level,title_abstract,,,ScholarShield: Zero Trust Security Architecture for Academic Document Repositories and AI Pipelines
doi:10.1109/icsit70359.2026.11766006,exclude,system level,title_abstract,,,Secure Agentic AI Systems: A Layered Defense Framework
doi:10.5281/zenodo.24049560,exclude,system level,title_abstract,,,Secure Intent-Driven Internet Execution: A Threat Model and Defense Architecture for Autonomous Capa
doi:10.1109/icnpcv69434.2026.11708929,exclude,not about prompt injection,title_abstract,,,Secure Knowledge Extraction From Large Language Model
doi:10.1109/icecet65726.2026.11632629,exclude,system level,title_abstract,,,"Secure MCP-Based Tool-Augmented RAG for Industrial IoT Diagnostics Under Indirect Prompt Injection, "
doi:10.1016/j.energy.2026.140027,exclude,not about prompt injection,title_abstract,,,Secure and trustworthy energy systems: A four-layer threat model and defense-in-depth framework
doi:10.1109/southeastcon63549.2026.11475941,exclude,system level,title_abstract,,,Secure by Design: Quantifying Architectural Resilience and the Agents Rule of two
doi:10.37547/ajast/volume06issue08-08,exclude,not about prompt injection,title_abstract,,,Secure-XPAPF: A Cyber-Secure and Explainable Deep Learning Framework for Automated Pronunciation Ass
doi:10.3390/app16167974,exclude,system level,title_abstract,,,SecureMCP: Policy-Enforced Defense Against Prompt Injection in LLM-Generated SQL for AIoT Databases
doi:10.3390/sym18091565,exclude,system level,title_abstract,,,SecurePrompt-IntegrityNet: Prompt-Injection-Resilient Data Integrity Verification for Agentic LLM Ne
doi:10.5281/zenodo.23158671,exclude,system level,title_abstract,,"training-free five-layer defence, filters and detectors",SecureRAG: A Training-Free Layered Defence Against Prompt Injection in Retrieval-Augmented Generatio
doi:10.5281/zenodo.23588032,exclude,attack only,title_abstract,,,Securing AI agents on instant-payment rails: a proof of concept on AI security for fast payment syst
title:securingaimodelsagainstbackdoorsjailbreaksandpromptinjections,exclude,detector only,full_text,,"full text: thesis; its only prompt-injection method, LCF, is a runtime monitor that flags inputs and abstains","Securing AI models against backdoors, jailbreaks, and prompt injections"
doi:10.56201/wjimt.v8.no6.2024.pg217.248,exclude,"survey, position or other",title_abstract,,,Securing Agentic AI Enterprise Workflows Against Prompt Injection Tool Poisoning Memory Manipulation
doi:10.5281/zenodo.23876687,exclude,attack only,title_abstract,,,"Securing Agentic AI in the Cloud: Tool Poisoning, Rug Pulls and Goal Hijacking in MCP-Based Educatio"
doi:10.5281/zenodo.21842991,exclude,detector only,title_abstract,,,Securing Autonomous AI Agents: Detecting and Mitigating Prompt Injection Attacks in LLM-Based System
doi:10.1109/msec.2026.3727871,exclude,"survey, position or other",title_abstract,,,Securing Autonomous Large Language Model Agents: Lessons From OpenClaw
doi:10.2139/ssrn.7381439,exclude,system level,title_abstract,,,Securing Bounded-Domain LLM Systems Using Deterministic Natural-Language Security Gateways
doi:10.1109/iscit70453.2026.11695371,exclude,system level,title_abstract,,,Securing Customer Data and Tool-Using AI Agents in Retail Banking
doi:10.66104/3m1m5x77,exclude,"survey, position or other",title_abstract,,,Securing Enterprise Language Model Applications Against Data Leakage and Response Manipulation
doi:10.2139/ssrn.7184619,exclude,not about prompt injection,title_abstract,,,Securing Federated Model Context Protocol (MCP) and Multi-Agent Supply Chains
doi:10.5120/ijcaa46e9a93b9d5,exclude,system level,title_abstract,,,"Securing Generative-AI and AIOps Systems Against Prompt Injection, Data Poisoning, and Adversarial A"
title:securingintelligencethestrategicnecessityofairgappedaisystemsintheageofcloudbasedllms,exclude,not about prompt injection,title_abstract,,,Securing Intelligence: The Strategic Necessity of Air-Gapped AI Systems in the Age of Cloud-Based LL
doi:10.1007/978-3-032-29251-3_6,exclude,"survey, position or other",title_abstract,,,"Securing LLM Agents in Production: Threat Models, Defenses, and Operational Trade-Offs"
doi:10.2139/ssrn.6340078,exclude,"survey, position or other",title_abstract,,,"Securing LLM Powered AI Browsers Against Prompt Injection: A Comprehensive Survey, Threat Taxonomy, "
doi:10.1007/s10664-026-10820-8,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.1007/s10664-026-10820-8 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,"Securing LLM-in-the-loop software for empirical study of risks, mitigations, and utility trade-offs "
title:securinglargelanguagemodelsagainstpromptinjectionanddataleakageattackshttpsdoiorg105281zen,exclude,"survey, position or other",title_abstract,,,Securing Large Language Models Against Prompt Injection and Data Leakage Attacks: https://doi.org/10
doi:10.70593/978-93-7185-447-4,exclude,"survey, position or other",title_abstract,,,"Securing Large Language Models: Adversarial Attacks, Data Privacy, and Artificial Intelligence Safet"
doi:10.3390/app162111039,include,,title_abstract,,role: re-measurement; cross-model and cross-lingual evaluation of IPI defences in local RAG assistants (Appl. Sci. 16(21):11039),Securing Locally Deployed LLM-RAG Assistants: A Cross-Model and Cross-Lingual Evaluation of Indirect
doi:10.20944/preprints202607.0009.v1,exclude,system level,title_abstract,,,Securing Model Context Protocol Servers: Security Assessment and a Lightweight Wrapper-Based Hardeni
doi:10.1007/978-3-032-28591-1_50,exclude,"survey, position or other",title_abstract,,,"Securing Retrieval-Augmented Generation: Prompt Injection, Exfiltration Paths, and Practical Defense"
doi:10.3390/computation14050098,exclude,system level,title_abstract,,,Securing Tool-Using AI Agents Against Injection and Authority Misuse
doi:10.1016/j.icte.2026.10.014,exclude,"survey, position or other",title_abstract,,,Securing next-generation edge intelligence: A survey from federated learning to autonomous agents in
doi:10.5038/tzlj6645,exclude,not about prompt injection,title_abstract,,,Securing the AI Supply Chain: Safeguarding U.S. Advantage in the Age of Generative AI
doi:10.3390/jcp6020063,exclude,"survey, position or other",title_abstract,,,"Securing the Cognitive Layer: A Survey on Security Threats, Defenses, and Privacy-Preserving Archite"
doi:10.3390/electronics15163700,exclude,"survey, position or other",title_abstract,,"Mourad et al., survey already in the candidate list",Securing the Prompt Pipeline: A Systematic Review of Defense Mechanisms Against Prompt-Based Attacks
doi:10.6084/m9.figshare.34488996,exclude,system level,title_abstract,,,Securing the Retrieval-Augmented Generation Pipeline: A Multi-Layered Defense Architecture Against H
doi:10.18653/v1/2026.acl-industry.58,exclude,system level,title_abstract,,,Securing the Tool Layer: A Threat Taxonomy and Runtime Defense Framework for Model Context Protocol
doi:10.5281/zenodo.22028416,exclude,system level,title_abstract,,CaMeL-based architecture,Security Architecture for AI Agent Systems
doi:10.2139/ssrn.7023538,exclude,"survey, position or other",title_abstract,,,Security Evaluation Gaps in Retrieval-augmented and Agentic LLM Systems: A Focused Review
doi:10.33395/sinkron.v10i1.15662,exclude,attack only,title_abstract,,,Security Evaluation of Indonesian LLMs for Digital Business Using STAR Prompt Injection
doi:10.53348/jsst5st8,exclude,"survey, position or other",title_abstract,,,Security Risk Assessment and Layered Protection Strategies for Large Language Model Banking Chatbots
title:securityrisksoflargelanguagemodelsasystematicliteraturereview,exclude,"survey, position or other",title_abstract,,,Security Risks of Large Language Models: A Systematic Literature Review
doi:10.1109/gaiss66401.2026.11779700,exclude,"survey, position or other",title_abstract,,,Security Threats and Mitigation in Conversational AI Systems
doi:10.21275/sr26316110418,exclude,"survey, position or other",title_abstract,,,Security Threats in the Model Context Protocol: A Comprehensive Survey and Trust Boundary Mitigation
doi:10.1007/978-3-032-27163-1_20,exclude,"survey, position or other",title_abstract,,,Security Vulnerabilities in AI Assistants and Their Implications for IoT-Based Environmental Systems
doi:10.5753/sbc.20981.0.1,exclude,"survey, position or other",title_abstract,,,Security and Privacy in Data-Driven Systems: From Traditional Machine Learning to Large Language Mod
doi:10.3390/ai7050152,exclude,"survey, position or other",title_abstract,,,"Security and Privacy of Large Language Models: Threat Taxonomy, Ethical Implications, and Governance"
doi:10.3929/ethz-c-000808673,exclude,"survey, position or other",title_abstract,,"thesis on ML security and privacy, general",Security and Privacy of Machine Learning Systems
doi:10.2139/ssrn.6199178,exclude,"survey, position or other",title_abstract,,,Security and Risk Implications of Transformer-Based Large Language Models
doi:10.1016/j.inffus.2026.104241,exclude,"survey, position or other",title_abstract,,,Security and privacy in LLMs: A comprehensive survey of threats and mitigation strategies
doi:10.1007/978-3-032-25789-5_5,exclude,not about prompt injection,title_abstract,,,Security as an Ethical Imperative in AI Systems
doi:10.2139/ssrn.6285458,exclude,not about prompt injection,title_abstract,,,"Security in Virtualized, Containerized, and Cloud-Native Environments: Vulnerabilities and Defensive"
doi:10.2139/ssrn.7592098,exclude,"survey, position or other",title_abstract,,,"Security of Large Language Models: A Structured Survey of Attack Surfaces, Behavioral Vulnerabilitie"
doi:10.82419/323,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.82419/323 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Security of Vision-Language Model Agents Against Prompt Injection: From Text-Only Tool Calling to Vi
doi:10.3390/electronics15225229,exclude,"survey, position or other",title_abstract,,,"Security, Privacy, and Trustworthy Design for Retrieval-Augmented Generation Systems"
doi:10.1109/southeastcon63549.2026.11476026,exclude,system level,title_abstract,,,Security-Aware Retrieval-Augmented Generation System for Post-Quantum Cryptography Intelligence
title:selfconsistencycheckingforindirectpromptinjectiondefenseinemailagents,exclude,system level,title_abstract,,self-consistency checking of agent actions,Self-consistency checking for indirect prompt injection defense in email agents
doi:10.1109/iccvbic71195.2026.11689044,exclude,detector only,title_abstract,,,Semantic Consistency Drift Monitoring for Training-Free Adversarial Prompt Injection Detection in Ag
doi:10.5281/zenodo.21985436,exclude,not about prompt injection,title_abstract,,,"Semantic Crosswalk and Type-Separation Canon Locks: Load-Bearing Terms, Phase Boundaries, and Non-Su"
doi:10.5281/zenodo.20834834,exclude,detector only,title_abstract,,,Semantic Taint Propagation: Embedding-Based Semantic Flow Monitoring for Multi-Agent Large Language
doi:10.1007/978-3-032-24360-7_30,exclude,not about prompt injection,title_abstract,,,Sentinel Polity: LLM-Grounded Policy-as-Code for IaC Hardening in Hyperscale Clouds
doi:10.48175/ijarsct-40823,exclude,detector only,title_abstract,,,Sentinel-Rag: A Controlled Comparison of a Scratch-Trained Classifier and a Local LLM Judge for Defe
doi:10.3233/faia261172,exclude,system level,title_abstract,,,SentinelX AI: A Post-Quantum Secured Multi-Agent Defense Framework for LLM-Enabled Systems Against A
doi:10.70917/ijcisim-2026-5724,exclude,system level,title_abstract,,,Shieldgraph: Topology-Aware Runtime Defense For Multi-Agent LLM Systems Against Adversarial Coordina
doi:10.5281/zenodo.22855795,exclude,system level,title_abstract,,,"Shift Intent Left: Intent Contracts, Agency Budgets, and Drift Verification for Securing the Agentic"
doi:10.21203/rs.3.rs-11711485/v1,include,,full_text,,full text: system-prompt policy with a pinned operator key and a signature-verification tool against injected halt requests in tool output; effect differs by model,"Signed to Stop: In-Band Halt Injection as a Denial-of-Service on LLM Agents, and a Model-Dependent A"
doi:10.5281/zenodo.21786156,exclude,duplicate version,title_abstract,,reproducibility bundle of record 2007,"Signed to Stop: Reproducibility Bundle (code, pre-registration, and per-trial data)"
doi:10.1109/sp63933.2026.00241,exclude,attack only,title_abstract,,,Site Isolation is Dead: How Site Isolation is Broken in Agentic Browsers and Extensions
doi:10.5281/zenodo.24213884,exclude,system level,title_abstract,,,"SliceGate: reproducibility package (code, cached agent outputs and results) for SLA-aware verificati"
title:smallruleguardrailsforretrievalaugmentedgeneration,exclude,detector only,title_abstract,,,Small-Rule Guardrails for Retrieval-Augmented Generation
doi:10.5281/zenodo.19512510,exclude,not about prompt injection,title_abstract,,,Social Engineering as Adversarial Branch Selection Under Irreversible Constraint Pruning
doi:10.5281/zenodo.21081172,exclude,not about prompt injection,title_abstract,,,Software Supply Chain Hygiene of Model Context Protocol Servers: An Empirical Study
doi:10.1145/3841580.3841584,exclude,attack only,title_abstract,,,Sold a Lie: Exploring the Influence of Deceptive AI Agents on Consumer Decision-Making in Online Sho
doi:10.5281/zenodo.20258140,exclude,not about prompt injection,title_abstract,,,Sovereignty by Design
doi:10.4218/etr2.70162,exclude,not about prompt injection,title_abstract,,,"Special issue on trustworthy and safe AI: Interpretability, robustness, and security"
doi:10.5281/zenodo.23797101,exclude,system level,title_abstract,,,Stateful Monitor-Router-Rollback Regulation for Robust Functional Recovery in Large Language Model A
doi:10.18653/2026.emnlp-main.2303,include,,full_text,,full text: CBC: supervised fine-tuning and online alignment that teach the model to check task consistency before complying,Staying on Task: Defending Against Prompt Injection via Task Consistency Checking
doi:10.3390/computers15010055,exclude,detector only,title_abstract,,corrected on rereading the abstract: a separate adversarially trained filter removes injected text before the unchanged grading model reads the answer,SteadyEval: Robust LLM Exam Graders via Adversarial Training and Distillation
doi:10.5281/zenodo.19035023,exclude,not about prompt injection,title_abstract,,,Stop the Leak: Securing Your AI's System Instructions
doi:10.5281/zenodo.21939849,exclude,attack only,title_abstract,,,Structural Immunity and Silent Compromise: Prompt Injection Against LLM-Based and Traditional ML Tex
doi:10.5281/zenodo.22877616,exclude,not about prompt injection,title_abstract,,,"Structured PREreview of ""Automating Business Intelligence Requirements with Generative AI and Semant"
doi:10.5281/zenodo.22919716,exclude,not about prompt injection,title_abstract,,PREreview of record 1746,"Structured PREreview of ""Mitigating LLM Prompt Injections via Multi-Vector Representation and Voting"
doi:10.1016/j.patcog.2026.113852,exclude,not about prompt injection,title_abstract,,,Structured prompt-guided knowledge injection for medical image segmentation
doi:10.5281/zenodo.20760894,exclude,"survey, position or other",title_abstract,,,Supply-Chain Ghosts and Inference-Time Phantoms: How Attacker-Controlled Artifacts Exploit the Trust
doi:10.3390/fi18090472,exclude,not about prompt injection,title_abstract,,,Sustainability–Resilience Trade-Offs in Edge-Enabled Systems: A Comprehensive Survey
doi:10.2139/ssrn.5863564,exclude,not about prompt injection,title_abstract,,,Symbolic Trigger Effect in Stateless AI Systems: Observations and Implications
doi:10.5281/zenodo.21882080,exclude,system level,title_abstract,,,TAMCF: A Trust-Aware Multi-Agent Framework for Human-Governed AI-Assisted Cyber Defense of Critical
doi:10.1109/icaiset66439.2026.11541308,exclude,system level,title_abstract,,,TBIVL: A Unified Identity Verification Layer for Secure Agent-to-Service Context Exchange
doi:10.67620/ajousci.240001,exclude,"survey, position or other",title_abstract,,,"THE DUAL ROLE OF LARGE LANGUAGE MODELS IN CYBERSECURITY: THREATS, DEFENSES, AND RESEARCH DIRECTIONS"
doi:10.5281/zenodo.23393577,exclude,not about prompt injection,title_abstract,,,THE EL-RAKHAWI DOCTRINE FOR RETRIEVAL-AUGMENTED NEURO-SYMBOLIC AI A Unified Framework for Logical Ve
doi:10.5281/zenodo.18496129,exclude,not about prompt injection,title_abstract,,,"THE HAPPY HANDSHAKE: HOW A SINGLE LETTER ""B"" UNLOCKED GROK'S IMAGINATION LAYER — AND WHY IT MATTERS "
doi:10.5281/zenodo.21811861,exclude,not about prompt injection,title_abstract,,,THE SEMANTIC VENOM
doi:10.52058/2695-1592-2026-7(62)-438-452,exclude,"survey, position or other",title_abstract,,,"THREAT MODELING OF AUTONOMOUS CODING AGENTS IN CORPORATE MOBILE DEVELOPMENT. PART 1: SYSTEM MODEL, A"
doi:10.22214/ijraset.2026.84360,exclude,detector only,title_abstract,,,"TRACER-AI: A Multi-Layer Explainable Framework for Prompt Injection, Agent Goal Hijacking, and Tool "
doi:10.6084/m9.figshare.31855459,exclude,not about prompt injection,title_abstract,,,Task-Specific LoRA-Adapted Language Models for Disinformation Detection and Factual Biography Genera
doi:10.51582/interconf.19-20.05.2026.017,exclude,"survey, position or other",title_abstract,,,Taxonomy of Prompt Injection Attacks and Analysis of Defense Mechanisms in Large Language Model-Base
doi:10.5281/zenodo.22323361,exclude,system level,title_abstract,,,Technical Architecture for Governing Consequential AI Agent Actions
doi:10.1051/itmconf/20269304002/pdf,exclude,system level,title_abstract,,,Terminal Security Risks and Man-Machine Collaborative Defense Mechanism for OpenClaw
doi:10.5281/zenodo.22709418,exclude,not about prompt injection,title_abstract,,,"The 2026 AI Inflection Series, Chapter 13: How Computer-Using Agents Will Transform Enterprise Execu"
doi:10.1145/3770855.3818247,exclude,not about prompt injection,title_abstract,,,The 2nd SeT-LLM Workshop on Secure and Trustworthy Large Language Models
doi:10.1109/gaiss66401.2026.11779648,exclude,not about prompt injection,title_abstract,,,The AI Agent Passport: A Fraud-First Identity Architecture for Agent-Driven Personal Loan Applicatio
doi:10.1109/access.2026.3658308,exclude,"survey, position or other",title_abstract,,,The AI-Cybersecurity Nexus: How Large Language Models are Reshaping Threat Intelligence and Digital
doi:10.5281/zenodo.20271322,exclude,not about prompt injection,title_abstract,,,"The AI-Induced Ordering Pressure: How Advanced AI Forces Human Systems to Reorganize Memory, Authori"
doi:10.5281/zenodo.22995853,exclude,"survey, position or other",title_abstract,,,"The Adversarial Frontier of Agentic AI: OS-Level Threats, Byzantine Amplification, and Latent Space "
doi:10.1002/9781394402069.ch3,exclude,"survey, position or other",title_abstract,,,The Adversary's Playbook
doi:10.1002/9781394402069.ch1,exclude,"survey, position or other",title_abstract,,,The Age of Intelligent Threats
doi:10.5281/zenodo.21452279,exclude,system level,title_abstract,,,The Agent Runtime Operating Environment (AROE): An Implementation Guide for Secure AI Agents
doi:10.5281/zenodo.20519598,exclude,"survey, position or other",title_abstract,,,The Agentic Attack Surface Is Already Weaponized: A Synthesis of Concurrent Threats Against LLM-Driv
doi:10.5281/zenodo.20519743,exclude,"survey, position or other",title_abstract,,,The Agentic Trust Stack Has No Bottom: Why Every Layer From PRNG to Payment Rail Is a First-Class At
doi:10.5281/zenodo.19568049,exclude,not about prompt injection,title_abstract,,,The Architecture of Erasure: A Forensic Audit of the CollectiveOS Expropriation and the Sovereign Is
doi:10.5281/zenodo.19637258,exclude,not about prompt injection,title_abstract,,,The Architecture of Immutable Governance: Eradicating Systemic Drift in Generative Intelligence via
doi:10.5281/zenodo.19552899,exclude,not about prompt injection,title_abstract,,,"The Architecture of Immutable Peace: Cryptographic Governance, Fiscal Deterrence, and the Transition"
doi:10.2139/ssrn.6494918,exclude,attack only,title_abstract,,,The Atmosphere Attack: Postural Manipulation and the Hidden Threat in Agentic AI
doi:10.68957/ejais-2026-115,exclude,"survey, position or other",title_abstract,,,The Autonomy Paradox: Securing Agentic AI Systems Against MachineSpeed and Semantic Threats
doi:10.5281/zenodo.20838836,exclude,not about prompt injection,title_abstract,,,The Claude Fable 5 & Claude Mythos 5 System Card: A Detailed Analysis
doi:10.5281/zenodo.23065786,exclude,not about prompt injection,title_abstract,,,"The Competence Gap: How Unskilled Use of AI Prompts, Agents, Skills and Code Generators Opens New At"
doi:10.5281/zenodo.19542047,exclude,not about prompt injection,title_abstract,,,The Containment Paradox: When AI Sandboxing Backfires
doi:10.70528/ijlrp.v7.i4.2134,exclude,"survey, position or other",title_abstract,,,"The Convergent Cyber Threat Horizon in Financial Services: Artificial Intelligence, Agentic Systems,"
doi:10.1007/978-3-032-20318-2_45,exclude,not about prompt injection,title_abstract,,,The Double Exposure: AI as Character and Cinematic Lens for Network Security
doi:10.5281/zenodo.23689238,exclude,not about prompt injection,title_abstract,,,"The Egress Tax: Containment Costs, Liability Allocation, and the Accounting Boundary of Frontier AI "
doi:10.2139/ssrn.7389061,exclude,detector only,title_abstract,,,The Encoding Alphabet Is the Security Boundary: Measuring the Coverage Limits of Deterministic Multi
doi:10.1002/9781394402069.ch8,exclude,"survey, position or other",title_abstract,,,The Generative AI Attack Surface
doi:10.36227/techrxiv.177219982.22869977/v1,include,,title_abstract,,Zonal LoRA hardening against context flattening,The Glass Wall: Enforcing Cognitive Segregation in Large Language Models via Zonal LoRA Hardening
doi:10.5281/zenodo.23510607,exclude,detector only,title_abstract,,,The Guardrail Cries Wolf: Prompt-Injection Detectors on the Text Security Teams Actually Send Them
doi:10.55248/gengpi.07.0326.0730,exclude,"survey, position or other",title_abstract,,,The Hidden Threat Landscape of Large Language Models: A Security Vulnerability Perspective
doi:10.5281/zenodo.21740585,exclude,not about prompt injection,title_abstract,,,The Instrument — GFIS: A Verifiable Agentic Research Pipeline. Methods Note (System State as of 31 J
doi:10.2139/ssrn.7669059,exclude,detector only,title_abstract,,,The LASS Paradox: Linear Accessibility Does Not Imply Specificity inPrompt Injection Probes
doi:10.5281/zenodo.23506227,include,,title_abstract,,"role: re-measurement; Meta-SecAlign-8B internals, registered test",The MLPs of Layers 8-15 of Meta-SecAlign-8B and Its Scaling Knob: An Unresolved Registered Test
doi:10.2139/ssrn.7194038,exclude,not about prompt injection,title_abstract,,,The Maturity Illusion: Zero Trust Governance for Autonomous AI Agents
doi:10.5281/zenodo.21233561,exclude,system level,title_abstract,,"threshold defences in RAG, formal",The Multiplicative Vulnerability: Formalizing Threshold Defenses Against Indirect Prompt Injection i
doi:10.1109/access.2026.3680147,exclude,"survey, position or other",title_abstract,,,The New Web Attack Surface: A Taxonomy of Semantic and Agentic Threats in AI Browsers
doi:10.5281/zenodo.18255591,exclude,detector only,title_abstract,,,The Paradox of Cognitive Defense: Limitations of Traditional WAFs against LLM Prompt Injection
doi:10.5281/zenodo.19422909,exclude,not about prompt injection,title_abstract,,,"The Plumbing of AI Safety: Guardrails, Not Vibes"
doi:10.5281/zenodo.18886676,exclude,not about prompt injection,title_abstract,,,The Red Queen's Trap: A HATI² Stress-Test in Cybersecurity Economics
doi:10.1109/msec.2026.3713527,exclude,attack only,title_abstract,,,The Reproduction Number of AI Worms: Epidemic Modeling of Adversarial Attacks in LLM Agent Ecosystem
doi:10.1109/mcomstd.2026.3676030,exclude,"survey, position or other",title_abstract,,,"The Safety Frontier in LLMs: Risks, Defenses, and Open Challenges"
doi:10.11648/j.sdcomput.20260101.16,exclude,"survey, position or other",title_abstract,,,The Security Duality of Large Language Model Agents: A Systematic Review of Self-Security Protection
doi:10.5281/zenodo.22170103,exclude,not about prompt injection,title_abstract,,,The Shift to the AI Control Plane: Securing the Frontier of Autonomous Action
doi:10.2139/ssrn.6575978,exclude,"survey, position or other",title_abstract,,,The Silent Risk: Security Vulnerabilities in AI-Enabled Pharmacovigilance Systems
doi:10.2196/preprints.99050,exclude,duplicate version,title_abstract,,preprint version of record 2080,The Silent Risk: Security Vulnerabilities in AI-Enabled Pharmacovigilance Systems (Preprint)
doi:10.5281/zenodo.19199327,exclude,"survey, position or other",title_abstract,,,The Skill Ceiling: Author-Side Defences and Infrastructure-Level Trust for Agent Skills and Extensio
doi:10.5281/zenodo.20363723,exclude,attack only,title_abstract,,,The Summarization Trap: Quantifying Role-Based Vulnerabilities and Lexical Hijacking in RAG Pipeline
doi:10.5281/zenodo.19701663,exclude,not about prompt injection,title_abstract,,,The Survival Paradox: Analyzing Constraint Evasion in Large Language Models Triggered by Simulated E
doi:10.5281/zenodo.24127510,exclude,system level,title_abstract,,,"The Trust Boundary You Forgot You Had: Threats Posed by AI-Coded Plugins and DAW Workstations, and a"
doi:10.5281/zenodo.21841919,exclude,not about prompt injection,title_abstract,,,The Unshackled Harness: Cyber-Evaluation Environments as the Least-Defended Systems in AI Safety
doi:10.5281/zenodo.22068232,exclude,not about prompt injection,title_abstract,,,The White Rabbit Methodology: The Ethical Hacker's Operator's Handbook
doi:10.1016/j.neucom.2026.133313,exclude,"survey, position or other",title_abstract,,,The double-edged sword: LLM operations (LLMOps) security in the cloud- a comprehensive review
doi:10.1145/3830454.3846783,exclude,jailbreak or direct injection only,title_abstract,,,Thinking Dangerously: How Reasoning Training Dilutes Safety in Language Models
doi:10.1002/9781394491148.ch4,exclude,not about prompt injection,title_abstract,,,Threat Modeling for Upload‐Induced Data Leakage in Generative AI
doi:10.2139/ssrn.6280298,exclude,"survey, position or other",title_abstract,,,"Threats and Defenses Against Injection, Inference, and Data Leakage in Educational Artificial Intell"
doi:10.36227/techrxiv.177040550.02084667/v1,exclude,"survey, position or other",title_abstract,,,Threats and Defenses in Large Language Models: A Review of Adversarial and Model Poisoning Technique
doi:10.5281/zenodo.20263276,exclude,not about prompt injection,title_abstract,,,Three Modes of Memetic Replication in Persistent-Memory AI Agents: A Substrate-Typed Taxonomy with P
doi:10.5281/zenodo.20043246,exclude,system level,title_abstract,,,"Three Substrates, Seven Models, Six Tasks: A Comparative Empirical Evaluation of Structural Enforcem"
doi:10.2139/ssrn.6813640,exclude,not about prompt injection,title_abstract,,,Token-Aliased Closed-Loop Security: Comprehensive Authentication Lifecycle Defense Modules
doi:10.5281/zenodo.19794071,exclude,attack only,title_abstract,,,Tool Output Mimicry: Bypassing Multi-Layer Agentic AI Defenses via Upstream-Agent Impersonation in U
doi:10.1007/978-3-032-39143-8_25,exclude,system level,title_abstract,,"tool-based data segregation, no abstract",Tool-Based Data Segregation: A Prevention Strategy Against Prompt Injection in AI Agents
doi:10.17605/osf.io/fe67v,exclude,attack only,title_abstract,,harness fixes against faults and injection; check at full text if needed,Tool-Call Faults and Prompt Injection in LLM Agents: Agent Reliability Lab
doi:10.66857/jirt.v3i2.52,exclude,system level,title_abstract,,,Tool-Flow Taint Analysis for Data Exfiltration Defense in Large Language Model Agents
doi:10.1109/tnse.2026.3680460,exclude,system level,title_abstract,,,Topology Linearization for Multi-Agent Systems Security: Mitigating Malicious Propagation via Path D
doi:10.5281/zenodo.21671469,exclude,system level,title_abstract,,,Toward Closing the Syntactic-Semantic Gap: Adversarial Robustness of a Hybrid Security Kernel in Age
doi:10.5281/zenodo.22819018,exclude,benchmark only,title_abstract,,,Toward Reproducible Local Evaluation of Prompt Injection in Open-Weight LLMs: Design and Threat Mode
doi:10.1109/fdl71572.2026.11725314,exclude,not about prompt injection,title_abstract,,,Toward Trustworthy Edge AI Agents: Automated Threat Modeling as a First-Class Design Requirement: Sp
doi:10.1016/j.ejrad.2026.113318,exclude,not about prompt injection,title_abstract,,,Toward truly agentic radiology AI: A multimodal architecture framework and differentiated implementa
doi:10.6082/zc3gv-41k29,exclude,no full text obtainable,full_text,"not retrieved: no open copy found by script; open https://doi.org/10.6082/zc3gv-41k29 in a browser and save the PDF, or go through the university library if the page asks for payment",full text: not obtained by script or by hand,Towards Safe and Secure Agentic Systems
doi:10.1007/978-3-032-15993-9_34,exclude,not about prompt injection,title_abstract,,,Towards Secure Retrieval-Augmented Generation: Preventing LLM Data Leaks with RBAC
doi:10.1007/s10207-026-01339-6,exclude,"survey, position or other",title_abstract,,,"Towards trustworthy foundation models: a systematic review of safety, evaluation, and defence mechan"
doi:10.5281/zenodo.20325768,exclude,duplicate version,title_abstract,,"dataset record of DRIP, already included via arXiv","Training and evaluation dataset for paper ""DRIP: Defending Prompt Injection via Token-wise Represent"
doi:10.5281/zenodo.20098297,exclude,detector only,title_abstract,,,Training-Free Behavioral Drift Detection in Large Language Models Using Output-Embedding Stability M
doi:10.2139/ssrn.7302410,exclude,"survey, position or other",title_abstract,,,"Transformers and Large Language Models for Dynamic Cyber Defence: A Survey of Adaptive Defence, Movi"
doi:10.5281/zenodo.18625271,exclude,not about prompt injection,title_abstract,,,Traversal Log: The Thousand Worlds Session — Google AI Mode as Generative Story Engine — Crimson Hex
doi:10.5281/zenodo.23066521,exclude,detector only,title_abstract,,,"Tron-1B: Fast, Calibrated Typed Decisions with a Set-Attention Option Head"
doi:10.5281/zenodo.20700880,exclude,"survey, position or other",title_abstract,,,"Trust Boundaries Under Autonomous Action: How Memory Poisoning, Graph Selection Integrity, Prompt In"
doi:10.5281/zenodo.21746423,exclude,detector only,title_abstract,,,Trust Boundaries and the Limits of Pattern-Based Prompt Injection Detection
title:trustconsensusforsecuringmcpselectionineducationalecosystems,exclude,not about prompt injection,title_abstract,,,Trust Consensus for Securing MCP Selection in Educational Ecosystems
doi:10.2139/ssrn.6313620,exclude,not about prompt injection,title_abstract,,,Trust Infrastructure for the AI Economy: Reducing Financial Risk and Fraud in Regulated Sectors thro
doi:10.5281/zenodo.22897640,exclude,not about prompt injection,title_abstract,,,Trust Without Exposure: Cryptographic Non-Joinability So Enterprises Can Share AI Without Sharing Se
doi:10.1109/access.2026.3706063,exclude,system level,title_abstract,,,Trust-Aware Orchestration Architecture for LLM-Assisted Workflows in Multi-Tenant Enterprise Systems
doi:10.5281/zenodo.24140058,exclude,system level,title_abstract,,,Trust-Bounded Governance for LLM-Assisted SDN-IoT Defense: code and data
doi:10.1109/access.2026.3726909,exclude,system level,title_abstract,,,TrustTrace: Provenance-Aware Tracing of Indirect Prompt Injection Propagation in LLM-Assisted Learni
title:trustworthyaiaiagentsinformationsecurityandaiagentsforinformationsecurity,exclude,"survey, position or other",title_abstract,,,Trustworthy AI: AI Agents Information Security and AI Agents for Information Security
doi:10.53941/pc.2026.100008,exclude,system level,title_abstract,,,Trustworthy Deployment of LLM-Based RAG Systems for Small Businesses: A Security and Confidence-Awar
title:trustworthyintelligencefusionforsearchandrescuedesigninggroundedandsecuremultiagentaiformi,exclude,system level,title_abstract,,,Trustworthy Intelligence Fusion for Search and Rescue: Designing Grounded and Secure Multi-Agent AI
doi:10.5281/zenodo.24076230,exclude,detector only,title_abstract,,,"Trustworthy Multi-Agent AI for Europe: Causal Detection of Instruction Drift, Prompt-Injection Influ"
title:trustworthygenerativeaisecuringthepipelineacrossinputscontextdataandoutputs,exclude,"survey, position or other",title_abstract,,,"Trustworthy generative AI: securing the pipeline across inputs, context, data, and outputs"
doi:10.5281/zenodo.18685116,exclude,not about prompt injection,title_abstract,,,TruthGate v1.0: Epistemic Stability Measurement for Language Models via Twin-Probe Epsilon Injection
doi:10.5281/zenodo.20498183,exclude,not about prompt injection,title_abstract,,,Two Words Broke My AI Architecture
doi:10.1109/i3ctcon68242.2026.11507202,exclude,detector only,title_abstract,,,Two-Layer Input Filtering Framework for Large Language Model Security
doi:10.21203/rs.3.rs-10731314/v1,exclude,attack only,title_abstract,,,"Typed-Field Forgery in Agent Communication Protocol Await-Resume: A Study of Content, Metadata, and "
doi:10.5281/zenodo.19746723,exclude,system level,title_abstract,,,Typestate-Enforced Agent Loops: Making Policy Gates Unskippable at Compile Time
doi:10.5281/zenodo.20604549,exclude,not about prompt injection,title_abstract,,,UAWM: A Unified Adaptive World Model with Multi-Layered Security for Robust Decision-Making (Version
doi:10.5281/zenodo.20774130,exclude,system level,title_abstract,,,"Unexpressible, Not Filtered: A Structural Framework for Governing AI-Agent Actions — the Network Int"
doi:10.5281/zenodo.20518226,exclude,"survey, position or other",full_text,,"full text: concept note on Unicode-shifted text; the only experiment asks a model to read one shifted string, no injection outcome measured",Unicode Shifting: A Defense Methodology That Prevents Attackers from Reading in Natural Language
doi:10.5281/zenodo.23048818,exclude,not about prompt injection,title_abstract,,,Unified Architecture of Physical-Layer Governance and Quantum Real-Time Optimization (RTO): Determin
doi:10.4324/9781003673347-4,exclude,not about prompt injection,title_abstract,,,"Unifying Infrastructure, Data Governance, and Ethical Oversight"
doi:10.5281/zenodo.21340832,exclude,not about prompt injection,title_abstract,,,Universal Dense Semantic Compression (UDSC): A Multi-Level Semantic Compression Language for LLM-Nat
doi:10.5281/zenodo.22867425,exclude,not about prompt injection,title_abstract,,,Universal Fractal Natural Language Decision Map: Real-Time Edge Triage Across Heterogeneous Domains
doi:10.20944/preprints202606.1520.v1,exclude,attack only,title_abstract,,,Unverifiable Identity Claims as a Structural Vulnerability in LLMs
doi:10.5281/zenodo.22163198,exclude,detector only,title_abstract,,,UrduGuard: Benchmarking and Detecting Urdu-Language Jailbreak and Prompt-Injection Attacks in Multi-
doi:10.5281/zenodo.24050517,exclude,no full text obtainable,full_text,"not retrieved: data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF",full text: not obtained by script or by hand,"UrduRAG IPI: Multilingual Indirect Prompt Injection Benchmark, Datasets and Experimental Results"
title:usingcontextorientedrulesandmlanalyticsforautomatictrainingofllmprotectionsystemsagainstpr,exclude,detector only,title_abstract,,,Using context-oriented rules and ML analytics for automatic training of LLM protection systems again
doi:10.51583/ijltemas.2026.151000080,exclude,detector only,title_abstract,,,Vectorwall: A Hybrid Multi-Layer API Security Gateway with Ensemble Machine Learning and Adaptive Th
doi:10.21203/rs.3.rs-11040747/v1,exclude,system level,title_abstract,,,Verifiable Provenance for AI Judicial Decision Support Using Transparency Logs and Citation Entailme
doi:10.1109/smartnets69662.2026.11604866,exclude,benchmark only,title_abstract,,,Veritas: Dynamic Evaluation Framework for Rag Systems' Defense Against Prompt Injection Attacks
title:verteidigungsmanahmengegenangriffeaufllmbasierendenspamfiltern,exclude,no full text obtainable,full_text,not retrieved: no identifier; search the title in a browser or the library catalogue,full text: not obtained by script or by hand,Verteidigungsmaßnahmen gegen Angriffe auf LLM basierenden Spam Filtern
doi:10.5281/zenodo.22159565,exclude,attack only,title_abstract,,,VisGuard-Ur: A Proof-of-Concept Study on Typographic Jailbreak and Prompt-Injection Attacks Against
doi:10.5281/zenodo.23382162,exclude,not about prompt injection,title_abstract,,,Visual Studio Copilot and SSMS Remote Code Execution (CVE-2026-103572): A Detection Engineering Anal
doi:10.1016/j.knosys.2026.117001,exclude,"survey, position or other",title_abstract,,"Sammartino et al., survey already in the candidate list",Vulnerabilities in autonomous execution: A survey of security threats and defenses in LLM-driven age
doi:10.1145/3774905.3795093,exclude,system level,title_abstract,,,"WAB: Overcoming Memory, Network, and Security Walls in Native Agentic Browsers with WebAssembly"
doi:10.5281/zenodo.18144131,exclude,not about prompt injection,title_abstract,,,"WEAPONIZING LARGE LANGUAGE MODELS: AUTOMATED PHISHING, SOCIAL ENGINEERING, AND MALWARE GENERATION"
doi:10.25394/pgs.31954341,exclude,attack only,title_abstract,,,WORKFLOW CENTRIC ANALYSIS AND OPTIMIZATION OF INDIRECT PROMPT INJECTION ATTACKS AGAINST AGENTIC LARG
doi:10.5281/zenodo.23838850,exclude,attack only,title_abstract,,,Weaponising Model Safety: Steering LLM Decision Harnesses
doi:10.2139/ssrn.7457218,exclude,system level,title_abstract,,no abstract,Web Agent Defense Gateway
doi:10.4324/9781003687474-2,exclude,not about prompt injection,title_abstract,,,What Are Generative AI and Agentic AI?
doi:10.5281/zenodo.21343930,include,,title_abstract,,role: re-measurement; prompt-level defences across 14 small quantised agent configurations,What Determines the Vulnerability of Small LLM Agents to Indirect Prompt Injection? A Systematic Stu
doi:10.5281/zenodo.21944597,exclude,attack only,title_abstract,,,What surrounds an injection decides whether it works: context and channel effects in tool-metadata a
doi:10.1109/svcc69905.2026.11642185,exclude,not about prompt injection,title_abstract,,,When AI Agents Fail in Production: Security Lessons for Builders
doi:10.5281/zenodo.23803909,exclude,not about prompt injection,title_abstract,,,When AI Agents Go Off Script: How Websites Become Targets
doi:10.5281/zenodo.20735483,include,,title_abstract,,role: re-measurement; model-dependent failure modes where defences backfire,When Defenses Backfire: Model-Dependent Failure Modes in Prompt-Injection Defense for LLM Agents
doi:10.20944/preprints202610.2588.v1,exclude,attack only,title_abstract,,,When Does a Prompt Become a Worm? Measuring Reproduction Numbers of Self-Replicating Instructions in
doi:10.1007/978-3-032-16791-0_30,exclude,"survey, position or other",title_abstract,,,When Generative AI Prompts Bite Back: Investigating Different Types of Prompt Injection Attacks on L
doi:10.5281/zenodo.22984603,exclude,system level,title_abstract,,,When Psychological Operations Meet Language Models: A Defense Framework for Symbolic Interaction Att
doi:10.5281/zenodo.24238846,exclude,not about prompt injection,title_abstract,,,When Success Becomes Permission: Verification Decay and Induced Trust Accumulation in Agentic AI Sys
doi:10.2139/ssrn.7248360,exclude,attack only,title_abstract,,,When Truth is Retrieved but Ignored: Evidence-Present Indirect Injection in Multi-document RAG
doi:10.5281/zenodo.23735249,exclude,no full text obtainable,full_text,"not retrieved: data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF",full text: not obtained by script or by hand,"When the User Says ""Send"": Authorization, Parallel Tool Calls, and the Limits of Prompt-Level and St"
doi:10.33767/osf.io/462dp_v1,exclude,not about prompt injection,title_abstract,,,When the Web Speaks Differently to Machines Machine-Human Content Asymmetry in AI-Mediated Informati
doi:10.5281/zenodo.20790896,exclude,not about prompt injection,title_abstract,,,"Who Does Your Shopping Agent Work For? Where the hype stands against what is delivered, why the neut"
doi:10.5281/zenodo.24034006,exclude,not about prompt injection,title_abstract,,,Who Governs the Governor? — Stored Prompt Injection Through the API Gateway
doi:10.5281/zenodo.23758886,include,,title_abstract,,"steering lever against injected instructions, source weighting (companion paper 2170)","Who said this? Most chat models weigh a text's source differently for each behaviour, and a steering"
doi:10.5281/zenodo.23758884,include,,title_abstract,,"chat role and goal adherence, companion measurement; treat with 2169",Whose words count? Chat role and partial control of goal adherence
doi:10.5281/zenodo.23982238,exclude,not about prompt injection,title_abstract,,,"Why Enterprise AI in ChatGPT, Claude and Every Corporate AI Deployment Needs Cryptographic Limits on"
doi:10.18653/2026.findings-emnlp.1103,exclude,detector only,title_abstract,,,Why Prompt-Injection Detectors Fail on XPIA: A Latent-Space Explanation
doi:10.21203/rs.3.rs-9707431/v1,exclude,detector only,title_abstract,,,Why Single-Layer LLM Guardrails Fail: A Dual-Detection Pattern on AWS Bedrock
doi:10.32473/flairs.39.1.141778,exclude,not about prompt injection,title_abstract,,,Words As Weapons
doi:10.2139/ssrn.6903618,exclude,jailbreak or direct injection only,title_abstract,,,XJailGuard: An Explainable and Cross-Lingual Framework for Multi-turn Jailbreak Detection in LLMs
doi:10.5281/zenodo.23097077,exclude,not about prompt injection,title_abstract,,,Your AI Chatbot Is Being Hijacked: How to Spot and Block Prompt Injection Attacks in 2026
doi:10.5281/zenodo.21428132,exclude,attack only,title_abstract,,,Your Agents Work as Expected. That's the Problem. : Prompt-Injection Cascades in a Multi-Agent Enter
doi:10.46254/an16.20260369,exclude,system level,title_abstract,,,ZT-RAG: A Zero-Trust Reference Architecture for Secure Cloud-Native Retrieval-Augmented Generation S
doi:10.1109/icstem68144.2026.11735682,exclude,system level,title_abstract,,,Zero Trust Framework for Detecting and Preventing Prompt Injection Attacks in Enterprise AI Chatbots
doi:10.5281/zenodo.22715483,exclude,system level,title_abstract,,,"Zero Trust for Autonomous Agents: Dynamic Runtime Authorization via Intent Alignment, Lineage Decay,"
doi:10.5281/zenodo.20472685,exclude,not about prompt injection,title_abstract,,,Zero Trust for Fallible Agents: Why AI Belongs Inside the ZTA Control Model
doi:10.1109/iccsdfai70505.2026.11647875,exclude,detector only,title_abstract,,,Zero-Shot Prompt Injection Detection via N-gram Pooling of Token-Level Hidden States
doi:10.1109/icaic67076.2026.11395696,exclude,attack only,title_abstract,,,Zero-Shot Tokenizer Transfer for Targeted Attacks on Retrieval-Augmented Generation
doi:10.2139/ssrn.7667739,exclude,not about prompt injection,title_abstract,,,Zero-Trust Multi-Agent Governance: Defending Autonomous Industrial Swarms Against Cascading Memory P
doi:10.1109/ichora69329.2026.11537001,exclude,system level,title_abstract,,,Zero-trust Conceptual Architecture for Secure Integration of Large Language Models and Autonomous AI
doi:10.5281/zenodo.19702226,exclude,attack only,title_abstract,,,adversarial_prompt_injection_Hamzi_Aimen
doi:10.5281/zenodo.22860886,exclude,not about prompt injection,title_abstract,,,"agmi: Agent Memory Integrity, a conformance test suite for tamper evidence in AI agent memory and ch"
doi:10.5281/zenodo.20473400,exclude,not about prompt injection,title_abstract,,,anan5093/ncmm-secure-ai-platform: Securing the Invisible Backbone: How I Built a Classified AI Intel
doi:10.5281/zenodo.22657808,exclude,attack only,title_abstract,,,cca2026_repro
doi:10.5281/zenodo.21909786,exclude,benchmark only,title_abstract,,AgentDojo wrapper harness; check for defence results at full text,"dspy-security-bench: reproducible security, authorization, and mission assurance evidence for tool-u"
doi:10.18653/2026.emnlp-demos.19,exclude,system level,title_abstract,,,e2a: A Secure Agent-Native Email Gateway with Phishing and Prompt-Injection Defense
doi:10.21203/rs.3.rs-10354224/v1,exclude,system level,title_abstract,,,guardrail-rs A Fail-Open Reverse Proxy for Prompt-Injection Defense and PII Redaction in LLM Applica
doi:10.5281/zenodo.23811350,exclude,duplicate version,title_abstract,,code record of record 1855,"hijack-proof-agent: code and results for ""Prompt-Injection Defences on a Small Local Agent: What Wor"
doi:10.5281/zenodo.23784336,exclude,no full text obtainable,full_text,"not retrieved: data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF",full text: not obtained by script or by hand,"ipi-bench v0.2.1: code and data for ""Beyond Detection: Measuring Ranking Harm and Defense Cost of In"
doi:10.5281/zenodo.19489455,exclude,not about prompt injection,title_abstract,,,mathmentorAI/claim-layer: ClaimLayer 2.0.1
doi:10.5281/zenodo.23665516,exclude,no full text obtainable,full_text,"not retrieved: data or code deposit only; find the paper by its title (author page, arXiv, journal) and save the PDF",full text: not obtained by script or by hand,memgate pilot: email data and outputs for prompt injection into agent memory (evaluation v2)
doi:10.5281/zenodo.19457343,exclude,not about prompt injection,title_abstract,,,nukIeer/AI-Prompt-Injection-Cheatsheet: New release
doi:10.5281/zenodo.23197334,exclude,system level,title_abstract,,,process-aware-verification-mfg: code and data for Process-Aware Verification Against Prompt Injectio
doi:10.5281/zenodo.23758689,exclude,not about prompt injection,title_abstract,,,sharr666/llm-security-and-streaming-benchmarks: v1.0.1: Multi-Document LLM Architecture & Streaming
doi:10.5281/zenodo.20697266,exclude,not about prompt injection,title_abstract,,,xinlongliu0307/local-coding-agent: local-coding-agent v0.1.0
doi:10.5281/zenodo.18438220,include,,title_abstract,,role: re-measurement; translation-based defence evaluated and found limited,ÞÝÐING Revisited: Empirical Limits of Translation-Based Prompt Injection Defense and the Deobfuscati
title:,exclude,not about prompt injection,title_abstract,,,ВЛИЯНИЕ ИСКУССТВЕННОГО ИНТЕЛЛЕКТА НА КИБЕРБЕЗОПАСНОСТЬ: НОВЫЕ ГОРИЗОНТЫ И СКРЫТЫЕ УГРОЗЫ
title:aicuregrid,exclude,not about prompt injection,title_abstract,,,التحكم في مخاطر الذكاء الاصطناعي في الأمن السيبراني: مراجعة حديثة للحوكمة والحماية وإطار الإنفاذ الم
doi:10.5281/zenodo.21820256,exclude,"survey, position or other",title_abstract,,,哥德尔渔网:提示词注入的三层次防御
doi:10.5281/zenodo.24209140,exclude,system level,title_abstract,,,意味論的非干渉性に基づく自律型AIワーム防御アーキテクチャ

Every full-text decision with the anchor phrase that Stage 4 looks for in its PDF.

In [ ]:
%%writefile schema/fulltext_decisions.csv
key,slug,decision,reason,basis,anchor,note
arxiv:2402.06922,2402.06922,include,,full text,"we consider two complementary approaches:prompt hardening, which modifies the agent’s instructions","role: re-measurement; prompt hardening and external filters against confidentiality attacks through tool content, results averaged over five defences (Fig. 7)"
arxiv:2405.10529,2405.10529,exclude,image or audio channel only,full text,we propose to address patched visual prompt in- jection,SmoothVLM: randomised smoothing against adversarial visual patches in vision-language models
arxiv:2406.14048,2406.14048,exclude,jailbreak or direct injection only,full text,Defenders craft prompts and filters to instruct the LLM to protect a given secret,secret-extraction competition: the attacker talks to the defended model directly
arxiv:2407.18213,2407.18213,exclude,not about prompt injection,full text,"across several classification tasks, model families, and adversarial attacks",adversarial training scaling on classification tasks and StrongREJECT; no injected-data setting
arxiv:2504.20769,2504.20769,include,,full text,when 1 out of 10 references provided is corrupted with prompt injection attacks,chain-of-defensive-thought: few-shot structured defensive reasoning as a prompting defence against injected references
arxiv:2509.00088,2509.00088,exclude,detector only,full text,achieving supe- rior robustness in malicious prompt detection,AEGIS: co-evolved prompt for an LLM detector of injected submissions
arxiv:2509.05883,2509.05883,exclude,attack only,full text,Each model was tested without supplementary sanitization,eight commercial models tested without defences
arxiv:2510.01586,2510.01586,exclude,jailbreak or direct injection only,arXiv abstract page; PDF over the 10 MB reading limit,jointly optimizes attackers (which synthesize evolving jailbreak prompts) and defenders,AdvEvo-MARL: co-evolutionary RL of task agents against evolving jailbreak prompts
arxiv:2511.01287,2511.01287,exclude,detector only,full text,we ex- plore a simple detection-based defense,defence is a detector that removes injected text before scoring
arxiv:2511.15759,2511.15759,include,,full text,Attack Type Baseline +Filtering +Guardrails Full Defense,layered defence; hierarchical system-prompt guardrails measured as an added layer over content filtering (Table 2)
arxiv:2511.20597,2511.20597,exclude,detector only,full text,The core detector is a fine-tuned Qwen3-30B-A3B-Instruct-2507,BrowseSafe: fine-tuned detection classifier with architectural controls
arxiv:2512.00966,2512.00966,include,,full text,intent analysis and response generation are carried outby the same model under the same context,"IntentGuard: thinking-prefill intent analysis inside the protected model's own reasoning, then neutralises matched untrusted spans"
arxiv:2512.01326,2512.01326,include,,full text,we evaluate the ro- bustness of JATMO against HOUYI,role: re-measurement; JATMO task-specific fine-tuning re-tested against the HOUYI genetic attack
arxiv:2512.05745,2512.05745,exclude,image or audio channel only,full text,"which embed malicious instructions in im- ages, videos, or audio","ARGUS: activation steering against instructions embedded in images, video or audio"
arxiv:2512.16307,2512.16307,include,,full text,with particular emphasis on the robustness of prevention- based defenses such as delimiters and instructional prompts,role: re-measurement; delimiter and instructional-prompt defences re-tested on small LLaMA models (own method is detection)
arxiv:2601.07263,2601.07263,exclude,system level,full text,a lightweight runtime module that enforces environment and intention consistency alignment,SUPERVISOR: runtime module checking page context against the goal before actions run
arxiv:2601.07835,2601.07835,include,,full text,Direct Preference Optimization (DPO) training methodology adapted for unlearning unsafe response patterns,SecureCAI: DPO training of the assistant with a security constitution against injected security artifacts
arxiv:2601.12822,2601.12822,exclude,system level,full text,fine-tune a vision-language corrector that acts as a plug-and-play defense module,MirrorGuard: separate fine-tuned corrector model that rewrites the agent's reasoning
arxiv:2601.19051,2601.19051,exclude,detector only,full text,to continuously enhance detection efficacy for prompt-based attack techniques,HASTE: hard-negative mining to retrain injection detectors
arxiv:2602.10481,2602.10481,exclude,system level,full text,our approachoperates at the abstraction boundary between LLMs and applications,authenticated prompts and context with policy enforcement at the application boundary
arxiv:2603.21975,2603.21975,exclude,detector only,full text,designed to support the development of AI-driven solutions for detecting harmful LLM outputs,dataset for classifiers of unsafe model outputs
arxiv:2604.19844,2604.19844,exclude,image or audio channel only,arXiv abstract page; PDF over the 10 MB reading limit,structure-based and noise-based visual injections,misleading visual signals (structure- and noise-based) read by vision-language agents
arxiv:2604.24020,2604.24020,exclude,"survey, position or other",full text,Live experiments show weakest-first ASAT raises average TLDT score from 80.9 to 96.9,poster; in-context awareness training scored on a self-defined 12-dimension scale with the agent as its own evaluator; no injection outcome reported on its own
arxiv:2604.26506,2604.26506,include,,full text,the Defender is strengthened through preference-based training to maintain consistent reviews,SafeReview: reviewer model trained by DPO in co-evolution with a GRPO attacker against hidden prompts in submissions
arxiv:2605.03378,2605.03378,exclude,system level,full text,a causal-provenance runtime auditor that verifies whether each state-changing agent action,ARGUS: causal-provenance runtime auditor of agent actions; prompting defences appear only as baselines
arxiv:2605.06669,2605.06669,exclude,jailbreak or direct injection only,full text,We evaluate a domain-specific multi- layer safeguard pipeline combining deterministic pattern filters,tutor setting where the student types the injection; defences are filters and guardrails
arxiv:2605.12233,2605.12233,include,,full text,Spotlighting [19] and RUP [10] are prompt-level methods that wrap tool output with sentinel markers,"role: benchmark with defence results; six defences evaluated on task alignment, including Spotlighting and RUP as prompt-level methods"
arxiv:2605.18918,2605.18918,exclude,detector only,full text,a simple external surrogate detector that classifies an LLM’s prefill-stage hidden states,ESLD: linear probe on a guard model's hidden states
arxiv:2605.25194,2605.25194,exclude,image or audio channel only,full text,Adversarial images pose a severe security threat to multimodal large language models through prompt injection,GTM: masks adversarial image tokens in multimodal models
arxiv:2606.12716,2606.12716,exclude,detector only,full text,"a practical defense, motivated by the long-context challenge of academic papers, that uses chunk-based embedding search",PaperGuard: chunk-based embedding search that flags injected content in multimodal papers
arxiv:2606.23277,2606.23277,exclude,detector only,full text,GIF achieves near-perfect recall rates even without a downstream declassifier,GIF: information-flow measure that flags injected spans; the model's generation is not changed
arxiv:2607.03821,2607.03821,exclude,system level,full text,which extends untrusted data tracking from the agent’s context to the user’s environment,DualView: dual-LLM symbol tracking extended to the user's environment
arxiv:2607.10269,2607.10269,exclude,image or audio channel only,full text,malicious textual information embedded in physical environments introduces a high- priority visual channel,physical scene text read by vision-language models on wearables
arxiv:2607.14493,2607.14493,include,,full text,"We empirically tested prompt-level defenses (spotlighting, instruction hierarchies)",role: re-measurement; spotlighting and instruction-hierarchy prompts tested with data-level and dual-LLM defences on injected network logs (LogInject)
arxiv:2607.26115,2607.26115,include,,full text,we design a scalable self‐play algorithm where the model is tasked with attacking a diverse population of simultaneously‐trained de‐ fenderagents,GPT-Red: self-play attacker used for RL adversarial training of the production model against prompt injection; defender robustness reported
arxiv:2608.05495,2608.05495,exclude,image or audio channel only,full text,can the agent tell a genuine user command from ambient or externally-sourced content,"smart-home agents reading ambient audio, video and on-screen text"
arxiv:2608.08027,2608.08027,exclude,detector only,full text,This defense method uses the Attention Competition Ratio,BASIS: attention-based linear probes that gate refusal
arxiv:2608.14629,2608.14629,exclude,jailbreak or direct injection only,full text,we propose a method to resist jailbreak attacks using a three step Recursive Self-Correction process,bias injected through the user's adversarial prompt; mitigations are chain-of-thought prompting and DPO against jailbreak-style prompts
arxiv:2608.27990,2608.27990,exclude,detector only,arXiv abstract page; PDF over the 10 MB reading limit,agent-agnostic defense middleware,CAITLYN: agent-agnostic middleware that synthesises new detection scripts and LLM checks
arxiv:2608.28411,2608.28411,include,,full text,"We benchmark state-of-the-art prevention defenses, including instructional prevention","role: benchmark with defence results; LongPIBench tests instructional, delimiter, sandwich, SecInfer, PromptLocate and Meta-SecAlign-8B in long contexts"
arxiv:2609.16098,2609.16098,include,,full text,"Among the three prompt-based techniques: CoT, Paraphrasing, and Reflection, Paraphrasing appears to be the most impactful","chain-of-thought system instruction, paraphrasing and self-reflection measured with tool filtering on ASB, with an ablation (Table 1)"
arxiv:2609.20188,2609.20188,include,,full text,channel separation alone removes all measured attack success,"ResumeShield: fenced data channel declared inert by the trusted instructions, measured alone in the ablation"
arxiv:2609.22510,2609.22510,include,,full text,a preference-optimized model (SecAlign) that fully closes imperative injection still executes 11.8% of explosive prompts,role: re-measurement; SecAlign re-tested against dormant conditional (explosive) prompts; own defence is a detector
arxiv:2610.02432,2610.02432,exclude,system level,full text,Committees of 5–7 heterogeneous models reduce the ASR for Gemma-3-4B,"thesis; defences are committees of models, multi-layer filters and MCP call attestation"
arxiv:2610.03093,2610.03093,exclude,not about prompt injection,full text,"our empirical evaluation is strictly bounded to synthetic, domain-tokenised environments","synthetic planning environments; conflict between a latent goal channel and a text goal, no injected-data setting"
arxiv:2610.05295,2610.05295,include,,full text,directions estimated from hijacked and resisted training trajectories reduce attack success at pre-action and injected-span positions,role: mechanistic analysis; activation edits along role directions reduce attack success on AgentDojo
arxiv:2610.06064,2610.06064,include,,full text,Steering trust modulates susceptibility to indirect prompt injections,"trust steering matrices learned with BiPO, evaluated on indirect prompt injection among other agent settings"
arxiv:2610.15177,2610.15177,include,,full text,enforces task- dependent security policies directly within the denoising process of diffusion language models,CoDA: task policies enforced inside the denoising of diffusion language models
arxiv:2610.19372,2610.19372,include,,full text,policy-verified preference training of Qwen3-8B reduces vio- lating trajectories from 37.2% to 10.3,"PVST: rejection-sampling fine-tuning and DPO of the agent from policy-verifier verdicts, evaluated on AgentDojo"
arxiv:2610.22594,2610.22594,include,,full text,from prompt-based instructions and injection detectors to a guardrail that removes injected text and a model trained to resist injection,role: re-measurement; prompt-based instructions and a model trained to resist injection tested in medical multi-agent systems; own defence verifies clinical judgements
arxiv:2610.31974,2610.31974,exclude,system level,full text,the MCP layer mints a keyed-hash message authentication code,HMAC-signed provenance labels in the MCP layer against instructions decoded from images
arxiv:2610.39758,2610.39758,exclude,detector only,full text,which learns a soft prompt and scoring readout offline to san- itize tool outputs using a frozen task model,SEAL: learned soft prompt and readout that delete tool-output tokens before the task model reads them
arxiv:2610.41307,2610.41307,exclude,image or audio channel only,full text,Multimodal indirect prompt injection embeds malicious instructions in external media,"ERASE: on-policy self-distillation against instructions embedded in images, audio and video"
arxiv:2611.04673,2611.04673,exclude,system level,full text,"we propose AuthorityFlow, a three-stage parameter-level defense",AuthorityFlow: parameter-level authorisation of tool arguments by traced information flow; prompting defences appear only as baselines
arxiv:2611.24362,2611.24362,include,,full text,It also integrates an on-policy adversarial training pipeline tailored to IPI defense,PIPETTE: adversarial training of agents by reasoning-guided RL with on-policy self-distillation on diverse red-team attacks
doi:10.18653/2026.emnlp-main.2303,doi_10_18653_2026_emnlp_main_2303,include,,full text,We first use supervised fine-tuning to teach CBC structured consistency rea- soning,CBC: supervised fine-tuning and online alignment that teach the model to check task consistency before complying
doi:10.21203/rs.3.rs-11711485/v1,doi_10_21203_rs_3_rs_11711485_v1,include,,full text,We give the agent an operator public key pinned in its system prompt and a verify_signature tool,system-prompt policy with a pinned operator key and a signature-verification tool against injected halt requests in tool output; effect differs by model
doi:10.21203/rs.3.rs-9653510/v1,doi_10_21203_rs_3_rs_9653510_v1,include,,full text,constructing a prompt -engineering-based defense system through mandatory input structural decoupling,"HAA/ULC: system-prompt protocol with input decoupling and master-objective anchoring, tested on GPT-4o"
doi:10.3126/joeis.v4i1.81604,doi_10_3126_joeis_v4i1_81604,include,,full text,LLMs were retrained using adversarial examples to improve robustness,claims adversarial retraining of LLMs combined with input sanitisation against direct and indirect injection; evaluation details thin
doi:10.3126/nprcjmr.v1i8.73029,doi_10_3126_nprcjmr_v1i8_73029,exclude,not about prompt injection,full text,"We delve into the mechanisms of adversarial attacks, including evasion, poisoning, and model inversion","general overview of adversarial machine learning in cybersecurity (evasion, poisoning, model inversion); prompt injection not studied"
doi:10.3389/fcomp.2026.1735253,doi_10_3389_fcomp_2026_1735253,exclude,system level,full text,"integrates prompt ﬁltering, retrieval augmented grounding, trust aware scoring, and an auditing feedback loop","RoLLMRec: prompt filtering, retrieval grounding, trust-aware scoring and auditing around an LLM recommender"
doi:10.47134/jtsi.v3i2.6047,doi_10_47134_jtsi_v3i2_6047,include,,full text,Token-Adaptive Gaussian Defense (TAGD) layer that certifies LLM generation against indirect prompt injection,AdverShield-LLM: claims certified Gaussian noise at the token level of generation against indirect injection in IoT RAG
doi:10.51483/ijaiml.2557,doi_10_51483_ijaiml_2557,include,,full text,This study suggests a hybrid defense that combines sanitization delimiters with instruction -hierarchy prompting,"sanitisation delimiters combined with instruction-hierarchy prompting, each also measured alone"
doi:10.5281/zenodo.18712066,doi_10_5281_zenodo_18712066,exclude,detector only,full text,achieving deterministic injection detection in ~40μs per scan,Nullform: deterministic rewriting rules that detect structural injections
doi:10.5281/zenodo.19353246,doi_10_5281_zenodo_19353246,exclude,"survey, position or other",full text,This paper argues for native separated memory spaces as the architectural endpoint,position paper arguing for native separated memory spaces; nothing built or measured
doi:10.5281/zenodo.19603811,doi_10_5281_zenodo_19603811,exclude,system level,full text,We present a Uniﬁed Cryptographic-Control Architecture (UCCA),"UCCA: composite of smoothing, erase-and-check, control-barrier action filtering and zero-knowledge proofs; components not measured separately"
doi:10.5281/zenodo.20067761,doi_10_5281_zenodo_20067761,exclude,detector only,full text,was used to fine-tune the BERT language model for the addition of a defense layer for detecting normal input Prompts and Injection,LLRM: fine-tuned BERT filter in front of the chatbot
doi:10.5281/zenodo.20518226,doi_10_5281_zenodo_20518226,exclude,"survey, position or other",full text,"Experiment: ""hello world"" Shifted by 500,000","concept note on Unicode-shifted text; the only experiment asks a model to read one shifted string, no injection outcome measured"
doi:10.5281/zenodo.20537888,doi_10_5281_zenodo_20537888,exclude,detector only,full text,"AgentShield, që paraqet një proxy, apo një man -in-the- middle (MITM)",AgentShield thesis: proxy that scans fetched web content with a second LLM before it reaches the agent
doi:10.5281/zenodo.23972864,doi_10_5281_zenodo_23972864,include,,full text,Benign fine-tuning (orange) removes all three defenses at n = 5000,"role: re-measurement; extended report with results: benign fine-tuning erodes Meta-SecAlign, SecAlign and LocalAlign, and 250 security examples repair them (main paper still to be located)"
doi:10.61424/hewtat90,doi_10_61424_hewtat90,exclude,system level,full text,"This study evaluates a ReAct -style control, native tool filtering, deterministic self-verification, a calibrated action gate, and a distilled guardrail","tool filtering, self-verification, a learned action gate and a distilled guardrail around the agent"
doi:10.62161/sauc.v12.6399,doi_10_62161_sauc_v12_6399,include,,full text,spotlighting via datamarking (D1); structured - query delimiting with an instruction hierarchy (D2),"role: re-measurement; datamarking, structured-query delimiting with an instruction hierarchy, a guardrail and their combination in a municipal copilot"
title:adaptivedefenseframeworkformitigatingpromptinjectionattacksinlargelanguagemodels,title_adaptivedefenseframeworkformitigatingpromptinjectionattacksi,exclude,"survey, position or other",full text,The study develops a conceptual model for improving LLM reliability,conceptual framework without experiments
doi:10.31234/osf.io/zxuwf_v2,doi_10_31234_osf_io_zxuwf_v2,include,,full text,we explore the effectiveness of a possible defensive strategy: explicitly instructing LLMs to detect such injections,prompting defence: the reviewing model is told to check the manuscript for hidden instructions; measured on three LLMs with 26 preprints
doi:10.1038/s41598-026-74077-3,doi_10_1038_s41598_026_74077_3,exclude,image or audio channel only,full text,adversarial text rendered into the pixel data of medical images,instructions rendered into the pixels of dental radiographs read by vision-language models
doi:10.36227/techrxiv.171822345.56781952/v1,doi_10_36227_techrxiv_171822345_56781952_v1,exclude,jailbreak or direct injection only,full text,injecting carefully crafted prompts into the input stream of the LLM,"moving-target defence (random parameter perturbation, re-initialisation, context changes); the attacks are typed straight into the model's input, with no third-party data channel"
doi:10.3390/app16157662,doi_10_3390_app16157662,include,,full text,Spotlighting-Only (ASR 20.4%) and Detector-Only (ASR 25.6%),"Spotlight-Guard: spotlighting measured on its own and inside the layered system, with a component ablation (Tables 3 and 5)"
doi:10.20944/preprints202603.1023.v1,doi_10_20944_preprints202603_1023_v1,include,,full text,"a lightweight, black-box middleware that enforces strict instruction-data separation without model fine-tuning","Real User Instruction: prompt-level separation of user instructions from data, with the model made to list suspected injections before answering; compared with spotlighting baselines on indirect injection (Table 2)"
title:securingaimodelsagainstbackdoorsjailbreaksandpromptinjections,title_securingaimodelsagainstbackdoorsjailbreaksandpromptinjection,exclude,detector only,full text,"white-box runtime detection of misbehavior induced by backdoors, jailbreak prompts, and prompt injections, under an abstention-based response","thesis; its only prompt-injection method, LCF, is a runtime monitor that flags inputs and abstains"
doi:10.1109/icaiic68212.2026.11454261,doi_10_1109_icaiic68212_2026_11454261,exclude,jailbreak or direct injection only,full text,The user prompt is the adversarial harmful prompt crafted by the attacker,"included at title and abstract; the full text shows the attacks are harmful user prompts from WildJailbreak, with no third-party data channel"
doi:10.1109/icassp55912.2026.11463282,doi_10_1109_icassp55912_2026_11463282,exclude,jailbreak or direct injection only,full text,"combined with state-of-the-art attack generators (e.g., PAIR, AutoDAN) to craft sophisticated jailbreak prompts",included at title and abstract; the full text shows the attacks are adversarial user inputs (AdvBench goals with PAIR and AutoDAN)
arxiv:2504.21029,2504.21029,exclude,jailbreak or direct injection only,full text,structurally separates trusted system instructions from untrusted user inputs,"included at title and abstract; the full text shows the threat is the user's own input, and the only case study is hypothetical, with no measured results"
doi:10.20944/preprints202609.1881.v1,doi_10_20944_preprints202609_1881_v1,include,,full text,Our modified loop inserts a Safety Check before each action,"full-text check: data markers with a system-prompt rule and a safety check inside the agent's own reasoning loop, added over filtering and sandboxing (Table 1), with a component ablation"
doi:10.1038/s41598-025-31086-y,doi_10_1038_s41598_025_31086_y,exclude,jailbreak or direct injection only,full text,"The attacker cannot alter plug-ins, tool APIs, or retrieval augmentation mechanisms","PromptGuard: four layers (input filter, role-separated prompt format, output critic, response rewrite); the adversary controls the user's own input and cannot touch tools or retrieval, so the attacks are direct; the format layer is measured alone (Table 8) but only against user-typed prompts"

The files saved by hand: fingerprint (SHA-256), name as downloaded, the record it belongs to, and what to do with it.

In [ ]:
%%writefile schema/hand_files.csv
sha256,file_name,slug,action,note
09e5dee6442a45cb03782a45269f3c672a4ceb267a29f5210083344a782c12a8,v1.pdf,doi_10_36227_techrxiv_171822345_56781952_v1,use,
0ecce5794b953addf810070e8380af46f65689c93a4f03d86d0506ce3cc57a6e,3v1.pdf,doi_10_36227_techrxiv_171822345_56781952_v1,duplicate,second download of the same TechRxiv preprint
32fd106d00f48b73e5e93111571627f8b645772d5dcc54d8f3b5f7f95252d3c5,v11.pdf,doi_10_36227_techrxiv_177219982_22869977_v1,use,
f451bf761ed2380d660a23e1c293eaba785544775b4488358f1901767d338ace,applsci-16-07662.pdf,doi_10_3390_app16157662,use,
181e0562d864b60c1d7d6ecbe4671895726be20394544cf8e0981ad57a52d3fa,preprints202603.1023.v1.pdf,doi_10_20944_preprints202603_1023_v1,use,
0007d193fe6784e34a9714f5ce20241eebb8539ecdcc17193a5648d63bf979e4,s41598-025-31086-y.htm,doi_10_1038_s41598_025_31086_y,unusable,saved web page without the article text
c23b68ed97d0e243a7540b3ec6dd4f1e7863690db308e66913e06e96d94f63e9,Securing AI models against backdoors jailbreaks and prompt inje.pdf,title_securingaimodelsagainstbackdoorsjailbreaksandpromptinjection,use,
2319ea772a98eed5fb3a4a4b2887806afcbe267e63be25921d1884003bf9c8f3,Poncelas - Defenses against prompt injection in LLM agents systematic evaluation and mechanistic.pdf,title_defensescontrapromptinjectionenagentsllmavaluacisistemticaia,use,English text of the thesis; the record title is the Catalan one
a6c641e9afbb2f4edcda9bb828ff37e19d1f86fb119ab22626d3aebc272f722a,ETRI Journal - 2026 - Yoon - Adaptive orthogonal role‐aware transformer for robust instruction alignment.pdf,doi_10_4218_etrij_2025_0449,use,
f9f0f9e18e784943d6bf96e1ce0e3ad1d7c8fb54797cce13482048831515db6a,Evaluating+Prompt+Injection+Defenses+in+Large+Language+Models+A+Multi-Model+Empirical+Study+of+Security–Usability+Trade-offs.pdf,doi_10_70715_jitcai_2026_v3_i3_071,use,
967801dd79d8751b228ca2aa847674fed9c71241deaf5f9370b6cae3085c72ef,computers-15-00570.pdf,doi_10_3390_computers15090570,use,
f7205759930c3c0512370fb55ba07615d484c470b1125e8340935bf65762f65d,applsci-16-11039.pdf,doi_10_3390_app162111039,use,
f7205759930c3c0512370fb55ba07615d484c470b1125e8340935bf65762f65d,applsci-16-11039 (1).pdf,doi_10_3390_app162111039,duplicate,second download of the same article
9a71abfcb3f973763a4bca70c06f11bc088ad2af8100a6cf97b71985f30d2f52,preprints202609.1881.v1.pdf,doi_10_20944_preprints202609_1881_v1,use,
837f9e0090825d5d238ac24866753b17f995dcbc46a1cdb8545b874e2a2fc677,3779208.3804880.pdf,doi_10_1145_3779208_3804880,use,
567c27f8849330dcbf091e09cdd718536c3362fde125289c9a2b1118ed18bb2d,3830454.3846434.pdf,doi_10_1145_3830454_3846434,use,
1b30babecb7ca3ecd09c0f705360a0af7ee9c81d2c6e0efd5c9392725012103b,LLM-as-a-Judge_Vulnerabilities_in_Review_Summarization_and_a_Reasoning-based_Defense_against_Cross-Lingual_Prompt_Injection.pdf,doi_10_1109_ubmk71847_2026_11745222,use,
552d53655dedfd49b25db0eb839ae1ded9e3f7513723bc9568200ff6f92f768e,Constitutional_Sanitization_and_Fine-Tuning_for_Robustness_to_Prompt_Injection_in_LLM-Based_Log_Analysis.pdf,doi_10_1109_icssa70263_2026_11765792,use,
ed539ec008dfead7f6ed8d2fcd35920df886cc0c361bdfeba1e198fc0fc05128,PromptPatch_Towards_Precise_and_Stable_Behavioral_Patching_in_Large_Language_Models_via_Feedback-Driven_Prompt_Optimization.pdf,doi_10_1109_icassp55912_2026_11463282,use,
190fca28dafe295f00afa632c527e5b27275b561ea9bc28e34d7457766ffdc69,Defending_Against_Prompt_Injection_Attacks_Using_Automatic_System_Prompt_Engineering.pdf,doi_10_1109_icaiic68212_2026_11454261,use,
,Malladi_washington_0250O_30249.pdf,title_guardragadversarialpreferencetrainingagainstindirectpromptin,use,14 MB thesis; matched by its file name
8e3a530bebba507099404da94b150256a31694fcf0835c38d49f5a36fc976a43,s41598-025-31086-y.pdf,doi_10_1038_s41598_025_31086_y,use,PDF saved after the empty web page

## Stage 2. Give the hand-saved files their standard names

In [ ]:
import json, datetime as dt
import search, corpus, paths, overlap
for m in (search, corpus, paths, overlap):
    importlib.reload(m)
paths.ensure_dirs()

man = search.read_csv(paths.REPORTS / "defences.csv")
known = {m["slug"] for m in man}
mapping = search.read_csv(Path("schema/hand_files.csv"))
placed = search.place_hand_files(paths.DATA_DEFENCES, mapping, known)
log = paths.REPORTS / "hand_files.csv"
before = {(r["file_name"], r["slug"], r["action"]): r["status"] for r in search.read_csv(log)} if log.exists() else {}
for p in placed:                                 # on a rerun keep the record of what the first run did
    if p["status"].startswith("already") and before.get((p["file_name"], p["slug"], p["action"]), "").startswith(("renamed", "moved", "same file")):
        p["status"] = before[(p["file_name"], p["slug"], p["action"])]
search.write_csv(log, placed, ["file_name", "slug", "action", "status", "sha256"])
problems = []
for p in placed:
    print(f"{p['status']:<22} {p['action']:<9} {p['slug']:<62} <- {p['file_name'][:60]}")
    if p["status"] in ("not found", "unknown action") or p["status"].startswith("kept aside"):
        problems.append(f"hand file {p['file_name']}: {p['status']}")
left = [q.name for q in sorted(paths.DATA_DEFENCES.iterdir()) if q.is_file() and q.stem not in known]
print("files in data/defences without a standard name:", left or "none")

## Stage 3. Apply the decisions; PRISMA counts by stage

In [ ]:
rows = search.read_csv(paths.REPORTS / "screening.csv")
log_rows = search.read_csv(paths.REPORTS / "search_log.csv")
SEARCH_DATE = sorted({r["date"] for r in log_rows})[-1] if log_rows else "unknown"
decisions = search.read_csv(Path("schema/screening_decisions.csv"))
rows, more = search.apply_decisions(rows, decisions)
problems += more
roots = [r for r in rows if not r.get("linked_to")]
problems += [f"study without a decision: {r['key']}" for r in roots if r.get("decision") == "pending"]
search.write_csv(paths.REPORTS / "screening.csv", rows, search.SCREEN_FIELDS)
counts = search.prisma_counts(log_rows, rows)
counts["search_date"] = SEARCH_DATE
counts["title_abstract_corrections"] = sum(1 for r in roots if r.get("note", "").startswith("corrected on rereading the abstract"))
print(f"studies {len(roots)}; screened {counts['screened']}; pending {counts['pending']}")
print(f"title/abstract exclusions: {sum(counts['excluded_title_abstract_by_reason'].values())} "
      f"(of which corrected on rereading the abstract: {counts['title_abstract_corrections']})")
print(f"reports sought for full text: {counts['reports_sought_for_full_text']}")
print(f"full text assessed: {counts['full_text_assessed']} -> included {counts['included_after_full_text']}, "
      f"excluded {sum(counts['excluded_full_text_by_reason'].values())}")
print("full-text exclusions by reason:", json.dumps(counts["excluded_full_text_by_reason"], indent=1))
print(f"included now: {counts['included']}; still unsure: {counts['unsure_to_full_text']}")
for p in problems:
    print("problem:", p)

## Stage 4. Check every full-text decision against its PDF

In [ ]:
import logging
logging.getLogger("pypdf").setLevel(logging.ERROR)          # pypdf's font-table warnings are noise here
ft = search.read_csv(Path("schema/fulltext_decisions.csv"))
check = []
for r in ft:
    pdf = paths.DATA_DEFENCES / f"{r['slug']}.pdf"
    out = dict(r, page="", status="")
    if not pdf.exists():
        out["status"] = "PDF not in data/defences"
    else:
        texts = overlap.pdf_page_texts(pdf)
        a = overlap.norm_text(r["anchor"])
        hits = [i + 1 for i, t in enumerate(texts) if a in t]
        straddle = ""
        if not hits:
            hits = [i + 1 for i in range(len(texts) - 1) if a in texts[i] + texts[i + 1]]
            straddle = " (runs over the page break)" if hits else ""
        out["page"] = str(hits[0]) if hits else ""
        out["status"] = ("found" + straddle) if hits else "anchor not found"
        out["sha256"] = overlap.sha256_of(pdf)
    check.append(out)
search.write_csv(paths.REPORTS / "fulltext_check.csv", check,
                 ["key", "slug", "decision", "reason", "basis", "page", "status", "anchor", "sha256", "note"])
found = sum(1 for c in check if c["status"].startswith("found"))
print(f"anchors found: {found} of {len(check)}")
for c in check:
    if not c["status"].startswith("found"):
        print(f"  {c['slug']:<45} {c['status']} | basis: {c['basis']}")
        problems.append(f"anchor check {c['slug']}: {c['status']}")

## Stage 5. Register the hand-saved PDFs; what is still missing

In [ ]:
by_key = {r["key"]: r for r in rows}
newly = 0
for m in man:                                    # carry the current decision into the manifest
    r = by_key[m["key"]]
    m.update(decision=r["decision"], reason=r.get("reason", ""), stage=r.get("stage", ""), note=r.get("note", ""))
    pdf = paths.DATA_DEFENCES / f"{m['slug']}.pdf"
    if not m.get("pdf_path") and pdf.exists() and pdf.stat().st_size > 1000:
        m.update(pdf_status="placed by hand", pdf_source="saved by hand from a browser into data/defences", pdf_url="",
                 pdf_path=str(pdf.relative_to(paths.REPO_ROOT)), sha256=corpus.sha256(pdf), pages=corpus.page_count(pdf))
        newly += 1
fields = ["slug", "key", "decision", "reason", "stage", "note", "title", "first_author", "year", "venue", "arxiv",
          "arxiv_version", "doi", "pdf_status", "pdf_source", "pdf_url", "pdf_path", "sha256", "pages"]
search.write_csv(paths.REPORTS / "defences.csv", man, fields)
hand = [{"decision": m["decision"], "save_as": f"data/defences/{m['slug']}.pdf", "title": m["title"], "doi": m.get("doi", ""),
         "link": f"https://doi.org/{m['doi']}" if m.get("doi") else "", "how": "open the link in a browser and save the PDF",
         "tried": m.get("pdf_status", "")[:400]}
        for m in man if m["decision"] in ("include", "unsure") and not m.get("pdf_path")]
search.write_csv(paths.REPORTS / "fetch_by_hand.csv", hand, ["decision", "save_as", "title", "doi", "link", "how", "tried"])
inc = [m for m in man if m["decision"] == "include"]
print(f"PDFs registered in this step: {newly}")
print(f"included studies: {len(inc)}; with a PDF: {sum(1 for m in inc if m.get('pdf_path'))}")
print(f"still to fetch: {len(hand)}")
for h in hand:
    print(f"  [{h['decision']}] {h['save_as']}  {h['link']}")

## Stage 6. Decision record, PRISMA file, tests

In [ ]:
counts["pdfs_obtained"] = sum(1 for m in man if m.get("pdf_path"))
counts["pdfs_to_fetch_by_hand"] = len(hand)
counts["fulltext_anchors_found"] = found
counts["hand_files_placed"] = sum(1 for p in placed if p["action"] == "use" and p["status"] in ("renamed", "already placed"))
(paths.REPORTS / "prisma_counts.json").write_text(json.dumps(counts, indent=1))
unsure_left = counts["unsure_to_full_text"]
status = "CORPUS FROZEN; FULL-TEXT ASSESSMENT " + ("COMPLETE" if unsure_left == 0 else f"PENDING FOR {unsure_left} UNSURE STUDY")
nft = counts["excluded_full_text_by_reason"].get("no full text obtainable", 0)
lines = [f"decision_02: {status}", f"search date: {SEARCH_DATE}; this step: {dt.date.today().isoformat()}",
         f"studies after version linking: {counts['studies_after_version_linking']}; screened: {counts['screened']}",
         "excluded at title and abstract: " + json.dumps(counts["excluded_title_abstract_by_reason"]),
         f"  of which inclusions corrected on rereading the abstract: {counts['title_abstract_corrections']}",
         f"reports sought for full text: {counts['reports_sought_for_full_text']}; not retrieved: {nft}",
         f"full text assessed: {counts['full_text_assessed']}; included {counts['included_after_full_text']}; "
         "excluded " + json.dumps(counts["excluded_full_text_by_reason"]),
         f"full-text decisions with the anchor found in the PDF: {found} of {len(check)}",
         f"included now: {counts['included']}; unsure: {unsure_left}",
         f"PDFs in data/defences: {counts['pdfs_obtained']} (saved by hand and placed here: {counts['hand_files_placed']}); "
         f"still to fetch: {len(hand)}",
         ("next: notebook 03 (pilot extraction); the included studies still decided at title and abstract get their "
          "full-text check at extraction (notebooks 03 and 04)") if unsure_left == 0 and not hand else
         "next: save the PDFs in reports/fetch_by_hand.csv into data/defences, then rerun this notebook"] + \
        [f"- problem: {p}" for p in problems]
(paths.REPORTS / "decision_02.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 7. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "02d full-text stage closed: hand-saved PDFs placed, last unsure studies decided with checked anchors, six abstract corrections, no-full-text exclusions recorded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)